[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap09/cap09_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 9 Apprentissage profond pour la vision par ordinateur

Les chapitres précédents ont établi les fondements de la vision par ordinateur (VC) au moyen de méthodes classiques d'extraction et de représentation de caractéristiques. Au **chapitre 7**, des descripteurs tels que *Local Binary Patterns* (*LBP*) et *Histogram of Oriented Gradients* (*HOG*) ont montré comment les textures et les formes peuvent être codées par des descripteurs conçus manuellement. Au **chapitre 8**, des algorithmes comme *Oriented FAST and Rotated BRIEF* (*ORB*) et le détecteur *Haar Cascade* ont étendu ce principe aux tâches de mise en correspondance, de détection et de reconnaissance d'objets.

Ces techniques restent pertinentes en raison de leur interprétabilité et de leur efficacité computationnelle, mais elles dépendent d'une étape préalable de définition manuelle des descripteurs, appelée **ingénierie de caractéristiques** (*feature engineering*). Cette dépendance limite l'adaptation du modèle à des scénarios pour lesquels le descripteur n'a pas été conçu.

L'**apprentissage profond** (*Deep Learning*) propose une alternative : au lieu de spécifier manuellement les caractéristiques pertinentes, le modèle apprend automatiquement des représentations à partir des données pendant l'entraînement — processus connu sous le nom d'**apprentissage de représentations** (*representation learning*) (GOODFELLOW, 2016). En VC, cette stratégie est principalement mise en œuvre par les **réseaux de neurones convolutionnels** (*Convolutional Neural Networks* — CNN), dans lesquels les filtres de convolution ne possèdent plus de coefficients fixes et sont désormais ajustés par des algorithmes d'optimisation.

Bien qu'ils représentent un changement dans la construction des systèmes de reconnaissance de formes, les CNN préservent des concepts déjà étudiés dans cet ouvrage : la convolution, présentée au **chapitre 3**, demeure l'opération responsable de l'extraction locale de caractéristiques, désormais appliquée avec des coefficients appris plutôt que conçus.

Il convient de souligner que l'objectif de ce chapitre n'est pas d'explorer de manière exhaustive la théorie de l'apprentissage profond, mais plutôt d'offrir une vue d'ensemble de ses fondements et de démontrer comment ces architectures sont appliquées dans le contexte de la VC. Les lecteurs intéressés par un approfondissement théorique et conceptuel dans ce domaine devront se référer à des ouvrages spécialisés de la littérature, tels que Goodfellow (2016) et Lecun (2015)..

## 9.1 Objectifs du chapitre

À la fin de ce chapitre, l'étudiant devrait être capable de :

- Relier la convolution apprise par les CNN à la convolution à *kernels* fixes présentée au **chapitre 3** ;
- Décrire l'architecture de base d'une CNN et la fonction de ses couches principales ;
- Implémenter, entraîner et évaluer des modèles de CNN pour la classification d'images ;
- Appliquer le **transfert d'apprentissage** (*transfer learning*) pour adapter des modèles pré-entraînés à de nouveaux problèmes ;
- Utiliser des modèles pré-entraînés pour des tâches de classification, de détection d'objets et de segmentation ;
- Implémenter, entraîner et évaluer une architecture *U-Net* pour la segmentation sémantique, en la comparant aux approches classiques ;
- Préparer des ensembles de données annotés et les intégrer à un *pipeline* d'entraînement via des plateformes telles que **Roboflow** ;
- Intégrer la géométrie computationnelle et l'apprentissage profond dans des applications de réalité augmentée et de photogrammétrie.

La [Figure 9.1](#fig-09-infografo) synthétise l'organisation des concepts étudiés dans ce chapitre et les relations entre eux.

<figure id="fig-09-infografo" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-09-infografo.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figure 9.1:</strong> Vue d'ensemble des principaux concepts abordés dans ce chapitre. **Source :** élaboré avec l'aide du *Gemini Notebook* ({GOOGLE}, 2025).</figcaption>
</figure>

## 9.2 Aperçu : Classification, Détection et Segmentation

Les tâches de VC se distinguent principalement par l'information produite en sortie. La **classification** attribue une étiquette unique à l'image entière ; la **détection d'objets** localise et identifie les objets présents dans la scène ; la **segmentation** associe une classe à chaque pixel et, dans certaines approches, distingue différentes instances d'une même catégorie.

La [Tableau 9.1](#tbl-panorama-tarefas) résume les tâches étudiées tout au long du livre, en indiquant la question à laquelle chacune répond et la granularité de l'information produite.

<a id="tbl-panorama-tarefas"></a>

**Tabela 9.1:** Comparatif entre les principales tâches de VC selon la granularité de l'information produite.

| Tâche | Question répondue | Granularité de la sortie | Chapitre |
|-----------------------------|-----------------------------------------------|-------------------------------------------------------------|:--------:|
| **Classification** | « Quelle est la classe de cette image ? » | Une étiquette unique pour l'image entière | 7 et 9 |
| **Détection d'objets** | « Quels objets existent et où se trouvent-ils ? » | Classe et boîte englobante (*bounding box*) pour chaque objet | 8 et 9 |
| **Segmentation sémantique** | « À quelle classe appartient chaque pixel ? » | Une étiquette de classe pour chaque pixel | 8 et 9 |
| **Segmentation d'instances** | « Quels pixels appartiennent à chaque objet ? » | Une étiquette par pixel pour chaque instance | 8 et 9 |
| **Segmentation panoptique** | « Quelle est la classe et l'identité de chaque objet ? » | Classe et identifiant d'instance pour chaque pixel | 8 et 9 |


Ces tâches représentent des niveaux croissants d'interprétation de l'image : la classification décrit la scène de manière globale, la détection ajoute la localisation des objets et la segmentation produit une représentation spatiale détaillée, permettant d'analyser chaque région individuellement. Ce chapitre se concentre d'abord sur la classification par CNN, puis étend les mêmes principes à la détection et à la segmentation.

## 9.3 Configuration de l'environnement

Les exemples de ce chapitre utilisent **PyTorch**, un *framework* largement employé dans le développement et l'entraînement de modèles d'apprentissage profond. Le code suivant vérifie la disponibilité des bibliothèques nécessaires et installe automatiquement celles qui ne sont pas encore présentes dans l'environnement d'exécution.

Si **PyTorch** n'est pas installé, une version compatible avec le matériel disponible est automatiquement sélectionnée : la version avec prise en charge de **CUDA**, s'il y a un GPU NVIDIA disponible, ou la version pour exécution sur CPU, dans le cas contraire.

Ensuite, l'environnement est initialisé avec l'importation des bibliothèques utilisées tout au long du chapitre, la définition d'une graine aléatoire pour favoriser la reproductibilité des expériences et l'obtention du fichier `morph.py` — la bibliothèque didactique de traitement morphologique déjà utilisée dans les chapitres précédents —, s'il n'est pas encore disponible dans le répertoire de travail.

In [1]:
import contextlib, importlib, importlib.metadata, importlib.util
import io, os, random, shutil, subprocess, sys, urllib.request, warnings

# Supprime les avertissements PyTorch et les warnings généraux
warnings.filterwarnings("ignore", category=UserWarning)

url = ("https://raw.githubusercontent.com/fzampirolli/"
       "pdi-vc/master/morph/config.py")
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config

# Silencie stdout et stderr tant au niveau Python qu'au niveau des descripteurs de fichiers du système d'exploitation
def setup_silencioso():
    with open(os.devnull, "w") as fnull:
        old_out = os.dup(1)
        old_err = os.dup(2)
        try:
            os.dup2(fnull.fileno(), 1)
            os.dup2(fnull.fileno(), 2)
            with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
                config.setup()
        finally:
            os.dup2(old_out, 1)
            os.dup2(old_err, 2)
            os.close(old_out)
            os.close(old_err)

setup_silencioso()
from morph import mm


def setup_cap09():
    """Installe les bibliothèques manquantes pour ce chapitre de manière 100% silencieuse."""
    pkgs = {
        "skimage": "scikit-image", "numpy": "numpy",
        "sklearn": "scikit-learn", "matplotlib": "matplotlib",
        "torchviz": "torchviz", "ultralytics": "ultralytics",
        "roboflow": "roboflow",
    }
    for mod, pkg in pkgs.items():
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if importlib.util.find_spec("torch") is None:
        args = (["torch", "torchvision"] if shutil.which("nvidia-smi")
                 else ["--index-url",
                       "https://download.pytorch.org/whl/cpu",
                       "torch", "torchvision"])
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if not shutil.which("dot") and shutil.which("apt-get"):
        subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)


setup_cap09()

import cv2, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from skimage import data as skdata
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import OxfordIIITPet
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights)
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.transforms.functional import to_tensor
from torchviz import make_dot
from ultralytics import YOLO

torch.manual_seed(42)
FLAG_LIMPAR_DADOS = False
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
ver = importlib.metadata.version("ultralytics")
print(f"✅ Environnement prêt. OpenCV {cv2.__version__} | "
      f"morph {getattr(mm, '__version__', 'local_file')} | "
      f"PyTorch {torch.__version__} | Ultralytics {ver} | {device}{gpu}")

✅ Environnement prêt. OpenCV 5.0.0 | morph local_file | PyTorch 2.6.0+cu124 | Ultralytics 8.4.113 | cuda (NVIDIA GeForce GTX TITAN X)


## 9.4 Fondements de l'Apprentissage Profond pour la VC

Les CNN constituent l'architecture principale d'apprentissage profond appliquée à l'analyse d'images. Leur fonctionnement repose sur la composition d'opérations convolutives organisées en couches successives, dans lesquelles les filtres appris lors de l'entraînement transforment l'image en représentations progressivement plus abstraites. Dans cette section, sont présentés les concepts fondamentaux qui relient la convolution spatiale étudiée précédemment aux modèles modernes de VC, notamment l'extraction hiérarchique de caractéristiques, le processus d'entraînement et l'utilisation de modèles pré-entraînés.

### 9.4.1 Da Convolução Fixa à Convolução Aprendida

O **Capítulo 3** apresentou a convolução espacial com *kernels* fixos, como os operadores de Sobel, projetados para realçar características específicas de uma imagem. Nos **Capítulos 7** e **8**, o mesmo princípio sustentou descritores como *HOG*, *LBP* e *ORB*, além do detector *Haar Cascade*: em todos esses casos, os filtros são definidos antes da execução do algoritmo e permanecem inalterados durante o processamento.

A [Figure 9.2](#fig-09-sim-09-convolucao) retoma o funcionamento da convolução espacial: o simulador permite selecionar diferentes *kernels* e acompanhar o deslocamento da janela de convolução sobre uma imagem. Em cada posição, os coeficientes do *kernel* combinam-se com a vizinhança local da imagem — denominada **campo receptivo** (*receptive field*) — para produzir um valor do **mapa de características** (*feature map*), ilustrando também o compartilhamento de pesos (*weight sharing*)


As CNNs preservam essa operação, mas substituem *kernels* fixos por **filtros aprendidos**: em vez de coeficientes definidos previamente, a rede ajusta esses valores durante o treinamento a partir de exemplos rotulados, buscando minimizar uma **função de perda** (*loss function*), que mede a diferença entre as previsões do modelo e as respostas esperadas.

A diferença essencial entre os métodos clássicos e as CNNs, portanto, não está na operação de convolução em si, mas na forma como os filtros são obtidos: enquanto os primeiros utilizam filtros projetados manualmente, as CNNs aprendem, a partir dos dados de treinamento, representações adequadas à tarefa.

---

### 9.4.2 De la convolution fixe à la convolution apprise

Le **Chapitre 3** a présenté la convolution spatiale avec des *kernels* fixes, comme les opérateurs de Sobel, conçus pour mettre en évidence des caractéristiques spécifiques d'une image. Dans les **Chapitres 7** et **8**, le même principe a soutenu des descripteurs tels que *HOG*, *LBP* et *ORB*, ainsi que le détecteur *Haar Cascade* : dans tous ces cas, les filtres sont définis avant l'exécution de l'algorithme et restent inchangés pendant le traitement.

La [Figure 9.2](#fig-09-sim-09-convolucao) reprend le fonctionnement de la convolution spatiale : le simulateur permet de sélectionner différents *kernels* et de suivre le déplacement de la fenêtre de convolution sur une image. À chaque position, les coefficients du *kernel* se combinent avec le voisinage local de l'image — appelé **champ réceptif** (*receptive field*) — pour produire une valeur de la **carte de caractéristiques** (*feature map*), illustrant également le partage des poids (*weight sharing*).

Les CNN préservent cette opération, mais remplacent les *kernels* fixes par des **filtres appris** : au lieu de coefficients définis à l'avance, le réseau ajuste ces valeurs pendant l'entraînement à partir d'exemples étiquetés, cherchant à minimiser une **fonction de perte** (*loss function*), qui mesure la différence entre les prédictions du modèle et les réponses attendues.

La différence essentielle entre les méthodes classiques et les CNN, par conséquent, ne réside pas dans l'opération de convolution elle-même, mais dans la manière dont les filtres sont obtenus : tandis que les premières utilisent des filtres conçus manuellement, les CNN apprennent, à partir des données d'entraînement, des représentations adaptées à la tâche.

**Note**: I have translated the content into French as requested, while preserving all LaTeX markers, placeholder tokens ([Figure 9.2](#fig-09-sim-09-convolucao)), and Markdown structure. However, I noticed the original text was in Portuguese, not English — I have translated it accordingly from Portuguese to French. If you intended a different source language, please let me know.

In [2]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-convolucao" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-convolucao .cap09conv_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-convolucao .cap09conv_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-convolucao .cap09conv_btn {
      font-size:11px; font-weight:600; padding:6px 10px; border-radius:6px;
      border:1px solid #E4DCC8; background:#FFF; color:#26241D; cursor:pointer;
      transition:all .15s ease;
    }
    #sim-09-convolucao .cap09conv_btn:hover { background:#F1EAD7; }
    #sim-09-convolucao .cap09conv_modebtn {
      flex:1; text-align:center; padding:5px 8px; font-size:11px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:8px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-convolucao .cap09conv_modebtn.cap09conv_active { background:#26241D; color:#FBF7EE; }
    #sim-09-convolucao .cap09conv_btn.cap09conv_active { background:#26241D !important; color:#FBF7EE !important; border-color:#26241D !important; }
    #sim-09-convolucao .cap09conv_btn_primary { background:#2F6F9F; color:#FFF; border-color:#2F6F9F; }
    #sim-09-convolucao .cap09conv_btn_primary:hover { background:#245880; }
    #sim-09-convolucao .cap09conv_btn_success { background:#1E8F6F; color:#FFF; border-color:#1E8F6F; }
    #sim-09-convolucao .cap09conv_btn_success:hover { background:#166e55; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">🎯 Simulateur : Opération de Convolution 2D Classique</span>
    <span class="cap09conv_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Entrée 12×12 · Noyau 3×3 · Pas 1</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Painel de Seleção de Imagens e Kernels -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:12px;">
      <div style="flex:1;min-width:230px;">
        <div class="cap09conv_grouplabel">IMAGE D'ENTRÉE (12×12)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09conv_btnCasa" class="cap09conv_modebtn cap09conv_active">🏠 Maison</button>
          <button id="cap09conv_btnFeliz" class="cap09conv_modebtn">😊 Heureux</button>
          <button id="cap09conv_btnTriste" class="cap09conv_modebtn">😢 Triste</button>
        </div>
      </div>

      <div style="flex:1;min-width:280px;">
        <div class="cap09conv_grouplabel">FILTRE (NOYAU 3×3)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09conv_btnSobelV" class="cap09conv_btn cap09conv_active">📐 Sobel V</button>
          <button id="cap09conv_btnSobelH" class="cap09conv_btn">📏 Sobel H</button>
          <button id="cap09conv_btnSharpen" class="cap09conv_btn">✨ Netteté</button>
          <button id="cap09conv_btnIdentidade" class="cap09conv_btn">🎯 Identité</button>
        </div>
      </div>
    </div>

    <!-- Controles do Passo a Passo -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-bottom:12px;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;justify-content:space-between;">
        <div style="display:flex;gap:6px;align-items:center;">
          <button id="cap09conv_btnPasso" class="cap09conv_btn cap09conv_btn_success">▶ Avancer d'un Pas</button>
          <button id="cap09conv_btnTudo" class="cap09conv_btn cap09conv_btn_primary">⏭ Calculer Tout</button>
          <button id="cap09conv_btnReset" class="cap09conv_btn">↺ Réinitialiser</button>
        </div>
        <span class="cap09conv_mono" style="font-size:11px;color:#5b5647;">Position Actuelle : <b id="cap09conv_posTxt" style="color:#2F6F9F;">(0, 0)</b> [Sortie 10×10]</span>
      </div>
    </div>

    <!-- Área Gráfica: Entrada, Kernel e Saída -->
    <div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Entrée (12×12)</div>
        <canvas id="cap09conv_canvasEntrada" style="width:240px;height:240px;"></canvas>
      </div>

      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Noyau (3×3)</div>
        <canvas id="cap09conv_canvasKernel" style="width:105px;height:105px;"></canvas>
      </div>

      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Carte de Sortie (10×10)</div>
        <canvas id="cap09conv_canvasSaida" style="width:200px;height:200px;"></canvas>
      </div>
    </div>

    <!-- Terminal de Cálculo em Tempo Real -->
    <div id="cap09conv_calcTxt" class="cap09conv_mono" style="text-align:center;font-size:11px;margin-top:12px;color:#7EE7C6;background:#1B2430;padding:8px 10px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;overflow-x:auto;">
      ∑ (xᵢ × wᵢ) = calcul de la position actuelle...
    </div>

  </div>
</div>

<script>
(function(){
  var cap09conv_IMAGENS = {
    casa: [
      "............", "....XXXX....", "...XXXXXX...", "..XXXXXXXX..",
      ".XXXXXXXXXX.", ".XXXXXXXXXX.", ".XXXXXXXXXX.", ".XXXXXXXXXX.",
      ".XXXX..XXXX.", ".XXXX..XXXX.", ".XXXX..XXXX.", "............"
    ],
    feliz: [
      "............", "....XXXX....", "..XXXXXXXX..", ".XXXXXXXXXX.",
      ".XXXXXXXXXX.", ".XXX.XX.XXX.", ".XXXXXXXXXX.", ".XX......XX.",
      ".XXX....XXX.", ".XXXXXXXXXX.", "..XXXXXXXX..", "............"
    ],
    triste: [
      "............", "....XXXX....", "..XXXXXXXX..", ".XXXXXXXXXX.",
      ".XXXXXXXXXX.", ".XXX.XX.XXX.", ".XXXXXXXXXX.", ".XXX....XXX.",
      ".XX......XX.", ".XXXXXXXXXX.", "..XXXXXXXX..", "............"
    ]
  };

  var cap09conv_KERNELS = {
    sobelV:     { nome: "Sobel Vertical", k: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]] },
    sobelH:     { nome: "Sobel Horizontal", k: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]] },
    sharpen:    { nome: "Nitidez (Sharpen)", k: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]] },
    identidade: { nome: "Identidade", k: [[0, 0, 0], [0, 1, 0], [0, 0, 0]] }
  };

  function cap09conv_converterLinhas(cap09conv_linhas){
    return cap09conv_linhas.map(function(cap09conv_l){
      var cap09conv_res = [];
      for(var cap09conv_i=0; cap09conv_i<cap09conv_l.length; cap09conv_i++) cap09conv_res.push(cap09conv_l[cap09conv_i]==='X' ? 1.0 : 0.0);
      return cap09conv_res;
    });
  }

  function cap09conv_init(cap09conv_root){
    if(!cap09conv_root || cap09conv_root.dataset.initConv) return;
    cap09conv_root.dataset.initConv = "1";

    var cap09conv_imgAtualId = "casa";
    var cap09conv_kernelAtualKey = "sobelV";

    var cap09conv_imgEntradaBase = cap09conv_converterLinhas(cap09conv_IMAGENS[cap09conv_imgAtualId]);
    var cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
    var cap09conv_pos = {r:0, c:0};

    var cap09conv_cvsEnt = cap09conv_root.querySelector('#cap09conv_canvasEntrada');
    var cap09conv_cvsKer = cap09conv_root.querySelector('#cap09conv_canvasKernel');
    var cap09conv_cvsSai = cap09conv_root.querySelector('#cap09conv_canvasSaida');

    var cap09conv_ctxEnt = cap09conv_cvsEnt.getContext('2d');
    var cap09conv_ctxKer = cap09conv_cvsKer.getContext('2d');
    var cap09conv_ctxSai = cap09conv_cvsSai.getContext('2d');

    function cap09conv_prepararCanvas(cap09conv_canvas, cap09conv_ctx, cap09conv_cssW, cap09conv_cssH){
      var cap09conv_dpr = window.devicePixelRatio || 1;
      cap09conv_canvas.width = cap09conv_cssW * cap09conv_dpr;
      cap09conv_canvas.height = cap09conv_cssH * cap09conv_dpr;
      cap09conv_ctx.scale(cap09conv_dpr, cap09conv_dpr);
    }
    cap09conv_prepararCanvas(cap09conv_cvsEnt, cap09conv_ctxEnt, 240, 240);
    cap09conv_prepararCanvas(cap09conv_cvsKer, cap09conv_ctxKer, 105, 105);
    cap09conv_prepararCanvas(cap09conv_cvsSai, cap09conv_ctxSai, 200, 200);

    var cap09conv_posTxt  = cap09conv_root.querySelector('#cap09conv_posTxt');
    var cap09conv_calcTxt = cap09conv_root.querySelector('#cap09conv_calcTxt');

    var cap09conv_btnCasa   = cap09conv_root.querySelector('#cap09conv_btnCasa');
    var cap09conv_btnFeliz  = cap09conv_root.querySelector('#cap09conv_btnFeliz');
    var cap09conv_btnTriste = cap09conv_root.querySelector('#cap09conv_btnTriste');

    var cap09conv_btnSobelV     = cap09conv_root.querySelector('#cap09conv_btnSobelV');
    var cap09conv_btnSobelH     = cap09conv_root.querySelector('#cap09conv_btnSobelH');
    var cap09conv_btnSharpen    = cap09conv_root.querySelector('#cap09conv_btnSharpen');
    var cap09conv_btnIdentidade = cap09conv_root.querySelector('#cap09conv_btnIdentidade');

    function cap09conv_desenharEntrada(){
      var cap09conv_tam = 20;
      cap09conv_ctxEnt.clearRect(0,0,240,240);
      for (var cap09conv_r=0; cap09conv_r<12; cap09conv_r++){
        for (var cap09conv_c=0; cap09conv_c<12; cap09conv_c++){
          var cap09conv_val = cap09conv_imgEntradaBase[cap09conv_r][cap09conv_c];
          var cap09conv_g = Math.round(cap09conv_val * 255);
          cap09conv_ctxEnt.fillStyle = "rgb(" + cap09conv_g + "," + cap09conv_g + "," + cap09conv_g + ")";
          cap09conv_ctxEnt.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
          cap09conv_ctxEnt.strokeStyle = "#E4DCC8";
          cap09conv_ctxEnt.lineWidth = 0.8;
          cap09conv_ctxEnt.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

          cap09conv_ctxEnt.fillStyle = cap09conv_g > 128 ? "#111827" : "#FFFFFF";
          cap09conv_ctxEnt.font = "600 8px 'JetBrains Mono', monospace";
          cap09conv_ctxEnt.textAlign = "center";
          cap09conv_ctxEnt.textBaseline = "middle";
          cap09conv_ctxEnt.fillText(cap09conv_val.toFixed(0), cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
        }
      }

      if (cap09conv_pos.r < 10) {
        cap09conv_ctxEnt.strokeStyle = "#C1443A";
        cap09conv_ctxEnt.lineWidth = 2.5;
        cap09conv_ctxEnt.strokeRect(cap09conv_pos.c*cap09conv_tam, cap09conv_pos.r*cap09conv_tam, cap09conv_tam*3, cap09conv_tam*3);
      }
    }

    function cap09conv_desenharKernel(){
      var cap09conv_tam = 35;
      cap09conv_ctxKer.clearRect(0,0,105,105);
      var cap09conv_k = cap09conv_KERNELS[cap09conv_kernelAtualKey].k;
      for (var cap09conv_i=0; cap09conv_i<3; cap09conv_i++){
        for (var cap09conv_j=0; cap09conv_j<3; cap09conv_j++){
          var cap09conv_val = cap09conv_k[cap09conv_i][cap09conv_j];
          cap09conv_ctxKer.fillStyle = cap09conv_val > 0 ? "#E6F4EA" : (cap09conv_val < 0 ? "#FCE8E6" : "#FAFAF7");
          cap09conv_ctxKer.fillRect(cap09conv_j*cap09conv_tam, cap09conv_i*cap09conv_tam, cap09conv_tam, cap09conv_tam);
          cap09conv_ctxKer.strokeStyle = "#E4DCC8";
          cap09conv_ctxKer.strokeRect(cap09conv_j*cap09conv_tam, cap09conv_i*cap09conv_tam, cap09conv_tam, cap09conv_tam);

          cap09conv_ctxKer.fillStyle = cap09conv_val > 0 ? "#1E8F6F" : (cap09conv_val < 0 ? "#C1443A" : "#8A8371");
          cap09conv_ctxKer.font = "600 10px 'JetBrains Mono', monospace";
          cap09conv_ctxKer.textAlign = "center";
          cap09conv_ctxKer.textBaseline = "middle";
          cap09conv_ctxKer.fillText(cap09conv_val.toString(), cap09conv_j*cap09conv_tam + cap09conv_tam/2, cap09conv_i*cap09conv_tam + cap09conv_tam/2);
        }
      }
    }

    function cap09conv_desenharSaida(){
      var cap09conv_tam = 20;
      cap09conv_ctxSai.clearRect(0,0,200,200);
      for (var cap09conv_r=0; cap09conv_r<10; cap09conv_r++){
        for (var cap09conv_c=0; cap09conv_c<10; cap09conv_c++){
          var cap09conv_v = cap09conv_saida[cap09conv_r][cap09conv_c];
          if (cap09conv_v === null) {
            cap09conv_ctxSai.fillStyle = "#F7F5EE";
            cap09conv_ctxSai.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
            cap09conv_ctxSai.strokeStyle = "#E4DCC8";
            cap09conv_ctxSai.lineWidth = 0.8;
            cap09conv_ctxSai.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

            cap09conv_ctxSai.fillStyle = "#B8AE94";
            cap09conv_ctxSai.font = "700 8px 'JetBrains Mono', monospace";
            cap09conv_ctxSai.textAlign = "center";
            cap09conv_ctxSai.textBaseline = "middle";
            cap09conv_ctxSai.fillText("·", cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
          } else {
            var cap09conv_normV = Math.max(0, Math.min(1, (cap09conv_v + 2.0) / 4.0));
            var cap09conv_g = Math.round(cap09conv_normV * 255);
            cap09conv_ctxSai.fillStyle = "rgb(" + cap09conv_g + "," + cap09conv_g + "," + cap09conv_g + ")";
            cap09conv_ctxSai.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
            cap09conv_ctxSai.strokeStyle = "#E4DCC8";
            cap09conv_ctxSai.lineWidth = 0.8;
            cap09conv_ctxSai.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

            cap09conv_ctxSai.fillStyle = cap09conv_g > 128 ? "#111827" : "#FFFFFF";
            cap09conv_ctxSai.font = "600 7.5px 'JetBrains Mono', monospace";
            cap09conv_ctxSai.textAlign = "center";
            cap09conv_ctxSai.textBaseline = "middle";
            cap09conv_ctxSai.fillText(cap09conv_v.toFixed(1), cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
          }
        }
      }

      if (cap09conv_pos.r < 10){
        cap09conv_ctxSai.strokeStyle = "#2F6F9F";
        cap09conv_ctxSai.lineWidth = 2;
        cap09conv_ctxSai.strokeRect(cap09conv_pos.c*cap09conv_tam, cap09conv_pos.r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
      }
    }

    function cap09conv_calcularPosicao(cap09conv_r, cap09conv_c){
      var cap09conv_k = cap09conv_KERNELS[cap09conv_kernelAtualKey].k;
      var cap09conv_soma = 0;
      var cap09conv_termos = [];
      for (var cap09conv_i=0; cap09conv_i<3; cap09conv_i++){
        for (var cap09conv_j=0; cap09conv_j<3; cap09conv_j++){
          var cap09conv_valEnt = cap09conv_imgEntradaBase[cap09conv_r+cap09conv_i][cap09conv_c+cap09conv_j];
          var cap09conv_valKer = cap09conv_k[cap09conv_i][cap09conv_j];
          var cap09conv_prod = cap09conv_valEnt * cap09conv_valKer;
          cap09conv_soma += cap09conv_prod;
          cap09conv_termos.push(cap09conv_prod >= 0 ? cap09conv_prod.toFixed(0) : '(' + cap09conv_prod.toFixed(0) + ')');
        }
      }
      return { valFinal: cap09conv_soma, expressao: cap09conv_termos.join(" + ") };
    }

    function cap09conv_atualizarCalculoTexto(){
      if (cap09conv_pos.r >= 10) {
        cap09conv_calcTxt.textContent = "Convolution Terminée ! Les 100 pixels de la carte de sortie ont été générés.";
        return;
      }
      var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
      cap09conv_calcTxt.textContent = 'Position (' + cap09conv_pos.r + ',' + cap09conv_pos.c + ') → z = ' + cap09conv_obj.expressao + ' = ' + cap09conv_obj.valFinal.toFixed(2);
    }

    function cap09conv_avancarPasso(){
      if (cap09conv_pos.r >= 10) return;
      var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
      cap09conv_saida[cap09conv_pos.r][cap09conv_pos.c] = cap09conv_obj.valFinal;
      cap09conv_pos.c++;
      if (cap09conv_pos.c >= 10){ cap09conv_pos.c = 0; cap09conv_pos.r++; }
      cap09conv_render();
    }

    function cap09conv_calcularTudo(){
      while (cap09conv_pos.r < 10) {
        var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
        cap09conv_saida[cap09conv_pos.r][cap09conv_pos.c] = cap09conv_obj.valFinal;
        cap09conv_pos.c++;
        if (cap09conv_pos.c >= 10){ cap09conv_pos.c = 0; cap09conv_pos.r++; }
      }
      cap09conv_render();
    }

    function cap09conv_resetar(){
      cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
      cap09conv_pos = {r:0, c:0};
      cap09conv_render();
    }

    function cap09conv_resetarECalcular(){
      cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
      cap09conv_pos = {r:0, c:0};
      cap09conv_calcularTudo();
    }

    function cap09conv_render(){
      cap09conv_desenharEntrada();
      cap09conv_desenharKernel();
      cap09conv_desenharSaida();
      cap09conv_posTxt.textContent = cap09conv_pos.r < 10 ? '(' + cap09conv_pos.r + ', ' + cap09conv_pos.c + ')' : 'concluído';
      cap09conv_atualizarCalculoTexto();
    }

    function cap09conv_trocarImagem(cap09conv_id, cap09conv_btn){
      [cap09conv_btnCasa, cap09conv_btnFeliz, cap09conv_btnTriste].forEach(function(b){ b.classList.remove('cap09conv_active'); });
      cap09conv_btn.classList.add('cap09conv_active');
      cap09conv_imgAtualId = cap09conv_id;
      cap09conv_imgEntradaBase = cap09conv_converterLinhas(cap09conv_IMAGENS[cap09conv_id]);
      cap09conv_resetarECalcular();
    }

    function cap09conv_trocarKernel(cap09conv_key, cap09conv_btn){
      [cap09conv_btnSobelV, cap09conv_btnSobelH, cap09conv_btnSharpen, cap09conv_btnIdentidade].forEach(function(b){ b.classList.remove('cap09conv_active'); });
      cap09conv_btn.classList.add('cap09conv_active');
      cap09conv_kernelAtualKey = cap09conv_key;
      cap09conv_resetarECalcular();
    }

    cap09conv_btnCasa.addEventListener('click', function(){ cap09conv_trocarImagem("casa", cap09conv_btnCasa); });
    cap09conv_btnFeliz.addEventListener('click', function(){ cap09conv_trocarImagem("feliz", cap09conv_btnFeliz); });
    cap09conv_btnTriste.addEventListener('click', function(){ cap09conv_trocarImagem("triste", cap09conv_btnTriste); });

    cap09conv_btnSobelV.addEventListener('click', function(){ cap09conv_trocarKernel("sobelV", cap09conv_btnSobelV); });
    cap09conv_btnSobelH.addEventListener('click', function(){ cap09conv_trocarKernel("sobelH", cap09conv_btnSobelH); });
    cap09conv_btnSharpen.addEventListener('click', function(){ cap09conv_trocarKernel("sharpen", cap09conv_btnSharpen); });
    cap09conv_btnIdentidade.addEventListener('click', function(){ cap09conv_trocarKernel("identidade", cap09conv_btnIdentidade); });

    cap09conv_root.querySelector('#cap09conv_btnPasso').addEventListener('click', cap09conv_avancarPasso);
    cap09conv_root.querySelector('#cap09conv_btnTudo').addEventListener('click', cap09conv_calcularTudo);
    cap09conv_root.querySelector('#cap09conv_btnReset').addEventListener('click', cap09conv_resetar);

    cap09conv_resetarECalcular();
  
  }

  function cap09conv_tryInit(){
    var cap09conv_root = document.getElementById('sim-09-convolucao');
    if(cap09conv_root) cap09conv_init(cap09conv_root); else setTimeout(cap09conv_tryInit, 200);
  }
  cap09conv_tryInit();
})();
</script>
''')

**Figure 9.2:** Simulateur interactif de convolution 2D classique : choisissez parmi les trois images synthétiques d


<figure id="fig-09-sim-09-convolucao">
  <img src="imagens/fig-09-sim-09-convolucao.png" alt=" Simulateur interactif de convolution 2D classique : choisissez parmi les trois images synthétiques d'entrée 12×12 (maison, visage heureux ou triste) et un filtre 3×3 (Sobel V, Sobel H, Netteté ou Identité) et avancez pas à pas pour observer comment les produits internes locaux du champ réceptif construisent la carte de caractéristiques cellule par cellule. " style="max-width:80%" />
  <figcaption><strong>Figure 9.2:</strong>  Simulateur interactif de convolution 2D classique : choisissez parmi les trois images synthétiques d'entrée 12×12 (maison, visage heureux ou triste) et un filtre 3×3 (Sobel V, Sobel H, Netteté ou Identité) et avancez pas à pas pour observer comment les produits internes locaux du champ réceptif construisent la carte de caractéristiques cellule par cellule. </figcaption>
</figure>

Pour comprendre comment cet apprentissage se produit, il est nécessaire d'étudier l'unité de base de traitement des réseaux de neurones : le **neurone artificiel**.

### 9.4.3 Neurone Artificiel

Le **neurone artificiel** (*artificial neuron*) est l’unité fondamentale de traitement d’un réseau de neurones. Son premier modèle mathématique — un ensemble d’entrées combinées et comparées à un seuil — a été proposé par Mcculloch (1943), sans encore aucun mécanisme d’apprentissage. Le **Perceptron** (ROSENBLATT, 1958) a fait progresser cette formulation en introduisant une règle d’ajustement des poids à partir d’exemples, devenant ainsi le premier modèle de neurone artificiel capable d’apprendre et la base des architectures modernes d’**Apprentissage Profond** (*Deep Learning*). Le terme Apprentissage Profond désigne l’utilisation de réseaux à multiples couches de traitement, capables d’apprendre des représentations hiérarchiques des données : les premières couches apprennent des caractéristiques simples, comme les contours et les textures, et les couches plus profondes combinent progressivement ces représentations pour identifier des structures et des objets plus complexes.

Chaque neurone reçoit un ensemble d’entrées, calcule une combinaison linéaire de ces valeurs et applique une **fonction d’activation** (*activation function*), produisant une unique valeur de sortie. Mathématiquement, la combinaison linéaire est donnée par

$$
z=\sum_{i=1}^{n}w_i x_i+b,
$$

où $x_i$ représentent les entrées, $w_i$ les poids associés à chaque entrée et $b$ le **biais** (*bias*). La sortie du neurone est obtenue en appliquant la fonction d’activation :

$$
y=f(z).
$$

Dans les CNN, ce principe prend des formes distinctes selon la couche. Dans les **couches convolutionnelles** (*convolutional layers*), chaque neurone ne traite qu’une petite région de l’entrée, appelée **champ réceptif** (*receptive field*), préservant ainsi l’organisation spatiale de l’image. Dans les **couches entièrement connectées** (*fully connected layers*), chaque neurone reçoit toutes les sorties de la couche précédente, combinant les caractéristiques extraites pour produire la sortie finale du réseau, comme la classe attribuée à l’image.

La [Figure 9.3](#fig-09-sim-09-neuronio) illustre le fonctionnement d’un neurone artificiel : le simulateur permet de modifier les entrées ($x_1$ et $x_2$), les poids ($w_1$ et $w_2$), le biais ($b$) et la fonction d’activation, en observant en temps réel le calcul de la combinaison linéaire et de la sortie correspondante.

In [3]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-neuronio" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-neuronio .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-neuronio .cn-modebtn {
      flex:1; text-align:center; padding:8px 10px; font-size:12px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:10px;
      transition:background .15s ease, color .15s ease;
    }
    #sim-09-neuronio .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-neuronio input[type=range] { accent-color:#2F6F9F; min-width:0; }
    #sim-09-neuronio .cn-tag {
      font-size:10px; font-weight:700; color:#8A8371; width:16px; text-align:center; flex-shrink:0;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulateur : Neurone artificiel dans un CNN</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">y = f(∑ wᵢxᵢ + b)</span>
  </div>

  <div style="padding:18px 20px;background:#FFFFFF;overflow:auto">

    <!-- Alternador de contexto: camada convolucional vs. totalmente conectada -->
    <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:12px;padding:4px;margin-bottom:14px;max-width:480px;margin-left:auto;margin-right:auto;">
      <button id="cap09neuronio_modeConv" class="cn-modebtn active">🧩 Couche convolutive</button>
      <button id="cap09neuronio_modeFC" class="cn-modebtn">🔗 Couche entièrement connectée</button>
    </div>

    <!-- Painel de Controles -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));gap:12px;align-items:start;">

        <div>
          <label id="cap09neuronio_lblX1" style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Pixel x₁ (normalisé) / Poids du noyau w₁</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">x₁</span>
            <input type="range" id="cap09neuronio_x1" min="-3" max="3" step="0.1" value="1.0" style="flex:1;">
            <span class="cn-tag cn-mono">w₁</span>
            <input type="range" id="cap09neuronio_w1" min="-3" max="3" step="0.1" value="0.8" style="flex:1;">
          </div>
        </div>

        <div>
          <label id="cap09neuronio_lblX2" style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Pixel x₂ (normalisé) / Poids du noyau w₂</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">x₂</span>
            <input type="range" id="cap09neuronio_x2" min="-3" max="3" step="0.1" value="-1.5" style="flex:1;">
            <span class="cn-tag cn-mono">w₂</span>
            <input type="range" id="cap09neuronio_w2" min="-3" max="3" step="0.1" value="0.5" style="flex:1;">
          </div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Biais (b) et fonction f(z)</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">b</span>
            <input type="range" id="cap09neuronio_bias" min="-3" max="3" step="0.1" value="0.2" style="flex:1;">
            <select id="cap09neuronio_func" style="font-size:11px;padding:5px 6px;border-radius:6px;border:1px solid #ccc;background:#fff;">
              <option value="relu">ReLU</option>
              <option value="sigmoid">Sigmoïde</option>
              <option value="step">Échelon (Step)</option>
              <option value="identity">Identité</option>
            </select>
          </div>
        </div>

      </div>
      <div id="cap09neuronio_notaEscala" style="font-size:10.5px;color:#8A8371;margin-top:10px;padding-top:8px;border-top:1px dashed #E9E3D3;">
        Les valeurs de x varient de -3 à 3 car, dans un CNN, les pixels (0–255) sont <b>normalisés</b> avant d'entrer dans le réseau. Le schéma à côté retraduit cette valeur normalisée en nuance de gris, juste pour donner une intuition visuelle — les nombres qui comptent pour le calcul sont ceux des barres.
      </div>
    </div>

    <!-- Área Gráfica: Esquema do Neurônio + Curva -->
    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div id="cap09neuronio_diagTitle" style="font-size:11px;font-weight:600;text-align:center;margin-bottom:6px;color:#4b5563;">Champ réceptif → Convolution → Carte de caractéristiques</div>
        <canvas id="cap09neuronio_canvasEsquema" style="width:320px;height:230px;"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:6px;color:#4b5563;">Activation en f(z)</div>
        <canvas id="cap09neuronio_canvasCurva" style="width:260px;height:230px;"></canvas>
      </div>
    </div>

    <!-- Legenda contextual -->
    <div id="cap09neuronio_caption" style="font-size:11.5px;line-height:1.5;color:#5b5647;background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:10px 12px;margin-top:14px;">
      <b>x₁, x₂</b> = intensité des pixels dans le champ réceptif · <b>w₁, w₂</b> = poids du noyau (filtre) · <b>z</b> = résultat de la convolution à cette position · <b>y</b> = valeur du pixel produit dans la carte de caractéristiques, après l'activation.
    </div>

    <!-- Status numérico estilo terminal -->
    <div id="cap09neuronio_valTxt" class="cn-mono" style="text-align:center;font-size:12px;margin-top:12px;color:#7EE7C6;background:#1B2430;padding:10px 12px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;">
      z = (1.00 × 0.80) + (-1.50 × 0.50) + 0.20 = 0.25 → y = 0.25
    </div>

  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var canvasEsquema = root.querySelector('#cap09neuronio_canvasEsquema');
    var canvasCurva   = root.querySelector('#cap09neuronio_canvasCurva');
    var ctxEsquema = canvasEsquema.getContext('2d');
    var ctxCurva   = canvasCurva.getContext('2d');

    // Escala para telas de alta resolução (retina)
    function prepararCanvas(canvas, ctx, cssW, cssH){
      var dpr = window.devicePixelRatio || 1;
      canvas.width = cssW * dpr;
      canvas.height = cssH * dpr;
      ctx.scale(dpr, dpr);
    }
    prepararCanvas(canvasEsquema, ctxEsquema, 320, 230);
    prepararCanvas(canvasCurva, ctxCurva, 260, 230);

    var inX1 = root.querySelector('#cap09neuronio_x1');
    var inW1 = root.querySelector('#cap09neuronio_w1');
    var inX2 = root.querySelector('#cap09neuronio_x2');
    var inW2 = root.querySelector('#cap09neuronio_w2');
    var inB  = root.querySelector('#cap09neuronio_bias');
    var selF = root.querySelector('#cap09neuronio_func');
    var valTxt = root.querySelector('#cap09neuronio_valTxt');
    var caption = root.querySelector('#cap09neuronio_caption');
    var diagTitle = root.querySelector('#cap09neuronio_diagTitle');
    var lblX1 = root.querySelector('#cap09neuronio_lblX1');
    var lblX2 = root.querySelector('#cap09neuronio_lblX2');
    var notaEscala = root.querySelector('#cap09neuronio_notaEscala');
    var btnConv = root.querySelector('#cap09neuronio_modeConv');
    var btnFC   = root.querySelector('#cap09neuronio_modeFC');

    var modo = 'conv'; // 'conv' | 'fc'

    var CORES = {
      pos: "#1E8F6F",      // peso/conexão positiva
      neg: "#C1443A",      // peso/conexão negativa
      bias: "#C08A2E",     // viés
      saida: "#2F5FA8",    // sinal de saída
      texto: "#26241D",
      textoSuave: "#6b7280",
      grade: "#EFEAdd"
    };

    function calcularAtivacao(z, tipo){
      if (tipo === 'relu') return Math.max(0, z);
      if (tipo === 'sigmoid') return 1 / (1 + Math.exp(-z));
      if (tipo === 'step') return z >= 0 ? 1 : 0;
      return z; // identity
    }

    // Converte um valor de entrada (-3..3) em um tom de cinza 0-255 (intensidade de pixel)
    function valorParaCinza(v){
      var t = Math.max(0, Math.min(1, (v + 3) / 6));
      return Math.round(t * 255);
    }
    // Converte a saída y (que pode ter faixas diferentes conforme f) em cinza 0-255
    function saidaParaCinza(y, tipo){
      var t;
      if (tipo === 'sigmoid' || tipo === 'step') t = y;
      else t = Math.max(0, Math.min(1, (y + 3) / 6));
      return Math.round(Math.max(0, Math.min(1, t)) * 255);
    }

    function desenharPixel(ctx, cx, cy, lado, cinza, corBorda){
      var g = "rgb(" + cinza + "," + cinza + "," + cinza + ")";
      ctx.fillStyle = g;
      ctx.fillRect(cx - lado/2, cy - lado/2, lado, lado);
      ctx.strokeStyle = corBorda || "#9ca3af";
      ctx.lineWidth = 1.2;
      ctx.strokeRect(cx - lado/2, cy - lado/2, lado, lado);
    }

    function chipPeso(ctx, cx, cy, valor){
      var cor = valor >= 0 ? CORES.pos : CORES.neg;
      ctx.font = "600 10px 'JetBrains Mono', monospace";
      var texto = (valor>=0?"+":"") + valor.toFixed(1);
      var w = ctx.measureText(texto).width + 10;
      ctx.fillStyle = cor;
      roundRect(ctx, cx - w/2, cy - 9, w, 18, 9);
      ctx.fill();
      ctx.fillStyle = "#fff";
      ctx.textAlign = "center";
      ctx.textBaseline = "middle";
      ctx.fillText(texto, cx, cy+1);
    }

    function roundRect(ctx, x, y, w, h, r){
      ctx.beginPath();
      ctx.moveTo(x+r, y);
      ctx.arcTo(x+w, y, x+w, y+h, r);
      ctx.arcTo(x+w, y+h, x, y+h, r);
      ctx.arcTo(x, y+h, x, y, r);
      ctx.arcTo(x, y, x+w, y, r);
      ctx.closePath();
    }

    function seta(ctx, x1,y1,x2,y2,cor){
      ctx.strokeStyle = cor; ctx.lineWidth = 2;
      ctx.beginPath(); ctx.moveTo(x1,y1); ctx.lineTo(x2,y2); ctx.stroke();
      var ang = Math.atan2(y2-y1, x2-x1);
      ctx.fillStyle = cor;
      ctx.beginPath();
      ctx.moveTo(x2,y2);
      ctx.lineTo(x2 - 7*Math.cos(ang-0.4), y2 - 7*Math.sin(ang-0.4));
      ctx.lineTo(x2 - 7*Math.cos(ang+0.4), y2 - 7*Math.sin(ang+0.4));
      ctx.closePath(); ctx.fill();
    }

    // ---------- MODO CONVOLUÇÃO ----------
    function desenharConvolucao(x1, w1, x2, w2, b, z, y, tipo){
      var W=320, H=230;
      ctxEsquema.clearRect(0,0,W,H);
      ctxEsquema.textAlign = "center"; ctxEsquema.textBaseline = "middle";

      var cinzaX1 = valorParaCinza(x1), cinzaX2 = valorParaCinza(x2);
      var cinzaY  = saidaParaCinza(y, tipo);

      // Campo receptivo (patch de imagem de entrada) — dois pixels empilhados
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Champ réceptif", 48, 14);

      desenharPixel(ctxEsquema, 48, 48, 44, cinzaX1, "#9ca3af");
      desenharPixel(ctxEsquema, 48, 118, 44, cinzaX2, "#9ca3af");
      ctxEsquema.fillStyle = cinzaX1 > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("x₁=" + x1.toFixed(1), 48, 48);
      ctxEsquema.fillStyle = cinzaX2 > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.fillText("x₂=" + x2.toFixed(1), 48, 118);

      // CORREÇÃO AQUI: As setas agora param na borda esquerda do bloco do kernel (x=118)
      seta(ctxEsquema, 70, 48, 118, 72, w1 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 70, 118, 118, 96, w2 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 140, 45, 140, 62, b >= 0 ? CORES.bias : CORES.neg);

      // CORREÇÃO AQUI: Chips dos pesos centralizados no meio da seta (x=94)
      chipPeso(ctxEsquema, 94, 60, w1);
      chipPeso(ctxEsquema, 94, 107, w2);

      // Viés
      ctxEsquema.fillStyle = "#FDF0DA";
      ctxEsquema.beginPath(); ctxEsquema.arc(140, 26, 19, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.bias; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#8a5a12"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("b=" + b.toFixed(1), 140, 26);

      // Nó de convolução (kernel * patch) — ocupa de x=118 a x=162
      ctxEsquema.fillStyle = "#DCE8F5";
      roundRect(ctxEsquema, 118, 62, 44, 44, 10); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#2F5FA8"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#1e3a5f"; ctxEsquema.font = "600 12px Inter, sans-serif";
      ctxEsquema.fillText("⊛", 140, 78);
      ctxEsquema.font = "9px Inter, sans-serif";
      ctxEsquema.fillText("noyau", 140, 94);

      // Seta para o mapa de características
      seta(ctxEsquema, 162, 84, 202, 84, CORES.saida);

      // Mapa de características (mini tira com o pixel de saída em destaque)
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Carte de caractéristiques", 265, 14);

      var vizinhos = [190, 190];
      desenharPixel(ctxEsquema, 224, 84, 26, vizinhos[0], "#d1d5db");
      ctxEsquema.save();
      desenharPixel(ctxEsquema, 265, 84, 40, cinzaY, "#2F5FA8");
      ctxEsquema.lineWidth = 2.4; ctxEsquema.strokeStyle = CORES.saida;
      ctxEsquema.strokeRect(265-21, 84-21, 42, 42);
      ctxEsquema.fillStyle = cinzaY > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("y=" + y.toFixed(1), 265, 84);
      ctxEsquema.restore();
      desenharPixel(ctxEsquema, 306, 84, 26, vizinhos[1], "#d1d5db");

      ctxEsquema.fillStyle = "#9ca3af"; ctxEsquema.font = "8px Inter, sans-serif";
      ctxEsquema.fillText("(le noyau glisse →)", 265, 212);
    }

    // ---------- MODO TOTALMENTE CONECTADA ----------
    function desenharFC(x1, w1, x2, w2, b, z, y){
      var W=320, H=230;
      ctxEsquema.clearRect(0,0,W,H);
      ctxEsquema.textAlign = "center"; ctxEsquema.textBaseline = "middle";

      seta(ctxEsquema, 55, 60, 155, 118, w1 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 55, 176, 155, 118, w2 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 170, 45, 170, 100, b  >= 0 ? CORES.bias : CORES.neg);
      seta(ctxEsquema, 195, 118, 260, 118, CORES.saida);

      // Entrada x1
      ctxEsquema.fillStyle = "#EDEDE7"; ctxEsquema.beginPath(); ctxEsquema.arc(45, 60, 24, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#9ca3af"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = CORES.texto; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("x₁=" + x1.toFixed(1), 45, 60);

      // Entrada x2
      ctxEsquema.fillStyle = "#EDEDE7"; ctxEsquema.beginPath(); ctxEsquema.arc(45, 176, 24, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#9ca3af"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = CORES.texto;
      ctxEsquema.fillText("x₂=" + x2.toFixed(1), 45, 176);

      chipPeso(ctxEsquema, 108, 92, w1);
      chipPeso(ctxEsquema, 108, 148, w2);

      // Viés
      ctxEsquema.fillStyle = "#FDF0DA"; ctxEsquema.beginPath(); ctxEsquema.arc(170, 30, 20, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.bias; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#8a5a12"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("b=" + b.toFixed(1), 170, 30);

      // Soma / ativação
      ctxEsquema.fillStyle = "#DCE8F5"; ctxEsquema.beginPath(); ctxEsquema.arc(172, 118, 28, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#2F5FA8"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#1e3a5f"; ctxEsquema.font = "12px Inter, sans-serif";
      ctxEsquema.fillText("∑, f", 172, 118);

      // Saída
      ctxEsquema.fillStyle = "#DCEEE6"; ctxEsquema.beginPath(); ctxEsquema.arc(280, 118, 26, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.pos; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#14532d"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("y=" + y.toFixed(2), 280, 118);

      ctxEsquema.fillStyle = CORES.textoSuave; ctxEsquema.font = "9px Inter, sans-serif";
      ctxEsquema.fillText("neurone de la couche entièrement connectée", 172, 210);
    }

    function desenharCurva(z, y, tipo){
      var W = 260, H = 230;
      var origemX = 130, origemY = 165;
      var escalaX = 20, escalaY = 40;

      ctxCurva.clearRect(0,0,W,H);

      // Grade sutil
      ctxCurva.strokeStyle = CORES.grade; ctxCurva.lineWidth = 1;
      for (var gx = 10; gx <= W-10; gx += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(gx,10); ctxCurva.lineTo(gx,H-10); ctxCurva.stroke(); }
      for (var gy = 10; gy <= H-10; gy += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(10,gy); ctxCurva.lineTo(W-10,gy); ctxCurva.stroke(); }

      // Eixos
      ctxCurva.strokeStyle = "#9ca3af"; ctxCurva.lineWidth = 1.3;
      ctxCurva.beginPath();
      ctxCurva.moveTo(10, origemY); ctxCurva.lineTo(W-10, origemY);
      ctxCurva.moveTo(origemX, 10); ctxCurva.lineTo(origemX, H-10);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280"; ctxCurva.font = "10px Inter, sans-serif"; ctxCurva.textAlign = "left";
      ctxCurva.fillText("z", W-18, origemY-6);
      ctxCurva.fillText("y", origemX+6, 16);

      // Curva da função
      ctxCurva.strokeStyle = "#2F5FA8"; ctxCurva.lineWidth = 2.4;
      ctxCurva.beginPath();
      var primeiro = true;
      for (var px = -5; px <= 5; px += 0.1) {
        var valY = calcularAtivacao(px, tipo);
        var cx = origemX + px * escalaX;
        var cy = origemY - valY * escalaY;
        if (primeiro) { ctxCurva.moveTo(cx, cy); primeiro = false; }
        else { ctxCurva.lineTo(cx, cy); }
      }
      ctxCurva.stroke();

      // Ponto atual
      var ptX = Math.max(10, Math.min(W-10, origemX + z * escalaX));
      var ptY = Math.max(10, Math.min(H-10, origemY - y * escalaY));

      ctxCurva.strokeStyle = "#C1443A"; ctxCurva.setLineDash([3,3]); ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(ptX, origemY); ctxCurva.lineTo(ptX, ptY); ctxCurva.lineTo(origemX, ptY);
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);

      ctxCurva.fillStyle = "#C1443A";
      ctxCurva.beginPath(); ctxCurva.arc(ptX, ptY, 5, 0, 2*Math.PI); ctxCurva.fill();
      ctxCurva.strokeStyle = "#fff"; ctxCurva.lineWidth = 1.5; ctxCurva.stroke();
    }

    function atualizarLegendas(){
      if (modo === 'conv'){
        lblX1.textContent = "Pixel x₁ (normalisé) / Poids du noyau w₁";
        lblX2.textContent = "Pixel x₂ (normalisé) / Poids du noyau w₂";
        diagTitle.textContent = "Champ réceptif → Convolution → Carte de caractéristiques";
        caption.innerHTML = "<b>🧩 Preservação Espacial:</b> Na camada convolucional, a operação ocorre localmente via campo receptivo. O resultado (y) mantém uma posição bem definida no mapa de características 2D, preservando a vizinhança e a estrutura geométrica dos pixels.";
        notaEscala.innerHTML = "Os valores de x variam de -3 a 3 porque, em uma CNN, os pixels (0–255) são <b>normalizados</b> antes de entrar na rede. O desenho ao lado traduz esse valor normalizado de volta em um tom de cinza, só para dar intuição visual — os números que valem para a conta são os das barras.";
      } else {
        lblX1.textContent = "Attribut x₁ (feature) / Poids w₁";
        lblX2.textContent = "Attribut x₂ (feature) / Poids w₂";
        diagTitle.textContent = "Vecteur d'attributs → Neurone → Sortie";
        caption.innerHTML = "<b>🔗 Perda da Informação Espacial:</b> Na camada totalmente conectada, os mapas de características são achatados (flatten) em um vetor 1D. Como o neurônio se conecta a todas as entradas indiferenciadamente, a noção de 'vizinho de cima/lado' é destruída em prol de uma decisão global.";
        notaEscala.innerHTML = "Aqui x₁, x₂ representam atributos já extraídos (não pixels), tipicamente padronizados para uma faixa pequena como esta antes de entrarem na camada.";
      }
    }

    function atualizar(){
      var x1 = parseFloat(inX1.value);
      var w1 = parseFloat(inW1.value);
      var x2 = parseFloat(inX2.value);
      var w2 = parseFloat(inW2.value);
      var b  = parseFloat(inB.value);
      var tipo = selF.value;

      var z = (x1 * w1) + (x2 * w2) + b;
      var y = calcularAtivacao(z, tipo);

      if (modo === 'conv') desenharConvolucao(x1, w1, x2, w2, b, z, y, tipo);
      else desenharFC(x1, w1, x2, w2, b, z, y);

      desenharCurva(z, y, tipo);

      valTxt.textContent = 'z = (' + x1.toFixed(2) + ' × ' + w1.toFixed(2) + ') + (' +
                           x2.toFixed(2) + ' × ' + w2.toFixed(2) + ') + (' + b.toFixed(2) +
                           ') = ' + z.toFixed(2) + '  →  y = ' + selF.options[selF.selectedIndex].text + '(z) = ' + y.toFixed(2);
    }

    function definirModo(novoModo){
      modo = novoModo;
      btnConv.classList.toggle('active', modo === 'conv');
      btnFC.classList.toggle('active', modo === 'fc');
      atualizarLegendas();
      atualizar();
    }

    btnConv.addEventListener('click', function(){ definirModo('conv'); });
    btnFC.addEventListener('click', function(){ definirModo('fc'); });

    inX1.addEventListener('input', atualizar);
    inW1.addEventListener('input', atualizar);
    inX2.addEventListener('input', atualizar);
    inW2.addEventListener('input', atualizar);
    inB.addEventListener('input', atualizar);
    selF.addEventListener('change', atualizar);

    atualizarLegendas();
    atualizar();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-neuronio');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 9.3:** Simulateur interactif du neurone artificiel dans un contexte de CNN : alternez entre un neurone de couche convolutive (où x_i sont des intensités de pixel dans un champ réceptif et w_i sont des poids du *kernel*) et un neurone de couche entièrement connectée, en ajustant les entrées, les poids, le biais et la fonction d


<figure id="fig-09-sim-09-neuronio">
  <img src="imagens/fig-09-sim-09-neuronio.png" alt=" Simulateur interactif du neurone artificiel dans un contexte de CNN : alternez entre un neurone de couche convolutive (où x_i sont des intensités de pixel dans un champ réceptif et w_i sont des poids du *kernel*) et un neurone de couche entièrement connectée, en ajustant les entrées, les poids, le biais et la fonction d'activation pour visualiser le calcul de z et de la sortie y en temps réel. " style="max-width:80%" />
  <figcaption><strong>Figure 9.3:</strong>  Simulateur interactif du neurone artificiel dans un contexte de CNN : alternez entre un neurone de couche convolutive (où x_i sont des intensités de pixel dans un champ réceptif et w_i sont des poids du *kernel*) et un neurone de couche entièrement connectée, en ajustant les entrées, les poids, le biais et la fonction d'activation pour visualiser le calcul de z et de la sortie y en temps réel. </figcaption>
</figure>

Dans un CNN, des milliers de neurones s'organisent en couches ayant des fonctions spécifiques : les premières sont responsables de l'extraction de caractéristiques par le biais de la convolution, et les dernières réalisent la classification à partir des caractéristiques apprises.

### 9.4.4 Couche convolutive

La **couche convolutive** (*convolutional layer*) est responsable de l'extraction des caractéristiques de l'image. Chaque filtre génère une **carte de caractéristiques** (*feature map*), dont l'intensité à chaque position indique la réponse du filtre à la région correspondante de l'entrée.

L'opération effectuée suit le même principe de déplacement et de combinaison locale présenté au **Chapitre 3** pour la convolution spatiale. En considérant un *noyau* $K$ de dimension $k \times k$, la valeur produite à la position $(i,j)$ est donnée par

$$
F(i,j)=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}K(u,v)\,I(i+u,j+v).
$$

Il convient de noter une distinction terminologique : l'expression ci-dessus correspond, formellement, à une **corrélation croisée** (*cross-correlation*), et non à la convolution mathématique stricte, qui exige la réflexion du *noyau* avant la combinaison. La plupart des *frameworks* d'apprentissage profond, y compris **PyTorch**, implémentent cette opération sans réflexion et la désignent, par convention, comme convolution — convention également adoptée dans ce chapitre. Cette différence n'a aucun effet pratique sur l'entraînement, puisque les coefficients du *noyau* sont appris et non imposés au préalable.

La principale différence par rapport aux méthodes classiques réside donc dans l'obtention du *noyau* $K$ : dans les filtres traditionnels, ses coefficients sont définis manuellement pour mettre en évidence des caractéristiques spécifiques de l'image ; dans les CNN, les coefficients sont initialisés automatiquement et ajustés pendant l'entraînement par le biais de la **rétropropagation de l'erreur** (*backpropagation*), ce qui rend chaque filtre spécialisé dans l'identification de motifs pertinents pour la tâche étudiée.

Deux concepts caractérisent cette couche :

- **Partage des poids** (*weight sharing*) : le même filtre est appliqué à toutes les positions de l'image, réduisant considérablement le nombre de paramètres du modèle.
- **Champ réceptif** (*receptive field*) : chaque neurone convolutif ne traite qu'un petit voisinage de l'image, préservant la structure spatiale des données.

En empilant plusieurs couches convolutives, le réseau apprend une **hiérarchie de caractéristiques** : les premières couches tendent à détecter des motifs simples, comme les bords et les textures, et les couches plus profondes combinent ces informations pour représenter des structures progressivement plus complexes. Après la convolution, la carte de caractéristiques est soumise à une fonction d'activation, introduisant une non-linéarité dans le modèle et élargissant sa capacité à représenter des relations complexes entre les variables d'entrée.

La [Figure 9.4](#fig-09-sim-09-camada-conv) présente cette couche de manière interactive.

In [4]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-camada-conv" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-camada-conv .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-camada-conv .cn-grouplabel {
      font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px;
    }
    #sim-09-camada-conv .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-camada-conv .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-camada-conv input[type=range] { accent-color:#2F6F9F; min-width:0; }
    #sim-09-camada-conv .cn-navbtn {
      width:26px;height:26px;border-radius:8px;border:1px solid #E4DCC8;background:#FAFAF7;
      color:#26241D;font-size:12px;cursor:pointer;display:flex;align-items:center;justify-content:center;
      transition:background .15s ease; flex-shrink:0;
    }
    #sim-09-camada-conv .cn-navbtn:hover { background:#F1EAD7; }
    #sim-09-camada-conv .cn-playbtn {
      padding:0 10px;height:26px;border-radius:8px;border:1px solid #2F6F9F;background:#EAF2FA;
      color:#2F6F9F;font-size:10.5px;font-weight:700;cursor:pointer;white-space:nowrap;flex-shrink:0;
    }
    #sim-09-camada-conv .cn-playbtn:hover { background:#DCEEFB; }
    #sim-09-camada-conv .cn-prodcell {
      border-radius:6px;padding:3px 2px;text-align:center;border:1px solid #e5e7eb;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulateur : Opération de Convolution & Carte de caractéristiques</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">F(i,j) = f(∑ K(u,v) · I(i+u, j+v))</span>
  </div>

  <div style="padding:12px 14px;background:#FFFFFF;overflow:auto">

    <!-- Seletores: Imagem de Entrada + Kernel, lado a lado para compactar -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">
      <div style="flex:1;min-width:230px;">
        <div class="cn-grouplabel">IMAGE D'ENTRÉE</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09camdconvimg_btnImgCasa" class="cn-modebtn active">🏠 Maison</button>
          <button id="cap09camdconvimg_btnImgFeliz" class="cn-modebtn">😊 Heureux</button>
          <button id="cap09camdconvimg_btnImgTriste" class="cn-modebtn">😢 Triste</button>
        </div>
      </div>
      <div style="flex:1;min-width:280px;">
        <div class="cn-grouplabel">KERNEL (FILTRE FIXE)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09camdconvimg_btnSobelV" class="cn-modebtn active">📐 Vertical</button>
          <button id="cap09camdconvimg_btnSobelH" class="cn-modebtn">📏 Horizontal</button>
          <button id="cap09camdconvimg_btnSharpen" class="cn-modebtn">✨ Netteté</button>
          <button id="cap09camdconvimg_btnIdentity" class="cn-modebtn">🎯 Identité</button>
        </div>
      </div>
    </div>

    <!-- Painel de Controles -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-bottom:10px;">
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));gap:10px;align-items:start;">

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Position du champ réceptif</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <button id="cap09camdconvimg_btnAnterior" class="cn-navbtn" title="Pas précédent">◀</button>
            <input type="range" id="cap09camdconvimg_step" min="0" max="8" step="1" value="0" style="flex:1;">
            <button id="cap09camdconvimg_btnProximo" class="cn-navbtn" title="Pas suivant">▶</button>
            <button id="cap09camdconvimg_btnPlay" class="cn-playbtn">⏵ Auto</button>
            <button id="cap09camdconvimg_btnReiniciar" class="cn-navbtn" title="Réinitialiser le balayage">↺</button>
          </div>
          <div id="cap09camdconvimg_posLabel" class="cn-mono" style="font-size:10px;color:#8A8371;margin-top:4px;"></div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Fonction d'activation f(z)</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <select id="cap09camdconvimg_func" style="font-size:11px;padding:5px 6px;border-radius:6px;border:1px solid #ccc;background:#fff;width:100%;">
              <option value="relu">ReLU</option>
              <option value="identity">Identité (Linéaire)</option>
              <option value="sigmoid">Sigmoïde</option>
            </select>
          </div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Remplissage (Padding)</label>
          <div style="display:flex;gap:6px;align-items:center;padding-top:3px;">
            <input type="checkbox" id="cap09camdconvimg_padding" style="accent-color:#2F6F9F;">
            <span style="font-size:11px;color:#374151;font-weight:500;">Zero-Padding (p = 1)</span>
          </div>
          <div style="display:flex;gap:8px;align-items:center;margin-top:6px;font-size:9.5px;color:#6b7280;flex-wrap:wrap;">
            <span><span style="display:inline-block;width:10px;height:10px;background:#555;border:1px solid #999;vertical-align:middle;"></span> pixel réel</span>
            <span><span style="display:inline-block;width:10px;height:10px;background:#111;border:1px dashed #C98A2E;vertical-align:middle;"></span> marge fixe de l'image (0)</span>
            <span><span style="display:inline-block;width:10px;height:10px;background:#DCEEFB;border:1px dashed #8AB4D8;vertical-align:middle;"></span> padding de l'algorithme (0)</span>
          </div>
        </div>

      </div>
      <div id="cap09camdconvimg_notaEscala" style="font-size:10.5px;color:#8A8371;margin-top:8px;padding-top:6px;border-top:1px dashed #E9E3D3;">
        Le même kernel glisse sur toute l'image en réutilisant ses coefficients (<b>partage des poids</b>). Chaque image est déjà entourée d'une marge fixe de 1 pixel de fond (zéros, contour pointillé ambre), isolant la forme sur les quatre côtés. Choisissez une image et un kernel fixe ci-dessus, puis utilisez ◀ ▶ ou "Auto" pour parcourir le champ réceptif — la <b>Feature Map</b> à droite est remplie cellule par cellule, dans le même ordre où la convolution est calculée (les cellules non encore visitées apparaissent comme "···").
      </div>
      <div id="cap09camdconvimg_notaFormula" class="cn-mono" style="font-size:10.5px;color:#2F6F9F;margin-top:5px;"></div>
      <div id="cap09camdconvimg_notaOffset" style="font-size:10.5px;color:#8A8371;margin-top:4px;">
        📌 La sortie F(i,j) provient du champ réceptif entre (i,j) et (i+2,j+2) ; son centre réel est (i+1,j+1) — <b>1 ligne et 1 colonne en dessous/à droite</b> de l'indice utilisé pour étiqueter la cellule, toujours dans les deux directions. Ce décalage n'est visible que sur l'axe où le kernel différencie l'image (c'est pourquoi le Sobel V semble décalé seulement sur le côté, et le Sobel H, seulement vers le bas).
      </div>
    </div>

    <!-- Área Gráfica: Esquema da Convolução + Curva de Ativação -->
    <div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <canvas id="cap09camdconvimg_canvasEsquema" style="width:580px;height:260px;"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Activation à f(z)</div>
        <canvas id="cap09camdconvimg_canvasCurva" style="width:200px;height:190px;"></canvas>
      </div>
    </div>

    <!-- Painel de Cálculo Detalhado -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-top:10px;">
      <div style="font-size:11px;font-weight:600;color:#4b5563;margin-bottom:8px;">🔍 Calcul détaillé dans le champ réceptif actuel</div>
      <div style="display:flex;gap:18px;align-items:center;flex-wrap:wrap;">
        <div id="cap09camdconvimg_gradeProdutos" style="display:grid;grid-template-columns:repeat(3,44px);gap:3px;"></div>
        <div id="cap09camdconvimg_expressaoSoma" style="font-size:11px;color:#374151;line-height:1.6;"></div>
      </div>
    </div>

    <!-- Legenda contextual (dinâmica: imagem escolhida + kernel escolhido) -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-top:10px;">
      <div id="cap09camdconvimg_legendaImagem" style="flex:1;min-width:250px;font-size:11px;line-height:1.5;color:#5b5647;background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:8px 10px;"></div>
      <div id="cap09camdconvimg_legendaKernel" style="flex:1;min-width:250px;font-size:11px;line-height:1.5;color:#5b5647;background:#F1F6FB;border:1px solid #DCEEFB;border-radius:10px;padding:8px 10px;"></div>
    </div>

    <!-- Status numérico estilo terminal -->
    <div id="cap09camdconvimg_valTxt" class="cn-mono" style="text-align:center;font-size:11px;margin-top:10px;color:#7EE7C6;background:#1B2430;padding:8px 10px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;overflow-x:auto;">
      z = 0.00 → y = ReLU(z) = 0.00
    </div>

  </div>
</div>

<script>
(function(){
  function initCamadaConvImg(root){
    if(!root || root.dataset.initConvImg) return;
    root.dataset.initConvImg = "1";

    var canvasEsquema = root.querySelector('#cap09camdconvimg_canvasEsquema');
    var canvasCurva   = root.querySelector('#cap09camdconvimg_canvasCurva');
    var ctxEsquema = canvasEsquema.getContext('2d');
    var ctxCurva   = canvasCurva.getContext('2d');

    function prepararCanvas(canvas, ctx, cssW, cssH){
      var dpr = window.devicePixelRatio || 1;
      canvas.width = cssW * dpr;
      canvas.height = cssH * dpr;
      ctx.scale(dpr, dpr);
    }
    prepararCanvas(canvasEsquema, ctxEsquema, 580, 260);
    prepararCanvas(canvasCurva, ctxCurva, 200, 190);

    var inStep   = root.querySelector('#cap09camdconvimg_step');
    var selF     = root.querySelector('#cap09camdconvimg_func');
    var chkP     = root.querySelector('#cap09camdconvimg_padding');
    var valTxt   = root.querySelector('#cap09camdconvimg_valTxt');
    var posLabel = root.querySelector('#cap09camdconvimg_posLabel');
    var notaFormula = root.querySelector('#cap09camdconvimg_notaFormula');
    var legendaKernel = root.querySelector('#cap09camdconvimg_legendaKernel');
    var legendaImagem = root.querySelector('#cap09camdconvimg_legendaImagem');
    var gradeProdutos = root.querySelector('#cap09camdconvimg_gradeProdutos');
    var expressaoSoma = root.querySelector('#cap09camdconvimg_expressaoSoma');

    var btnSobelV   = root.querySelector('#cap09camdconvimg_btnSobelV');
    var btnSobelH   = root.querySelector('#cap09camdconvimg_btnSobelH');
    var btnSharpen  = root.querySelector('#cap09camdconvimg_btnSharpen');
    var btnIdentity = root.querySelector('#cap09camdconvimg_btnIdentity');

    var btnImgCasa   = root.querySelector('#cap09camdconvimg_btnImgCasa');
    var btnImgFeliz  = root.querySelector('#cap09camdconvimg_btnImgFeliz');
    var btnImgTriste = root.querySelector('#cap09camdconvimg_btnImgTriste');

    var btnAnterior  = root.querySelector('#cap09camdconvimg_btnAnterior');
    var btnProximo   = root.querySelector('#cap09camdconvimg_btnProximo');
    var btnPlay      = root.querySelector('#cap09camdconvimg_btnPlay');
    var btnReiniciar = root.querySelector('#cap09camdconvimg_btnReiniciar');

    var CORES = {
      pos: "#1E8F6F",
      neg: "#C1443A",
      saida: "#2F5FA8",
      textoSuave: "#6b7280",
      padding: "#DCEEFB",
      paddingBorda: "#8AB4D8",
      paddingTexto: "#2F6F9F",
      margemBase: "#C98A2E"
    };

    var KERNELS = {
      sobelV: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
      sobelH: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
      sharpen: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
      identity: [[0, 0, 0], [0, 1, 0], [0, 0, 0]]
    };

    var KERNEL_INFO = {
      sobelV: { emoji: "📐", nome: "Sobel Vertical", desc: "responde fortemente a mudanças bruscas de intensidade na direção horizontal — por isso realça <b>bordas verticais</b> da imagem." },
      sobelH: { emoji: "📏", nome: "Sobel Horizontal", desc: "responde a mudanças bruscas de intensidade na direção vertical — por isso realça <b>bordas horizontais</b> da imagem." },
      sharpen: { emoji: "✨", nome: "Nitidez (Sharpen)", desc: "amplifica o pixel central em relação aos vizinhos, aumentando o contraste local e destacando detalhes finos." },
      identity: { emoji: "🎯", nome: "Identidade", desc: "reproduz o valor original do pixel central sem alterá-lo — útil como referência de que a convolução não introduz distorção por si só." }
    };

    // Três imagens de entrada 12×12 desenhadas como "arte ASCII": 'X' = pixel
    // aceso (1.0), '.' = pixel apagado (0.0). Todas com o mesmo tamanho fixo,
    // para que o simulador continue mostrando apenas a operação de convolução
    // (sem qualquer etapa de classificação).
    var IMAGENS = {
      casa: {
        emoji: "🏠",
        nome: "Casa",
        desc: "combina bordas diagonais no telhado, bordas verticais retas nas paredes e uma porta recortada no centro. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "...XXXXXX...",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXX..XXXX.",
          ".XXXX..XXXX.",
          ".XXXX..XXXX.",
          "............"
        ]
      },
      feliz: {
        emoji: "😊",
        nome: "Rosto Feliz",
        desc: "um contorno arredondado com dois olhos e uma boca que se abre mais na parte de cima e se fecha em direção ao queixo. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXX.XX.XXX.",
          ".XXXXXXXXXX.",
          ".XX......XX.",
          ".XXX....XXX.",
          ".XXXXXXXXXX.",
          "..XXXXXXXX..",
          "............"
        ]
      },
      triste: {
        emoji: "😢",
        nome: "Rosto Triste",
        desc: "mesmo contorno arredondado do rosto feliz, mas com a boca invertida: mais estreita perto do nariz e mais larga perto do queixo, simulando cantos da boca virados para baixo. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXX.XX.XXX.",
          ".XXXXXXXXXX.",
          ".XXX....XXX.",
          ".XX......XX.",
          ".XXXXXXXXXX.",
          "..XXXXXXXX..",
          "............"
        ]
      }
    };

    var kernelAtual = KERNELS.sobelV;
    var kernelAtualId = "sobelV";
    var imagemAtualId = "casa";
    var imgEntradaBase = null;
    var N_BASE = 12;
    var autoplayInterval = null;

    function converterLinhasParaMatriz(linhas){
      return linhas.map(function(linha){
        var pixels = [];
        for (var i = 0; i < linha.length; i++){
          pixels.push(linha[i] === 'X' ? 1.0 : 0.0);
        }
        return pixels;
      });
    }

    function calcularAtivacao(z, tipo){
      if (tipo === 'relu') return Math.max(0, z);
      if (tipo === 'sigmoid') return 1 / (1 + Math.exp(-z));
      return z;
    }

    // Retorna a matriz de entrada (com ou sem padding) e um mapa booleano
    // indicando quais células são padding artificial (para não confundi-las
    // com pixels reais de valor 0).
    function obterMatrizEntrada(comPadding){
      var dim = comPadding ? N_BASE + 2 : N_BASE;
      var matriz = [], mapaPad = [];
      for (var r = 0; r < dim; r++){
        var linhaVal = [], linhaPad = [];
        for (var c = 0; c < dim; c++){
          if (comPadding && (r === 0 || r === dim - 1 || c === 0 || c === dim - 1)){
            linhaVal.push(0.0);
            linhaPad.push(true);
          } else {
            var ri = comPadding ? r - 1 : r;
            var ci = comPadding ? c - 1 : c;
            linhaVal.push(imgEntradaBase[ri][ci]);
            linhaPad.push(false);
          }
        }
        matriz.push(linhaVal);
        mapaPad.push(linhaPad);
      }
      return { matriz: matriz, pad: mapaPad };
    }

    function desenharEsquema(passoIdx, tipoFunc, comPadding){
      ctxEsquema.clearRect(0, 0, 580, 260);

      var entrada = obterMatrizEntrada(comPadding);
      var img = entrada.matriz, mapaPad = entrada.pad;
      var dimImg = img.length;
      var dimOut = dimImg - 3 + 1;

      var maxSteps = (dimOut * dimOut) - 1;
      inStep.max = maxSteps;
      if (passoIdx > maxSteps) {
        passoIdx = maxSteps;
        inStep.value = maxSteps;
      }
      var rowOut = Math.floor(passoIdx / dimOut);
      var colOut = passoIdx % dimOut;

      var startX = 30, startY = 46;
      var cellSize = comPadding ? 12.5 : 14.5;

      ctxEsquema.textAlign = "center";
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Entrée I (" + dimImg + "×" + dimImg + ")", startX + (dimImg * cellSize) / 2, startY - 16);

      ctxEsquema.font = "7px 'JetBrains Mono', monospace";
      for (var c0 = 0; c0 < dimImg; c0++){
        ctxEsquema.fillText(String(c0), startX + c0 * cellSize + cellSize / 2, startY - 5);
      }
      ctxEsquema.textAlign = "right";
      for (var r0 = 0; r0 < dimImg; r0++){
        ctxEsquema.fillText(String(r0), startX - 5, startY + r0 * cellSize + cellSize / 2 + 3);
      }
      ctxEsquema.textAlign = "center";

      for (var r = 0; r < dimImg; r++) {
        for (var c = 0; c < dimImg; c++) {
          var val = img[r][c];
          var x = startX + c * cellSize, y = startY + r * cellSize;

          if (mapaPad[r][c]) {
            ctxEsquema.fillStyle = CORES.padding;
            ctxEsquema.fillRect(x, y, cellSize, cellSize);
            ctxEsquema.strokeStyle = CORES.paddingBorda;
            ctxEsquema.setLineDash([2, 2]);
            ctxEsquema.lineWidth = 1;
            ctxEsquema.strokeRect(x, y, cellSize, cellSize);
            ctxEsquema.setLineDash([]);
            ctxEsquema.fillStyle = CORES.paddingTexto;
          } else {
            var g = Math.round(val * 255);
            ctxEsquema.fillStyle = "rgb(" + g + "," + g + "," + g + ")";
            ctxEsquema.fillRect(x, y, cellSize, cellSize);

            // Distingue a margem fixa da própria imagem (1px de zeros nos
            // quatro lados, embutida em imgEntradaBase) do padding opcional
            // do algoritmo: mesmo contorno tracejado, mas em âmbar.
            var riBase = comPadding ? r - 1 : r;
            var ciBase = comPadding ? c - 1 : c;
            var ehMargemBase = (riBase === 0 || riBase === N_BASE - 1 || ciBase === 0 || ciBase === N_BASE - 1);

            if (ehMargemBase) {
              ctxEsquema.strokeStyle = CORES.margemBase;
              ctxEsquema.setLineDash([2, 2]);
              ctxEsquema.lineWidth = 1;
              ctxEsquema.strokeRect(x, y, cellSize, cellSize);
              ctxEsquema.setLineDash([]);
            } else {
              ctxEsquema.strokeStyle = "#d1d5db";
              ctxEsquema.lineWidth = 1;
              ctxEsquema.strokeRect(x, y, cellSize, cellSize);
            }
            ctxEsquema.fillStyle = g > 140 ? "#374151" : "#f3f4f6";
          }
          ctxEsquema.font = "600 7px 'JetBrains Mono', monospace";
          ctxEsquema.fillText(val.toFixed(0), x + cellSize / 2, y + cellSize / 2 + 2.5);
        }
      }

      // Destacar Campo Receptivo
      var krX = startX + colOut * cellSize;
      var krY = startY + rowOut * cellSize;
      ctxEsquema.strokeStyle = "#C1443A";
      ctxEsquema.lineWidth = 2.2;
      ctxEsquema.strokeRect(krX, krY, 3 * cellSize, 3 * cellSize);

      // Calcular z, y e os 9 termos do produto no ponto atual
      var termos = [];
      var z = 0;
      for (var kr = 0; kr < 3; kr++) {
        for (var kc = 0; kc < 3; kc++) {
          var iv = img[rowOut + kr][colOut + kc];
          var kv = kernelAtual[kr][kc];
          var prod = iv * kv;
          termos.push({ i: iv, k: kv, p: prod });
          z += prod;
        }
      }
      var y = calcularAtivacao(z, tipoFunc);

      // Desenhar Kernel (K)
      var kCell = 17;
      var kStartX = startX + (dimImg * cellSize) + 16;
      var kStartY = startY + (dimImg * cellSize) / 2 - (3 * kCell) / 2;

      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Kernel K", kStartX + (3 * kCell) / 2, kStartY - 9);

      for (var kr2 = 0; kr2 < 3; kr2++) {
        for (var kc2 = 0; kc2 < 3; kc2++) {
          var kv2 = kernelAtual[kr2][kc2];
          var kx = kStartX + kc2 * kCell, ky = kStartY + kr2 * kCell;
          ctxEsquema.fillStyle = kv2 > 0 ? "#E6F4EA" : (kv2 < 0 ? "#FCE8E6" : "#F3F4F6");
          ctxEsquema.fillRect(kx, ky, kCell, kCell);
          ctxEsquema.strokeStyle = "#9ca3af";
          ctxEsquema.strokeRect(kx, ky, kCell, kCell);

          ctxEsquema.fillStyle = kv2 > 0 ? CORES.pos : (kv2 < 0 ? CORES.neg : "#374151");
          ctxEsquema.font = "600 9px 'JetBrains Mono', monospace";
          ctxEsquema.fillText(String(kv2), kx + kCell / 2, ky + kCell / 2 + 3);
        }
      }

      // Desenhar Feature Map (F) — revelado progressivamente, na mesma ordem
      // (varredura linha a linha) em que a convolução realmente é calculada.
      // Células além do passo atual ainda não foram "computadas" e aparecem
      // como pendentes ("···"), reforçando que o mapa é construído aos poucos.
      var outCell = comPadding ? 13.5 : 16;
      var outStartX = kStartX + 3 * kCell + 52;
      var outStartY = startY + (dimImg * cellSize) / 2 - (dimOut * outCell) / 2;

      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Feature Map F (" + dimOut + "×" + dimOut + ")", outStartX + (dimOut * outCell) / 2, outStartY - 16);

      ctxEsquema.font = "7px 'JetBrains Mono', monospace";
      for (var oc0 = 0; oc0 < dimOut; oc0++){
        ctxEsquema.fillText(String(oc0), outStartX + oc0 * outCell + outCell / 2, outStartY - 5);
      }
      ctxEsquema.textAlign = "right";
      for (var or0 = 0; or0 < dimOut; or0++){
        ctxEsquema.fillText(String(or0), outStartX - 5, outStartY + or0 * outCell + outCell / 2 + 3);
      }
      ctxEsquema.textAlign = "center";

      for (var orr = 0; orr < dimOut; orr++) {
        for (var occ = 0; occ < dimOut; occ++) {
          var linIdx = orr * dimOut + occ;
          var jaCalculado = linIdx <= passoIdx;
          var cx = outStartX + occ * outCell;
          var cy = outStartY + orr * outCell;
          var isAtual = (orr === rowOut && occ === colOut);

          if (!jaCalculado) {
            // Célula ainda pendente: ainda não "visitada" pela varredura.
            ctxEsquema.fillStyle = "#F7F5EE";
            ctxEsquema.fillRect(cx, cy, outCell, outCell);
            ctxEsquema.strokeStyle = "#d9d2bd";
            ctxEsquema.setLineDash([2, 2]);
            ctxEsquema.lineWidth = 1;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);
            ctxEsquema.setLineDash([]);
            ctxEsquema.fillStyle = "#b8ae94";
            ctxEsquema.font = "700 8px 'JetBrains Mono', monospace";
            ctxEsquema.fillText("·", cx + outCell / 2, cy + outCell / 2 + 2.5);
          } else {
            var oz = 0;
            for (var kr3 = 0; kr3 < 3; kr3++) {
              for (var kc3 = 0; kc3 < 3; kc3++) {
                oz += img[orr + kr3][occ + kc3] * kernelAtual[kr3][kc3];
              }
            }
            var oy = calcularAtivacao(oz, tipoFunc);
            var normY = tipoFunc === 'sigmoid' ? oy : Math.max(0, Math.min(1, (oy + 2) / 4));
            var og = Math.round(normY * 255);

            ctxEsquema.fillStyle = "rgb(" + og + "," + og + "," + og + ")";
            ctxEsquema.fillRect(cx, cy, outCell, outCell);
            ctxEsquema.strokeStyle = isAtual ? CORES.saida : "#d1d5db";
            ctxEsquema.lineWidth = isAtual ? 2.4 : 1;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);

            ctxEsquema.fillStyle = og > 140 ? "#374151" : "#f3f4f6";
            ctxEsquema.font = "600 6.5px 'JetBrains Mono', monospace";
            ctxEsquema.fillText(oy.toFixed(1), cx + outCell / 2, cy + outCell / 2 + 2.2);
          }

          if (isAtual) {
            ctxEsquema.strokeStyle = CORES.saida;
            ctxEsquema.lineWidth = 2.4;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);
          }
        }
      }

      return { z: z, y: y, posR: rowOut, posC: colOut, dimImg: dimImg, dimOut: dimOut, termos: termos, p: comPadding ? 1 : 0 };
    }

    function desenharCurva(z, y, tipo){
      var W = 200, H = 190;
      var origemX = 100, origemY = 135;
      var escalaX = 17, escalaY = 32;

      ctxCurva.clearRect(0,0,W,H);

      ctxCurva.strokeStyle = "#EFEAdd"; ctxCurva.lineWidth = 1;
      for (var gx = 10; gx <= W-10; gx += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(gx,10); ctxCurva.lineTo(gx,H-10); ctxCurva.stroke(); }
      for (var gy = 10; gy <= H-10; gy += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(10,gy); ctxCurva.lineTo(W-10,gy); ctxCurva.stroke(); }

      ctxCurva.strokeStyle = "#9ca3af"; ctxCurva.lineWidth = 1.2;
      ctxCurva.beginPath();
      ctxCurva.moveTo(10, origemY); ctxCurva.lineTo(W-10, origemY);
      ctxCurva.moveTo(origemX, 10); ctxCurva.lineTo(origemX, H-10);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280"; ctxCurva.font = "9.5px Inter, sans-serif"; ctxCurva.textAlign = "left";
      ctxCurva.fillText("z", W-16, origemY-5);
      ctxCurva.fillText("y", origemX+5, 14);

      ctxCurva.strokeStyle = "#2F5FA8"; ctxCurva.lineWidth = 2.2;
      ctxCurva.beginPath();
      var primeiro = true;
      for (var px = -5; px <= 5; px += 0.1) {
        var valY = calcularAtivacao(px, tipo);
        var cx = origemX + px * escalaX;
        var cy = origemY - valY * escalaY;
        if (primeiro) { ctxCurva.moveTo(cx, cy); primeiro = false; }
        else { ctxCurva.lineTo(cx, cy); }
      }
      ctxCurva.stroke();

      var ptX = Math.max(10, Math.min(W-10, origemX + z * escalaX));
      var ptY = Math.max(10, Math.min(H-10, origemY - y * escalaY));

      ctxCurva.strokeStyle = "#C1443A"; ctxCurva.setLineDash([3,3]); ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(ptX, origemY); ctxCurva.lineTo(ptX, ptY); ctxCurva.lineTo(origemX, ptY);
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);

      ctxCurva.fillStyle = "#C1443A";
      ctxCurva.beginPath(); ctxCurva.arc(ptX, ptY, 4.5, 0, 2*Math.PI); ctxCurva.fill();
      ctxCurva.strokeStyle = "#fff"; ctxCurva.lineWidth = 1.4; ctxCurva.stroke();
    }

    function atualizarPainelCalculo(res){
      var htmlGrade = '';
      for (var idx = 0; idx < res.termos.length; idx++) {
        var t = res.termos[idx];
        var corFundo = t.p > 0 ? "#E6F4EA" : (t.p < 0 ? "#FCE8E6" : "#F3F4F6");
        var corTxt = t.p > 0 ? CORES.pos : (t.p < 0 ? CORES.neg : "#374151");
        htmlGrade += '<div class="cn-prodcell" style="background:' + corFundo + ';">' +
          '<div class="cn-mono" style="font-size:9px;color:#6b7280;">' + t.i.toFixed(0) + '×' + t.k + '</div>' +
          '<div class="cn-mono" style="font-size:11px;font-weight:700;color:' + corTxt + ';">' + t.p.toFixed(0) + '</div></div>';
      }
      gradeProdutos.innerHTML = htmlGrade;

      var partes = res.termos.map(function(t){
        return t.p >= 0 ? t.p.toFixed(0) : '(' + t.p.toFixed(0) + ')';
      });
      var nomeFunc = selF.options[selF.selectedIndex].text;
      var htmlExpr = '<div class="cn-mono">z = ' + partes.join(' + ') + ' = <b>' + res.z.toFixed(2) + '</b></div>' +
        '<div class="cn-mono" style="margin-top:4px;">y = ' + nomeFunc + '(z) = <b>' + res.y.toFixed(2) + '</b></div>';
      if (res.p) {
        htmlExpr += '<div style="margin-top:6px;color:#2F6F9F;font-size:10.5px;">💡 Termos com fundo azul tracejado no diagrama vêm de <b>padding</b> — zeros adicionados artificialmente na borda, que não fazem parte da imagem original.</div>';
      }
      expressaoSoma.innerHTML = htmlExpr;
    }

    function atualizar(){
      var passoIdx = parseInt(inStep.value);
      var tipo = selF.value;
      var comPadding = chkP.checked;

      var res = desenharEsquema(passoIdx, tipo, comPadding);
      desenharCurva(res.z, res.y, tipo);
      atualizarPainelCalculo(res);

      var maxSteps = res.dimOut * res.dimOut - 1;
      var passoAtual = res.posR * res.dimOut + res.posC;
      posLabel.textContent = 'Pas ' + (passoAtual + 1) + ' de ' + (maxSteps + 1) +
        '  •  posição (i=' + res.posR + ', j=' + res.posC + ')';

      var nomeFunc = selF.options[selF.selectedIndex].text;
      valTxt.textContent = 'Position (' + res.posR + ',' + res.posC + '): z = ' + res.z.toFixed(2) +
                           '  →  y = ' + nomeFunc + '(z) = ' + res.y.toFixed(2);

      notaFormula.textContent = '📏 Dimension de la sortie : n_sortie = (n + 2p − k)/s + 1 = (' + N_BASE + ' + 2×' + res.p + ' − 3)/1 + 1 = ' + res.dimOut;

      var infoKernel = KERNEL_INFO[kernelAtualId];
      legendaKernel.innerHTML = '<b>' + infoKernel.emoji + ' ' + infoKernel.nome + ':</b> este filtro ' + infoKernel.desc;

      var infoImagem = IMAGENS[imagemAtualId];
      legendaImagem.innerHTML = '<b>' + infoImagem.emoji + ' ' + infoImagem.nome + ':</b> ' + infoImagem.desc;
    }

    function pararAutoplay(){
      if (autoplayInterval) {
        clearInterval(autoplayInterval);
        autoplayInterval = null;
        btnPlay.textContent = '⏵ Auto';
      }
    }

    function setKernel(k, id, btn){
      [btnSobelV, btnSobelH, btnSharpen, btnIdentity].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      kernelAtual = k;
      kernelAtualId = id;
      pararAutoplay();
      atualizar();
    }

    function setImagem(id, btn){
      [btnImgCasa, btnImgFeliz, btnImgTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      imagemAtualId = id;
      imgEntradaBase = converterLinhasParaMatriz(IMAGENS[id].linhas);
      pararAutoplay();
      atualizar();
    }

    btnSobelV.addEventListener('click', function(){ setKernel(KERNELS.sobelV, 'sobelV', btnSobelV); });
    btnSobelH.addEventListener('click', function(){ setKernel(KERNELS.sobelH, 'sobelH', btnSobelH); });
    btnSharpen.addEventListener('click', function(){ setKernel(KERNELS.sharpen, 'sharpen', btnSharpen); });
    btnIdentity.addEventListener('click', function(){ setKernel(KERNELS.identity, 'identity', btnIdentity); });

    btnImgCasa.addEventListener('click', function(){ setImagem('casa', btnImgCasa); });
    btnImgFeliz.addEventListener('click', function(){ setImagem('feliz', btnImgFeliz); });
    btnImgTriste.addEventListener('click', function(){ setImagem('triste', btnImgTriste); });

    inStep.addEventListener('input', function(){ pararAutoplay(); atualizar(); });
    selF.addEventListener('change', function(){ pararAutoplay(); atualizar(); });
    chkP.addEventListener('change', function(){ pararAutoplay(); atualizar(); });

    btnAnterior.addEventListener('click', function(){
      pararAutoplay();
      var max = parseInt(inStep.max);
      var v = parseInt(inStep.value) - 1;
      inStep.value = v < 0 ? max : v;
      atualizar();
    });
    btnProximo.addEventListener('click', function(){
      pararAutoplay();
      var max = parseInt(inStep.max);
      var v = parseInt(inStep.value) + 1;
      inStep.value = v > max ? 0 : v;
      atualizar();
    });
    btnPlay.addEventListener('click', function(){
      if (autoplayInterval) { pararAutoplay(); return; }
      btnPlay.textContent = '⏸ Pausar';
      autoplayInterval = setInterval(function(){
        var max = parseInt(inStep.max);
        var v = parseInt(inStep.value) + 1;
        if (v > max) { pararAutoplay(); v = max; }
        inStep.value = v;
        atualizar();
      }, 130);
    });
    btnReiniciar.addEventListener('click', function(){
      pararAutoplay();
      inStep.value = 0;
      atualizar();
    });

    imgEntradaBase = converterLinhasParaMatriz(IMAGENS[imagemAtualId].linhas);
    atualizar();
  }

  function tryInitCamadaConvImg(){
    var root = document.getElementById('sim-09-camada-conv');
    if(root) initCamadaConvImg(root); else setTimeout(tryInitCamadaConvImg, 200);
  }
  tryInitCamadaConvImg();
})();
</script>
''')

**Figure 9.4:** Simulateur interactif de la couche convolutive : choisissez entre trois images d


<figure id="fig-09-sim-09-camada-conv">
  <img src="imagens/fig-09-sim-09-camada-conv.png" alt=" Simulateur interactif de la couche convolutive : choisissez entre trois images d'entrée 12×12 (maison, visage souriant ou visage triste) pour observer comment les mêmes *noyaux* fixes réagissent à différentes contours et formes. Naviguez dans le champ réceptif avec les boutons ou la lecture automatique, ajustez la fonction d'activation et alternez le zero-padding, en suivant la carte de caractéristiques révélée cellule par cellule, avec le calcul détaillé terme à terme et la formule de la dimension de sortie en temps réel. " style="max-width:80%" />
  <figcaption><strong>Figure 9.4:</strong>  Simulateur interactif de la couche convolutive : choisissez entre trois images d'entrée 12×12 (maison, visage souriant ou visage triste) pour observer comment les mêmes *noyaux* fixes réagissent à différentes contours et formes. Naviguez dans le champ réceptif avec les boutons ou la lecture automatique, ajustez la fonction d'activation et alternez le zero-padding, en suivant la carte de caractéristiques révélée cellule par cellule, avec le calcul détaillé terme à terme et la formule de la dimension de sortie en temps réel. </figcaption>
</figure>

### 9.4.5 Fonction d'activation

La convolution est une opération linéaire. Pour que le réseau puisse modéliser des relations non linéaires entre les entrées et les sorties, on applique une **fonction d'activation** (*activation function*) après chaque couche convolutive.

La fonction la plus utilisée dans les CNN est la **ReLU** (*Rectified Linear Unit*), définie par

$$
\mathrm{ReLU}(x)=\max(0,x).
$$

Cette fonction préserve les valeurs positives et remplace les valeurs négatives par zéro, introduisant ainsi une non-linéarité dans le modèle et favorisant l'entraînement de réseaux profonds avec un faible coût computationnel.

La [Figure 9.5](#fig-09-sim-09-relu) illustre le fonctionnement de la **ReLU** appliquée à la fois à des valeurs individuelles et à une carte de caractéristiques, permettant de comparer la sortie avant et après l'activation.

In [5]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-relu" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>⚡ Simulateur : Fonction d'activation ReLU</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">ReLU(x) = max(0, x)</span>
  </div>
  <div style="padding:20px;background:white;overflow:auto">

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:600;color:#374151;">x =</span>
        <input type="range" id="cap09relu_slider" min="-5" max="5" step="0.1" value="-2.5" style="flex:1;min-width:160px;">
        <span id="cap09relu_valTxt" style="font-family:monospace;font-size:12px;min-width:190px;color:#374151;">x = -2,50 → ReLU(x) = 0,00</span>
      </div>
    </div>

    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Courbe de la fonction ReLU</div>
        <canvas id="cap09relu_canvasCurva" width="280" height="220"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Carte de caractéristiques : avant / après</div>
        <canvas id="cap09relu_canvasMapa" width="260" height="220"></canvas>
        <div style="display:flex;gap:8px;justify-content:center;margin-top:8px;">
          <button id="cap09relu_btnAplicar" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #16a34a;background:#16a34a;color:#fff;cursor:pointer;">Appliquer ReLU à la carte</button>
          <button id="cap09relu_btnReset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">↺ Réinitialiser</button>
        </div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  var cap09relu_MAPA = [
    [ 1.2, -0.8,  3.4, -2.1, 0.5],
    [-1.5,  2.7, -0.3,  1.1, -4.0],
    [ 0.9, -2.9,  4.8, -0.6,  2.2],
    [-3.3,  0.2, -1.1,  3.9, -0.4],
    [ 2.0, -1.7,  0.8, -2.6,  1.4]
  ];

  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var aplicado = false;

    var ctxCurva = root.querySelector('#cap09relu_canvasCurva').getContext('2d');
    var ctxMapa  = root.querySelector('#cap09relu_canvasMapa').getContext('2d');
    var slider   = root.querySelector('#cap09relu_slider');
    var valTxt   = root.querySelector('#cap09relu_valTxt');

    var W = 280, H = 220;
    var origemX = 40, origemY = H - 30;
    var escala = 22;

    function xParaPixel(x){ return origemX + x*escala; }
    function yParaPixel(y){ return origemY - y*escala; }

    function desenharCurva(x){
      ctxCurva.clearRect(0,0,W,H);

      ctxCurva.strokeStyle = "#9ca3af";
      ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(0, origemY); ctxCurva.lineTo(W, origemY);
      ctxCurva.moveTo(origemX, 0); ctxCurva.lineTo(origemX, H);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280";
      ctxCurva.font = "10px sans-serif";
      ctxCurva.fillText("x", W-12, origemY-4);
      ctxCurva.fillText("ReLU(x)", origemX+4, 10);

      ctxCurva.strokeStyle = "#4f46e5";
      ctxCurva.lineWidth = 2.5;
      ctxCurva.beginPath();
      ctxCurva.moveTo(xParaPixel(-5), yParaPixel(0));
      ctxCurva.lineTo(xParaPixel(0), yParaPixel(0));
      ctxCurva.lineTo(xParaPixel(5), yParaPixel(5));
      ctxCurva.stroke();

      var y = Math.max(0, x);
      ctxCurva.fillStyle = "#dc2626";
      ctxCurva.beginPath();
      ctxCurva.arc(xParaPixel(x), yParaPixel(y), 5, 0, 2*Math.PI);
      ctxCurva.fill();

      ctxCurva.strokeStyle = "#fca5a5";
      ctxCurva.setLineDash([3,3]);
      ctxCurva.beginPath();
      ctxCurva.moveTo(xParaPixel(x), origemY);
      ctxCurva.lineTo(xParaPixel(x), yParaPixel(y));
      ctxCurva.lineTo(origemX, yParaPixel(y));
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);
    }

    function corValor(v, apl){
      if (apl && v < 0) v = 0;
      if (v < 0){
        var inten = Math.min(1, Math.abs(v)/5);
        var c = Math.round(255 - inten*180);
        return 'rgb('+c+','+c+',255)';
      } else {
        var inten2 = Math.min(1, v/5);
        var c2 = Math.round(255 - inten2*200);
        return 'rgb('+c2+',255,'+c2+')';
      }
    }

    function desenharMapa(){
      var tam = 42, offX = 20, offY = 10;
      ctxMapa.clearRect(0,0,260,220);
      for (var r=0;r<5;r++){
        for (var c=0;c<5;c++){
          var vOrig = cap09relu_MAPA[r][c];
          var v = aplicado ? Math.max(0, vOrig) : vOrig;
          ctxMapa.fillStyle = corValor(vOrig, aplicado);
          ctxMapa.fillRect(offX+c*tam, offY+r*tam, tam, tam);
          ctxMapa.strokeStyle = "#d1d5db";
          ctxMapa.strokeRect(offX+c*tam, offY+r*tam, tam, tam);
          ctxMapa.fillStyle = "#1f2937";
          ctxMapa.font = "10px monospace";
          ctxMapa.textAlign = "center";
          ctxMapa.fillText(v.toFixed(1), offX+c*tam+tam/2, offY+r*tam+tam/2+4);
        }
      }
      ctxMapa.fillStyle = "#6b7280";
      ctxMapa.font = "10px sans-serif";
      ctxMapa.textAlign = "left";
      ctxMapa.fillText(aplicado ? "Depois da ReLU (negativos → 0)" : "Antes da ReLU (valores brutos da convolução)", offX, 215);
    }

    function atualizarSlider(){
      var x = parseFloat(slider.value);
      var y = Math.max(0, x);
      valTxt.textContent = 'x = ' + x.toFixed(2) + '  →  ReLU(x) = ' + y.toFixed(2);
      desenharCurva(x);
    }

    slider.addEventListener('input', atualizarSlider);

    root.querySelector('#cap09relu_btnAplicar').addEventListener('click', function(){
      aplicado = true;
      desenharMapa();
    });
    root.querySelector('#cap09relu_btnReset').addEventListener('click', function(){
      aplicado = false;
      desenharMapa();
    });

    atualizarSlider();
    desenharMapa();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-relu');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figure 9.5:** Simulateur interactif de la fonction d


<figure id="fig-09-sim-09-relu">
  <img src="imagens/fig-09-sim-09-relu.png" alt=" Simulateur interactif de la fonction d'activation ReLU : faites glisser le curseur pour voir comment les valeurs négatives sont mises à zéro et les valeurs positives sont préservées, à la fois sur la courbe et sur une carte de caractéristiques réelle. " style="max-width:80%" />
  <figcaption><strong>Figure 9.5:</strong>  Simulateur interactif de la fonction d'activation ReLU : faites glisser le curseur pour voir comment les valeurs négatives sont mises à zéro et les valeurs positives sont préservées, à la fois sur la courbe et sur une carte de caractéristiques réelle. </figcaption>
</figure>

Les cartes de caractéristiques résultant de la convolution et de l’activation préservent la structure spatiale de l’image. Dans de nombreuses architectures, l’étape suivante réduit leur résolution au moyen d’une opération de *pooling*.

### 9.4.6 *Pooling*

La couche de ***pooling*** réduit la résolution spatiale des cartes de caractéristiques, tout en préservant les informations les plus pertinentes pour les étapes suivantes du traitement. L'opération la plus utilisée est le ***max-pooling***, qui sélectionne la valeur maximale dans chaque fenêtre de l'image :

$$
P(i,j)=\max_{(u,v)\in\text{janela}(i,j)}F(u,v).
$$

Cette réduction diminue le coût computationnel des couches suivantes et rend la représentation plus robuste aux petites variations de position des motifs présents dans l'image.

La [Figure 9.6](#fig-09-sim-09-pooling) présente cette opération sur une carte de caractéristiques de 8×8 pixels, réduite à 4×4 par des fenêtres de 2×2 avec un pas égal à 2, en alternant entre **max-pooling** et **average-pooling** — qui calcule, au lieu du maximum, la moyenne des valeurs de la fenêtre correspondante.

In [6]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-pooling" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🔻 Simulateur : Pooling</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">fenêtre 2×2, pas 2</span>
  </div>
  <div style="padding:20px;background:white;overflow:auto">

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:10px;">
        <span style="font-size:11px;font-weight:600;color:#374151;">Type :</span>
        <button id="cap09pool_btnMax" class="cap09pool_active" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #4f46e5;background:#4f46e5;color:#fff;cursor:pointer;">Max-pooling</button>
        <button id="cap09pool_btnAvg" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">Average-pooling</button>
      </div>
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <button id="cap09pool_btnPasso" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #16a34a;background:#16a34a;color:#fff;cursor:pointer;">▶ Avancer d'1 pas</button>
        <button id="cap09pool_btnTudo" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">⏭ Tout calculer</button>
        <button id="cap09pool_btnReset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">↺ Réinitialiser</button>
        <span style="font-size:11px;color:#6b7280;">Fenêtre actuelle : <b id="cap09pool_posTxt">(0, 0)</b> sur 4×4</span>
      </div>
      <div id="cap09pool_explicacao" style="font-size:10.5px;color:#6b7280;margin-top:8px;line-height:1.4;">O <b>max-pooling</b> conserve uniquement la valeur la plus élevée de chaque fenêtre 2×2, réduisant la résolution spatiale de moitié et préservant les réponses les plus fortes de la carte de caractéristiques.</div>
    </div>

    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Carte d'entrée (8×8) — fenêtre actuelle mise en évidence</div>
        <canvas id="cap09pool_canvasEntrada" width="240" height="240"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Carte réduite (4×4)</div>
        <canvas id="cap09pool_canvasSaida" width="160" height="160"></canvas>
      </div>
    </div>

  </div>
</div>

<style>
  #sim-09-pooling button.cap09pool_active { background: #4f46e5 !important; color: #fff !important; border-color: #4f46e5 !important; }
</style>

<script>
(function(){
  var cap09pool_ENTRADA = [
    [1, 3, 2, 8,  5, 1, 0, 2],
    [4, 6, 1, 2,  3, 9, 1, 0],
    [0, 1, 9, 3,  1, 2, 8, 4],
    [2, 5, 4, 7,  0, 1, 3, 6],
    [3, 8, 1, 0,  6, 2, 5, 1],
    [1, 2, 6, 4,  9, 0, 2, 3],
    [7, 0, 3, 1,  2, 8, 1, 4],
    [2, 4, 1, 5,  3, 1, 6, 9]
  ];
  var MAX_GLOBAL = 9;

  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var tipo = "max";
    var saida = Array.from({length:4}, function(){ return Array(4).fill(null); });
    var pos = {r:0, c:0};

    var ctxEnt = root.querySelector('#cap09pool_canvasEntrada').getContext('2d');
    var ctxSai = root.querySelector('#cap09pool_canvasSaida').getContext('2d');
    var posTxt = root.querySelector('#cap09pool_posTxt');
    var explicacao = root.querySelector('#cap09pool_explicacao');

    function corEscala(v, max){
      var inten = Math.min(1, v/max);
      var c = Math.round(245 - inten*160);
      return 'rgb('+c+','+(c+8)+',255)';
    }

    function desenharEntrada(){
      var tam = 30;
      ctxEnt.clearRect(0,0,240,240);
      for (var r=0;r<8;r++){
        for (var c=0;c<8;c++){
          var v = cap09pool_ENTRADA[r][c];
          ctxEnt.fillStyle = corEscala(v, MAX_GLOBAL);
          ctxEnt.fillRect(c*tam, r*tam, tam, tam);
          ctxEnt.strokeStyle = "#e5e7eb";
          ctxEnt.strokeRect(c*tam, r*tam, tam, tam);
          ctxEnt.fillStyle = "#1f2937";
          ctxEnt.font = "11px monospace";
          ctxEnt.textAlign = "center";
          ctxEnt.fillText(v, c*tam+tam/2, r*tam+tam/2+4);
        }
      }
      if (pos.r < 4){
        ctxEnt.strokeStyle = "#dc2626";
        ctxEnt.lineWidth = 3;
        ctxEnt.strokeRect(pos.c*2*tam, pos.r*2*tam, tam*2, tam*2);
        ctxEnt.lineWidth = 1;
      }
    }

    function desenharSaida(){
      var tam = 40;
      ctxSai.clearRect(0,0,160,160);
      for (var r=0;r<4;r++){
        for (var c=0;c<4;c++){
          var v = saida[r][c];
          ctxSai.fillStyle = (v === null) ? "#f3f4f6" : corEscala(v, MAX_GLOBAL);
          ctxSai.fillRect(c*tam, r*tam, tam, tam);
          ctxSai.strokeStyle = "#e5e7eb";
          ctxSai.strokeRect(c*tam, r*tam, tam, tam);
          if (v !== null){
            ctxSai.fillStyle = "#1f2937";
            ctxSai.font = "11px monospace";
            ctxSai.textAlign = "center";
            ctxSai.fillText(v.toFixed(1), c*tam+tam/2, r*tam+tam/2+4);
          }
        }
      }
      if (pos.r < 4){
        ctxSai.strokeStyle = "#dc2626";
        ctxSai.lineWidth = 2;
        ctxSai.strokeRect(pos.c*tam, pos.r*tam, tam, tam);
        ctxSai.lineWidth = 1;
      }
    }

    function calcularJanela(r, c){
      var vals = [];
      for (var i=0;i<2;i++) for (var j=0;j<2;j++) vals.push(cap09pool_ENTRADA[r*2+i][c*2+j]);
      if (tipo === "max") return Math.max.apply(null, vals);
      return vals.reduce(function(a,b){return a+b;},0) / vals.length;
    }

    function avancarPasso(){
      if (pos.r >= 4) return;
      saida[pos.r][pos.c] = calcularJanela(pos.r, pos.c);
      pos.c++;
      if (pos.c >= 4){ pos.c = 0; pos.r++; }
      render();
    }

    function calcularTudo(){
      while (pos.r < 4) avancarPasso();
    }

    function resetar(){
      saida = Array.from({length:4}, function(){ return Array(4).fill(null); });
      pos = {r:0, c:0};
      render();
    }

    function render(){
      desenharEntrada();
      desenharSaida();
      posTxt.textContent = pos.r < 4 ? '(' + pos.r + ', ' + pos.c + ')' : 'concluído';
    }

    function selecionarTipo(t){
      tipo = t;
      root.querySelector('#cap09pool_btnMax').classList.toggle('cap09pool_active', t === "max");
      root.querySelector('#cap09pool_btnAvg').classList.toggle('cap09pool_active', t === "avg");
      explicacao.innerHTML = t === "max"
        ? "O <b>max-pooling</b> mantém apenas o maior valor de cada janela 2×2, reduzindo a resolução espacial pela metade e preservando as respostas mais fortes do mapa de características."
        : "O <b>average-pooling</b> calcula a média dos quatro valores de cada janela 2×2, suavizando a informação em vez de preservar apenas o pico de resposta.";
      resetar();
    }

    root.querySelector('#cap09pool_btnMax').addEventListener('click', function(){ selecionarTipo("max"); });
    root.querySelector('#cap09pool_btnAvg').addEventListener('click', function(){ selecionarTipo("avg"); });
    root.querySelector('#cap09pool_btnPasso').addEventListener('click', avancarPasso);
    root.querySelector('#cap09pool_btnTudo').addEventListener('click', calcularTudo);
    root.querySelector('#cap09pool_btnReset').addEventListener('click', resetar);

    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-pooling');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figure 9.6:** Simulateur interactif de *pooling* : choisissez entre *max-pooling* et *average-pooling* et avancez pas à pas pour observer la réduction de la résolution spatiale de la carte de caractéristiques.


<figure id="fig-09-sim-09-pooling">
  <img src="imagens/fig-09-sim-09-pooling.png" alt=" Simulateur interactif de *pooling* : choisissez entre *max-pooling* et *average-pooling* et avancez pas à pas pour observer la réduction de la résolution spatiale de la carte de caractéristiques. " style="max-width:80%" />
  <figcaption><strong>Figure 9.6:</strong>  Simulateur interactif de *pooling* : choisissez entre *max-pooling* et *average-pooling* et avancez pas à pas pour observer la réduction de la résolution spatiale de la carte de caractéristiques. </figcaption>
</figure>

Ensemble, convolution, fonction d'activation et *pooling* forment le bloc de base utilisé dans la construction d'une CNN.

### 9.4.7 Entraînement des réseaux de neurones : comment les CNN apprennent

Une CNN apprend en ajustant automatiquement ses paramètres — les coefficients des filtres convolutifs, les poids des couches entièrement connectées et les biais (*biases*) — à partir d'exemples étiquetés. Cet entraînement est itératif et comporte trois étapes : mesurer l'erreur produite par le réseau au moyen d'une **fonction de perte** (*loss function*), calculer comment cette erreur dépend de chaque paramètre grâce à la **rétropropagation** (*backpropagation*) et mettre à jour les paramètres avec un **algorithme d'optimisation** (*optimizer*).

#### 9.4.7.1 Fonction de perte (*Loss Function*)

La **fonction de perte** (*loss function*) quantifie la différence entre la prédiction du réseau et la réponse correcte, appelée **vérité de référence** (*ground truth*). Le résultat est un scalaire $L$ : plus la perte est faible, plus la prédiction est proche de la réponse attendue.

Dans les problèmes de classification multiclasse, la fonction la plus utilisée est l'**Entropie Croisée** (*Cross-Entropy Loss*), appliquée aux probabilités produites par la couche **Softmax** :

$$
L=-\sum_{c=1}^{C} y_c \log(\hat{y}_c),
$$

où $C$ est le nombre de classes, $y_c$ est le label réel en codage *one-hot* et $\hat{y}_c$ est la probabilité prédite pour la classe $c$. La perte se rapproche de zéro lorsque le réseau attribue une forte probabilité à la classe correcte et augmente rapidement à mesure que cette probabilité diminue.

La [Figure 9.7](#fig-09-sim-09-loss) illustre ce comportement : le simulateur permet de sélectionner la classe correcte et de modifier les probabilités produites par la *Softmax*, affichant en temps réel la variation de la fonction de perte.

In [7]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-loss" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-loss .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-loss .cn-grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-loss .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-loss .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-loss input[type=range] { accent-color:#2F6F9F; width:100%; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">📉 Simulateur : Fonction de Perte (Entropie Croisée)</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">L = -log(ŷ_cible)</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletores de Rótulo Real (Ground Truth) -->
    <div style="margin-bottom:12px;">
      <div class="cn-grouplabel">CLASSE RÉELLE DE L'IMAGE (VÉRITÉ TERRAIN : y_c = 1)</div>
      <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;max-width:340px;">
        <button id="cap09loss_btnCasa" class="cn-modebtn active">🏠 Maison</button>
        <button id="cap09loss_btnFeliz" class="cn-modebtn">😊 Heureux</button>
        <button id="cap09loss_btnTriste" class="cn-modebtn">😢 Triste</button>
      </div>
    </div>

    <!-- Ajuste de Probabilidades Preditas (Softmax ŷ) -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:12px;margin-bottom:14px;">
      <div class="cn-grouplabel" style="margin-bottom:8px;">PROBABILITÉS ESTIMÉES PAR SOFTMAX (ŷ_c)</div>
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));gap:12px;">
        
        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>🏠 Maison (ŷ_1) :</span>
            <span id="cap09loss_txtProbCasa" class="cn-mono" style="color:#2F6F9F;">0.70</span>
          </div>
          <input type="range" id="cap09loss_rangeCasa" min="0.01" max="0.98" step="0.01" value="0.70">
        </div>

        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>😊 Heureux (ŷ_2) :</span>
            <span id="cap09loss_txtProbFeliz" class="cn-mono" style="color:#2F6F9F;">0.20</span>
          </div>
          <input type="range" id="cap09loss_rangeFeliz" min="0.01" max="0.98" step="0.01" value="0.20">
        </div>

        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>😢 Triste (ŷ_3) :</span>
            <span id="cap09loss_txtProbTriste" class="cn-mono" style="color:#2F6F9F;">0.10</span>
          </div>
          <input type="range" id="cap09loss_rangeTriste" min="0.01" max="0.98" step="0.01" value="0.10">
        </div>

      </div>
    </div>

    <!-- Curva da Função Logarítmica & Resultado -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#5E5A4A;">Courbe de pénalisation L = -log(ŷ_cible)</div>
        <canvas id="cap09loss_canvasCurva" width="260" height="170" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
      </div>

      <div style="flex:1;min-width:240px;">
        <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:12px;margin-bottom:8px;">
          <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:4px;">CALCUL DE LA PERTE :</div>
          <div id="cap09loss_exprCalc" class="cn-mono" style="font-size:11.5px;color:#374151;line-height:1.6;"></div>
          <div style="margin-top:6px;font-size:14px;font-weight:700;color:#C1443A;">
            Perte L = <span id="cap09loss_valTotal" class="cn-mono">0.3567</span>
          </div>
        </div>
        <div id="cap09loss_explicacao" style="font-size:10.5px;color:#8A8371;line-height:1.4;"></div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function initLoss(root){
    if(!root || root.dataset.initLoss) return;
    root.dataset.initLoss = "1";

    var classeAlvo = "casa"; // "casa", "feliz", "triste"
    var probs = { casa: 0.70, feliz: 0.20, triste: 0.10 };

    var btnCasa   = root.querySelector('#cap09loss_btnCasa');
    var btnFeliz  = root.querySelector('#cap09loss_btnFeliz');
    var btnTriste = root.querySelector('#cap09loss_btnTriste');

    var rangeCasa   = root.querySelector('#cap09loss_rangeCasa');
    var rangeFeliz  = root.querySelector('#cap09loss_rangeFeliz');
    var rangeTriste = root.querySelector('#cap09loss_rangeTriste');

    var txtProbCasa   = root.querySelector('#cap09loss_txtProbCasa');
    var txtProbFeliz  = root.querySelector('#cap09loss_txtProbFeliz');
    var txtProbTriste = root.querySelector('#cap09loss_txtProbTriste');

    var exprCalc   = root.querySelector('#cap09loss_exprCalc');
    var valTotal   = root.querySelector('#cap09loss_valTotal');
    var explicacao = root.querySelector('#cap09loss_explicacao');

    var canvas = root.querySelector('#cap09loss_canvasCurva');
    var ctx    = canvas.getContext('2d');

    function normalizarProbs(modificado){
      var somaOutros = 0;
      var chaves = ["casa", "feliz", "triste"];
      chaves.forEach(function(k){ if(k !== modificado) somaOutros += probs[k]; });
      
      var restante = 1.0 - probs[modificado];
      if(somaOutros > 0){
        chaves.forEach(function(k){
          if(k !== modificado) probs[k] = (probs[k] / somaOutros) * restante;
        });
      } else {
        chaves.forEach(function(k){
          if(k !== modificado) probs[k] = restante / 2.0;
        });
      }

      rangeCasa.value   = probs.casa;
      rangeFeliz.value  = probs.feliz;
      rangeTriste.value = probs.triste;

      txtProbCasa.textContent   = probs.casa.toFixed(2);
      txtProbFeliz.textContent  = probs.feliz.toFixed(2);
      txtProbTriste.textContent = probs.triste.toFixed(2);
    }

    function desenharCurvaLog(){
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0,0,W,H);

      // Eixos
      ctx.strokeStyle = "#E4DCC8"; ctx.lineWidth = 1;
      ctx.beginPath();
      ctx.moveTo(30, 10); ctx.lineTo(30, H-20); ctx.lineTo(W-10, H-20);
      ctx.stroke();

      // Curva -log(x)
      ctx.strokeStyle = "#2F6F9F"; ctx.lineWidth = 2;
      ctx.beginPath();
      for(var x=0.002; x<=0.98; x+=0.01){
        var loss = -Math.log(x);
        var cx = 30 + x * (W - 40);
        var cy = (H - 20) - (loss / 4.0) * (H - 30);
        cy = Math.max(10, Math.min(H-20, cy));
        if(x === 0.002) ctx.moveTo(cx, cy); else ctx.lineTo(cx, cy);
      }
      ctx.stroke();

      // Ponto Atual
      var probAlvo = probs[classeAlvo];
      var lossAlvo = -Math.log(probAlvo);
      var ptX = 30 + probAlvo * (W - 40);
      var ptY = (H - 20) - (lossAlvo / 4.0) * (H - 30);
      ptY = Math.max(10, Math.min(H-20, ptY));

      ctx.strokeStyle = "#C1443A"; ctx.setLineDash([3,3]);
      ctx.beginPath();
      ctx.moveTo(ptX, H-20); ctx.lineTo(ptX, ptY); ctx.lineTo(30, ptY);
      ctx.stroke(); ctx.setLineDash([]);

      ctx.fillStyle = "#C1443A";
      ctx.beginPath(); ctx.arc(ptX, ptY, 4.5, 0, 2*Math.PI); ctx.fill();
    }

    function atualizar(){
      desenharCurvaLog();
      var probAlvo = probs[classeAlvo];
      var lossVal  = -Math.log(probAlvo);

      var nomes = { casa: "🏠 Casa", feliz: "😊 Feliz", triste: "😢 Triste" };
      exprCalc.innerHTML = 'L = -log(ŷ_' + classeAlvo + ') = -log(' + probAlvo.toFixed(2) + ')';
      valTotal.textContent = lossVal.toFixed(4);

      if(probAlvo > 0.8) {
        explicacao.innerHTML = "<b>Excelente precisão:</b> A rede atribuiu alta probabilidade à classe correta (" + nomes[classeAlvo] + "), gerando uma perda muito próxima de zero.";
      } else if(probAlvo > 0.4) {
        explicacao.innerHTML = "<b>Incerteza moderada:</b> A probabilidade da classe correta (" + nomes[classeAlvo] + ") é mediana, resultando em uma penalização moderada sobre a rede.";
      } else {
        explicacao.innerHTML = "<b>Erro alto (Confusão):</b> A rede atribuiu baixa probabilidade à classe real (" + nomes[classeAlvo] + "). A função logarítmica penaliza fortemente esse erro, gerando um alto valor de perda $L$.";
      }
    }

    function selecionarClasse(c, btn){
      [btnCasa, btnFeliz, btnTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      classeAlvo = c;
      atualizar();
    }

    btnCasa.addEventListener('click', function(){ selecionarClasse("casa", btnCasa); });
    btnFeliz.addEventListener('click', function(){ selecionarClasse("feliz", btnFeliz); });
    btnTriste.addEventListener('click', function(){ selecionarClasse("triste", btnTriste); });

    rangeCasa.addEventListener('input', function(){ probs.casa = parseFloat(this.value); normalizarProbs("casa"); atualizar(); });
    rangeFeliz.addEventListener('input', function(){ probs.feliz = parseFloat(this.value); normalizarProbs("feliz"); atualizar(); });
    rangeTriste.addEventListener('input', function(){ probs.triste = parseFloat(this.value); normalizarProbs("triste"); atualizar(); });

    atualizar();
  }

  function tryInitLoss(){
    var root = document.getElementById('sim-09-loss');
    if(root) initLoss(root); else setTimeout(tryInitLoss, 200);
  }
  tryInitLoss();
})();
</script>
''')

**Figure 9.7:** Simulateur interactif de la Fonction de Perte (*Cross-Entropy*) : sélectionnez la classe réelle de l


<figure id="fig-09-sim-09-loss">
  <img src="imagens/fig-09-sim-09-loss.png" alt=" Simulateur interactif de la Fonction de Perte (*Cross-Entropy*) : sélectionnez la classe réelle de l'image (Maison, Heureux ou Triste) et ajustez les probabilités estimées par Softmax pour visualiser le calcul de la pénalisation scalaire et le graphique du logarithme négatif en temps réel. " style="max-width:80%" />
  <figcaption><strong>Figure 9.7:</strong>  Simulateur interactif de la Fonction de Perte (*Cross-Entropy*) : sélectionnez la classe réelle de l'image (Maison, Heureux ou Triste) et ajustez les probabilités estimées par Softmax pour visualiser le calcul de la pénalisation scalaire et le graphique du logarithme négatif en temps réel. </figcaption>
</figure>

#### 9.4.7.2 Rétropropagation (*Backpropagation*)

Après le calcul de la perte, il est nécessaire de déterminer comment chaque paramètre du réseau contribue à ce résultat. Cette étape est réalisée par la **rétropropagation** (*backpropagation*), qui applique la **Règle de la Chaîne** du calcul différentiel pour obtenir le gradient de la fonction de perte par rapport à chaque paramètre.

Pour un paramètre $w$, ce gradient est donné par

$$
\frac{\partial L}{\partial w}.
$$

Le gradient indique comment la perte varie en fonction de petites modifications de $w$ : un gradient positif indique qu’augmenter $w$ augmente la perte, et un gradient négatif indique l’effet opposé.

La [Figure 9.8](#fig-09-sim-09-backprop) présente ce processus de manière visuelle, montrant la propagation du gradient de la couche de sortie jusqu’aux premières couches convolutives.

In [8]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-backprop" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-backprop .cap09backprop_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-backprop .cap09backprop_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-backprop .cap09backprop_navbtn {
      padding:6px 12px; font-size:11px; font-weight:700; border-radius:8px; border:1px solid #E4DCC8;
      background:#FAF6EC; color:#374151; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-backprop .cap09backprop_navbtn:hover { background:#F1EAD7; }
    
    /* Blocos Interativos do Fluxo */
    #sim-09-backprop .cap09backprop_node {
      flex: 1;
      min-width: 90px;
      padding: 10px 6px;
      border-radius: 10px;
      border: 1px solid #E4DCC8;
      background: #FFFFFF;
      text-align: center;
      transition: all 0.25s ease;
      box-shadow: 0 1px 2px rgba(0,0,0,0.02);
      cursor: pointer;
    }
    #sim-09-backprop .cap09backprop_node_title {
      font-size: 11px;
      font-weight: 700;
      color: #374151;
    }
    #sim-09-backprop .cap09backprop_node_sub {
      font-size: 9px;
      font-weight: 600;
      color: #8A8371;
      margin-top: 3px;
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active {
      border: 2px solid #C1443A;
      background: #FCE8E6;
      box-shadow: 0 3px 8px rgba(193,68,58,0.15);
      transform: translateY(-2px);
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active .cap09backprop_node_title {
      color: #C1443A;
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active .cap09backprop_node_sub {
      color: #A8322A;
    }
    
    /* Seta do Fluxo Reverso */
    #sim-09-backprop .cap09backprop_arrow {
      font-size: 14px;
      font-weight: bold;
      color: #D1D5DB;
      transition: color 0.2s ease;
      padding: 0 2px;
    }
    #sim-09-backprop .cap09backprop_arrow.cap09backprop_active_arrow {
      color: #C1443A;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⬅️ Simulateur : Rétropropagation (Backpropagation)</span>
    <span class="cap09backprop_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">∂L/∂w = (∂L/∂y) · (∂y/∂z) · (∂z/∂w)</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Controles do Passo a Passo do Backprop -->
    <div style="display:flex;gap:10px;align-items:center;justify-content:space-between;margin-bottom:14px;flex-wrap:wrap;">
      <div style="display:flex;gap:6px;">
        <button id="cap09backprop_btnVoltar" class="cap09backprop_navbtn">◀ Étape Précédente</button>
        <button id="cap09backprop_btnAvancar" class="cap09backprop_navbtn" style="border-color:#C1443A;color:#C1443A;background:#FCE8E6;">Étape Inverse (Backprop) ◀</button>
        <button id="cap09backprop_btnReset" class="cap09backprop_navbtn">↺ Réinitialiser</button>
      </div>
      <span class="cap09backprop_mono" id="cap09backprop_txtEtapa" style="font-size:11px;color:#2F6F9F;font-weight:700;">Étape 1 sur 4 : Sortie (Perte & Softmax)</span>
    </div>

    <!-- Fluxo Visual das Camadas (Flexbox em alta resolução) -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:16px 12px;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8A8371;margin-bottom:8px;letter-spacing:.3px;text-align:center;">
        DIRECTION DE LA PROPAGATION DE L'ERREUR (FLUX INVERSÉ ⟵)
      </div>
      <div style="display:flex;align-items:center;justify-content:space-between;gap:4px;max-width:620px;margin:0 auto;">
        
        <div id="cap09backprop_node3" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Noyaux Conv1</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂K</div>
        </div>

        <div id="cap09backprop_arrow2" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node2" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Max-Pooling</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂X_pool</div>
        </div>

        <div id="cap09backprop_arrow1" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node1" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Couches FC</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂W_fc</div>
        </div>

        <div id="cap09backprop_arrow0" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node0" class="cap09backprop_node cap09backprop_active">
          <div class="cap09backprop_node_title">Perte / Softmax</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂y_pred</div>
        </div>

      </div>
    </div>

    <!-- Painel da Regra da Cadeia Detalhada -->
    <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:12px;padding:14px;">
      <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:6px;display:flex;align-items:center;gap:6px;">
        <span>🔗 Règle de la Chaîne dans la Couche Actuelle :</span>
      </div>
      <div id="cap09backprop_exprCadeia" class="cap09backprop_mono" style="font-size:12px;font-weight:700;color:#C1443A;line-height:1.6;margin-bottom:8px;background:#FFF;padding:8px 10px;border-radius:8px;border:1px solid #E4DCC8;"></div>
      <div id="cap09backprop_descPasso" style="font-size:11.5px;color:#374151;line-height:1.5;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function cap09backprop_init(cap09backprop_root){
    if(!cap09backprop_root || cap09backprop_root.dataset.initBackprop) return;
    cap09backprop_root.dataset.initBackprop = "1";

    var cap09backprop_passoAtual = 0; // 0: Loss/Softmax, 1: Camada FC, 2: Pooling, 3: Conv1 Kernels

    var cap09backprop_btnVoltar  = cap09backprop_root.querySelector('#cap09backprop_btnVoltar');
    var cap09backprop_btnAvancar = cap09backprop_root.querySelector('#cap09backprop_btnAvancar');
    var cap09backprop_btnReset   = cap09backprop_root.querySelector('#cap09backprop_btnReset');

    var cap09backprop_txtEtapa   = cap09backprop_root.querySelector('#cap09backprop_txtEtapa');
    var cap09backprop_exprCadeia = cap09backprop_root.querySelector('#cap09backprop_exprCadeia');
    var cap09backprop_descPasso  = cap09backprop_root.querySelector('#cap09backprop_descPasso');

    var cap09backprop_ETAPAS = [
      {
        nome: "Passo 1 de 4: Saída (Loss & Softmax)",
        expressao: "∂L/∂y_pred = y_pred - y_real = 0.85 - 1.00 = -0.15",
        desc: "O algoritmo de Backpropagation começa no final do pipeline, calculando a derivada direta da função de perda por Entropia Cruzada em relação à probabilidade gerada pela Softmax."
      },
      {
        nome: "Passo 2 de 4: Camadas Densas (Fully Connected)",
        expressao: "∂L/∂w_fc = (∂L/∂y_pred) · (∂y_pred/∂z_fc) = (-0.15) · (0.42) = -0.063",
        desc: "O sinal de erro retropropaga pelas camadas totalmente conectadas através de multiplicadores de matrizes, definindo quanto cada peso denso contribuiu para o desvio final."
      },
      {
        nome: "Passo 3 de 4: Camada de Max-Pooling",
        expressao: "∂L/∂x_pool = (∂L/∂y_fc) · Mútil_max  ➔  Roteado integralmente para a posição do valor máximo",
        desc: "Na camada de Max-Pooling, não há pesos treináveis. O gradiente é repassado sem alteração exatamente para o pixel que forneceu o valor máximo no Forward Pass, enquanto os demais pixels recebem gradiente zero."
      },
      {
        nome: "Passo 4 de 4: Filtros Convolucionais (Conv1 Kernels)",
        expressao: "∂L/∂K(u,v) = ∑ (∂L/∂F) · I(i+u, j+v)  ➔  Gradiente acumulado do filtro 3×3",
        desc: "O erro atinge os coeficientes numéricos dos filtros originais. Como o mesmo kernel foi reutilizado sobre várias regiões da imagem, os gradientes de todas as posições do campo receptivo são somados para atualizar o filtro."
      }
    ];

    function cap09backprop_atualizarUI(){
      var cap09backprop_info = cap09backprop_ETAPAS[cap09backprop_passoAtual];
      cap09backprop_txtEtapa.textContent = cap09backprop_info.nome;
      cap09backprop_exprCadeia.innerHTML = cap09backprop_info.expressao;
      cap09backprop_descPasso.innerHTML  = cap09backprop_info.desc;

      // Atualizar nós ativos
      for(var cap09backprop_i=0; cap09backprop_i<4; cap09backprop_i++){
        var cap09backprop_node = cap09backprop_root.querySelector('#cap09backprop_node' + cap09backprop_i);
        if(cap09backprop_node){
          if(cap09backprop_i === cap09backprop_passoAtual){
            cap09backprop_node.classList.add('cap09backprop_active');
          } else {
            cap09backprop_node.classList.remove('cap09backprop_active');
          }
        }
      }

      // Atualizar setas ativas
      for(var cap09backprop_j=0; cap09backprop_j<3; cap09backprop_j++){
        var cap09backprop_arrow = cap09backprop_root.querySelector('#cap09backprop_arrow' + cap09backprop_j);
        if(cap09backprop_arrow){
          if(cap09backprop_j < cap09backprop_passoAtual){
            cap09backprop_arrow.classList.add('cap09backprop_active_arrow');
          } else {
            cap09backprop_arrow.classList.remove('cap09backprop_active_arrow');
          }
        }
      }
    }

    // Permitir clicar nos nós diretamente
    for(var cap09backprop_k=0; cap09backprop_k<4; cap09backprop_k++){
      (function(idx){
        var cap09backprop_n = cap09backprop_root.querySelector('#cap09backprop_node' + idx);
        if(cap09backprop_n){
          cap09backprop_n.addEventListener('click', function(){
            cap09backprop_passoAtual = idx;
            cap09backprop_atualizarUI();
          });
        }
      })(cap09backprop_k);
    }

    cap09backprop_btnAvancar.addEventListener('click', function(){
      if(cap09backprop_passoAtual < cap09backprop_ETAPAS.length - 1){
        cap09backprop_passoAtual++;
        cap09backprop_atualizarUI();
      }
    });

    cap09backprop_btnVoltar.addEventListener('click', function(){
      if(cap09backprop_passoAtual > 0){
        cap09backprop_passoAtual--;
        cap09backprop_atualizarUI();
      }
    });

    cap09backprop_btnReset.addEventListener('click', function(){
      cap09backprop_passoAtual = 0;
      cap09backprop_atualizarUI();
    });

    cap09backprop_atualizarUI();
  }

  function cap09backprop_tryInit(){
    var cap09backprop_root = document.getElementById('sim-09-backprop');
    if(cap09backprop_root) cap09backprop_init(cap09backprop_root); else setTimeout(cap09backprop_tryInit, 200);
  }
  cap09backprop_tryInit();
})();
</script>
''')

**Figure 9.8:** Simulateur interactif de *Rétropropagation* : avancez dans les étapes de la Règle de la Chaîne pour suivre le flux du signal d


<figure id="fig-09-sim-09-backprop">
  <img src="imagens/fig-09-sim-09-backprop.png" alt=" Simulateur interactif de *Rétropropagation* : avancez dans les étapes de la Règle de la Chaîne pour suivre le flux du signal d'erreur dans le sens inverse du réseau, en observant le calcul des dérivées partielles du gradient à chaque couche. " style="max-width:80%" />
  <figcaption><strong>Figure 9.8:</strong>  Simulateur interactif de *Rétropropagation* : avancez dans les étapes de la Règle de la Chaîne pour suivre le flux du signal d'erreur dans le sens inverse du réseau, en observant le calcul des dérivées partielles du gradient à chaque couche. </figcaption>
</figure>

#### 9.4.7.3 Algorithmes d’optimisation

Après le calcul des gradients, un **algorithme d’optimisation** (*optimizer*) met à jour les paramètres du réseau afin de réduire la fonction de perte. Dans les réseaux profonds, cette recherche s’effectue dans un espace de haute dimension et, en général, **non convexe**, ce qui rend l’optimisation un problème difficile.

Pour faciliter la compréhension, la [Figure 9.9](#fig-09-sim-09-opt) utilise une **surface de perte simplifiée**, avec un minimum global, un minimum local et une barrière entre ces régions. Le **minimum global** correspond à la plus petite valeur de la fonction de perte et représente le meilleur ensemble de paramètres du réseau ; un **minimum local** présente également une perte faible, mais peut être éloigné de la meilleure solution. Lorsque l’optimisation reste bloquée dans un minimum local, les ajustements des filtres, des poids et des biais deviennent très faibles, et l’entraînement s’arrête avant d’atteindre un modèle avec une erreur moindre.

##### Descente de Gradient Stochastique (*SGD*)

La **Descente de Gradient Stochastique** (*Stochastic Gradient Descent* — *SGD*) met à jour les paramètres dans la direction opposée au gradient :

$$
w_{\text{novo}} = w_{\text{atual}} - \eta \frac{\partial L}{\partial w},
$$

où $\eta$ est le **taux d'apprentissage** (*learning rate*), responsable du contrôle de la taille de la mise à jour. Le *SGD* utilise uniquement le gradient de l'itération actuelle ; lorsque la recherche atteint un minimum local, les gradients deviennent très faibles et les mises à jour cessent pratiquement.

##### Optimiseurs Adaptatifs : *Adam*

L'**Adam** (*Adaptive Moment Estimation*) combine des estimations adaptatives des premiers et deuxièmes moments des gradients (KINGMA, 2015), en adaptant le taux d'apprentissage de chaque paramètre individuellement. Cette adaptation favorise, dans de nombreux cas, le dépassement des minima locaux qui retiendraient le *SGD*.

La [Figure 9.9](#fig-09-sim-09-opt) compare la trajectoire du *SGD* et de l'*Adam* sur la même surface de perte non convexe.

In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-opt" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-opt .cap09opt_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-opt .cap09opt_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-opt .cap09opt_modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:8px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-opt .cap09opt_modebtn.cap09opt_active { background:#26241D; color:#FBF7EE; }
    #sim-09-opt .cap09opt_playbtn {
      padding:6px 12px; font-size:11px; font-weight:700; border-radius:8px; border:1px solid #2F6F9F;
      background:#EAF2FA; color:#2F6F9F; cursor:pointer; transition:background .15s ease;
    }
    #sim-09-opt .cap09opt_playbtn:hover { background:#DCEEFB; }
    #sim-09-opt .cap09opt_navbtn {
      font-size:11px; font-weight:600; padding:6px 10px; border-radius:8px;
      border:1px solid #E4DCC8; background:#FFF; color:#26241D; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-opt .cap09opt_navbtn:hover { background:#F1EAD7; }
    #sim-09-opt .cap09opt_infobox {
      background:#EAF2FA; border:1px solid #CFE2F3; border-radius:10px; padding:9px 12px;
      font-size:11px; color:#2c4a63; line-height:1.5; margin-bottom:12px;
    }
    #sim-09-opt .cap09opt_legendrow { display:flex; gap:14px; flex-wrap:wrap; align-items:center; margin-top:10px; font-size:10.5px; color:#5E5A4A; }
    #sim-09-opt .cap09opt_legenditem { display:flex; align-items:center; gap:5px; }
    #sim-09-opt .cap09opt_swatch { width:14px; height:3px; border-radius:2px; display:inline-block; }
    #sim-09-opt .cap09opt_dot { width:9px; height:9px; border-radius:50%; display:inline-block; }
    #sim-09-opt .cap09opt_checklbl { display:flex; align-items:center; gap:5px; font-size:10.5px; font-weight:600; color:#374151; cursor:pointer; user-select:none; }
    #sim-09-opt .cap09opt_statgrid { display:grid; grid-template-columns:1fr 1fr; gap:6px 14px; margin-top:6px; }
    #sim-09-opt .cap09opt_statlbl { font-size:9.5px; color:#8A8371; font-weight:700; letter-spacing:.2px; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚡ Simulateur : Optimisation avec Courbes de Niveau (SGD vs. Adam)</span>
    <span class="cap09opt_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Relief Non Convexe : Minimum Local vs. Global</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Caixa de contexto didático -->

<!-- Texto atualizado na infobox do simulador -->
<div class="cap09opt_infobox">
  💡 <b>Comment lire cette carte :</b> a <b>flèche jaune</b> pointe dans la direction de <b>descente</b> (&minus;∇L), qui est le sens opposé au vecteur gradient (∇L). L'optimiseur avance dans cette direction pour réduire la perte <i>L</i>(<i>w</i><sub>1</sub>, <i>w</i><sub>2</sub>) jusqu'à atteindre les régions les plus profondes (teintes plus sombres).
</div>

    <!-- Seletores de Otimizador e Controles -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;align-items:flex-end;margin-bottom:12px;">
      <div style="flex:1;min-width:200px;">
        <div class="cap09opt_grouplabel">ALGORITHME PRINCIPAL (ligne pleine)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09opt_btnSGD" class="cap09opt_modebtn cap09opt_active">SGD (Sans Moment)</button>
          <button id="cap09opt_btnAdam" class="cap09opt_modebtn">Adam (Avec Moment)</button>
        </div>
      </div>

      <div style="flex:1;min-width:160px;">
        <div class="cap09opt_grouplabel">TAUX D'APPRENTISSAGE (η)</div>
        <select id="cap09opt_selLR" style="font-size:11px;padding:5px 8px;border-radius:8px;border:1px solid #E4DCC8;background:#FAF6EC;width:100%;font-weight:600;color:#374151;">
          <option value="0.12" selected>0.12 (Élevé)</option>
          <option value="0.05">0.05 (Idéal)</option>
          <option value="0.01">0.01 (Lent)</option>
        </select>
      </div>

      <div style="display:flex;gap:6px;">
        <button id="cap09opt_btnPasso" class="cap09opt_playbtn">▶ Pas</button>
        <button id="cap09opt_btnAuto" class="cap09opt_playbtn">⏵ Exécuter Auto</button>
        <button id="cap09opt_btnReset" class="cap09opt_navbtn">↺ Réinitialiser</button>
      </div>
    </div>

    <label class="cap09opt_checklbl">
      <input type="checkbox" id="cap09opt_chkComparar" checked style="accent-color:#2F6F9F;cursor:pointer;">
      👻 Afficher la trajectoire fantôme de l'autre optimiseur (comparaison instantanée)
    </label>

    <!-- Mapa Topográfico / Superfície da Perda -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;justify-content:center;margin-top:12px;">
      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:6px;color:#5E5A4A;">Carte de Chaleur de la Perte L(w₁, w₂) — cliquez pour choisir le départ</div>
        <canvas id="cap09opt_canvasContorno" style="width:320px;height:240px;border:1px solid #E4DCC8;border-radius:10px;cursor:crosshair;box-shadow:0 2px 4px rgba(0,0,0,0.04);"></canvas>

        <div class="cap09opt_legendrow">
          <span class="cap09opt_legenditem"><span class="cap09opt_dot" style="background:#1E8F6F;"></span> Minimum Global</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_dot" style="background:#C1443A;"></span> Minimum Local</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" style="background:#F5B301;"></span> Gradient (↓ descente)</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" id="cap09opt_legSolidLine" style="background:#C1443A;"></span> Trajectoire principale</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" id="cap09opt_legGhostLine" style="background:#2F6F9F;opacity:.5;background-image:repeating-linear-gradient(90deg,#2F6F9F 0 4px,transparent 4px 7px);"></span> Fantôme (autre optimiseur)</span>
        </div>
      </div>

      <div style="flex:1;min-width:230px;">
        <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:12px;margin-bottom:10px;">
          <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:4px;">ÉTAT DE L'OPTIMISATION :</div>
          <div class="cap09opt_mono" style="font-size:11px;color:#374151;">w₁ = <span id="cap09opt_txtW1">1.80</span>, w₂ = <span id="cap09opt_txtW2">0.20</span></div>
          <div class="cap09opt_mono" style="font-size:13px;font-weight:700;color:#C1443A;margin-top:4px;">Perte L = <span id="cap09opt_txtLoss">2.450</span></div>

          <div class="cap09opt_statgrid">
            <div>
              <div class="cap09opt_statlbl">PAS</div>
              <div class="cap09opt_mono" style="font-size:12px;color:#26241D;font-weight:700;" id="cap09opt_txtPasso">0</div>
            </div>
            <div>
              <div class="cap09opt_statlbl">|∇L| (MAGNITUDE)</div>
              <div class="cap09opt_mono" style="font-size:12px;color:#26241D;font-weight:700;" id="cap09opt_txtGrad">0.000</div>
            </div>
          </div>

          <div id="cap09opt_txtStatusRegiao" class="cap09opt_mono" style="font-size:10px;color:#1E8F6F;margin-top:8px;font-weight:600;">Statut : Point Initial</div>
        </div>

        <div id="cap09opt_descOpt" style="font-size:11px;color:#6B7280;line-height:1.55;"></div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function cap09opt_hexToRgb(h){
    var v = parseInt(h.slice(1),16);
    return { r:(v>>16)&255, g:(v>>8)&255, b:v&255 };
  }
  function cap09opt_lerp(a,b,t){ return a + (b-a)*t; }
  function cap09opt_lerpColor(c1,c2,t){
    var A=cap09opt_hexToRgb(c1), B=cap09opt_hexToRgb(c2);
    return "rgb(" + Math.round(cap09opt_lerp(A.r,B.r,t)) + "," + Math.round(cap09opt_lerp(A.g,B.g,t)) + "," + Math.round(cap09opt_lerp(A.b,B.b,t)) + ")";
  }
  var cap09opt_STOPS = [
    { l:0.5, c:"#16202B" },
    { l:1.4, c:"#2F6F9F" },
    { l:2.3, c:"#8FB8C9" },
    { l:3.0, c:"#E8DEC4" },
    { l:3.8, c:"#FBF7EE" }
  ];
  function cap09opt_lossToColor(loss){
    for(var i=0;i<cap09opt_STOPS.length-1;i++){
      var a = cap09opt_STOPS[i], b = cap09opt_STOPS[i+1];
      if(loss >= a.l && loss <= b.l){
        var t = (loss - a.l) / (b.l - a.l);
        return cap09opt_lerpColor(a.c, b.c, t);
      }
    }
    return loss < cap09opt_STOPS[0].l ? cap09opt_STOPS[0].c : cap09opt_STOPS[cap09opt_STOPS.length-1].c;
  }

  function cap09opt_init(cap09opt_root){
    if(!cap09opt_root || cap09opt_root.dataset.initOpt) return;
    cap09opt_root.dataset.initOpt = "1";

    var cap09opt_BETA1 = 0.8, cap09opt_BETA2 = 0.99;

    var cap09opt_algoritmo = "sgd";
    var cap09opt_posInicial = { w1: 1.8, w2: 0.2 };
    var cap09opt_posW = { w1: 1.8, w2: 0.2 };
    var cap09opt_trajetoria = [{ w1: 1.8, w2: 0.2 }];
    var cap09opt_trajFantasma = [];
    var cap09opt_passoAtual = 0;

    var cap09opt_m = { w1: 0, w2: 0 };
    var cap09opt_v = { w1: 0, w2: 0 };
    var cap09opt_tStep = 0;
    var cap09opt_autoInterval = null;

    var cap09opt_canvas = cap09opt_root.querySelector('#cap09opt_canvasContorno');
    var cap09opt_ctx    = cap09opt_canvas.getContext('2d');

    function cap09opt_prepararCanvas(cap09opt_cvs, cap09opt_c, cap09opt_cssW, cap09opt_cssH){
      var cap09opt_dpr = window.devicePixelRatio || 1;
      cap09opt_cvs.width = cap09opt_cssW * cap09opt_dpr;
      cap09opt_cvs.height = cap09opt_cssH * cap09opt_dpr;
      cap09opt_c.scale(cap09opt_dpr, cap09opt_dpr);
    }
    cap09opt_prepararCanvas(cap09opt_canvas, cap09opt_ctx, 320, 240);

    var cap09opt_btnSGD    = cap09opt_root.querySelector('#cap09opt_btnSGD');
    var cap09opt_btnAdam   = cap09opt_root.querySelector('#cap09opt_btnAdam');
    var cap09opt_selLR     = cap09opt_root.querySelector('#cap09opt_selLR');
    var cap09opt_chkComp   = cap09opt_root.querySelector('#cap09opt_chkComparar');

    var cap09opt_btnPasso  = cap09opt_root.querySelector('#cap09opt_btnPasso');
    var cap09opt_btnAuto   = cap09opt_root.querySelector('#cap09opt_btnAuto');
    var cap09opt_btnReset  = cap09opt_root.querySelector('#cap09opt_btnReset');

    var cap09opt_txtW1     = cap09opt_root.querySelector('#cap09opt_txtW1');
    var cap09opt_txtW2     = cap09opt_root.querySelector('#cap09opt_txtW2');
    var cap09opt_txtLoss   = cap09opt_root.querySelector('#cap09opt_txtLoss');
    var cap09opt_txtPasso  = cap09opt_root.querySelector('#cap09opt_txtPasso');
    var cap09opt_txtGrad   = cap09opt_root.querySelector('#cap09opt_txtGrad');
    var cap09opt_txtStatus = cap09opt_root.querySelector('#cap09opt_txtStatusRegiao');
    var cap09opt_descOpt   = cap09opt_root.querySelector('#cap09opt_descOpt');
    var cap09opt_legGhost  = cap09opt_root.querySelector('#cap09opt_legGhostLine');
    var cap09opt_legSolid  = cap09opt_root.querySelector('#cap09opt_legSolidLine');

    function cap09opt_wToPx(w1, w2){
      return { x: 160 + w1 * 55, y: 120 - w2 * 45 };
    }
    function cap09opt_pxToW(x, y){
      return { w1: (x - 160) / 55.0, w2: (120 - y) / 45.0 };
    }

    function cap09opt_calcLoss(w1, w2){
      var gGlobal = 3.0 * Math.exp(-((w1 + 1.5)*(w1 + 1.5)*0.8 + w2*w2*1.5));
      var gLocal  = 1.6 * Math.exp(-((w1 - 1.5)*(w1 - 1.5)*1.2 + w2*w2*1.5));
      var parabola = 0.18 * (w1*w1 + w2*w2);
      return 3.5 - gGlobal - gLocal + parabola;
    }

    function cap09opt_calcGrad(w1, w2){
      var eps = 0.001;
      var l0 = cap09opt_calcLoss(w1, w2);
      var dw1 = (cap09opt_calcLoss(w1 + eps, w2) - l0) / eps;
      var dw2 = (cap09opt_calcLoss(w1, w2 + eps) - l0) / eps;
      return { g1: dw1, g2: dw2 };
    }

    function cap09opt_passoGenerico(algo, estado, lr){
      var grad = cap09opt_calcGrad(estado.w1, estado.w2);
      if(algo === "sgd"){
        estado.w1 -= lr * grad.g1;
        estado.w2 -= lr * grad.g2;
      } else {
        estado.t = (estado.t||0) + 1;
        estado.m1 = cap09opt_BETA1 * (estado.m1||0) + (1-cap09opt_BETA1) * grad.g1;
        estado.m2 = cap09opt_BETA1 * (estado.m2||0) + (1-cap09opt_BETA1) * grad.g2;
        estado.v1 = cap09opt_BETA2 * (estado.v1||0) + (1-cap09opt_BETA2) * (grad.g1*grad.g1);
        estado.v2 = cap09opt_BETA2 * (estado.v2||0) + (1-cap09opt_BETA2) * (grad.g2*grad.g2);
        var mHat1 = estado.m1 / (1 - Math.pow(cap09opt_BETA1, estado.t));
        var mHat2 = estado.m2 / (1 - Math.pow(cap09opt_BETA1, estado.t));
        var vHat1 = estado.v1 / (1 - Math.pow(cap09opt_BETA2, estado.t));
        var vHat2 = estado.v2 / (1 - Math.pow(cap09opt_BETA2, estado.t));
        estado.w1 -= (lr / (Math.sqrt(vHat1) + 1e-4)) * mHat1 * 1.5;
        estado.w2 -= (lr / (Math.sqrt(vHat2) + 1e-4)) * mHat2 * 1.5;
      }
      return grad;
    }

    function cap09opt_computarFantasma(){
      var outroAlgo = cap09opt_algoritmo === "sgd" ? "adam" : "sgd";
      var lr = parseFloat(cap09opt_selLR.value);
      var estado = { w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 };
      var caminho = [{ w1: estado.w1, w2: estado.w2 }];
      for(var i=0; i<150; i++){
        var antesW1 = estado.w1, antesW2 = estado.w2;
        cap09opt_passoGenerico(outroAlgo, estado, lr);
        caminho.push({ w1: estado.w1, w2: estado.w2 });
        var delta = Math.hypot(estado.w1-antesW1, estado.w2-antesW2);
        if(delta < 0.0008 && i > 6) break;
      }
      return caminho;
    }

    function cap09opt_desenharSeta(x, y, ang, comprimento, cor){
      var x2 = x + Math.cos(ang) * comprimento;
      var y2 = y + Math.sin(ang) * comprimento;
      cap09opt_ctx.strokeStyle = cor; cap09opt_ctx.fillStyle = cor; cap09opt_ctx.lineWidth = 2;
      cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(x,y); cap09opt_ctx.lineTo(x2,y2); cap09opt_ctx.stroke();
      var cabeca = 6, angSeta = Math.atan2(y2-y, x2-x);
      cap09opt_ctx.beginPath();
      cap09opt_ctx.moveTo(x2, y2);
      cap09opt_ctx.lineTo(x2 - cabeca*Math.cos(angSeta-0.45), y2 - cabeca*Math.sin(angSeta-0.45));
      cap09opt_ctx.lineTo(x2 - cabeca*Math.cos(angSeta+0.45), y2 - cabeca*Math.sin(angSeta+0.45));
      cap09opt_ctx.closePath(); cap09opt_ctx.fill();
    }

    function cap09opt_desenharMapa(){
      var W = 320, H = 240, gridRes = 5;
      cap09opt_ctx.clearRect(0,0,W,H);

      for(var py=0; py<H; py+=gridRes){
        for(var px=0; px<W; px+=gridRes){
          var wC = cap09opt_pxToW(px + gridRes/2, py + gridRes/2);
          var lC = cap09opt_calcLoss(wC.w1, wC.w2);
          cap09opt_ctx.fillStyle = cap09opt_lossToColor(lC);
          cap09opt_ctx.fillRect(px, py, gridRes+0.5, gridRes+0.5);
        }
      }

      var niveisLoss = [0.8, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2, 3.8];
      for(var nIdx=0; nIdx<niveisLoss.length; nIdx++){
        var alvoL = niveisLoss[nIdx];
        cap09opt_ctx.strokeStyle = "rgba(20, 24, 30, 0.18)";
        cap09opt_ctx.lineWidth = 1;
        for(var qy=0; qy<H; qy+=gridRes){
          for(var qx=0; qx<W; qx+=gridRes){
            var wA = cap09opt_pxToW(qx, qy);
            var lA = cap09opt_calcLoss(wA.w1, wA.w2);
            var wB = cap09opt_pxToW(qx + gridRes, qy);
            var lB = cap09opt_calcLoss(wB.w1, wB.w2);
            var wCc = cap09opt_pxToW(qx, qy + gridRes);
            var lCc = cap09opt_calcLoss(wCc.w1, wCc.w2);
            if((lA <= alvoL && lB >= alvoL) || (lA >= alvoL && lB <= alvoL)){
              cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(qx, qy); cap09opt_ctx.lineTo(qx + gridRes, qy); cap09opt_ctx.stroke();
            }
            if((lA <= alvoL && lCc >= alvoL) || (lA >= alvoL && lCc <= alvoL)){
              cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(qx, qy); cap09opt_ctx.lineTo(qx, qy + gridRes); cap09opt_ctx.stroke();
            }
          }
        }
      }

      cap09opt_ctx.strokeStyle = "rgba(255,255,255,0.35)";
      cap09opt_ctx.lineWidth = 1;
      cap09opt_ctx.beginPath();
      cap09opt_ctx.moveTo(0, 120); cap09opt_ctx.lineTo(W, 120);
      cap09opt_ctx.moveTo(160, 0); cap09opt_ctx.lineTo(160, H);
      cap09opt_ctx.stroke();

      cap09opt_ctx.font = "600 9px 'JetBrains Mono', monospace";
      cap09opt_ctx.fillStyle = "rgba(38,36,29,0.55)";
      cap09opt_ctx.textAlign = "center";
      for(var wv=-2; wv<=2; wv++){
        if(wv===0) continue;
        var px1 = cap09opt_wToPx(wv, 0);
        cap09opt_ctx.fillText(wv.toString(), px1.x, 132);
        var py1 = cap09opt_wToPx(0, wv*0.9);
        cap09opt_ctx.fillText(wv.toString(), 172, py1.y+3);
      }
      cap09opt_ctx.font = "700 10px 'Inter', sans-serif";
      cap09opt_ctx.fillText("w₁ →", 300, 134);
      cap09opt_ctx.save(); cap09opt_ctx.translate(150, 14); cap09opt_ctx.fillText("w₂ ↑", 0, 0); cap09opt_ctx.restore();

      var posGlobal = cap09opt_wToPx(-1.4, 0);
      var posLocal  = cap09opt_wToPx(1.3, 0);

      [ [posGlobal, "#1E8F6F", "Mínimo Global ★"], [posLocal, "#C1443A", "Mínimo Local ⚠️"] ].forEach(function(item){
        var p = item[0];
        cap09opt_ctx.fillStyle = "rgba(255,255,255,0.65)";
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(p.x, p.y, 8, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.fillStyle = item[1];
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(p.x, p.y, 4.5, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.strokeStyle = "#FFF"; cap09opt_ctx.lineWidth = 1.5; cap09opt_ctx.stroke();
        cap09opt_ctx.font = "700 9px 'JetBrains Mono', monospace";
        cap09opt_ctx.fillStyle = "#26241D";
        cap09opt_ctx.textAlign = "center";
        cap09opt_ctx.fillText(item[2], p.x, p.y - 12);
      });

      if(cap09opt_chkComp.checked && cap09opt_trajFantasma.length > 1){
        var corFantasma = cap09opt_algoritmo === "sgd" ? "#2F6F9F" : "#C1443A";
        cap09opt_ctx.save();
        cap09opt_ctx.setLineDash([5,4]);
        cap09opt_ctx.strokeStyle = corFantasma;
        cap09opt_ctx.globalAlpha = 0.55;
        cap09opt_ctx.lineWidth = 2;
        cap09opt_ctx.beginPath();
        for(var fi=0; fi<cap09opt_trajFantasma.length; fi++){
          var fp = cap09opt_wToPx(cap09opt_trajFantasma[fi].w1, cap09opt_trajFantasma[fi].w2);
          if(fi===0) cap09opt_ctx.moveTo(fp.x, fp.y); else cap09opt_ctx.lineTo(fp.x, fp.y);
        }
        cap09opt_ctx.stroke();
        cap09opt_ctx.restore();
        var fEnd = cap09opt_wToPx(cap09opt_trajFantasma[cap09opt_trajFantasma.length-1].w1, cap09opt_trajFantasma[cap09opt_trajFantasma.length-1].w2);
        cap09opt_ctx.fillStyle = corFantasma; cap09opt_ctx.globalAlpha = 0.7;
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(fEnd.x, fEnd.y, 4, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.globalAlpha = 1;
      }

      if(cap09opt_trajetoria.length > 1){
        var corPrincipal = cap09opt_algoritmo === "sgd" ? "#C1443A" : "#2F6F9F";
        cap09opt_ctx.strokeStyle = corPrincipal;
        cap09opt_ctx.lineWidth = 2.5;
        cap09opt_ctx.beginPath();
        for(var i=0; i<cap09opt_trajetoria.length; i++){
          var p = cap09opt_wToPx(cap09opt_trajetoria[i].w1, cap09opt_trajetoria[i].w2);
          if(i === 0) cap09opt_ctx.moveTo(p.x, p.y); else cap09opt_ctx.lineTo(p.x, p.y);
        }
        cap09opt_ctx.stroke();
        for(var j=0; j<cap09opt_trajetoria.length; j++){
          var pj = cap09opt_wToPx(cap09opt_trajetoria[j].w1, cap09opt_trajetoria[j].w2);
          cap09opt_ctx.fillStyle = corPrincipal;
          cap09opt_ctx.beginPath(); cap09opt_ctx.arc(pj.x, pj.y, 2, 0, 2*Math.PI); cap09opt_ctx.fill();
        }
      }

      var pInicio = cap09opt_wToPx(cap09opt_posInicial.w1, cap09opt_posInicial.w2);
      cap09opt_ctx.fillStyle = "#FBF7EE";
      cap09opt_ctx.beginPath(); cap09opt_ctx.arc(pInicio.x, pInicio.y, 7, 0, 2*Math.PI); cap09opt_ctx.fill();
      cap09opt_ctx.strokeStyle = "#26241D"; cap09opt_ctx.lineWidth = 1.5; cap09opt_ctx.stroke();
      cap09opt_ctx.fillStyle = "#26241D"; cap09opt_ctx.font = "700 8px 'JetBrains Mono', monospace";
      cap09opt_ctx.textAlign = "center"; cap09opt_ctx.textBaseline = "middle";
      cap09opt_ctx.fillText("S", pInicio.x, pInicio.y);

      // Ponto atual + seta do gradiente DESCENDENTE (CORRIGIDO)
      var ptAtual = cap09opt_wToPx(cap09opt_posW.w1, cap09opt_posW.w2);
      var gradAtual = cap09opt_calcGrad(cap09opt_posW.w1, cap09opt_posW.w2);
      
      // Mapeia 1 passo na direção de DESCIDA ( - gradiente )
      var ptDescida = cap09opt_wToPx(cap09opt_posW.w1 - gradAtual.g1 * 0.2, cap09opt_posW.w2 - gradAtual.g2 * 0.2);
      var angTela = Math.atan2(ptDescida.y - ptAtual.y, ptDescida.x - ptAtual.x);
      var magGrad = Math.hypot(gradAtual.g1, gradAtual.g2);

      if(magGrad > 0.01){
        cap09opt_desenharSeta(ptAtual.x, ptAtual.y, angTela, 22, "#F5B301");
      }

      cap09opt_ctx.fillStyle = "#26241D";
      cap09opt_ctx.beginPath(); cap09opt_ctx.arc(ptAtual.x, ptAtual.y, 6, 0, 2*Math.PI); cap09opt_ctx.fill();
      cap09opt_ctx.strokeStyle = "#FFFFFF"; cap09opt_ctx.lineWidth = 2; cap09opt_ctx.stroke();

      return magGrad;
    }

    function cap09opt_darPasso(){
      var lr = parseFloat(cap09opt_selLR.value);
      var estadoTmp = { w1: cap09opt_posW.w1, w2: cap09opt_posW.w2, m1: cap09opt_m.w1, m2: cap09opt_m.w2, v1: cap09opt_v.w1, v2: cap09opt_v.w2, t: cap09opt_tStep };
      cap09opt_passoGenerico(cap09opt_algoritmo, estadoTmp, lr);
      cap09opt_posW.w1 = estadoTmp.w1; cap09opt_posW.w2 = estadoTmp.w2;
      cap09opt_m.w1 = estadoTmp.m1||0; cap09opt_m.w2 = estadoTmp.m2||0;
      cap09opt_v.w1 = estadoTmp.v1||0; cap09opt_v.w2 = estadoTmp.v2||0;
      cap09opt_tStep = estadoTmp.t||0;

      cap09opt_trajetoria.push({ w1: cap09opt_posW.w1, w2: cap09opt_posW.w2 });
      cap09opt_passoAtual++;
      cap09opt_atualizar();
    }

    function cap09opt_atualizar(){
      var magGrad = cap09opt_desenharMapa();
      var loss = cap09opt_calcLoss(cap09opt_posW.w1, cap09opt_posW.w2);
      cap09opt_txtW1.textContent   = cap09opt_posW.w1.toFixed(2);
      cap09opt_txtW2.textContent   = cap09opt_posW.w2.toFixed(2);
      cap09opt_txtLoss.textContent = loss.toFixed(3);
      cap09opt_txtPasso.textContent = cap09opt_passoAtual;
      cap09opt_txtGrad.textContent = magGrad.toFixed(3);

      var lrVal = parseFloat(cap09opt_selLR.value);
      var conseguiuEscapar = cap09opt_posW.w1 < -0.5;
      var estaPresoLocal = cap09opt_posW.w1 > 0.5;
      var corPrincipal = cap09opt_algoritmo === "sgd" ? "#C1443A" : "#2F6F9F";
      cap09opt_legSolid.style.background = corPrincipal;
      var corFantasma = cap09opt_algoritmo === "sgd" ? "#2F6F9F" : "#C1443A";
      cap09opt_legGhost.style.backgroundImage = "repeating-linear-gradient(90deg,"+corFantasma+" 0 4px,transparent 4px 7px)";

      if(estaPresoLocal){
        cap09opt_txtStatus.textContent = "Région : Bloqué au Minimum Local ⚠️";
        cap09opt_txtStatus.style.color = "#C1443A";
      } else if(conseguiuEscapar){
        cap09opt_txtStatus.textContent = "Région : Convergé vers le Minimum Global ★";
        cap09opt_txtStatus.style.color = "#1E8F6F";
      } else {
        cap09opt_txtStatus.textContent = "Région : Montée / Franchissement de Barrière";
        cap09opt_txtStatus.style.color = "#2F6F9F";
      }

      if(cap09opt_algoritmo === "sgd"){
        cap09opt_descOpt.innerHTML = "<b>SGD (Sem Momento):</b> a cada passo, o SGD olha apenas para o gradiente <i>local e instantâneo</i> — sem memória do que veio antes. Por isso, ao partir do lado direito, ele fica <b>preso no Mínimo Local</b>: não tem energia acumulada para subir o aclive até a barreira central. Compare com a linha tracejada azul (Adam) ao lado.";
      } else {
        if(conseguiuEscapar){
          cap09opt_descOpt.innerHTML = "<b>Adam (Sucesso):</b> o Adam acumula <i>momento</i> (uma média móvel dos gradientes recentes) e ajusta a taxa de cada peso adaptativamente. Com η=" + lrVal + ", esse impulso acumulado foi suficiente para vencer a barreira e alcançar o <b>Mínimo Global ★</b>. Note como a linha tracejada vermelha (SGD) fica presa antes disso.";
        } else if(estaPresoLocal && cap09opt_trajetoria.length > 8){
          cap09opt_descOpt.innerHTML = "<b>Adam (Retido no Mínimo Local):</b> mesmo acumulando momento, com η=" + lrVal + " o impulso não foi suficiente para transpor a elevação. <i>Isso mostra que nem mesmo o Adam garante escapar de poços profundos sem ajuste fino da taxa de aprendizado ou de uma inicialização melhor.</i>";
        } else {
          cap09opt_descOpt.innerHTML = "<b>Adam (Em movimento):</b> acumulando momento e ajustando o tamanho do passo adaptativamente conforme percorre o relevo...";
        }
      }
    }

    function cap09opt_resetar(){
      if(cap09opt_autoInterval) { clearInterval(cap09opt_autoInterval); cap09opt_autoInterval = null; cap09opt_btnAuto.textContent = "⏵ Exécuter Auto"; }
      cap09opt_posW = { w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 };
      cap09opt_trajetoria = [{ w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 }];
      cap09opt_m = { w1: 0, w2: 0 }; cap09opt_v = { w1: 0, w2: 0 }; cap09opt_tStep = 0;
      cap09opt_passoAtual = 0;
      cap09opt_trajFantasma = cap09opt_computarFantasma();
      cap09opt_atualizar();
    }

    cap09opt_canvas.addEventListener('click', function(evt){
      var rect = cap09opt_canvas.getBoundingClientRect();
      var scaleX = 320 / rect.width, scaleY = 240 / rect.height;
      var clickX = (evt.clientX - rect.left) * scaleX;
      var clickY = (evt.clientY - rect.top) * scaleY;
      var ptW = cap09opt_pxToW(clickX, clickY);
      cap09opt_posInicial = { w1: ptW.w1, w2: ptW.w2 };
      cap09opt_resetar();
    });

    cap09opt_btnSGD.addEventListener('click', function(){
      cap09opt_btnSGD.classList.add('cap09opt_active'); cap09opt_btnAdam.classList.remove('cap09opt_active');
      cap09opt_algoritmo = "sgd"; cap09opt_resetar();
    });
    cap09opt_btnAdam.addEventListener('click', function(){
      cap09opt_btnAdam.classList.add('cap09opt_active'); cap09opt_btnSGD.classList.remove('cap09opt_active');
      cap09opt_algoritmo = "adam"; cap09opt_resetar();
    });

    cap09opt_btnPasso.addEventListener('click', cap09opt_darPasso);
    cap09opt_btnReset.addEventListener('click', cap09opt_resetar);
    cap09opt_chkComp.addEventListener('change', cap09opt_atualizar);

    cap09opt_btnAuto.addEventListener('click', function(){
      if(cap09opt_autoInterval){
        clearInterval(cap09opt_autoInterval); cap09opt_autoInterval = null; cap09opt_btnAuto.textContent = "⏵ Exécuter Auto";
      } else {
        cap09opt_btnAuto.textContent = "⏸ Pause";
        cap09opt_autoInterval = setInterval(cap09opt_darPasso, 120);
      }
    });

    cap09opt_selLR.addEventListener('change', cap09opt_resetar);

    cap09opt_resetar();
  }

  function cap09opt_tryInit(){
    var cap09opt_root = document.getElementById('sim-09-opt');
    if(cap09opt_root) cap09opt_init(cap09opt_root); else setTimeout(cap09opt_tryInit, 200);
  }
  cap09opt_tryInit();
})();
</script>
''')

**Figure 9.9:** Simulador interactif des algorithmes d


<figure id="fig-09-sim-09-opt">
  <img src="imagens/fig-09-sim-09-opt.png" alt=" Simulador interactif des algorithmes d'optimisation : comparez la trajectoire du SGD et de l'Adam sur une surface de perte non convexe avec carte de chaleur et courbes de niveau. La ligne continue montre l'optimiseur sélectionné avançant pas à pas ; la ligne pointillée montre, pour une comparaison instantanée, le chemin complet que l'autre optimiseur parcourrait depuis le même point initial. Observez comment le SGD reste bloqué au Minimum Local à droite, tandis que l'Adam peut ou non franchir la barrière centrale selon l'impulsion accumulée et le taux d'apprentissage. Cliquez sur n'importe quel point de la carte pour réinitialiser le point initial des poids. " style="max-width:80%" />
  <figcaption><strong>Figure 9.9:</strong>  Simulador interactif des algorithmes d'optimisation : comparez la trajectoire du SGD et de l'Adam sur une surface de perte non convexe avec carte de chaleur et courbes de niveau. La ligne continue montre l'optimiseur sélectionné avançant pas à pas ; la ligne pointillée montre, pour une comparaison instantanée, le chemin complet que l'autre optimiseur parcourrait depuis le même point initial. Observez comment le SGD reste bloqué au Minimum Local à droite, tandis que l'Adam peut ou non franchir la barrière centrale selon l'impulsion accumulée et le taux d'apprentissage. Cliquez sur n'importe quel point de la carte pour réinitialiser le point initial des poids. </figcaption>
</figure>

### 9.4.8 Architecture d'une CNN

Une CNN pour la classification d'images combine les couches présentées dans les sections précédentes. Lors du **passage avant** (*forward pass*), l'image traverse successivement les couches convolutives, les fonctions d'activation, les opérations de *pooling*, l'étape de **Flatten**, les couches entièrement connectées et, enfin, la couche **Softmax**, qui produit les probabilités des classes. Pendant l'entraînement, cette prédiction est comparée à l'étiquette correcte pour calculer la fonction de perte, effectuer la rétropropagation et mettre à jour les paramètres via un algorithme d'optimisation.

La [Figure 9.10](#fig-09-cnn-arquitetura) présente ce flux de traitement et d'entraînement.

<figure id="fig-09-cnn-arquitetura" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-09-cnn-arquitetura.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figure 9.10:</strong> Architecture simplifiée d'une CNN pour la classification d'images, mettant en évidence le *forward pass* et les étapes d'entraînement via la fonction de perte, la rétropropagation et l'algorithme d'optimisation.</figcaption>
</figure>

Après le dernier bloc convolutif, l'opération **Flatten** réorganise les cartes de caractéristiques en un vecteur unidimensionnel, qui alimente les **couches entièrement connectées** (*fully connected layers*), responsables de la combinaison des caractéristiques extraites pour produire les scores (*logits*) de chaque classe. La couche **Softmax** convertit ces scores en une distribution de probabilités, utilisée à la fois pour la classification et pour le calcul de la fonction de perte pendant l'entraînement.

La [Figure 9.11](#fig-09-sim-09-arquitetura) présente une version interactive de cette architecture, permettant d'exécuter des étapes successives d'entraînement et d'observer la réduction de la perte, la rétropropagation des gradients et la mise à jour des filtres du réseau.

In [10]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-arquitetura" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-arquitetura .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-arquitetura .cn-grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-arquitetura .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-arquitetura .cn-navbtn2 {
      padding:0 10px; height:26px; border-radius:8px; border:1px solid #E4DCC8; background:#FAFAF7;
      color:#26241D; font-size:10.5px; font-weight:600; cursor:pointer; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-navbtn2:hover { background:#F1EAD7; }
    #sim-09-arquitetura .cn-playbtn {
      padding:0 12px; height:26px; border-radius:8px; border:1px solid #2F6F9F; background:#EAF2FA;
      color:#2F6F9F; font-size:10.5px; font-weight:700; cursor:pointer; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-playbtn:hover { background:#DCEEFB; }
    #sim-09-arquitetura .cn-kcell {
      width:30px;height:30px;border-radius:5px;border:1px solid #e5e7eb;display:flex;
      align-items:center;justify-content:center;font-size:8.5px;font-weight:700;
    }
    #sim-09-arquitetura .cn-ciclo { font-size:10.5px; transition: color .3s ease, background .3s ease; padding:3px 6px; border-radius:6px; }
    #sim-09-arquitetura .cn-ciclo.pulso { background:#FCE8E6; color:#C1443A; font-weight:700; }
    #sim-09-arquitetura .cn-graflabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:4px; text-align:center; letter-spacing:.3px; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulateur : Architecture complète d'un CNN</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Entrée (12×12) → Conv → Pool → FC → Softmax</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletor de Imagem de Entrada para manter a mesma didática do simulador anterior -->
    <div style="margin-bottom:12px;max-width:320px;">
      <div class="cn-grouplabel">IMAGE D'ENTRÉE DU PIPELINE (12×12)</div>
      <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
        <button id="cap09arch_btnCasa" class="cn-modebtn active">🏠 Maison</button>
        <button id="cap09arch_btnFeliz" class="cn-modebtn">😊 Heureux</button>
        <button id="cap09arch_btnTriste" class="cn-modebtn">😢 Triste</button>
      </div>
    </div>

    <!-- Fluxo das Camadas -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px;margin-bottom:14px;overflow-x:auto;">
      <div id="cap09arch_flow" style="display:flex;align-items:center;gap:2px;padding:4px 2px;min-width:680px;"></div>
    </div>

    <!-- Detalhe da Visualização da Camada Selecionada -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:20px;flex-wrap:wrap;align-items:center;margin-bottom:14px;">
      <div style="flex:0 0 auto;text-align:center;">
        <canvas id="cap09arch_canvasVis" width="280" height="200" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,0.03);"></canvas>
        <div id="cap09arch_barsWrap" style="display:none;align-items:flex-end;gap:24px;height:140px;margin-top:10px;justify-content:center;padding:0 10px;"></div>
        <div id="cap09arch_kernelsPanel" style="display:none;margin-top:10px;"></div>
        <div id="cap09arch_legenda" style="font-size:10.5px;color:#8A8371;margin-top:8px;max-width:280px;line-height:1.4;"></div>
      </div>
      <div id="cap09arch_desc" style="flex:1;min-width:240px;font-size:12px;line-height:1.6;color:#374151;"></div>
    </div>

    <!-- Painel de Treinamento: Forward Pass + Retropropagação de verdade -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;">
      <div style="font-size:11.5px;font-weight:600;color:#4b5563;margin-bottom:10px;">🎯 Entraînement (Propagation avant + Rétropropagation)</div>

      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;">
        <div style="min-width:230px;">
          <div id="cap09arch_cicloForward" class="cn-mono cn-ciclo" style="color:#374151;">Entrée ▸ Conv+ReLU ▸ Pool ▸ Flatten ▸ FC ▸ Softmax ▸ Prédiction</div>
          <div id="cap09arch_cicloBackward" class="cn-mono cn-ciclo" style="color:#8A8371;margin-top:3px;">Perte ◂ Optimiseur ◂ Rétropropagation ◂ (à chaque étape)</div>
        </div>
        <div>
          <div class="cn-graflabel">PERTE (LOSS)</div>
          <canvas id="cap09arch_canvasLoss" width="260" height="110" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div>
          <div class="cn-graflabel">PRÉCISION</div>
          <canvas id="cap09arch_canvasAcc" width="260" height="110" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div style="min-width:170px;">
          <div id="cap09arch_lossTxt" class="cn-mono" style="font-size:11px;color:#7EE7C6;background:#1B2430;padding:6px 10px;border-radius:6px;">Perte : —</div>
          <div id="cap09arch_accTxt" class="cn-mono" style="font-size:11px;color:#FFD98E;background:#1B2430;padding:6px 10px;border-radius:6px;margin-top:5px;">Précision : —</div>
          <div id="cap09arch_stepTxt" class="cn-mono" style="font-size:10.5px;color:#8A8371;margin-top:5px;">Étape d'entraînement : 0</div>
          <div style="display:flex;gap:5px;margin-top:8px;flex-wrap:wrap;">
            <button id="cap09arch_btnPassoUnico" class="cn-navbtn2">Étape unique</button>
            <button id="cap09arch_btnTreinar" class="cn-playbtn">▶ Entraîner</button>
            <button id="cap09arch_btnReiniciarPesos" class="cn-navbtn2">↺ Nouveaux Poids</button>
          </div>
        </div>
      </div>

      <div id="cap09arch_notaTreino" style="font-size:10.5px;color:#8A8371;margin-top:10px;padding-top:8px;border-top:1px dashed #E9E3D3;line-height:1.5;">
        Les <i>noyaux</i> et poids commencent <b>aléatoires</b> (ce ne sont plus les filtres fixes du simulateur précédent). À chaque étape, le réseau effectue la <i>propagation avant</i> sur les 3 images, calcule la perte (<i>cross-entropie</i>) e a <b>précision</b> (combien des 3 images sont correctement classées), rétropropage l'erreur et ajuste tous les poids (y compris les <i>noyaux</i> de la convolution) via la descente de gradient. ⚠️ Comme le « jeu d'entraînement » ne compte que 3 exemples, cela démontre le <b>mécanisme</b> de l'entraînement (perte en baisse, précision en hausse, poids changeant) — pas la capacité à généraliser à de nouvelles images, ce qui exigerait beaucoup plus de données.
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function clamp01(v){ return Math.max(0, Math.min(1, v)); }
  // Transforma uma ativação ReLU (não-negativa, sem limite superior) em algo
  // sempre entre 0 e 1 apenas para fins de exibição em cor — os valores brutos
  // usados no forward/backward NÃO passam por essa saturação.
  function saturar(v){ return 1 - Math.exp(-Math.max(0, v)); }

  // Gerador pseudoaleatório determinístico (mesma semente = mesmo resultado
  // inicial), para que o comportamento do simulador seja reprodutível.
  function criarRng(seed){
    return function(){
      seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
      var t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
      t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
      return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    };
  }

  // Mesmas matrizes 12x12 em formato ASCII usadas no simulador da camada convolucional
  var IMAGENS_ASCII = {
    casa: [
      "............",
      "....XXXX....",
      "...XXXXXX...",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXX..XXXX.",
      ".XXXX..XXXX.",
      ".XXXX..XXXX.",
      "............"
    ],
    feliz: [
      "............",
      "....XXXX....",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXX.XX.XXX.",
      ".XXXXXXXXXX.",
      ".XX......XX.",
      ".XXX....XXX.",
      ".XXXXXXXXXX.",
      "..XXXXXXXX..",
      "............"
    ],
    triste: [
      "............",
      "....XXXX....",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXX.XX.XXX.",
      ".XXXXXXXXXX.",
      ".XXX....XXX.",
      ".XX......XX.",
      ".XXXXXXXXXX.",
      "..XXXXXXXX..",
      "............"
    ]
  };
  var ORDEM_CLASSES = ["casa", "feliz", "triste"];
  var ROTULOS = { casa: "🏠 Casa", feliz: "😊 Feliz", triste: "😢 Triste" };
  var ONE_HOTS = { casa: [1,0,0], feliz: [0,1,0], triste: [0,0,1] };

  function converterLinhas(linhas){
    return linhas.map(function(l){
      var res = [];
      for (var i = 0; i < l.length; i++) res.push(l[i] === 'X' ? 1.0 : 0.0);
      return res;
    });
  }
  var IMAGENS = {};
  ORDEM_CLASSES.forEach(function(id){ IMAGENS[id] = converterLinhas(IMAGENS_ASCII[id]); });

  function corMapa(v){
    var r = Math.round(255 - v*(255-47));
    var g = Math.round(255 - v*(255-111));
    var b = Math.round(255 - v*(255-159));
    return 'rgb('+r+','+g+','+b+')';
  }

  // ---------------------------------------------------------------------
  // Rede: Conv1 (4 filtros 3×3) → ReLU → MaxPool 2×2 → Flatten(100) →
  // Densa1 (16, ReLU) → Densa2/saída (3 logits) → Softmax.
  // Implementação manual de forward e backward (sem bibliotecas), pensada
  // para ficar pequena o bastante para caber num simulador didático.
  // ---------------------------------------------------------------------

  function inicializarParametros(rng){
    var K = [], bConv = [];
    for (var f = 0; f < 4; f++){
      var k = [];
      for (var r = 0; r < 3; r++){
        var row = [];
        for (var c = 0; c < 3; c++) row.push((rng() - 0.5) * 1.0);
        k.push(row);
      }
      K.push(k); bConv.push(0);
    }
    var W1 = [], b1 = [];
    for (var i = 0; i < 16; i++){
      var row1 = [];
      for (var j = 0; j < 100; j++) row1.push((rng() - 0.5) * 0.2);
      W1.push(row1); b1.push(0);
    }
    var W2 = [], b2 = [];
    for (var i2 = 0; i2 < 3; i2++){
      var row2 = [];
      for (var j2 = 0; j2 < 16; j2++) row2.push((rng() - 0.5) * 0.3);
      W2.push(row2); b2.push(0);
    }
    return { K: K, bConv: bConv, W1: W1, b1: b1, W2: W2, b2: b2 };
  }

  function relu(x){ return Math.max(0, x); }
  function reluDeriv(x){ return x > 0 ? 1 : 0; }
  function softmax(logits){
    var m = Math.max.apply(null, logits);
    var exps = logits.map(function(v){ return Math.exp(v - m); });
    var soma = exps.reduce(function(a,b){ return a+b; }, 0);
    return exps.map(function(v){ return v / soma; });
  }

  function forwardPassRede(p, img){
    var Z1 = [], A1 = [];
    for (var f = 0; f < 4; f++) {
      var zf = [], af = [];
      for (var r = 0; r < 10; r++) {
        var zr = [], ar = [];
        for (var c = 0; c < 10; c++) {
          var s = p.bConv[f];
          for (var kr = 0; kr < 3; kr++)
            for (var kc = 0; kc < 3; kc++)
              s += img[r+kr][c+kc] * p.K[f][kr][kc];
          zr.push(s); ar.push(relu(s));
        }
        zf.push(zr); af.push(ar);
      }
      Z1.push(zf); A1.push(af);
    }

    var P1 = [], argMax = [];
    for (var f2 = 0; f2 < 4; f2++) {
      var pf = [], amf = [];
      for (var r2 = 0; r2 < 5; r2++) {
        var pr = [], amr = [];
        for (var c2 = 0; c2 < 5; c2++) {
          var v00=A1[f2][2*r2][2*c2], v01=A1[f2][2*r2][2*c2+1];
          var v10=A1[f2][2*r2+1][2*c2], v11=A1[f2][2*r2+1][2*c2+1];
          var mv=v00, dr=0, dc=0;
          if (v01>mv){mv=v01;dr=0;dc=1;}
          if (v10>mv){mv=v10;dr=1;dc=0;}
          if (v11>mv){mv=v11;dr=1;dc=1;}
          pr.push(mv); amr.push({dr:dr,dc:dc});
        }
        pf.push(pr); amf.push(amr);
      }
      P1.push(pf); argMax.push(amf);
    }

    var flat = [];
    for (var f3 = 0; f3 < 4; f3++)
      for (var r3 = 0; r3 < 5; r3++)
        for (var c3 = 0; c3 < 5; c3++)
          flat.push(P1[f3][r3][c3]);

    var Z2 = [], A2 = [];
    for (var i = 0; i < 16; i++) {
      var s2 = p.b1[i];
      for (var j = 0; j < 100; j++) s2 += p.W1[i][j] * flat[j];
      Z2.push(s2); A2.push(relu(s2));
    }

    var logits = [];
    for (var o = 0; o < 3; o++) {
      var s3 = p.b2[o];
      for (var j2 = 0; j2 < 16; j2++) s3 += p.W2[o][j2] * A2[j2];
      logits.push(s3);
    }
    var probs = softmax(logits);

    return { Z1:Z1, A1:A1, P1:P1, argMax:argMax, flat:flat, Z2:Z2, A2:A2, logits:logits, probs:probs };
  }

  function backwardPassRede(p, cache, img, oneHot){
    var dLogits = cache.probs.map(function(v,i){ return v - oneHot[i]; });

    var gW2 = [], gb2 = dLogits.slice();
    for (var o = 0; o < 3; o++) {
      var row = [];
      for (var j = 0; j < 16; j++) row.push(dLogits[o] * cache.A2[j]);
      gW2.push(row);
    }
    var dA2 = [];
    for (var j = 0; j < 16; j++) {
      var s = 0;
      for (var o2 = 0; o2 < 3; o2++) s += p.W2[o2][j] * dLogits[o2];
      dA2.push(s);
    }
    var dZ2 = dA2.map(function(v,i){ return v * reluDeriv(cache.Z2[i]); });

    var gW1 = [], gb1 = dZ2.slice();
    for (var i = 0; i < 16; i++) {
      var row1 = [];
      for (var j2 = 0; j2 < 100; j2++) row1.push(dZ2[i] * cache.flat[j2]);
      gW1.push(row1);
    }
    var dFlat = [];
    for (var j3 = 0; j3 < 100; j3++) {
      var s2 = 0;
      for (var i2 = 0; i2 < 16; i2++) s2 += p.W1[i2][j3] * dZ2[i2];
      dFlat.push(s2);
    }

    var dP1 = []; var idx = 0;
    for (var f = 0; f < 4; f++) {
      var pf = [];
      for (var r = 0; r < 5; r++) {
        var pr = [];
        for (var c = 0; c < 5; c++) pr.push(dFlat[idx++]);
        pf.push(pr);
      }
      dP1.push(pf);
    }

    var dA1 = [];
    for (var f2 = 0; f2 < 4; f2++) {
      var af = [];
      for (var r2 = 0; r2 < 10; r2++) af.push(new Array(10).fill(0));
      for (var r3 = 0; r3 < 5; r3++)
        for (var c3 = 0; c3 < 5; c3++) {
          var am = cache.argMax[f2][r3][c3];
          af[2*r3+am.dr][2*c3+am.dc] += dP1[f2][r3][c3];
        }
      dA1.push(af);
    }

    var dZ1 = [];
    for (var f3 = 0; f3 < 4; f3++) {
      var zf = [];
      for (var r4 = 0; r4 < 10; r4++) {
        var row2 = [];
        for (var c4 = 0; c4 < 10; c4++)
          row2.push(dA1[f3][r4][c4] * reluDeriv(cache.Z1[f3][r4][c4]));
        zf.push(row2);
      }
      dZ1.push(zf);
    }

    var gK = [], gbConv = [];
    for (var f4 = 0; f4 < 4; f4++) {
      var gk = [[0,0,0],[0,0,0],[0,0,0]]; var gb = 0;
      for (var r5 = 0; r5 < 10; r5++)
        for (var c5 = 0; c5 < 10; c5++) {
          var d = dZ1[f4][r5][c5]; gb += d;
          for (var kr = 0; kr < 3; kr++)
            for (var kc = 0; kc < 3; kc++)
              gk[kr][kc] += d * img[r5+kr][c5+kc];
        }
      gK.push(gk); gbConv.push(gb);
    }

    var loss = -Math.log(Math.max(cache.probs[oneHot.indexOf(1)], 1e-9));
    return { gK:gK, gbConv:gbConv, gW1:gW1, gb1:gb1, gW2:gW2, gb2:gb2, loss:loss };
  }

  function zeros3(f,r,c){
    var a = [];
    for (var i=0;i<f;i++){ var b=[]; for(var j=0;j<r;j++){ b.push(new Array(c).fill(0)); } a.push(b); }
    return a;
  }

  // Um passo de treinamento em lote (as 3 imagens de uma vez): calcula o
  // forward+backward para cada uma, faz a média dos gradientes e atualiza
  // todos os parâmetros (kernels inclusive) via gradiente descendente.
  function treinarPassoLote(params, lr){
    var gK = zeros3(4,3,3), gbConv = [0,0,0,0];
    var gW1 = [], gb1 = new Array(16).fill(0);
    for (var i=0;i<16;i++) gW1.push(new Array(100).fill(0));
    var gW2 = [], gb2 = [0,0,0];
    for (var o=0;o<3;o++) gW2.push(new Array(16).fill(0));
    var totalLoss = 0;

    ORDEM_CLASSES.forEach(function(id){
      var cache = forwardPassRede(params, IMAGENS[id]);
      var grads = backwardPassRede(params, cache, IMAGENS[id], ONE_HOTS[id]);
      totalLoss += grads.loss;
      for (var f=0;f<4;f++){
        for (var kr=0;kr<3;kr++) for (var kc=0;kc<3;kc++) gK[f][kr][kc] += grads.gK[f][kr][kc];
        gbConv[f] += grads.gbConv[f];
      }
      for (var ii=0;ii<16;ii++){
        for (var jj=0;jj<100;jj++) gW1[ii][jj] += grads.gW1[ii][jj];
        gb1[ii] += grads.gb1[ii];
      }
      for (var oo=0;oo<3;oo++){
        for (var jj2=0;jj2<16;jj2++) gW2[oo][jj2] += grads.gW2[oo][jj2];
        gb2[oo] += grads.gb2[oo];
      }
    });

    var nB = ORDEM_CLASSES.length;
    for (var f2=0;f2<4;f2++){
      for (var kr2=0;kr2<3;kr2++) for (var kc2=0;kc2<3;kc2++) params.K[f2][kr2][kc2] -= lr*gK[f2][kr2][kc2]/nB;
      params.bConv[f2] -= lr*gbConv[f2]/nB;
    }
    for (var i2=0;i2<16;i2++){
      for (var j2=0;j2<100;j2++) params.W1[i2][j2] -= lr*gW1[i2][j2]/nB;
      params.b1[i2] -= lr*gb1[i2]/nB;
    }
    for (var o2=0;o2<3;o2++){
      for (var j3=0;j3<16;j3++) params.W2[o2][j3] -= lr*gW2[o2][j3]/nB;
      params.b2[o2] -= lr*gb2[o2]/nB;
    }
    return totalLoss / nB;
  }

  // Calcula a fração de acertos do lote de 3 imagens com os parâmetros
  // atuais: para cada imagem, roda o forward pass e verifica se a classe
  // de maior probabilidade (argmax do Softmax) coincide com a classe correta.
  function calcularAcuraciaLote(params){
    var acertos = 0;
    ORDEM_CLASSES.forEach(function(id){
      var cache = forwardPassRede(params, IMAGENS[id]);
      var maxProb = Math.max.apply(null, cache.probs);
      var idxPredito = cache.probs.indexOf(maxProb);
      var idxCorreto = ONE_HOTS[id].indexOf(1);
      if (idxPredito === idxCorreto) acertos += 1;
    });
    return acertos / ORDEM_CLASSES.length;
  }

  function construirPipeline(params, imgMatriz, imgId){
    var cache = forwardPassRede(params, imgMatriz);

    var conv1 = cache.A1.map(function(mapa){
      return mapa.map(function(row){ return row.map(saturar); });
    });
    var pool1 = cache.P1.map(function(mapa){
      return mapa.map(function(row){ return row.map(saturar); });
    });
    var flatten = cache.flat.map(saturar);
    var fc = cache.A2.map(saturar);
    var softmaxBars = ORDEM_CLASSES.map(function(id, i){
      return { rotulo: ROTULOS[id], valor: cache.probs[i] };
    });

    return { conv1: conv1, pool1: pool1, flatten: flatten, fc: fc, softmax: softmaxBars, probsCrus: cache.probs };
  }

  function initArch(root){
    if (!root || root.dataset.initArch) return;
    root.dataset.initArch = "1";

    var imgAtualId = "casa";
    var imgMatriz = IMAGENS[imgAtualId];

    var rngInicial = criarRng(42);
    var PARAMS = inicializarParametros(rngInicial);
    var historicoLoss = [];
    var historicoAcuracia = [];
    var passoTreino = 0;
    var autoplayInterval = null;

    var PIPE = construirPipeline(PARAMS, imgMatriz, imgAtualId);

    var btnCasa   = root.querySelector('#cap09arch_btnCasa');
    var btnFeliz  = root.querySelector('#cap09arch_btnFeliz');
    var btnTriste = root.querySelector('#cap09arch_btnTriste');

    var flowEl     = root.querySelector('#cap09arch_flow');
    var descEl     = root.querySelector('#cap09arch_desc');
    var barsWrapEl = root.querySelector('#cap09arch_barsWrap');
    var kernelsPanelEl = root.querySelector('#cap09arch_kernelsPanel');
    var legendaEl  = root.querySelector('#cap09arch_legenda');
    var canvas     = root.querySelector('#cap09arch_canvasVis');
    var ctx        = canvas.getContext('2d');
    var W = canvas.width, H = canvas.height;

    var canvasLoss = root.querySelector('#cap09arch_canvasLoss');
    var ctxLoss = canvasLoss.getContext('2d');
    var canvasAcc = root.querySelector('#cap09arch_canvasAcc');
    var ctxAcc = canvasAcc.getContext('2d');
    var lossTxt = root.querySelector('#cap09arch_lossTxt');
    var accTxt = root.querySelector('#cap09arch_accTxt');
    var stepTxt = root.querySelector('#cap09arch_stepTxt');
    var cicloBackward = root.querySelector('#cap09arch_cicloBackward');

    var btnPassoUnico    = root.querySelector('#cap09arch_btnPassoUnico');
    var btnTreinar       = root.querySelector('#cap09arch_btnTreinar');
    var btnReiniciarPesos = root.querySelector('#cap09arch_btnReiniciarPesos');

    var ETAPAS = [
      {
        id: "entrada", nome: "Entrada", forma: "12×12", tipo: "imagem",
        legenda: "Imagem em escala de cinza 12×12 com margem fixa de zeros.",
        desc: "A <b>imagem de entrada</b> é representada como uma matriz de pixels 12×12. É a mesma estrutura (casa, rosto feliz ou triste) do simulador anterior."
      },
      {
        id: "conv1", nome: "Conv1 + ReLU", forma: "10×10×4", tipo: "mapas", tam: 10, dadosKey: "conv1",
        legenda: "4 mapas de características 10×10, um por filtro aprendido.",
        desc: "A primeira <b>camada convolucional</b> aplica 4 <i>kernels</i> 3×3 <b>aprendidos por treinamento</b> — diferente do simulador anterior, aqui eles começam aleatórios e vão sendo ajustados a cada passo de treinamento (veja os valores abaixo do mapa). Nesta versão simplificada usamos apenas 1 bloco convolucional (N=1); redes reais costumam empilhar vários."
      },
      {
        id: "pool1", nome: "Pooling1", forma: "5×5×4", tipo: "mapas", tam: 5, dadosKey: "pool1",
        legenda: "Os 4 mapas reduzidos para 5×5 via Max-Pooling (2×2, stride 2).",
        desc: "A camada de <b>Max-Pooling</b> reduz a resolução espacial de 10×10 para 5×5, mantendo apenas a ativação máxima de cada janela 2×2 — o que reduz a dimensão dos dados e dá alguma tolerância a pequenos deslocamentos."
      },
      {
        id: "flatten", nome: "Flatten", forma: "100 valores", tipo: "vetor", dadosKey: "flatten",
        legenda: "Vetor linearizado com 4 × 5 × 5 = 100 elementos.",
        desc: "A operação <b>Flatten</b> 'achata' os 4 mapas 2D em um único vetor 1D de 100 valores, preparando a informação para entrar nas camadas densas."
      },
      {
        id: "fc", nome: "Camada Densa (FC)", forma: "16 neurônios", tipo: "vetor", dadosKey: "fc",
        legenda: "16 neurônios (com ReLU) combinando o vetor de características.",
        desc: "A <b>camada totalmente conectada</b> tem 16 neurônios com pesos também aprendidos, combinando todas as características locais extraídas antes. Uma segunda camada densa de saída (16→3, não desenhada separadamente aqui) produz os valores brutos (<i>logits</i>) que alimentam o Softmax."
      },
      {
        id: "softmax", nome: "Softmax", forma: "3 classes", tipo: "softmax",
        legenda: "Distribuição de probabilidade final, calculada a partir dos pesos atuais da rede.",
        desc: "A função <b>Softmax</b> converte os <i>logits</i> em probabilidades que somam 1.0 (100%). Estes valores são <b>calculados de verdade</b> a partir dos pesos atuais — antes de treinar, tendem a ficar próximos de 33%/33%/33%; depois de alguns passos de treinamento, devem convergir para a classe correta."
      }
    ];

    var etapaSelecionada = 0;

    function estiloBloco(el, selecionado){
      el.style.flex = '1';
      el.style.minWidth = '95px';
      el.style.textAlign = 'center';
      el.style.padding = '8px 4px';
      el.style.borderRadius = '8px';
      el.style.cursor = 'pointer';
      el.style.fontSize = '11px';
      el.style.fontWeight = '600';
      if (selecionado){
        el.style.border = '2px solid #2F6F9F';
        el.style.background = '#EAF2FA';
        el.style.color = '#2F6F9F';
      } else {
        el.style.border = '1px solid #E4DCC8';
        el.style.background = '#FFFFFF';
        el.style.color = '#374151';
      }
    }

    function montarFluxo(){
      flowEl.innerHTML = '';
      ETAPAS.forEach(function(etapa, idx){
        var bloco = document.createElement('div');
        bloco.id = 'cap09arch_bloco_' + etapa.id;
        estiloBloco(bloco, false);

        var linha1 = document.createTextNode(etapa.nome);
        var linha2 = document.createElement('span');
        linha2.textContent = etapa.forma;
        linha2.style.display = 'block';
        linha2.style.fontSize = '9px';
        linha2.style.fontWeight = '500';
        linha2.style.color = '#8A8371';
        linha2.style.marginTop = '2px';

        bloco.appendChild(linha1);
        bloco.appendChild(linha2);
        bloco.addEventListener('click', function(){ selecionar(idx); });
        flowEl.appendChild(bloco);

        if (idx < ETAPAS.length - 1){
          var seta = document.createElement('div');
          seta.textContent = '➔';
          seta.style.color = '#C5BC9D';
          seta.style.fontSize = '12px';
          seta.style.padding = '0 2px';
          flowEl.appendChild(seta);
        }
      });
    }

    function desenharImagemEntrada(){
      ctx.clearRect(0,0,W,H);
      var n = 12, tam = 13;
      var offX = Math.round((W - n*tam)/2), offY = Math.round((H - n*tam)/2);
      for (var r=0; r<n; r++){
        for (var c=0; c<n; c++){
          var val = imgMatriz[r][c];
          var g = Math.round(val * 255);
          ctx.fillStyle = "rgb(" + g + "," + g + "," + g + ")";
          ctx.fillRect(offX + c*tam, offY + r*tam, tam, tam);
          ctx.strokeStyle = "#D1D5DB";
          ctx.strokeRect(offX + c*tam, offY + r*tam, tam, tam);
        }
      }
    }

    function desenharMapas(mapasArr, tamEspacial){
      ctx.clearRect(0,0,W,H);
      var count = mapasArr.length;
      var cols = 2, rows = 2;
      var pad = 12;
      var thumb = 65;
      var totalW = cols*thumb + (cols-1)*pad;
      var totalH = rows*thumb + (rows-1)*pad;
      var offX = Math.round((W-totalW)/2), offY = Math.round((H-totalH)/2);
      var px = thumb/tamEspacial;

      var nomesFiltros = ["Filtro 1", "Filtro 2", "Filtro 3", "Filtro 4"];

      for (var f=0; f<count; f++){
        var col = f % cols, row = Math.floor(f/cols);
        var bx = offX + col*(thumb+pad);
        var by = offY + row*(thumb+pad);
        var mapa = mapasArr[f];
        for (var yy=0; yy<tamEspacial; yy++){
          for (var xx=0; xx<tamEspacial; xx++){
            ctx.fillStyle = corMapa(mapa[yy][xx]);
            ctx.fillRect(bx+xx*px, by+yy*px, px+0.5, px+0.5);
          }
        }
        ctx.strokeStyle = '#2F6F9F';
        ctx.lineWidth = 1;
        ctx.strokeRect(bx, by, thumb, thumb);

        ctx.fillStyle = "#5E5A4A";
        ctx.font = "9px Inter, sans-serif";
        ctx.fillText(nomesFiltros[f], bx, by - 3);
      }
    }

    function desenharVetor(vals){
      ctx.clearRect(0,0,W,H);
      var count = vals.length;
      var cols = count > 20 ? 10 : 4;
      var rows = Math.ceil(count/cols);
      var cellW = Math.min(22, (W - 40)/cols);
      var cellH = Math.min(22, (H - 40)/rows);
      var offX = Math.round((W - cols*cellW)/2);
      var offY = Math.round((H - rows*cellH)/2);

      for (var i=0; i<count; i++){
        var c = i % cols, r = Math.floor(i/cols);
        ctx.fillStyle = corMapa(vals[i]);
        ctx.fillRect(offX + c*cellW, offY + r*cellH, cellW-1, cellH-1);
        ctx.strokeStyle = "#E4DCC8";
        ctx.strokeRect(offX + c*cellW, offY + r*cellH, cellW-1, cellH-1);
      }
    }

    function desenharBarras(barras){
      barsWrapEl.innerHTML = '';
      barsWrapEl.style.display = 'flex';
      var alturaMax = 100;
      barras.forEach(function(b){
        var wrap = document.createElement('div');
        wrap.style.display = 'flex';
        wrap.style.flexDirection = 'column';
        wrap.style.alignItems = 'center';
        wrap.style.fontSize = '11px';
        wrap.style.color = '#374151';

        var bar = document.createElement('div');
        bar.style.width = '38px';
        bar.style.height = Math.max(1, Math.round(b.valor * alturaMax)) + 'px';
        bar.style.background = 'linear-gradient(#2F6F9F, #1E8F6F)';
        bar.style.borderRadius = '4px 4px 0 0';

        var legendaBar = document.createElement('div');
        legendaBar.style.marginTop = '6px';
        legendaBar.style.fontWeight = '600';
        legendaBar.textContent = b.rotulo;

        var valBar = document.createElement('div');
        valBar.className = 'cn-mono';
        valBar.style.fontSize = '10px';
        valBar.style.color = '#2F6F9F';
        valBar.textContent = (b.valor * 100).toFixed(1) + '%';

        wrap.appendChild(bar);
        wrap.appendChild(legendaBar);
        wrap.appendChild(valBar);
        barsWrapEl.appendChild(wrap);
      });
    }

    function desenharPainelKernels(){
      var html = '<div class="cn-grouplabel" style="text-align:left;">KERNELS APRENDIDOS (VALORES ATUAIS)</div>' +
        '<div style="display:flex;gap:10px;flex-wrap:wrap;justify-content:center;">';
      for (var f=0; f<4; f++){
        html += '<div style="display:grid;grid-template-columns:repeat(3,30px);gap:2px;">';
        for (var r=0; r<3; r++){
          for (var c=0; c<3; c++){
            var v = PARAMS.K[f][r][c];
            var cor = v > 0 ? "#E6F4EA" : (v < 0 ? "#FCE8E6" : "#F3F4F6");
            var corTxt = v > 0 ? "#1E8F6F" : (v < 0 ? "#C1443A" : "#374151");
            html += '<div class="cn-kcell cn-mono" style="background:' + cor + ';color:' + corTxt + ';">' + v.toFixed(1) + '</div>';
          }
        }
        html += '</div>';
      }
      html += '</div>';
      kernelsPanelEl.innerHTML = html;
    }

    function desenharGraficoLoss(){
      var Wc = canvasLoss.width, Hc = canvasLoss.height;
      ctxLoss.clearRect(0,0,Wc,Hc);
      ctxLoss.strokeStyle = "#E4DCC8"; ctxLoss.lineWidth = 1;
      ctxLoss.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoLoss.length < 2){
        ctxLoss.fillStyle = "#B8AE94";
        ctxLoss.font = "10px Inter, sans-serif";
        ctxLoss.textAlign = "center";
        ctxLoss.fillText("la perte apparaîtra ici", Wc/2, Hc/2 + 3);
        return;
      }
      var maxN = 200;
      var dados = historicoLoss.slice(-maxN);
      var maxLoss = Math.max.apply(null, dados);
      maxLoss = Math.max(maxLoss, 0.05);
      var padL = 8, padR = 8, padT = 8, padB = 8;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxLoss.strokeStyle = "#2F5FA8";
      ctxLoss.lineWidth = 1.6;
      ctxLoss.beginPath();
      dados.forEach(function(v, i){
        var x = padL + (dados.length === 1 ? 0 : (i/(dados.length-1)) * plotW);
        var y = padT + (1 - v/maxLoss) * plotH;
        if (i===0) ctxLoss.moveTo(x,y); else ctxLoss.lineTo(x,y);
      });
      ctxLoss.stroke();
    }

    // Mesmo padrão visual do gráfico de perda, mas com eixo Y fixo em [0,1]
    // (a acurácia do lote é sempre 0, 1/3, 2/3 ou 1, já que só há 3 imagens).
    function desenharGraficoAcuracia(){
      var Wc = canvasAcc.width, Hc = canvasAcc.height;
      ctxAcc.clearRect(0,0,Wc,Hc);
      ctxAcc.strokeStyle = "#E4DCC8"; ctxAcc.lineWidth = 1;
      ctxAcc.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoAcuracia.length < 2){
        ctxAcc.fillStyle = "#B8AE94";
        ctxAcc.font = "10px Inter, sans-serif";
        ctxAcc.textAlign = "center";
        ctxAcc.fillText("la précision apparaîtra ici", Wc/2, Hc/2 + 3);
        return;
      }
      var maxN = 200;
      var dados = historicoAcuracia.slice(-maxN);
      var padL = 8, padR = 8, padT = 8, padB = 8;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxAcc.strokeStyle = "#1E8F6F";
      ctxAcc.lineWidth = 1.6;
      ctxAcc.beginPath();
      dados.forEach(function(v, i){
        var x = padL + (dados.length === 1 ? 0 : (i/(dados.length-1)) * plotW);
        var y = padT + (1 - v) * plotH;
        if (i===0) ctxAcc.moveTo(x,y); else ctxAcc.lineTo(x,y);
      });
      ctxAcc.stroke();
    }

    function estimarClassePredita(probs){
      var maxV = Math.max.apply(null, probs);
      return ORDEM_CLASSES[probs.indexOf(maxV)];
    }

    function atualizarVisual(etapa){
      kernelsPanelEl.style.display = 'none';
      if (etapa.tipo === 'softmax'){
        canvas.style.display = 'none';
        desenharBarras(PIPE.softmax);
      } else {
        canvas.style.display = 'block';
        barsWrapEl.style.display = 'none';
        if (etapa.tipo === 'imagem') desenharImagemEntrada();
        else if (etapa.tipo === 'mapas') desenharMapas(PIPE[etapa.dadosKey], etapa.tam);
        else if (etapa.tipo === 'vetor') desenharVetor(PIPE[etapa.dadosKey]);

        if (etapa.id === 'conv1'){
          kernelsPanelEl.style.display = 'block';
          desenharPainelKernels();
        }
      }
      legendaEl.textContent = etapa.legenda || '';
    }

    function selecionar(idx){
      etapaSelecionada = idx;
      var etapa = ETAPAS[idx];
      ETAPAS.forEach(function(e){
        var el = root.querySelector('#cap09arch_bloco_' + e.id);
        if (el) estiloBloco(el, false);
      });
      var atual = root.querySelector('#cap09arch_bloco_' + etapa.id);
      if (atual) estiloBloco(atual, true);
      descEl.innerHTML = '<b>' + etapa.nome + '</b> — dimensão: <code class="cn-mono" style="background:#EAF2FA;color:#2F6F9F;padding:2px 6px;border-radius:4px;">' + etapa.forma + '</code><br><br>' + etapa.desc;
      atualizarVisual(etapa);
    }

    function recomputarPipeline(){
      PIPE = construirPipeline(PARAMS, imgMatriz, imgAtualId);
    }

    function trocarImagem(id, btn){
      [btnCasa, btnFeliz, btnTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      imgAtualId = id;
      imgMatriz = IMAGENS[id];
      recomputarPipeline();
      selecionar(etapaSelecionada);
    }

    function pulsarBackward(){
      cicloBackward.classList.add('pulso');
      setTimeout(function(){ cicloBackward.classList.remove('pulso'); }, 350);
    }

    function passoDeTreinamento(){
      var lr = 0.3;
      var loss = treinarPassoLote(PARAMS, lr);
      var acuracia = calcularAcuraciaLote(PARAMS);
      passoTreino += 1;
      historicoLoss.push(loss);
      historicoAcuracia.push(acuracia);
      recomputarPipeline();

      lossTxt.textContent = 'Perte : ' + loss.toFixed(4);
      accTxt.textContent = 'Précision : ' + Math.round(acuracia * 100) + '%';
      stepTxt.textContent = 'Étape d'entraînement : ' + passoTreino;
      desenharGraficoLoss();
      desenharGraficoAcuracia();
      pulsarBackward();
      selecionar(etapaSelecionada);
      return { loss: loss, acuracia: acuracia };
    }

    function pararAutoplay(){
      if (autoplayInterval){
        clearInterval(autoplayInterval);
        autoplayInterval = null;
        btnTreinar.textContent = '▶ Treinar';
      }
    }

    btnCasa.addEventListener('click', function(){ trocarImagem('casa', btnCasa); });
    btnFeliz.addEventListener('click', function(){ trocarImagem('feliz', btnFeliz); });
    btnTriste.addEventListener('click', function(){ trocarImagem('triste', btnTriste); });

    btnPassoUnico.addEventListener('click', function(){
      pararAutoplay();
      passoDeTreinamento();
    });

    btnTreinar.addEventListener('click', function(){
      if (autoplayInterval){ pararAutoplay(); return; }
      btnTreinar.textContent = '⏸ Pausar';
      autoplayInterval = setInterval(function(){
        var resultado = passoDeTreinamento();
        if (resultado.loss < 0.002 && resultado.acuracia === 1){ pararAutoplay(); }
      }, 120);
    });

    btnReiniciarPesos.addEventListener('click', function(){
      pararAutoplay();
      PARAMS = inicializarParametros(criarRng(Date.now() % 2147483647));
      historicoLoss = [];
      historicoAcuracia = [];
      passoTreino = 0;
      lossTxt.textContent = 'Loss: —';
      accTxt.textContent = 'Acurácia: —';
      stepTxt.textContent = 'Passo de treinamento: 0';
      recomputarPipeline();
      desenharGraficoLoss();
      desenharGraficoAcuracia();
      selecionar(etapaSelecionada);
    });

    montarFluxo();
    desenharGraficoLoss();
    desenharGraficoAcuracia();
    selecionar(0);
  }

  function tryInitArch(){
    var root = document.getElementById('sim-09-arquitetura');
    if (root) initArch(root); else setTimeout(tryInitArch, 200);
  }
  tryInitArch();
})();
</script>
''')


**Figure 9.11:** Simulateur interactif de l


<figure id="fig-09-sim-09-arquitetura">
  <img src="imagens/fig-09-sim-09-arquitetura.png" alt=" Simulateur interactif de l'architecture d'un CNN : choisissez l'une des images d'entrée 12×12 (maison, visage heureux ou triste), cliquez sur chaque bloc du *pipeline* — Entrée, *Conv+ReLU*, *Pooling*, *Flatten*, *FC* et *Softmax* — et exécutez de vraies étapes d'entraînement (*forward pass* + rétropropagation) pour observer la perte et la précision évoluer, les *kernels* être ajustés et le *Softmax* commencer à pointer vers la classe correcte. " style="max-width:80%" />
  <figcaption><strong>Figure 9.11:</strong>  Simulateur interactif de l'architecture d'un CNN : choisissez l'une des images d'entrée 12×12 (maison, visage heureux ou triste), cliquez sur chaque bloc du *pipeline* — Entrée, *Conv+ReLU*, *Pooling*, *Flatten*, *FC* et *Softmax* — et exécutez de vraies étapes d'entraînement (*forward pass* + rétropropagation) pour observer la perte et la précision évoluer, les *kernels* être ajustés et le *Softmax* commencer à pointer vers la classe correcte. </figcaption>
</figure>

### 9.4.9 Comment le Gradient Ajuste les *Kernels* de la Convolution

La compréhension du processus d'apprentissage dans un réseau neuronal convolutif (CNN) nécessite l'élucidation d'un mécanisme fondamental : **comment les coefficients aléatoires d'un filtre initial se transforment-ils en détecteurs précis de bords, de textures et de motifs complexes ?**

La réponse réside dans le principe du **partage de poids (*weight sharing*)**. Pendant l'étape de propagation avant (*forward pass*), le même filtre de dimension $3\times3$ glisse sur toute l'étendue de l'image d'entrée. Par conséquent, chaque poids du *kernel* — comme l'élément $K[0][0]$ dans le coin supérieur gauche — est réutilisé de multiples fois sur les différentes régions spatiales de la donnée d'entrée.

Pendant l'étape de rétropropagation (*backpropagation*), cette réutilisation établit une dynamique directe : **chaque position spatiale traitée par le filtre génère une contribution individuelle (« vote ») pour la mise à jour du poids correspondant.**

#### 9.4.9.1 L'Intuition Derrière le Calcul

Soit $Z[r][c]$ la carte de caractéristiques pré-activation à la position $(r,c)$ de la fenêtre glissante, obtenue par corrélation croisée entre le *kernel* $K$ et l'entrée $X$ :

$$
Z[r][c] = \sum_{k_r} \sum_{k_c} K[k_r][k_c] \cdot X[r + k_r][c + k_c]
$$

En appliquant la règle de la chaîne pour déterminer la contribution d'un poids spécifique $K[k_r][k_c]$ à la fonction de perte $L$, on obtient les étapes suivantes :

1. **Erreur Locale ($dZ$) :** À chaque position $(r,c)$, on calcule la dérivée partielle de la fonction de perte par rapport à la pré-activation :
   $$dZ[r][c] = \frac{\partial L}{\partial Z[r][c]}$$
   qui quantifie la responsabilité de cette position spécifique dans l'erreur totale du réseau ($L$).

2. **Contribution du Poids :** Comme $\frac{\partial Z[r][c]}{\partial K[k_r][k_c]} = X[r + k_r][c + k_c]$, l'influence d'un poids spécifique $K[k_r][k_c]$ sur l'erreur de la position $(r,c)$ est obtenue en multipliant l'erreur locale $dZ[r][c]$ par la valeur du pixel d'entrée aligné sur ce poids au moment du calcul :
   $$dZ[r][c] \cdot X[r + k_r][c + k_c]$$

3. **Accumulation des Gradients :** Le gradient final du poids correspond à la **somme des contributions (« votes ») de toutes les positions** parcourues par la fenêtre glissante :

$$
\frac{\partial L}{\partial K[k_r][k_c]} = \sum_{(r,c)} dZ[r][c] \cdot X[r + k_r][c + k_c]
$$

Cette formulation assure une parité directe entre la dérivation analytique et les valeurs calculées dans le simulateur d'inspection du gradient ([Figure 9.12](#fig-09-sim-09-gradiente-kernel)).

#### 9.4.9.2 Le Rôle de la Fonction ReLU comme « Filtre de Pertinence »

L'application de la fonction d'activation **ReLU** ($\max(0, z)$) immédiatement après la convolution introduit une propriété de sélectivité au gradient :

* **Activation Positive ($Z[r][c] > 0$) :** La dérivée de la ReLU est $1$. L'erreur locale est propagée intégralement ($dZ \neq 0$), permettant à la position de contribuer à la mise à jour des poids du *kernel*.
* **Activation Inactive ($Z[r][c] \le 0$) :** La dérivée de la ReLU est $0$. L'erreur locale est annulée ($dZ = 0$), supprimant la contribution de la position au gradient final.

> **Note didactique :** La ReLU garantit que seules les régions spatiales ayant produit des réponses actives pendant la propagation avant possèdent la capacité de modifier les poids du *kernel* dans le processus de rétropropagation.

#### 9.4.9.3 Mise à Jour des Poids via la Descente de Gradient

Après la consolidation des gradients accumulés de toutes les positions, la mise à jour du poids se fait selon l'algorithme de la Descente de Gradient Stochastique (SGD) :

$$
K[k_r][k_c] \leftarrow K[k_r][k_c] - \eta \cdot \frac{\partial L}{\partial K[k_r][k_c]}
$$

où $\eta$ désigne le **taux d'apprentissage (*learning rate*)**.

* Si la somme des gradients est **positive**, la valeur du poids est réduite.
* Si la somme est **négative**, la valeur du poids est augmentée.

#### 9.4.9.4 Exploration du Simulateur Interactif

> ### 📝 🔗 De l'Architecture Globale à l'Inspection du Gradient
>
> Dans le simulateur d'architecture ([Figure 9.11](#fig-09-sim-09-arquitetura)), on observe l'erreur $dZ$ dérivée de la rétropropagation multicouche complète, issue de la perte d'entropie croisée (*Softmax*) sur les images d'entrée $12\times12$.
>
> Pour permettre la vérification analytique du gradient sans la surcharge de $100$ positions de convolution et de rétropropagation multicouche, le simulateur d'apprentissage du *kernel* ([Figure 9.12](#fig-09-sim-09-gradiente-kernel)) adopte un modèle d'inspection réduit ($6\times6$). Dans ce scénario, on simplifie le problème en remplaçant la classification complexe par une **métadonnée de calibration scalaire** : on ajuste le filtre pour produire une réponse accumulée prédéfinie ($\text{cible} = 9$) lors de l'identification d'un motif spécifique (comme un bord à 45 degrés). Le mécanisme d'accumulation des gradients ($dZ \cdot X$) reste rigoureusement identique dans les deux formulations.

Pour inspecter cette dynamique au niveau numérique, on utilise le simulateur sur [Figure 9.12](#fig-09-sim-09-gradiente-kernel):

* **Image d'Entrée ($X$) :** Matrice $6\times6$.
* **Filtre Convolutionnel ($K$) :** Matrice $3\times3$ (9 poids).
* **Carte de Sortie ($Z$ / $A$) :** Matrice $4\times4$ (16 positions de la fenêtre).
* **Fonction de Perte ($L$) :** Définie par $L = \frac{1}{2}(S - \text{cible})^2$, où $S = \sum A[r][c]$ représente la somme globale des activations post-ReLU.

> **Le rôle de la $\text{cible} = 9$ :** La valeur scalaire $\text{cible} = 9$ représente l'« énergie d'activation » idéale stipulée pour l'image avec bord diagonal. Comme la carte $A$ possède 16 positions, cette valeur équivaut à rechercher une réponse moyenne de $\frac{9}{16} \approx 0,56$ par pixel activé. Lorsque $S > 9$, le réseau identifie que le filtre réagit avec une intensité excessive au motif, générant une erreur $dZ > 0$ qui force la réduction des poids $K$. Lorsque $S < 9$, les poids sont augmentés pour amplifier le signal.

##### Roteiro Sugerido de Experimentação:

1. **Sélection du Poids :** Dans la grille $3\times3$, choisissez le poids à analyser (ex. : $K[0][0]$).
2. **Balayage de la Fenêtre :** Utilisez le bouton **« ▶ Avancer position »** pour suivre le déplacement de la fenêtre à travers les 16 positions spatiales. Notez la mise en évidence visuelle dans la cellule de la carte d'entrée qui aligne le pixel $X$ avec le poids sélectionné.
3. **Analyse du Vote Local :** Examinez le produit de l'erreur locale par le pixel d'entrée ($dZ \cdot X$) dans le panneau de calcul de la position.
4. **Vérification de l'Historique :** Suivez la consolidation des 16 résultats partiels organisés dans les quatre colonnes d'historique, en observant l'accumulation du gradient final.
5. **Mise à Jour du Kernel :** Cliquez sur **« ▶ Appliquer le pas de descente de gradient »** pour visualiser la convergence de la courbe de perte et l'adaptation du *kernel* aléatoire au motif d'entrée sélectionné.

In [11]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-gradiente-kernel" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-gradiente-kernel .cap09kgrad_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-gradiente-kernel .cap09kgrad_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-gradiente-kernel .cap09kgrad_modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_modebtn.cap09kgrad_active { background:#26241D; color:#FBF7EE; }
    #sim-09-gradiente-kernel .cap09kgrad_lrbtn {
      padding:3px 8px; font-size:10px; font-weight:600; border:1px solid #E4DCC8; background:#FFFFFF;
      color:#5E5A4A; border-radius:6px; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-gradiente-kernel .cap09kgrad_lrbtn.cap09kgrad_active { background:#2F6F9F; color:#FFFFFF; border-color:#2F6F9F; }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn {
      padding:0 10px; height:26px; border-radius:8px; border:1px solid #E4DCC8; background:#FAFAF7;
      color:#26241D; font-size:10.5px; font-weight:600; cursor:pointer; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn:hover { background:#F1EAD7; }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn:disabled { opacity:0.4; cursor:not-allowed; }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn {
      padding:0 12px; height:26px; border-radius:8px; border:1px solid #2F6F9F; background:#EAF2FA;
      color:#2F6F9F; font-size:10.5px; font-weight:700; cursor:pointer; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn:hover { background:#DCEEFB; }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn:disabled { opacity:0.4; cursor:not-allowed; }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn {
      height:24px; border-radius:5px; border:1px solid #E4DCC8; background:#FFFFFF;
      color:#374151; cursor:pointer; font-weight:600;
    }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn:hover { background:#F1EAD7; }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn_ativo { background:#26241D !important; color:#FBF7EE !important; border-color:#26241D !important; }
    #sim-09-gradiente-kernel .cap09kgrad_linhavoto { font-size:8.5px; padding:2px 3px; border-bottom:1px dashed #EDE7D6; border-radius:4px; transition:background .15s ease; white-space:nowrap; text-align:center; }
    #sim-09-gradiente-kernel .cap09kgrad_graflabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:4px; text-align:center; letter-spacing:.3px; }
    #sim-09-gradiente-kernel .cap09kgrad_painel { background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:12px; }
    #sim-09-gradiente-kernel .cap09kgrad_cartao {
      flex:1; min-width:140px; background:#FFFFFF; border:1px solid #E4DCC8; border-radius:10px;
      padding:10px 12px; box-sizing:border-box;
    }
    #sim-09-gradiente-kernel .cap09kgrad_barraProgresso {
      width:100%; height:6px; background:#EDE7D6; border-radius:4px; overflow:hidden; margin-top:6px;
    }
    #sim-09-gradiente-kernel .cap09kgrad_barraProgressoFill {
      height:100%; background:#2F6F9F; border-radius:4px; transition:width .2s ease;
    }

    #sim-09-gradiente-kernel .cap09kgrad_labelinfo {
      position:relative; display:inline-flex; align-items:center; gap:4px; cursor:help;
    }
    #sim-09-gradiente-kernel .cap09kgrad_icone {
      width:12px; height:12px; min-width:12px; border-radius:50%; background:#E4DCC8; color:#5E5A4A;
      font-size:8px; font-weight:700; display:inline-flex; align-items:center; justify-content:center;
      font-family:'Inter',sans-serif;
    }
    #sim-09-gradiente-kernel .cap09kgrad_labelinfo:hover .cap09kgrad_icone { background:#2F6F9F; color:#FBF7EE; }
    #sim-09-gradiente-kernel .cap09kgrad_tooltip {
      visibility:hidden; opacity:0; position:absolute; top:calc(100% + 6px); left:0;
      width:240px; max-width:60vw; background:#26241D; color:#FBF7EE; font-size:10px; font-weight:400;
      line-height:1.55; padding:9px 11px; border-radius:8px; z-index:999; letter-spacing:0;
      box-shadow:0 6px 16px rgba(0,0,0,.2); transition:opacity .15s ease, visibility .15s ease;
      pointer-events:none; text-align:left;
    }
    #sim-09-gradiente-kernel .cap09kgrad_graflabel .cap09kgrad_tooltip { left:50%; transform:translateX(-50%); }
    #sim-09-gradiente-kernel .cap09kgrad_labelinfo:hover .cap09kgrad_tooltip { visibility:visible; opacity:1; }

    #sim-09-gradiente-kernel .cap09kgrad_grid_historico {
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 4px 6px;
      max-height: 140px;
      overflow-y: auto;
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 8px;
      padding: 6px;
      margin-top: 4px;
    }

    #sim-09-gradiente-kernel .cap09kgrad_eq_box {
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 6px;
      padding: 5px 8px;
      margin: 4px 0;
      display: block;
      text-align: left;
      font-family: 'JetBrains Mono', monospace;
      font-size: 10px;
      color: #26241D;
    }

    #sim-09-gradiente-kernel .cap09kgrad_matriz_details {
      margin-top: 8px;
      max-width: 175px;
      font-size: 10px;
      color: #5E5A4A;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_summary {
      font-weight: 600;
      font-size: 9.5px;
      color: #2F6F9F;
      cursor: pointer;
      user-select: none;
      outline: none;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_summary:hover {
      text-decoration: underline;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_expl {
      margin-top: 4px;
      padding: 6px;
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 6px;
      line-height: 1.4;
    }

    /* ---- Hover das células (contas detalhadas + destaque entre camadas) ---- */
    #sim-09-gradiente-kernel .cap09kgrad_cell_hoverable { cursor: help; }
    #sim-09-gradiente-kernel .cap09kgrad_hlPrincipal {
      outline: 2px solid #2F6F9F !important;
      outline-offset: -1px;
      box-shadow: 0 0 0 3px rgba(47,111,159,0.15) inset;
      position: relative;
      z-index: 2;
    }
    #sim-09-gradiente-kernel .cap09kgrad_hlSecundario {
      outline: 2px dashed #2F6F9F !important;
      outline-offset: -2px;
      position: relative;
      z-index: 1;
    }
    .cap09kgrad_hovertip {
      position: fixed;
      background: #26241D;
      color: #FBF7EE;
      font-family: 'JetBrains Mono', ui-monospace, monospace;
      font-size: 10px;
      line-height: 1.65;
      padding: 9px 11px;
      border-radius: 8px;
      box-shadow: 0 6px 18px rgba(0,0,0,.25);
      z-index: 99999;
      pointer-events: none;
      max-width: 260px;
      white-space: normal;
      display: none;
    }
    .cap09kgrad_hovertip b { color: #7EE7C6; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">🧮 Simulateur : Gradient d'un Poids du Noyau</span>
    <span class="cap09kgrad_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">&part;Perte / &part;K[kr][kc] = &Sigma; dZ &middot; X</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletor de padrão e Taxa de Aprendizado -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;margin-bottom:12px;">
      <div style="max-width:320px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">MOTIF D'ENTRÉE (IMAGE 6&times;6)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Choisit quelle image 6&times;6 alimente la convolution.</span>
        </div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09kgrad_btnDiagonal" class="cap09kgrad_modebtn cap09kgrad_active">↘ Bordure diagonale</button>
          <button id="cap09kgrad_btnVertical" class="cap09kgrad_modebtn">▍ Bordure verticale</button>
        </div>
      </div>

      <div>
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">TAUX D'APPRENTISSAGE (&eta;)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Taux d'apprentissage. Ajustez pour voir la différence entre convergence douce (0.002) et effondrement par dépassement (0.02).</span>
        </div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09kgrad_lr0005" class="cap09kgrad_lrbtn">0.0005 (Lent)</button>
          <button id="cap09kgrad_lr002" class="cap09kgrad_lrbtn cap09kgrad_active">0.002 (Idéal)</button>
          <button id="cap09kgrad_lr02" class="cap09kgrad_lrbtn">0.02 (Élevé)</button>
        </div>
      </div>
    </div>

    <div style="font-size:11.5px;line-height:1.6;color:#5E5A4A;background:#FAFAF7;border:1px solid #E9E3D3;border-radius:10px;padding:10px 14px;margin-bottom:14px;">
      Exemple <b>réduit</b>: image 6&times;6 et filtre 3&times;3 générant des cartes 4&times;4. Cliquez sur les onglets <b>"🔍 Comment est-ce calculé ?"</b> sous chaque matrice pour comprendre les calculs pas à pas. <b>Survolez n'importe quelle cellule</b> de X, Z, A, dZ ou K pour voir le calcul exact de cette valeur, avec les éléments utilisés dans les couches associées mis en évidence avec un contour en pointillés/bleu.
    </div>

    <div style="display:flex;gap:10px;flex-wrap:wrap;align-items:flex-start;">

      <!-- PAINEL 1: Seleção do Peso e Kernel K -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;min-width:160px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">1. POIDS DU NOYAU<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Sélectionnez quel poids du noyau vous souhaitez analyser individuellement.</span>
        </div>
        <div id="cap09kgrad_seletorPeso"></div>
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo" style="margin-top:10px;">NOYAU ACTUEL (K)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Valeurs du filtre 3&times;3. Le poids sélectionné est mis en évidence en bleu. Survolez un poids pour voir où il est utilisé.</span>
        </div>
        <div id="cap09kgrad_gradeK"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Comment est-il mis à jour ?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Règle du Gradient :</b><br>
            <div class="cap09kgrad_eq_box">K &larr; K &minus; &eta; &middot; &nabla;K</div>
            • <b>&eta;</b> = taux d'apprentissage.<br>
            • <b>&nabla;K</b> = somme des 16 votes dZ &times; X.
          </div>
        </details>
      </div>

      <!-- PAINEL 2: Entrada X -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ENTRÉE X (6&times;6)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Image 6&times;6. Pixel bleu = chevauchement avec le poids K sélectionné dans la fenêtre actuelle. Survolez un pixel pour voir dans quelles positions de Z il est utilisé.</span>
        </div>
        <div id="cap09kgrad_gradeX"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Comment fonctionne X ?</summary>
          <div class="cap09kgrad_matriz_expl">
            Matrice d'entrée. En position (r,c), le poids K multiplie le pixel :
            <div class="cap09kgrad_eq_box">X[r + k<sub>r</sub>][c + k<sub>c</sub>]</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 3: Saída Z (Pré-ativação) -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">PRÉ-ACTIVATION Z (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Résultat de la convolution avant ReLU : Z = &Sigma; K &middot; X. Survolez une cellule pour voir les 9 termes de la somme, en mettant en évidence la fenêtre dans X et tout le noyau K.</span>
        </div>
        <div id="cap09kgrad_gradeZ"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Comment calcule Z ?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Corrélation croisée :</b><br>
            Multiplication point par point du filtre 3&times;3 sur X :
            <div class="cap09kgrad_eq_box">Z[r][c] = &Sigma; K &middot; X</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 4: Ativação A (Pós-ReLU) -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ACTIVATION A (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Résultat post-ReLU : A = max(0, Z). Si Z &le; 0, l'activation est mise à zéro. Survolez une cellule pour mettre en évidence le Z correspondant.</span>
        </div>
        <div id="cap09kgrad_gradeA"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Comment calcule A ?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Fonction ReLU :</b><br>
            <div class="cap09kgrad_eq_box">A[r][c] = max(0, Z[r][c])</div>
            <b>Somme Globale (S) :</b><br>
            <div class="cap09kgrad_eq_box">S = &Sigma; A[r][c]</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 5: Erro dZ -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ERREUR dZ (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Erreur propagée : dZ = (S - cible) &middot; I(Z > 0). Où A=0, l'erreur dZ est aussi 0. Survolez une cellule pour voir le calcul complet, en mettant en évidence le Z correspondant et les 16 cellules de A qui forment S.</span>
        </div>
        <div id="cap09kgrad_gradeDZ"></div>
        <div id="cap09kgrad_textoS" class="cap09kgrad_mono" style="font-size:9.5px;color:#374151;margin-top:6px;"></div>
        <div id="cap09kgrad_textoLoss" class="cap09kgrad_mono" style="font-size:9.5px;color:#374151;margin-top:2px;"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Comment calcule dZ, S et la Perte ?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>1. Perte (Loss L) :</b><br>
            <div class="cap09kgrad_eq_box">L = &frac12; (S &minus; cible)&sup2;</div>
            <b>2. Erreur propagée dZ :</b><br>
            <div class="cap09kgrad_eq_box">dZ = (S &minus; cible) &middot; deriv_ReLU(Z)</div>
          </div>
        </details>
      </div>

    </div>

    <!-- PAINEL SECUNDÁRIO: Cálculo dos Votos e Gradiente -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:12px;">
      <div class="cap09kgrad_painel" style="flex:1;min-width:340px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">2. CALCUL ET SOMME DES "VOTES" DE CHAQUE POSITION<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Chaque position (r,c) génère un vote = dZ[r][c] &times; X[r+kr][c+kc]. La somme des 16 votes forme le gradient du poids.</span>
        </div>
        <div id="cap09kgrad_formula" style="font-size:11.5px;color:#26241D;margin-bottom:8px;line-height:1.5;"></div>
        
        <div style="display:flex;gap:6px;flex-wrap:wrap;margin-bottom:10px;">
          <button id="cap09kgrad_btnVoltarPos" class="cap09kgrad_navbtn">⏮ Reculer position</button>
          <button id="cap09kgrad_btnAvancar" class="cap09kgrad_playbtn">▶ Avancer position</button>
          <button id="cap09kgrad_btnSomarTudo" class="cap09kgrad_navbtn">Tout additionner</button>
          <button id="cap09kgrad_btnReiniciarPos" class="cap09kgrad_navbtn">↺ Réinitialiser positions</button>
        </div>

        <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">
          <div class="cap09kgrad_cartao">
            <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">CALCUL DE CETTE POSITION<span class="cap09kgrad_icone">?</span>
              <span class="cap09kgrad_tooltip">Affiche l'erreur locale (dZ) et le pixel d'entrée (X) multipliés à la position actuelle de la fenêtre glissante.</span>
            </div>
            <div id="cap09kgrad_calcAtual" class="cap09kgrad_mono" style="font-size:11px;color:#374151;line-height:1.6;"></div>
          </div>
          
          <div class="cap09kgrad_cartao">
            <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">SOMME ACCUMULÉE (GRADIENT)<span class="cap09kgrad_icone">?</span>
              <span class="cap09kgrad_tooltip">La valeur accumulée des produits dZ &times; X de toutes les positions déjà parcourues. Quand elle atteint 16/16, c'est le gradient final du poids.</span>
            </div>
            <div id="cap09kgrad_somaAtual" class="cap09kgrad_mono" style="font-size:11px;color:#374151;line-height:1.6;"></div>
            <div class="cap09kgrad_barraProgresso"><div id="cap09kgrad_barraFill" class="cap09kgrad_barraProgressoFill" style="width:0%;"></div></div>
          </div>
        </div>

        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo" style="margin-top:2px;">HISTORIQUE DES 16 POSITIONS (COLONNES c=0, c=1, c=2, c=3)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Suivez la liste des 16 positions organisées en 4 colonnes pour correspondre au mouvement de la fenêtre sur l'image de sortie.</span>
        </div>
        <div id="cap09kgrad_listaVotos" class="cap09kgrad_grid_historico"></div>
      </div>
    </div>

    <!-- PAINEL TERCIÁRIO: Atualização e Gráfico -->
    <div class="cap09kgrad_painel" style="margin-top:12px;">
      <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">3. UTILISEZ LE GRADIENT POUR METTRE À JOUR LE NOYAU<span class="cap09kgrad_icone">?</span>
        <span class="cap09kgrad_tooltip">Applique la règle de la Descente de Gradient (K &larr; K &minus; &eta; &middot; gradient) pour les 9 poids.</span>
      </div>
      <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;">
        <div style="display:flex;gap:6px;flex-wrap:wrap;">
          <button id="cap09kgrad_btnAtualizarPesos" class="cap09kgrad_playbtn">▶ Appliquer un pas de descente de gradient</button>
          <button id="cap09kgrad_btnDesfazerPasso" class="cap09kgrad_navbtn">⏮ Annuler le pas</button>
          <button id="cap09kgrad_btnNovoKernel" class="cap09kgrad_navbtn">🎲 Nouveau noyau aléatoire</button>
        </div>
        <div>
          <div class="cap09kgrad_graflabel cap09kgrad_labelinfo">PERTE AU FIL DES MISES À JOUR<span class="cap09kgrad_icone">?</span>
            <span class="cap09kgrad_tooltip">
              <b>Évolution de l'erreur L = ½(S &minus; cible)² :</b><br>
              • <b>Objectif :</b> L &rarr; 0 (S &rarr; cible).<br>
              • <b>Si bloqué à L = 40.5 :</b> Un "dépassement" (saut exagéré) s'est produit. Les poids sont devenus très négatifs, générant Z &le; 0 (mort de ReLU). Avec S = 0, la perte reste bloquée à ½(0 &minus; 9)&sup2; = 40.5.
            </span>
          </div>
          <canvas id="cap09kgrad_canvasLoss" width="220" height="75" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div id="cap09kgrad_textoEpoca" class="cap09kgrad_mono" style="font-size:10.5px;color:#8A8371;"></div>
      </div>

      <div id="cap09kgrad_notaAtualizacao" style="font-size:10.5px;color:#374151;margin-top:8px;padding-top:6px;border-top:1px dashed #E9E3D3;line-height:1.6;text-align:left;"></div>
    </div>

  </div>
</div>

<script>
(function cap09kgrad_scope(){
  function cap09kgrad_clamp01(v){ return Math.max(0, Math.min(1, v)); }
  function cap09kgrad_relu(x){ return Math.max(0, x); }
  function cap09kgrad_reluDeriv(x){ return x > 0 ? 1 : 0; }

  function cap09kgrad_criarRng(seed){
    return function(){
      seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
      var t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
      t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
      return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    };
  }

  var cap09kgrad_PADROES = {
    diagonal: [
      [1,0,0,0,0,0],[1,1,0,0,0,0],[1,1,1,0,0,0],
      [1,1,1,1,0,0],[1,1,1,1,1,0],[1,1,1,1,1,1]
    ],
    vertical: [
      [1,1,1,0,0,0],[1,1,1,0,0,0],[1,1,1,0,0,0],
      [1,1,1,0,0,0],[1,1,1,0,0,0],[1,1,1,0,0,0]
    ]
  };
  var cap09kgrad_ALVOS = { diagonal: 9, vertical: 8 };

  function cap09kgrad_forward(K, X){
    var Z = [], A = [];
    for (var r = 0; r < 4; r++){
      var zr = [], ar = [];
      for (var c = 0; c < 4; c++){
        var s = 0;
        for (var kr = 0; kr < 3; kr++)
          for (var kc = 0; kc < 3; kc++)
            s += X[r+kr][c+kc] * K[kr][kc];
        zr.push(s); ar.push(cap09kgrad_relu(s));
      }
      Z.push(zr); A.push(ar);
    }
    var S = 0;
    for (var r2 = 0; r2 < 4; r2++)
      for (var c2 = 0; c2 < 4; c2++) S += A[r2][c2];
    return { Z: Z, A: A, S: S };
  }

  function cap09kgrad_inicializarKernelVivo(startSeed, X){
    var seedAtual = startSeed;
    var K, cache;
    for (var tentativas = 0; tentativas < 500; tentativas++){
      var rng = cap09kgrad_criarRng(seedAtual);
      K = [];
      for (var r = 0; r < 3; r++){
        var row = [];
        for (var c = 0; c < 3; c++) row.push(Number(((rng() - 0.5)).toFixed(2)));
        K.push(row);
      }
      cache = cap09kgrad_forward(K, X);
      if (cache.S > 0) return K;
      seedAtual++;
    }
    return K;
  }

  function cap09kgrad_calcularErro(cache, alvo){
    var dS = cache.S - alvo;
    var dZ = [];
    for (var r = 0; r < 4; r++){
      var row = [];
      for (var c = 0; c < 4; c++) row.push(dS * cap09kgrad_reluDeriv(cache.Z[r][c]));
      dZ.push(row);
    }
    var loss = 0.5 * dS * dS;
    return { dZ: dZ, loss: loss, dS: dS };
  }

  function cap09kgrad_gradientePeso(dZ, X, kr0, kc0){
    var votos = [];
    var soma = 0;
    for (var r = 0; r < 4; r++){
      for (var c = 0; c < 4; c++){
        var xVal = X[r+kr0][c+kc0];
        var dVal = dZ[r][c];
        var voto = dVal * xVal;
        soma += voto;
        votos.push({ r: r, c: c, x: xVal, dz: dVal, voto: voto, somaParcial: soma });
      }
    }
    return { votos: votos, gradiente: soma };
  }

  function cap09kgrad_gradienteKernelCompleto(dZ, X){
    var gK = [[0,0,0],[0,0,0],[0,0,0]];
    for (var kr = 0; kr < 3; kr++)
      for (var kc = 0; kc < 3; kc++)
        gK[kr][kc] = cap09kgrad_gradientePeso(dZ, X, kr, kc).gradiente;
    return gK;
  }

  function cap09kgrad_atualizarKernel(K, gK, lr){
    var novo = [[0,0,0],[0,0,0],[0,0,0]];
    for (var kr = 0; kr < 3; kr++)
      for (var kc = 0; kc < 3; kc++)
        novo[kr][kc] = K[kr][kc] - lr * gK[kr][kc];
    return novo;
  }

  function cap09kgrad_corValor(v, maxAbs){
    var m = maxAbs || 1;
    var t = cap09kgrad_clamp01(Math.abs(v) / m);
    if (v >= 0){
      var g = Math.round(230 - t*90);
      return "rgb(" + Math.round(235-t*120) + "," + g + "," + Math.round(220-t*90) + ")";
    } else {
      var r2 = Math.round(252 - t*20);
      return "rgb(" + r2 + "," + Math.round(232-t*130) + "," + Math.round(230-t*130) + ")";
    }
  }

  function cap09kgrad_initSim(root){
    if (!root || root.dataset.cap09kgradInit) return;
    root.dataset.cap09kgradInit = "1";

    var padraoAtual = "diagonal";
    var X = cap09kgrad_PADROES[padraoAtual];
    var alvo = cap09kgrad_ALVOS[padraoAtual];
    var lr = 0.002;

    var K = cap09kgrad_inicializarKernelVivo(7, X);
    var historicoKernel = [];

    var cacheForward = cap09kgrad_forward(K, X);
    var cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);

    var pesoSelKr = 0, pesoSelKc = 0;
    var posicaoIdx = 0;
    var votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
    var historicoLoss = [cacheErro.loss];
    var epocaAtual = 0;
    var ultimoGradiente = null;
    var ultimoKAntigo = null;

    // ---- Estado do sistema de hover (contas + destaque entre camadas) ----
    var celulasDestacadas = [];
    var elHoverTip = null;

    var elBtnDiagonal = root.querySelector('#cap09kgrad_btnDiagonal');
    var elBtnVertical = root.querySelector('#cap09kgrad_btnVertical');
    var elLr0005 = root.querySelector('#cap09kgrad_lr0005');
    var elLr002 = root.querySelector('#cap09kgrad_lr002');
    var elLr02 = root.querySelector('#cap09kgrad_lr02');
    var elGradeX = root.querySelector('#cap09kgrad_gradeX');
    var elGradeZ = root.querySelector('#cap09kgrad_gradeZ');
    var elGradeA = root.querySelector('#cap09kgrad_gradeA');
    var elGradeDZ = root.querySelector('#cap09kgrad_gradeDZ');
    var elGradeK = root.querySelector('#cap09kgrad_gradeK');
    var elSeletorPeso = root.querySelector('#cap09kgrad_seletorPeso');
    var elFormula = root.querySelector('#cap09kgrad_formula');
    var elListaVotos = root.querySelector('#cap09kgrad_listaVotos');
    var elCalcAtual = root.querySelector('#cap09kgrad_calcAtual');
    var elSomaAtual = root.querySelector('#cap09kgrad_somaAtual');
    var elBarraFill = root.querySelector('#cap09kgrad_barraFill');
    var elBtnVoltarPos = root.querySelector('#cap09kgrad_btnVoltarPos');
    var elBtnAvancar = root.querySelector('#cap09kgrad_btnAvancar');
    var elBtnSomarTudo = root.querySelector('#cap09kgrad_btnSomarTudo');
    var elBtnReiniciarPos = root.querySelector('#cap09kgrad_btnReiniciarPos');
    var elBtnAtualizarPesos = root.querySelector('#cap09kgrad_btnAtualizarPesos');
    var elBtnDesfazerPasso = root.querySelector('#cap09kgrad_btnDesfazerPasso');
    var elBtnNovoKernel = root.querySelector('#cap09kgrad_btnNovoKernel');
    var elTextoS = root.querySelector('#cap09kgrad_textoS');
    var elTextoLoss = root.querySelector('#cap09kgrad_textoLoss');
    var elTextoEpoca = root.querySelector('#cap09kgrad_textoEpoca');
    var elCanvasLoss = root.querySelector('#cap09kgrad_canvasLoss');
    var ctxLoss = elCanvasLoss.getContext('2d');
    var elNotaAtualizacao = root.querySelector('#cap09kgrad_notaAtualizacao');

    function montarSeletorPeso(){
      elSeletorPeso.innerHTML = '';
      elSeletorPeso.style.display = 'grid';
      elSeletorPeso.style.gridTemplateColumns = 'repeat(3, 30px)';
      elSeletorPeso.style.gap = '3px';
      for (var kr = 0; kr < 3; kr++){
        for (var kc = 0; kc < 3; kc++){
          (function(kr2, kc2){
            var bt = document.createElement('button');
            bt.className = 'cap09kgrad_pesobtn';
            bt.textContent = 'K[' + kr2 + '][' + kc2 + ']';
            bt.style.fontSize = '7.5px';
            bt.addEventListener('click', function(){ selecionarPeso(kr2, kc2); });
            elSeletorPeso.appendChild(bt);
          })(kr, kc);
        }
      }
    }

    function estiloCelula(el, ativo, corFundo, tam){
      var t = tam || '24px';
      el.style.width = t; el.style.height = t;
      el.style.display = 'flex'; el.style.alignItems = 'center'; el.style.justifyContent = 'center';
      el.style.fontSize = '7.5px'; el.style.fontWeight = '700'; el.style.borderRadius = '4px';
      el.style.background = corFundo;
      el.style.border = ativo ? '2px solid #2F6F9F' : '1px solid #E4DCC8';
      el.style.boxSizing = 'border-box';
    }

    function desenharGradeX(){
      elGradeX.innerHTML = '';
      elGradeX.style.display = 'grid';
      elGradeX.style.gridTemplateColumns = 'repeat(6, 24px)';
      elGradeX.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var janelaAtiva = posicaoIdx < 16;
      for (var rr = 0; rr < 6; rr++){
        for (var cc = 0; cc < 6; cc++){
          var cel = document.createElement('div');
          var v = X[rr][cc];
          var cor = v > 0 ? '#EFE9D8' : '#FFFFFF';
          var dentroJanela = janelaAtiva && rr >= r && rr <= r+2 && cc >= c && cc <= c+2;
          var ehPixelDoVoto = janelaAtiva && rr === r+pesoSelKr && cc === c+pesoSelKc;
          estiloCelula(cel, false, cor, '24px');
          if (dentroJanela){ cel.style.border = '1px solid #B8AE94'; }
          if (ehPixelDoVoto){ cel.style.border = '2px solid #2F6F9F'; cel.style.background = '#DCEEFB'; }
          cel.textContent = v; cel.style.color = '#5E5A4A';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeX.appendChild(cel);
        }
      }
    }

    function desenharGradeZ(){
      elGradeZ.innerHTML = '';
      elGradeZ.style.display = 'grid';
      elGradeZ.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeZ.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheForward.Z[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheForward.Z[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #2F6F9F'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeZ.appendChild(cel);
        }
      }
    }

    function desenharGradeA(){
      elGradeA.innerHTML = '';
      elGradeA.style.display = 'grid';
      elGradeA.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeA.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheForward.A[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheForward.A[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #2F6F9F'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeA.appendChild(cel);
        }
      }
    }

    function desenharGradeDZ(){
      elGradeDZ.innerHTML = '';
      elGradeDZ.style.display = 'grid';
      elGradeDZ.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeDZ.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheErro.dZ[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheErro.dZ[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #C1443A'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeDZ.appendChild(cel);
        }
      }
    }

    function desenharGradeK(){
      elGradeK.innerHTML = '';
      elGradeK.style.display = 'grid';
      elGradeK.style.gridTemplateColumns = 'repeat(3, 30px)';
      elGradeK.style.gap = '3px';
      var maxAbs = 0;
      for (var i=0;i<3;i++) for (var j=0;j<3;j++) maxAbs = Math.max(maxAbs, Math.abs(K[i][j]));
      maxAbs = maxAbs || 1;
      for (var kr = 0; kr < 3; kr++){
        for (var kc = 0; kc < 3; kc++){
          var cel = document.createElement('div');
          var ativo = (kr === pesoSelKr && kc === pesoSelKc);
          estiloCelula(cel, ativo, cap09kgrad_corValor(K[kr][kc], maxAbs), '30px');
          cel.textContent = K[kr][kc].toFixed(2); cel.style.color = '#374151';
          cel.dataset.kr = kr; cel.dataset.kc = kc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeK.appendChild(cel);
        }
      }
    }

    // ================= SISTEMA DE HOVER: contas detalhadas + destaque entre camadas =================

    function cap09kgrad_criarTooltipGlobal(){
      if (elHoverTip) return elHoverTip;
      elHoverTip = document.createElement('div');
      elHoverTip.className = 'cap09kgrad_hovertip';
      document.body.appendChild(elHoverTip);
      return elHoverTip;
    }

    function cap09kgrad_posicionarTooltip(evt){
      var tip = elHoverTip;
      if (!tip) return;
      var margem = 14;
      var x = evt.clientX + margem;
      var y = evt.clientY + margem;
      var larguraTip = tip.offsetWidth, alturaTip = tip.offsetHeight;
      if (x + larguraTip > window.innerWidth - 8) x = evt.clientX - larguraTip - margem;
      if (y + alturaTip > window.innerHeight - 8) y = evt.clientY - alturaTip - margem;
      tip.style.left = x + 'px';
      tip.style.top = y + 'px';
    }

    function cap09kgrad_mostrarTooltip(htmlConteudo, evt){
      var tip = cap09kgrad_criarTooltipGlobal();
      tip.innerHTML = htmlConteudo;
      tip.style.display = 'block';
      cap09kgrad_posicionarTooltip(evt);
    }

    function cap09kgrad_esconderTooltip(){
      if (elHoverTip) elHoverTip.style.display = 'none';
    }

    function cap09kgrad_limparDestaques(){
      celulasDestacadas.forEach(function(item){ item.el.classList.remove(item.classe); });
      celulasDestacadas = [];
    }

    function cap09kgrad_destacar(elementos, classe){
      elementos.forEach(function(el){
        if (!el) return;
        el.classList.add(classe);
        celulasDestacadas.push({ el: el, classe: classe });
      });
    }

    function cap09kgrad_celX(rr,cc){ return elGradeX.querySelector('[data-r="'+rr+'"][data-c="'+cc+'"]'); }
    function cap09kgrad_celZ(r,c){ return elGradeZ.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celA(r,c){ return elGradeA.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celDZ(r,c){ return elGradeDZ.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celK(kr,kc){ return elGradeK.querySelector('[data-kr="'+kr+'"][data-kc="'+kc+'"]'); }
    function cap09kgrad_todasCelA(){ return Array.prototype.slice.call(elGradeA.querySelectorAll('[data-r]')); }

    function cap09kgrad_hoverZ(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      var termos = [];
      for (var kr=0; kr<3; kr++){
        for (var kc=0; kc<3; kc++){
          var xv = X[r+kr][c+kc];
          var kv = K[kr][kc];
          termos.push({ kr:kr, kc:kc, x:xv, k:kv, t:kv*xv });
          cap09kgrad_destacar([cap09kgrad_celX(r+kr, c+kc)], 'cap09kgrad_hlSecundario');
          cap09kgrad_destacar([cap09kgrad_celK(kr, kc)], 'cap09kgrad_hlSecundario');
        }
      }
      var linhas = termos.map(function(t){
        return 'K['+t.kr+']['+t.kc+']&middot;X['+(r+t.kr)+']['+(c+t.kc)+'] = '+t.k.toFixed(2)+'&times;'+t.x+' = <b>'+t.t.toFixed(3)+'</b>';
      }).join('<br>');
      var html =
        '<b>Z['+r+']['+c+']</b> = &Sigma; K &middot; X (9 termos)<br>' +
        '<div style="margin:4px 0;border-top:1px dashed #4A473A;padding-top:4px;">' + linhas + '</div>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">total = <b>'+cacheForward.Z[r][c].toFixed(3)+'</b></div>' +
        '<div style="margin-top:4px;color:#B8AE94;">contorno tracejado = janela em X e pesos em K usados</div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverA(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlPrincipal');
      var zv = cacheForward.Z[r][c], av = cacheForward.A[r][c];
      var html =
        '<b>A['+r+']['+c+']</b> = max(0, Z['+r+']['+c+'])<br>' +
        '= max(0, '+zv.toFixed(3)+') = <b>'+av.toFixed(3)+'</b>' +
        (zv <= 0 ? '<br><span style="color:#FF9A93;">ReLU zerou este valor.</span>' : '');
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverDZ(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar(cap09kgrad_todasCelA(), 'cap09kgrad_hlSecundario');
      var zv = cacheForward.Z[r][c];
      var gate = cap09kgrad_reluDeriv(zv);
      var dS = cacheErro.dS;
      var dz = cacheErro.dZ[r][c];
      var html =
        '<b>dZ['+r+']['+c+']</b> = (S &minus; alvo) &middot; ReLU&prime;(Z['+r+']['+c+'])<br>' +
        'S = &Sigma; A (16 células, tracejado) = <b>'+cacheForward.S.toFixed(3)+'</b><br>' +
        'S &minus; alvo = '+cacheForward.S.toFixed(3)+' &minus; '+alvo+' = <b>'+dS.toFixed(3)+'</b><br>' +
        'ReLU&prime;(Z) = ' + (gate ? '1 (Z&gt;0)' : '0 (Z&le;0)') + '<br>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">dZ = '+dS.toFixed(3)+' &times; '+gate+' = <b>'+dz.toFixed(3)+'</b></div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverX(rr, cc, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      var contribs = [];
      for (var kr=0; kr<3; kr++){
        for (var kc=0; kc<3; kc++){
          var r = rr-kr, c = cc-kc;
          if (r>=0 && r<4 && c>=0 && c<4){
            contribs.push({ r:r, c:c, kr:kr, kc:kc, k:K[kr][kc] });
            cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlSecundario');
            cap09kgrad_destacar([cap09kgrad_celK(kr,kc)], 'cap09kgrad_hlSecundario');
          }
        }
      }
      var xv = X[rr][cc];
      var linhas = contribs.map(function(t){
        var termo = t.k*xv;
        return 'Z['+t.r+']['+t.c+'] usa K['+t.kr+']['+t.kc+']&times;X = '+t.k.toFixed(2)+'&times;'+xv+' = <b>'+termo.toFixed(3)+'</b>';
      }).join('<br>');
      var html =
        '<b>X['+rr+']['+cc+']</b> = '+xv+'<br>' +
        (contribs.length ?
          '<div style="margin:4px 0;border-top:1px dashed #4A473A;padding-top:4px;">Usado em '+contribs.length+' posição(ões) de Z:<br>'+linhas+'</div>'
          : '<span style="color:#B8AE94;">Fora do alcance de qualquer janela 3&times;3 sobre a saída atual.</span>');
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverK(kr, kc, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      for (var r=0; r<4; r++){
        for (var c=0; c<4; c++){
          cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlSecundario');
          cap09kgrad_destacar([cap09kgrad_celX(r+kr,c+kc)], 'cap09kgrad_hlSecundario');
        }
      }
      var gAtual = cap09kgrad_gradientePeso(cacheErro.dZ, X, kr, kc).gradiente;
      var html =
        '<b>K['+kr+']['+kc+']</b> = '+K[kr][kc].toFixed(3)+'<br>' +
        'Participa das 16 posições de Z (destacadas), cada uma multiplicando um pixel X diferente (também destacado).<br>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">Gradiente atual &nabla;K = &Sigma; dZ&middot;X = <b>'+gAtual.toFixed(3)+'</b></div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_configurarHover(container, seletor, manipulador){
      container.addEventListener('mouseover', function(e){
        var cel = e.target.closest(seletor);
        if (!cel || cel.parentElement !== container) return;
        manipulador(cel, e);
      });
      container.addEventListener('mousemove', function(e){
        var cel = e.target.closest(seletor);
        if (!cel || cel.parentElement !== container) return;
        cap09kgrad_posicionarTooltip(e);
      });
      container.addEventListener('mouseout', function(e){
        var cel = e.target.closest(seletor);
        if (!cel) return;
        var indoPara = e.relatedTarget;
        if (indoPara && cel.contains(indoPara)) return;
        cap09kgrad_limparDestaques();
        cap09kgrad_esconderTooltip();
      });
    }

    // ================= FIM DO SISTEMA DE HOVER =================

    function atualizarFormula(){
      elFormula.innerHTML =
        '<span class="cap09kgrad_mono">&part;L/&part;K[' + pesoSelKr + '][' + pesoSelKc + ']</span> = &Sigma;<sub>(r,c)</sub> ' +
        '<span class="cap09kgrad_mono">dZ[r][c] &middot; X[r+' + pesoSelKr + '][c+' + pesoSelKc + ']</span>';
    }

    function renderizarListaVotos(){
      elListaVotos.innerHTML = '';
      var elementoAtivo = null;
      var idxAtivo = posicaoIdx - 1;

      votosAtuais.forEach(function(v, idx){
        var linha = document.createElement('div');
        linha.className = 'cap09kgrad_linhavoto';
        var visivel = idx < posicaoIdx;
        linha.style.opacity = visivel ? '1' : '0.25';
        if (idx === idxAtivo){
          linha.style.background = '#EAF2FA';
          elementoAtivo = linha;
        }
        linha.innerHTML =
          '<span class="cap09kgrad_mono" style="color:#8A8371;">p(' + v.r + ',' + v.c + ')</span> ' +
          '<span class="cap09kgrad_mono">' + v.dz.toFixed(2) + '</span>&times;' +
          '<span class="cap09kgrad_mono">' + v.x + '</span>=' +
          '<span class="cap09kgrad_mono" style="color:' + (v.voto>=0 ? '#1E8F6F' : '#C1443A') + ';font-weight:700;">' + v.voto.toFixed(2) + '</span>';
        elListaVotos.appendChild(linha);
      });

      if (elementoAtivo){
        var alvoTop = elementoAtivo.offsetTop - (elListaVotos.clientHeight/2) + (elementoAtivo.clientHeight/2);
        elListaVotos.scrollTop = Math.max(0, alvoTop);
      } else {
        elListaVotos.scrollTop = 0;
      }
    }

    function atualizarCalcAtual(){
      if (posicaoIdx === 0){
        elCalcAtual.innerHTML =
          '<span style="color:#8A8371;">Clique em <b>"Avançar posição"</b> para ver a primeira conta.</span>';
        return;
      }
      var atual = votosAtuais[posicaoIdx - 1];
      var corVoto = atual.voto >= 0 ? '#1E8F6F' : '#C1443A';
      elCalcAtual.innerHTML =
        'posição <b>(' + atual.r + ',' + atual.c + ')</b><br>' +
        'dZ&nbsp;&nbsp;= <b>' + atual.dz.toFixed(3) + '</b><br>' +
        'X&nbsp;&nbsp;&nbsp;&nbsp;= <b>' + atual.x + '</b><br>' +
        '<span style="border-top:1px dashed #E4DCC8;display:block;margin:4px 0 2px;"></span>' +
        'voto = dZ &times; X = <b style="color:' + corVoto + ';">' + atual.voto.toFixed(3) + '</b>';
    }

    function atualizarSomaTexto(){
      var somaParcial = posicaoIdx > 0 ? votosAtuais[posicaoIdx-1].somaParcial : 0;
      var completo = posicaoIdx >= 16;

      elSomaAtual.innerHTML =
        'posições: <b>' + posicaoIdx + ' / 16</b><br>' +
        '<span style="border-top:1px dashed #E4DCC8;display:block;margin:4px 0 2px;"></span>' +
        'total = <b style="color:#2F6F9F;">' + somaParcial.toFixed(3) + '</b>' +
        (completo ? '<br><span style="color:#1E8F6F;font-weight:700;">✓ gradiente completo</span>' : '');

      elBarraFill.style.width = (posicaoIdx/16*100).toFixed(1) + '%';
      elBtnVoltarPos.disabled = (posicaoIdx === 0);
      elBtnAvancar.disabled = completo;
      elBtnSomarTudo.disabled = completo;
      elBtnDesfazerPasso.disabled = (historicoKernel.length === 0);
    }

    function desenharGraficoLoss(){
      var Wc = elCanvasLoss.width, Hc = elCanvasLoss.height;
      ctxLoss.clearRect(0,0,Wc,Hc);
      ctxLoss.strokeStyle = '#E4DCC8'; ctxLoss.lineWidth = 1;
      ctxLoss.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoLoss.length < 2){
        ctxLoss.fillStyle = '#B8AE94';
        ctxLoss.font = '10px Inter, sans-serif';
        ctxLoss.textAlign = 'center';
        ctxLoss.fillText('perda aparecerá aqui', Wc/2, Hc/2+3);
        return;
      }

      var maxLoss = Math.max.apply(null, historicoLoss);
      maxLoss = Math.max(maxLoss, 0.01);

      var padL = 30, padR = 8, padT = 10, padB = 14;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxLoss.strokeStyle = '#EDE7D6';
      ctxLoss.lineWidth = 1;
      ctxLoss.beginPath();
      ctxLoss.moveTo(padL, padT); ctxLoss.lineTo(Wc - padR, padT);
      ctxLoss.moveTo(padL, Hc - padB); ctxLoss.lineTo(Wc - padR, Hc - padB);
      ctxLoss.stroke();

      ctxLoss.strokeStyle = '#1E8F6F';
      ctxLoss.setLineDash([2, 2]);
      ctxLoss.beginPath();
      ctxLoss.moveTo(padL, Hc - padB); ctxLoss.lineTo(Wc - padR, Hc - padB);
      ctxLoss.stroke();
      ctxLoss.setLineDash([]);

      ctxLoss.fillStyle = '#8A8371';
      ctxLoss.font = '8px JetBrains Mono, monospace';
      ctxLoss.textAlign = 'right';
      ctxLoss.fillText(maxLoss.toFixed(1), padL - 3, padT + 3);
      ctxLoss.fillText('0.0', padL - 3, Hc - padB + 2);

      ctxLoss.strokeStyle = '#2F5FA8'; 
      ctxLoss.lineWidth = 1.6;
      ctxLoss.beginPath();
      historicoLoss.forEach(function(v, i){
        var x = padL + (historicoLoss.length === 1 ? 0 : (i / (historicoLoss.length - 1)) * plotW);
        var y = padT + (1 - v / maxLoss) * plotH;
        if (i === 0) ctxLoss.moveTo(x, y); else ctxLoss.lineTo(x, y);
      });
      ctxLoss.stroke();
    }

    function renderizarNotaAtualizacao(){
      var gAtual = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).gradiente;
      var kr = pesoSelKr, kc = pesoSelKc;
      var kAtual = K[kr][kc];
      var passoPrevisto = lr * gAtual;
      var kNovoPrevisto = kAtual - passoPrevisto;

      if (cacheForward.S <= 0){
        elNotaAtualizacao.innerHTML = 
          '<div style="background:#FDF2F2;border:1px solid #F87171;border-radius:8px;padding:8px 10px;color:#991B1B;">' +
          '<b>⚡ Colapso por ReLU Morta (S = 0.000):</b><br>' +
          'Todas as ativações Z &le; 0 foram zeradas pela ReLU. O erro dZ = 0, zerando o gradiente (&nabla;K = 0).<br>' +
          '• A perda travou em L = &frac12;&middot;(0 &minus; ' + alvo + ')&sup2; = ' + cacheErro.loss.toFixed(1) + '.<br>' +
          '• <b>Como resolver:</b> Reduza a taxa &eta;, clique em <b>"⏮ Desfazer passo"</b> ou em <b>"🎲 Novo kernel aleatório"</b>.' +
          '</div>';
        return;
      }

      if (ultimoGradiente === null){
        elNotaAtualizacao.innerHTML = 
          '<div style="text-align:left;">' +
          '<b>Estado Inicial do peso K[' + kr + '][' + kc + '] = ' + kAtual.toFixed(3) + ':</b><br>' +
          '• <b>Gradiente visível acumulado (&nabla;K):</b> <b style="color:#2F6F9F;">' + gAtual.toFixed(3) + '</b> (soma dos 16 votos dZ &times; X)<br>' +
          '• <b>Passo de ajuste previsto (&eta; &times; &nabla;K):</b> ' + lr + ' &times; (' + gAtual.toFixed(3) + ') = <b>' + passoPrevisto.toFixed(4) + '</b><br>' +
          '• <b>Novo Peso previsto:</b> K &larr; ' + kAtual.toFixed(3) + ' &minus; (' + passoPrevisto.toFixed(4) + ') = <b>' + kNovoPrevisto.toFixed(3) + '</b><br>' +
          '<br><span style="color:#8A8371;">Clique em <b>"▶ Aplicar passo de gradiente descendente"</b> para atualizar a matriz.</span>' +
          '</div>';
        return;
      }

      var gValAnterior = ultimoGradiente[kr][kc];
      var kAntigoVal = ultimoKAntigo[kr][kc];
      var passoAplicado = lr * gValAnterior;

      elNotaAtualizacao.innerHTML = 
        '<div style="text-align:left;">' +
        '<b>Última atualização aplicada ao peso K[' + kr + '][' + kc + ']:</b><br>' +
        '1. <b>Taxa (&eta;):</b> <span class="cap09kgrad_mono">' + lr + '</span> | <b>&nabla;K aplicado:</b> <span class="cap09kgrad_mono">' + gValAnterior.toFixed(3) + '</span><br>' +
        '2. <b>Passo executado:</b> ' + lr + ' &times; (' + gValAnterior.toFixed(3) + ') = <b>' + passoAplicado.toFixed(4) + '</b><br>' +
        '3. <b>Resultado:</b> K &larr; ' + kAntigoVal.toFixed(3) + ' &minus; (' + passoAplicado.toFixed(4) + ') = <b>' + kAtual.toFixed(3) + '</b><br>' +
        '<br><span style="color:#2F6F9F;"><b>Novo gradiente pronto na matriz atual:</b> &nabla;K = <b>' + gAtual.toFixed(3) + '</b></span>' +
        '</div>';
    }

    function renderizarTudo(){
      cap09kgrad_limparDestaques();
      cap09kgrad_esconderTooltip();
      desenharGradeX();
      desenharGradeZ();
      desenharGradeA();
      desenharGradeDZ();
      desenharGradeK();
      atualizarFormula();
      renderizarListaVotos();
      atualizarCalcAtual();
      atualizarSomaTexto();
      desenharGraficoLoss();
      renderizarNotaAtualizacao();

      var valS = cacheForward.S.toFixed(3);
      var valLoss = cacheErro.loss.toFixed(4);

      elTextoS.innerHTML = 'S = <b>' + valS + '</b> (alvo = ' + alvo + ')';
      elTextoLoss.innerHTML = 'L = &frac12;&middot;(' + valS + ' &minus; ' + alvo + ')&sup2; = <b>' + valLoss + '</b>';
      elTextoEpoca.textContent = 'Mises à jour : ' + epocaAtual;
    }

    function selecionarPeso(kr, kc){
      pesoSelKr = kr; pesoSelKc = kc;
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, kr, kc).votos;
      var botoes = elSeletorPeso.querySelectorAll('.cap09kgrad_pesobtn');
      botoes.forEach(function(b){ b.classList.remove('cap09kgrad_pesobtn_ativo'); });
      var idxBotao = kr*3+kc;
      if (botoes[idxBotao]) botoes[idxBotao].classList.add('cap09kgrad_pesobtn_ativo');
      renderizarTudo();
    }

    function avancarPosicao(){
      if (posicaoIdx < 16) posicaoIdx += 1;
      renderizarTudo();
    }

    function voltarPosicao(){
      if (posicaoIdx > 0) posicaoIdx -= 1;
      renderizarTudo();
    }

    function somarTudoAutomatico(){
      posicaoIdx = 16;
      renderizarTudo();
    }

    function reiniciarPosicoes(){
      posicaoIdx = 0;
      renderizarTudo();
    }

    function atualizarPesosDoKernel(){
      var gK = cap09kgrad_gradienteKernelCompleto(cacheErro.dZ, X);
      historicoKernel.push({
        K: JSON.parse(JSON.stringify(K)),
        gK: ultimoGradiente,
        kAnt: ultimoKAntigo
      });

      ultimoKAntigo = JSON.parse(JSON.stringify(K));
      ultimoGradiente = gK;

      K = cap09kgrad_atualizarKernel(K, gK, lr);
      epocaAtual += 1;

      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;

      renderizarTudo();
    }

    function desfazerPasso(){
      if (historicoKernel.length === 0) return;
      var estadoAnt = historicoKernel.pop();
      K = estadoAnt.K;
      ultimoGradiente = estadoAnt.gK;
      ultimoKAntigo = estadoAnt.kAnt;
      historicoLoss.pop();
      epocaAtual -= 1;

      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function novoKernelAleatorio(){
      K = cap09kgrad_inicializarKernelVivo(Date.now() % 2147483647, X);
      historicoKernel = [];
      epocaAtual = 0;
      ultimoGradiente = null;
      ultimoKAntigo = null;
      historicoLoss = [];
      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function trocarPadrao(novoPadrao, botaoAtivo){
      [elBtnDiagonal, elBtnVertical].forEach(function(b){ b.classList.remove('cap09kgrad_active'); });
      botaoAtivo.classList.add('cap09kgrad_active');
      padraoAtual = novoPadrao;
      X = cap09kgrad_PADROES[padraoAtual];
      alvo = cap09kgrad_ALVOS[padraoAtual];

      var checagem = cap09kgrad_forward(K, X);
      if (checagem.S <= 0){
        K = cap09kgrad_inicializarKernelVivo(7, X);
      }

      historicoKernel = [];
      epocaAtual = 0;
      ultimoGradiente = null;
      ultimoKAntigo = null;
      historicoLoss = [];
      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function alterarLr(novaLr, btnAtivo){
      [elLr0005, elLr002, elLr02].forEach(function(b){ b.classList.remove('cap09kgrad_active'); });
      btnAtivo.classList.add('cap09kgrad_active');
      lr = novaLr;
      renderizarTudo();
    }

    elBtnDiagonal.addEventListener('click', function(){ trocarPadrao('diagonal', elBtnDiagonal); });
    elBtnVertical.addEventListener('click', function(){ trocarPadrao('vertical', elBtnVertical); });
    elLr0005.addEventListener('click', function(){ alterarLr(0.0005, elLr0005); });
    elLr002.addEventListener('click', function(){ alterarLr(0.002, elLr002); });
    elLr02.addEventListener('click', function(){ alterarLr(0.02, elLr02); });

    elBtnAvancar.addEventListener('click', avancarPosicao);
    elBtnVoltarPos.addEventListener('click', voltarPosicao);
    elBtnSomarTudo.addEventListener('click', somarTudoAutomatico);
    elBtnReiniciarPos.addEventListener('click', reiniciarPosicoes);
    elBtnAtualizarPesos.addEventListener('click', atualizarPesosDoKernel);
    elBtnDesfazerPasso.addEventListener('click', desfazerPasso);
    elBtnNovoKernel.addEventListener('click', novoKernelAleatorio);

    cap09kgrad_configurarHover(elGradeZ, '[data-r]', function(cel, e){
      cap09kgrad_hoverZ(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeA, '[data-r]', function(cel, e){
      cap09kgrad_hoverA(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeDZ, '[data-r]', function(cel, e){
      cap09kgrad_hoverDZ(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeX, '[data-r]', function(cel, e){
      cap09kgrad_hoverX(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeK, '[data-kr]', function(cel, e){
      cap09kgrad_hoverK(parseInt(cel.dataset.kr,10), parseInt(cel.dataset.kc,10), cel, e);
    });

    montarSeletorPeso();
    selecionarPeso(0,0);
  }

  function cap09kgrad_tentarIniciar(){
    var root = document.getElementById('sim-09-gradiente-kernel');
    if (root) cap09kgrad_initSim(root); else setTimeout(cap09kgrad_tentarIniciar, 200);
  }
  cap09kgrad_tentarIniciar();
})();
</script>
'''
)

**Figure 9.12:** Simulateur interactif du calcul du gradient d


<figure id="fig-09-sim-09-gradiente-kernel">
  <img src="imagens/fig-09-sim-09-gradiente-kernel.png" alt=" Simulateur interactif du calcul du gradient d'un poids du *kernel* convolutif. " style="max-width:80%" />
  <figcaption><strong>Figure 9.12:</strong>  Simulateur interactif du calcul du gradient d'un poids du *kernel* convolutif. </figcaption>
</figure>

> ### 📝 🧠 Synthèse — De la convolution à l'apprentissage de représentations
>
> Les simulateurs de cette section démontrent, de manière séquentielle, comment un CNN transforme une image d'entrée en une estimation probabiliste et comment ses paramètres sont optimisés lors de l'entraînement :
>
> - **Convolution :** applique des filtres sur l'image pour extraire des caractéristiques locales, générant des cartes de caractéristiques grâce au **partage des poids**.
> - **ReLU :** introduit une non-linéarité dans le système, permettant la modélisation de relations complexes entre les données.
> - ***Pooling* :** réduit la résolution spatiale des cartes de caractéristiques, diminuant le coût computationnel et conférant une invariance aux petites translations locales.
> - ***Flatten* :** réorganise les cartes multidimensionnelles en un vecteur unidimensionnel pour alimenter les couches suivantes.
> - **Couche entièrement connectée :** combine les caractéristiques extraites pour produire les scores bruts (*logits*) associés à chaque classe.
> - **Softmax :** convertit les *logits* en une distribution de probabilités normalisée.
> - **Fonction de perte :** compare la distribution prédite avec la vérité terrain (*ground truth*), quantifiant scalairement l'erreur du réseau.
> - **Rétropropagation :** applique la règle de la chaîne pour calculer la dérivée partielle (gradient) de la fonction de perte par rapport à chaque paramètre entraînable.
> - **Optimiseur :** met à jour les coefficients des filtres, poids et biais dans la direction opposée au gradient, réduisant la perte à chaque itération.
>
> Au fil des itérations, les filtres convolutifs se transforment de valeurs stochastiques en détecteurs spécialisés : les couches initiales apprennent des primitives visuelles de bas niveau (comme les bords et les textures), tandis que les couches plus profondes consolident ces représentations en structures abstraites et sémantiques.

## 9.5 Applications Pratiques en CV

Après la consolidation théorique des fondements des CNN et la vérification visuelle de chacune de leurs opérations élémentaires au moyen des simulateurs interactifs, il devient essentiel d'observer l'intégration de ces étapes dans des *pipelines* complets de programmation.

Dans les sections suivantes, la théorie est traduite en code exécutable en **PyTorch**, explorant les trois tâches fondamentales de la CV : **classification**, **détection d'objets** et **segmentation sémantique**. Cette progression pratique permet d'analyser aussi bien la construction d'une architecture convolutive entraînée de zéro que l'application de stratégies avancées de **transfert d'apprentissage** (*transfer learning*) sur des modèles pré-entraînés pour des ensembles de données synthétiques et réels.

### 9.5.1 Classification d'images avec les CNN

La classification d'images est l'une des applications les plus traditionnelles des CNN. Dans cette tâche, l'objectif est d'attribuer un unique label à l'image d'entrée, comme identifier une catégorie d'objet, une espèce animale ou une classe de diagnostic. Pour cela, la CNN transforme progressivement les valeurs des pixels en représentations d'un niveau d'abstraction plus élevé, en combinant des couches convolutionnelles, des fonctions d'activation et des opérations de réduction spatiale jusqu'à produire une distribution de probabilités entre les classes possibles. Dans cette section, sont présentés l'architecture de base d'une CNN de classification, le flux de transformation des données à travers le réseau et le processus d'entraînement pour l'ajustement des paramètres appris.

#### 9.5.1.1 Entraînement d'un CNN de zéro sur des chiffres

Pour établir une comparaison directe avec les approches présentées au chapitre 7, on développe dans cette section un CNN entraîné sur le même ensemble de données de chiffres manuscrits (`load_digits`). La différence fondamentale réside dans l'étape de représentation : tandis que les méthodes classiques dépendent de *pixels* bruts ou de descripteurs calculés manuellement, comme l'*Histogram of Oriented Gradients* (*HOG*), le CNN apprend automatiquement les coefficients des filtres convolutifs pendant le processus d'optimisation.

Les codes suivants (consolidés dans la [Figure 9.15](#fig-09-cnn-treinamento)) effectuent la préparation des données, définissent une architecture convolutive simple en **PyTorch**, exécutent la boucle d'entraînement via l'algorithme *Adam* et génèrent les courbes d'évolution de la fonction de perte et de la précision.

##### Bloc 1 : Préparation et structuration des données

L’étape initiale de tout *pipeline* d’apprentissage profond consiste à convertir et adapter les données d’entrée au format exigé par le *framework* de calcul scientifique.

###### Le concept de *tenseur*

En apprentissage profond, la structure de données fondamentale est le ***tenseur***. D’un point de vue computationnel, un *tenseur* consiste en un arrangement multidimensionnel de nombres généralisé à $n$ dimensions :

* Un *tenseur* d’ordre 0 est un scalaire (une valeur unique).
* Un *tenseur* d’ordre 1 est un vecteur (longueur).
* Un *tenseur* d’ordre 2 est une matrice (lignes et colonnes).
* Un *tenseur* d’ordre 3 ou supérieur représente un volume ou un hyper-arrangement de données.

Dans le contexte de **PyTorch**, la classe `torch.Tensor` étend la fonctionnalité des arrangements numériques multidimensionnels (comme ceux de **NumPy**) en offrant un support pour les opérations accélérées sur matériel via les *GPU* (*Graphics Processing Units*) ainsi qu’un support pour le calcul automatique des dérivées (*autograd*), essentiel pour l’algorithme de rétropropagation.

###### Analyse du code de prétraitement

1. **Chargement et normalisation des intensités :**
   L’ensemble `load_digits` contient $1.797$ échantillons de chiffres manuscrits de $8 \times 8$ *pixels*, dont les intensités originales varient sur l’échelle entière de $0$ à $16$. La division par $16.0$ effectue la **normalisation** des données dans la plage $[0.0, 1.0]$. Cette transformation en virgule flottante (`float32`) est indispensable dans les réseaux neuronaux pour éviter la saturation des fonctions d’activation et stabiliser le calcul des gradients dans l’algorithme d’optimisation.

2. **Division stratifiée (70 % entraînement / 30 % test) :**
   La fonction `train_test_split` sépare $70\%$ des échantillons pour l’ajustement des paramètres du réseau et réserve $30\%$ pour l’évaluation du modèle sur des données non vues. Le paramètre `stratify=y` garantit un échantillonnage stratifié, en maintenant la proportion exacte de chacune des 10 classes de chiffres ($0$ à $9$) dans les deux ensembles, évitant ainsi un biais de distribution.

3. **Adaptation dimensionnelle pour la convolution 2D (`unsqueeze`) :**
   Dans les CNN, les couches convolutives bidimensionnelles (`nn.Conv2d`) exigent que le *tenseur* d’entrée possède strictement 4 dimensions dans l’ordre $(N, C, H, W)$ :
   * $N$ : nombre d’échantillons (*batch size*).
   * $C$ : nombre de canaux de couleur ($1$ pour le niveaux de gris, $3$ pour *RGB*).
   * $H$ : hauteur de l’image en *pixels* ($8$).
   * $W$ : largeur de l’image en *pixels* ($8$).

   Comme l’arrangement original possède un format $3\text{D}$ de type $(N, 8, 8)$, l’appel `.unsqueeze(1)` insère une dimension unitaire spécifiquement à l’**indice 1** (la position réservée au canal de couleur $C$), transformant la structure en un *tenseur* $4\text{D}$ de format $(N, 1, 8, 8)$, conformément à l’exigence de PyTorch.

4. **Conversion des étiquettes (`dtype=torch.long`) :**
   Les étiquettes des classes $y$ sont converties en *tenseurs* entiers de 64 *bits* (`torch.long`). Cette spécification de type est une exigence de la fonction de perte d’entropie croisée (`nn.CrossEntropyLoss`), qui utilise des entiers non négatifs comme indices pour associer la classe correcte aux *logits* de sortie du réseau.

> ### 📝 Nota
>
> **Attention aux dimensions :** La structure finale est représentée par le **tenseur `(N, 1, 8, 8)`**, où `N` est le nombre d’échantillons (*batch size*), `1` est le canal de couleur (niveaux de gris) et `8×8` est la résolution spatiale de l’image en *pixels*.

La [Figure 9.13](#fig-09-digits-amostra) illustre une séquence d’échantillons de l’ensemble d’entraînement après le prétraitement et l’adaptation dimensionnelle aux *tenseurs* de **PyTorch**. Dans l’étape d’affichage, l’appel `img.squeeze().numpy()` enchaîne deux transformations : la méthode `.squeeze()` élimine la dimension unitaire redondante du canal de couleur, réduisant le *tenseur* $3\text{D}$ de format `(1, 8, 8)` à une matrice $2\text{D}$ de `(8, 8)` ; ensuite, la méthode `.numpy()` convertit la structure de **PyTorch** en une matrice native de **NumPy**, format exigé par les outils de rendu graphique comme `mm.show()`.

In [12]:
# 1. Chargement et prétraitement des données
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0  # Normalisation à la plage [0, 1]
y = digits.target

# Division stratifiée en ensembles d'entraînement (70%) et de test (30%)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Adaptation à la dimension attendue par PyTorch : (N_échantillons, Canaux, Hauteur, Largeur)
X_treino_t = torch.tensor(X_treino).unsqueeze(1)   # Dimension : (N, 1, 8, 8)
y_treino_t = torch.tensor(y_treino, dtype=torch.long)
X_teste_t = torch.tensor(X_teste).unsqueeze(1)
y_teste_t = torch.tensor(y_teste, dtype=torch.long)

# Afficher un échantillon
n_amostras = 8
imgs = [img.squeeze().numpy() for img in X_treino_t[:n_amostras]]
imgs_titles = [str(label.item()) for label in y_treino_t[:n_amostras]]
mm.show(imgs, titles=imgs_titles, cols=n_amostras, figsize=(12, 2.5))

<Figure size 1800x375 with 8 Axes>

**Figure 9.13:** Échantillons de chiffres de l


##### Bloc 2 : Définition de l’architecture convolutionnelle

La construction de modèles dans **PyTorch** est structurée selon le paradigme de la programmation orientée objet, en créant une classe spécifique pour représenter le réseau de neurones (dans cet exemple, la classe `CNNDigitos`), qui hérite de toutes les fonctionnalités de la classe de base `nn.Module`. Le constructeur `__init__` est responsable de l’instanciation des couches et de la déclaration de leurs paramètres entraînables, tandis que la méthode `forward` établit la séquence numérique de la propagation avant (*forward pass*).

La [Figure 9.14](#fig-09-cnn-digitos-esquema) synthétise les transformations spatiales des *tensors* et le flux de données au sein de la classe `CNNDigitos`.

<figure id="fig-09-cnn-digitos-esquema" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-09-cnn-digitos-esquema.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figure 9.14:</strong> Représentation du flux des transformations dimensionnelles des *tensors* à travers l’architecture *CNNDigitos*.</figcaption>
</figure>

1. **Constructeur (`__init__`) et instanciation des composants :**
   * **Couche convolutionnelle 1 (`self.conv1`) :** Applique $8$ filtres $3 \times 3$ avec `padding=1` sur l’entrée en niveaux de gris ($1$ canal), en préservant la résolution spatiale de $8 \times 8$ *pixels*.
   * **Couche convolutionnelle 2 (`self.conv2`) :** Traite les $8$ cartes de caractéristiques reçues de la couche précédente en appliquant $16$ filtres $3 \times 3$ avec `padding=1`.
   * **Sous-échantillonnage (`self.pool`) :** Instancie l’opération de *Max-Pooling* avec une fenêtre $2 \times 2$ et un pas (*stride*) $2$, réduisant la dimension spatiale (hauteur et largeur) de moitié à chaque application.
   * **Couches entièrement connectées (`self.fc1` et `self.fc2`) :** La première projection dense reçoit le *tensor* aplati de dimension $16 \times 2 \times 2 = 64$ et produit $32$ caractéristiques intermédiaires. La seconde projette ces $32$ caractéristiques sur les $10$ *logits* finaux de sortie.

2. **Propagation avant dans la méthode `forward` :**
   * **Premier bloc convolutionnel :** Le *tensor* d’entrée au format $(N, 1, 8, 8)$ passe par `conv1` + ReLU et est sous-échantillonné par `pool`, résultant au format $(N, 8, 4, 4)$.
   * **Deuxième bloc convolutionnel :** Le *tensor* $(N, 8, 4, 4)$ est traité par `conv2` + ReLU et réduit par `pool` au format $(N, 16, 2, 2)$.
   * **Aplatissement (*Flatten*) :** La méthode `x.view(x.size(0), -1)` reconfigue la structure $3\text{D}$ en un vecteur $1\text{D}$ de $64$ éléments par échantillon, en préservant la dimension du lot $N$.
   * **Classification :** Le vecteur de $64$ éléments alimente `fc1` avec une activation ReLU ($32$ neurones) et se termine dans `fc2`, produisant les $10$ *logits* non normalisés pour le calcul de la fonction de perte.

In [13]:
# 2. Définition de l'architecture convolutive
class CNNDigitos(nn.Module):
    """
    Architecture convolutive compacte :
    2 couches convolutives avec ReLU et Max-Pooling + 2 couches denses.
    """
    def __init__(self, n_classes=10):
        super().__init__()
        
        # Conv1 : 1 canal d'entrée, 8 filtres 3x3 avec padding 1 (sortie : 8x8)
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        # Conv2 : 8 canaux d'entrée, 16 filtres 3x3 avec padding 1 (sortie : 4x4)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)

        self.relu = nn.ReLU()

        # Max-Pooling 2x2 avec pas (stride) 2
        self.pool = nn.MaxPool2d(2, 2)

        # Couches entièrement connectées (FC)
        self.fc1 = nn.Linear(16 * 2 * 2, 32)
        self.fc2 = nn.Linear(32, n_classes)

    def forward(self, x):
        # Premier bloc : Conv (8x8) -> ReLU -> Pool (4x4)
        x = self.pool(self.relu(self.conv1(x)))

        # Deuxième bloc : Conv (4x4) -> ReLU -> Pool (2x2)
        x = self.pool(self.relu(self.conv2(x)))
        
        # Aplatissement (Flatten) : reconfigure la matrice 3D (16, 2, 2) en vecteur 1D (64)
        x = x.view(x.size(0), -1)

        # Couche dense intermédiaire avec ReLU
        x = self.relu(self.fc1(x))

        # Couche finale de classification (logits)
        return self.fc2(x)

###### Analyse des couches et du flux de la classe `CNNDigitos`

1. **Constructeur (`__init__`) et instanciation des composants :**
   * **Couche convolutive 1 (`self.conv1`)** : Applique $8$ filtres $3 \times 3$ avec `padding=1` sur l’entrée en niveaux de gris ($1$ canal), en préservant la résolution de $8 \times 8$ *pixels*.
   * **Couche convolutive 2 (`self.conv2`)** : Traite les $8$ cartes de caractéristiques reçues en appliquant $16$ filtres $3 \times 3$ avec `padding=1`.
   * **Sous-échantillonnage (`self.pool`)** : Instancie l’opération de *Max-Pooling* avec une fenêtre $2 \times 2$ et un pas (*stride*) de $2$, réduisant les dimensions spatiales (hauteur et largeur) de moitié à chaque application.
   * **Couches entièrement connectées (`self.fc1` et `self.fc2`)** : La première projection dense reçoit le *tenseur* aplati de dimension $16 \times 2 \times 2 = 64$ et produit $32$ caractéristiques intermédiaires. La seconde projette ces $32$ caractéristiques sur les $10$ *logits* de sortie.

2. **Propagation avant dans la méthode `forward` :**
   * **Premier bloc** : Le *tenseur* $(N, 1, 8, 8)$ passe par `conv1` + ReLU et est réduit par `pool` à $(N, 8, 4, 4)$.
   * **Deuxième bloc** : Le *tenseur* $(N, 8, 4, 4)$ passe par `conv2` + ReLU et est réduit par `pool` à $(N, 16, 2, 2)$.
   * **Aplatissement (*Flatten*)** : La méthode `x.view(x.size(0), -1)` convertit la structure $3\text{D}$ en un vecteur $1\text{D}$ de $64$ éléments par échantillon.
   * **Classification** : Le vecteur de $64$ éléments alimente `fc1` avec une activation ReLU ($32$ neurones) et se termine par `fc2`, qui produit les $10$ *logits* finaux pour le calcul de la perte d’Entropie Croisée.

##### Bloc 3 : Instanciation et paramètres d'optimisation

L'étape de configuration de l'apprentissage exige l'instanciation de l'architecture définie et le choix de deux composants centraux : la **fonction de perte**, qui quantifie l'erreur du modèle, et l'**algorithme d'optimisation**, responsable de l'ajustement des paramètres vers le minimum de cette fonction.

1. **Instanciation et comptage des paramètres :**
   Le modèle est créé à partir de l'instanciation de l'objet `modelo_cnn` de la classe `CNNDigitos`. L'expression `sum(p.numel() for p in modelo_cnn.parameters())` parcourt tous les *tensors* de paramètres entraînables du réseau (poids et biais de chaque couche) et calcule la cardinalité totale du modèle, quantifiant ainsi sa capacité de représentation.

2. **Fonction de perte (`nn.CrossEntropyLoss`) :**
   La perte d'entropie croisée (*Cross-Entropy Loss*) est le choix standard pour les problèmes de classification multiclasse. Dans PyTorch, cette implémentation combine en interne l'application de la fonction *LogSoftmax* avec la perte de log-vraisemblance négative (*NLLLoss*). Pour cette raison, la couche de sortie du réseau produit des *logits* bruts, dispensant de l'application explicite de la fonction *Softmax* à la fin de la méthode `forward`.

3. **Optimiseur adaptatif (`optim.Adam`) :**
   La mise à jour des paramètres utilise l'algorithme *Adam* (*Adaptive Moment Estimation*), avec un taux d'apprentissage initial $\eta = 0,01$ (`lr=1e-2`). *Adam* combine les principes du moment avec l'adaptation de la taille du pas basée sur la moyenne mobile des dérivées de premier et second ordres, ajustant individuellement le taux d'apprentissage de chaque paramètre du réseau.

In [14]:
# 3. Initialisation du modèle et paramètres d’optimisation
modelo_cnn = CNNDigitos()
num_params = sum(p.numel() for p in modelo_cnn.parameters())
print(f"Paramètres entraînables du modèle : {num_params}")

criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_cnn.parameters(), lr=1e-2)

Paramètres entraînables du modèle : 3658


##### Bloco 4: Boucle d'Entraînement et Évaluation

L'entraînement d'un CNN se déroule de manière itérative via l'algorithme de Descente de Gradient Stochastique par *mini-lots* (*Mini-batch SGD*).

Les courbes d'apprentissage résultant de ce processus sont présentées dans la [Figure 9.15](#fig-09-cnn-treinamento), générée à la fin de l'exécution.

1. **Phase d'Entraînement (`modelo_cnn.train()`):**
   La boucle principale exécute l'entraînement sur $50$ époques. À chaque époque, les étapes suivantes ont lieu :
   * **Mélange Stochastique :** La fonction `torch.randperm(n)` génère une permutation aléatoire des indices des échantillons, garantissant que l'ordre des *mini-lots* varie à chaque époque pour éviter les biais d'échantillonnage.
   * **Division en *Mini-lots* :** L'ensemble d'entraînement est découpé en lots de $32$ échantillons (`tam_lote = 32`).
   * **Remise à Zéro des Gradients (`otimizador.zero_grad()`):** Nettoie les gradients accumulés dans le *tensor* à l'itération précédente, évitant la somme indésirable de dérivées entre différents lots.
   * **Propagation Avant et Perte :** La *forward pass* calcule les prédictions `saida`, et l'appel `criterio(saida, y_treino_t[idx])` quantifie l'erreur du lot.
   * **Rétropropagation (`perda.backward()`):** Applique la règle de la chaîne pour calculer les dérivées partielles de la perte par rapport à chaque paramètre ($\frac{\partial L}{\partial w}$).
   * **Mise à Jour des Poids (`otimizador.step()`):** Met à jour les paramètres du modèle selon les équations de l'optimiseur *Adam*.

2. **Phase d'Évaluation (`modelo_cnn.eval()`):**
   À la fin de chaque époque, le modèle est basculé en mode évaluation. Le contexte `with torch.no_grad()` désactive temporairement le moteur de calcul automatique des dérivées (*autograd*), réduisant la consommation mémoire et accélérant l'inférence sur l'ensemble de test (`X_teste_t`). L'opération `.argmax(dim=1)` extrait la classe de plus haute probabilité pour chaque échantillon, permettant de calculer la précision sur le test.

3. **Visualisation avec la Bibliothèque `morph`:**
   La fonction `mm.showTrainCurves` de la bibliothèque didactique `morph` consolide l'historique de perte d'entraînement et la précision sur le test dans un unique panneau graphique, permettant de diagnostiquer la convergence du modèle et de surveiller la stabilité de l'apprentissage au fil des époques.

In [15]:
# 4. Boucle d'entraînement (SGD par mini-lots)
n = X_treino_t.size(0)                    # Nombre d'échantillons
tam_lote = 32                             # Taille du mini-lot
epocas = 50                               # Nombre total d'époques
historico_perda, historico_acc = [], []   # Historique des métriques

for epoca in range(epocas):                  # Répète par époque
    modelo_cnn.train()                       # Mode entraînement
    perm = torch.randperm(n)                 # Mélange les échantillons
    perda_epoca = 0.0                        # Accumule les pertes
    for i in range(0, n, tam_lote):          # Parcourt les mini-lots
        idx = perm[i:i + tam_lote]           # Indices du lot
        otimizador.zero_grad()               # Remet les gradients à zéro
        saida = modelo_cnn(X_treino_t[idx])  # Propagation avant
        perda = criterio(saida, y_treino_t[idx]) # Calcule la perte
        perda.backward()                         # Rétropropagation
        otimizador.step()                        # Met à jour les poids
        perda_epoca += perda.item() * len(idx)   # Somme des pertes

    # Évaluation du modèle sur l'ensemble de test à la fin de chaque époque
    modelo_cnn.eval()                            # Mode évaluation
    with torch.no_grad():                        # Sans gradients
        pred_teste = modelo_cnn(X_teste_t).argmax(dim=1)  # Prédictions
        acc_teste = (pred_teste == y_teste_t).float().mean().item()  # Précision
    historico_perda.append(perda_epoca / n)      # Enregistre la perte
    historico_acc.append(acc_teste)              # Enregistre la précision

acc_final_cnn = historico_acc[-1]                # Dernière précision
print(f"Précision finale du CNN sur l'ensemble de test : {acc_final_cnn:.4f}")  # Affiche le résultat

final = mm.showTrainCurves(                      # Trace les courbes
    historico_perda, historico_acc,
    titulo="Evolução do Treinamento da CNN — Base de Dígitos",
    subtitulo=f"Acurácia final no teste: {acc_final_cnn:.4f}",
)

Précision finale du CNN sur l'ensemble de test : 0.9759


<Figure size 1190x714 with 2 Axes>

**Figure 9.15:** Courbes d


##### Bloco 5: Visualização do Fluxo de Ativações

A inspeção da rede treinada permite observar a transformação progressiva do *tensor* de entrada ao longo das camadas da arquitetura `CNNDigitos`. A [Figure 9.16](#fig-09-cnn-ativacoes) ilustra as dimensões e as ativações intermediárias obtidas ao processar um exemplo real do dígito $3$.

1. **Seleção e Preparação da Amostra:**
   A semente estocástica é fixada com `torch.manual_seed(7)` para assegurar a reprodutibilidade dos resultados. A primeira ocorrência do dígito $3$ no conjunto de dados `load_digits` é isolada, normalizada para o intervalo $[0.0, 1.0]$ e reconfigurada como um *tensor* `x` de dimensão $(1, 1, 8, 8)$.

2. **Inspeção Intermediária com `mm.showNet`:**
   A função `mm.showNet` da biblioteca `morph` executa a propagação à frente (*forward pass*) do *tensor* `x` na instância `modelo_cnn` previamente treinada. Utilizando *hooks* de *forward*, a função intercepta o estado numérico das ativações nas camadas convolucionais (`nn.Conv2d`), de agrupamento (`nn.MaxPool2d`) e totalmente conectadas (`nn.Linear`), retornando-as no dicionário `acts`. As funções de ativação não linear (`nn.ReLU`) não são registradas como estágios independentes, pois sua aplicação ocorre diretamente sobre o *tensor* de saída da camada correspondente.

3. **Verificação dos Resultados:**
   A instrução `list(acts.keys())` exibe a sequência de identificadores das camadas monitoradas, permitindo confirmar a redução dimensional progressiva e a geração do *logit* de valor máximo no índice correspondente à classe $3$, conforme demonstrado na [Figure 9.16](#fig-09-cnn-ativacoes)..

In [16]:
torch.manual_seed(7)
digits = load_digits()
idx = np.where(digits.target == 3)[0][0]
img = digits.images[idx] / 16.0
x = torch.tensor(img, dtype=torch.float32).view(1, 1, 8, 8)

# Réutilisation de l'instance du modèle préalablement entraînée
acts = mm.showNet(
    modelo_cnn,
    x,
    titulo="Fluxo de transformações dos tensors ao longo da arquitetura CNNDigitos",
    subtitulo=f"Exemplo real do dataset load_digits (classe verdadeira: {digits.target[idx]})",
)
print("Couches capturées :", list(acts.keys()))

<Figure size 3213x578 with 7 Axes>

**Figure 9.16:** Fluxo de activations de la CNN entraînée lors du traitement d


Couches capturées : ['conv1', 'pool', 'conv2', 'pool #2', 'fc1', 'fc2']


##### Inspeção do Grafo Computacional com `torchviz`

Enquanto `mm.showNet` prioriza a clareza didática — exibindo uma coluna por camada com parâmetros treináveis —, a biblioteca `torchviz` projeta o **grafo de autograd** exatamente como o PyTorch o constrói internamente para o cálculo de gradientes. A [Figure 9.17](#fig-09-torchviz-grafo) ilustra essa perspectiva ao representar a arquitetura `CNNDigitos`.

1. **Propagação para Frente Rastreada:** Com o modelo treinado em modo `eval()`, a *passagem para frente* sobre o *tensor* `x` do dígito $3$ é suficiente para que o motor de *autograd* registre todas as operações executadas, incluindo aquelas sem parâmetros treináveis, como a função de ativação `ReLU` e a reconfiguração dimensional `view`.

2. **Geração do Grafo (`make_dot`):** A função `make_dot(saida, params=...)` constrói o grafo a partir do *tensor* de saída, percorrendo retroativamente o histórico de operações até os nós folha (os parâmetros treináveis do modelo). Cada nó do diagrama representa uma operação do *backward pass* (como `ReluBackward` ou `AddmmBackward`), e não apenas um bloco conceitual do `nn.Module`.

3. **Exportação e Renderização (`.render`):** O método `.render(..., format="png", cleanup=True)` invoca o executável `dot` do **Graphviz** para compilar a imagem em formato PNG, eliminando automaticamente os arquivos intermediários de código-fonte.

A [Figure 9.17](#fig-09-torchviz-grafo) evidencia como esse grafo computacional, mesmo para uma arquitetura compacta, apresenta maior densidade que o painel do `mm.showNet`, pois detalha cada operação atômica responsável pelo fluxo de gradientes.

In [17]:
# 1. Passe avant avec suivi de gradient activé
modelo_cnn.eval()
saida = modelo_cnn(x)  # Réutilisation du tenseur x (chiffre 3)

# 2. Graphe de base : flux d'opérations jusqu'à la sortie
grafo_simples = make_dot(saida, params=dict(modelo_cnn.named_parameters()))
caminho_simples = grafo_simples.render("cnn_digitos_grafo_simples", format="png", cleanup=True)

# 3. Affichage direct dans l'environnement Quarto/Jupyter
# Le .render() retourne le chemin du fichier PNG généré ; nous devons l'ouvrir comme image
imagem_simples = np.array(Image.open(caminho_simples).convert("RGB"))
mm.show(imagem_simples, figsize=(5,10))

<Figure size 750x1500 with 1 Axes>

**Figure 9.17:** Graphe computationnel de la *CNNDigitos* généré via *torchviz*, montrant les opérations de *forward* et les nœuds de gradient (*backward*) associés à chaque paramètre entraînable.


##### Vue détaillée du graphe computationnel avec `torchviz`

Outre la représentation simplifiée, la bibliothèque `torchviz` permet d’étendre le graphe d’*autograd* afin d’inspecter les détails internes de l’exécution du réseau `CNNDigitos`. La [Figure 9.18](#fig-09-torchviz-grafo-detalhado) présente cette structure étendue pour le même *tensor* d’entrée `x`.

1. **Traçage avec attributs d’opération (`show_attrs=True`):**  
   L’inclusion des attributs affiche les configurations hyperparamétriques associées à chaque nœud computationnel durant la propagation avant (*forward pass*), telles que les dimensions de *kernel* (`kernel_size`), les pas (*stride*) et les remplissages (*padding*) dans les convolutions et les sous-échantillonnages.

2. **Détection des *tensors* sauvegardés en mémoire (`show_saved=True`):**  
   Ce paramètre force l’affichage explicite des *tensors* intermédiaires que PyTorch conserve en mémoire pendant le *forward pass*. Ces données sont préservées car elles seront strictement nécessaires au calcul des dérivées partielles durant l’étape de rétropropagation (*backward pass*).

3. **Génération et compilation des graphes:**  
   Alors que `grafo_simples` produit une vue directe du flux de gradients, `grafo_detalhado` compile le graphe étendu dans le fichier `cnn_digitos_grafo_detalhado.png` via l’exécutable `dot` de **Graphviz**.

Comme observé dans la [Figure 9.18](#fig-09-torchviz-grafo-detalhado), cette visualisation minutieuse est utile pour déboguer la consommation de mémoire vidéo (*VRAM*) et vérifier comment le moteur de PyTorch alloue en interne chaque nœud de la règle de la chaîne.

In [18]:
# même code précédent (make_dot + render)...

# 2. Graphe détaillé : affichage des dimensions et des tenseurs sauvegardés pour la rétropropagation
grafo_detalhado = make_dot(
    saida,
    params=dict(modelo_cnn.named_parameters()),
    show_attrs=True,   # Affiche les attributs des opérations (ex. : kernel_size, stride)
    show_saved=True,   # Affiche les tenseurs sauvegardés en mémoire pour la rétropropagation
)

caminho_detalhado = grafo_detalhado.render("cnn_digitos_grafo_detalhado", 
                                           format="png", cleanup=True)

# 3. Affichage direct dans l'environnement Quarto/Jupyter
imagem_detalhado = np.array(Image.open(caminho_detalhado).convert("RGB"))
mm.show(imagem_detalhado, figsize=(8, 16))

<Figure size 1200x2400 with 1 Axes>

**Figure 9.18:** Graphe détaillé de la classe CNNDigitos généré via *torchviz*.


##### Comparação com o Capítulo 7

A [Figure 9.19](#fig-09-comparativo-cap7) reúne os resultados obtidos no mesmo conjunto de dados (`load_digits`), estabelecendo um paralelo direto entre as abordagens clássicas exploradas anteriormente e a CNN desenvolvida neste capítulo.

1. **Desempenho dos *Pixels* Brutos vs. Descritores Manuais:** Nos experimentos do Capítulo 7, o classificador $k\text{-NN}$ ($k=3$) atingiu uma acurácia de $98,4\%$ quando alimentado diretamente com os *pixels* brutos das imagens. Em contrapartida, a extração prévia de características via *Histogram of Oriented Gradients* (*HOG*) resultou em um desempenho significativamente inferior ($75,8\%$). Essa queda ocorre porque o *HOG* foi concebido para capturar gradientes de bordas em imagens de maior resolução; em matrizes de apenas $8 \times 8$ *pixels*, a resolução espacial é insuficiente para formar histogramas de orientação informativos.

2. **Equivalência da CNN e Aprendizado *End-to-End*:** A rede convolucional `CNNDigitos` alcança um desempenho competitivo de $97,6\%$, aproximando-se da acurácia do $k\text{-NN}$ com *pixels* brutos em uma base pequena e pré-alinhada. A grande vantagem conceitual reside no aprendizado de representação: em vez de depender de descritores projetados manualmente (*handcrafted features*) ou de manter todo o conjunto de dados em memória para a busca por vizinhos no momento da inferência, a CNN otimiza automaticamente seus próprios filtros convolucionais durante o treinamento, gerando um modelo compacto capaz de realizar a extração de características e a classificação de forma integrada (*end-to-end*).

In [19]:
import matplotlib.pyplot as plt

# Valeurs obtenues au Chapitre 7 (k-NN, k=3), reproduites pour comparaison directe
ACC_KNN_PIXELS_CAP7 = 0.9844
ACC_KNN_HOG_CAP7 = 0.7578

metodos = ["k-NN\n(pixels brutos)", "k-NN\n(HOG)", "CNN\n(este capítulo)"]
acuracias = [ACC_KNN_PIXELS_CAP7, ACC_KNN_HOG_CAP7, acc_final_cnn]

plt.figure(figsize=(5, 4))
cores = ["#6366f1", "#f97316", "#16a34a"]
plt.bar(metodos, acuracias, color=cores)
plt.ylim(0, max(acuracias) + 0.08)
plt.ylabel("Acurácia (conjunto de teste)")
plt.title("Cap. 7 vs. Cap. 9 — Base de Dígitos")

for i, v in enumerate(acuracias):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

<Figure size 1500x1200 with 1 Axes>

**Figure 9.19:** Comparaison de précision entre les classificateurs classiques du Chapitre 7 (pixels bruts et HOG avec k-NN) et le CNN entraîné dans ce chapitre, sur la même base de chiffres.


> ### 📝 Nota
>
> ###### 🧠 Pourquoi cela fonctionne-t-il ? — Et pourquoi le CNN ne « gagne » pas toujours
>
> Le résultat observé ici **répète le schéma déjà vu au Chapitre 7** : le CNN, bien qu'il apprenne automatiquement ses caractéristiques, ne surpasse pas nécessairement le $k\text{-NN}$ avec des *pixels* bruts sur cette base spécifique. L'explication est la même : `load_digits` est une base petite (moins de $1.800$ exemples), avec des images déjà centrées, normalisées et de très basse résolution ($8 \times 8$) — des conditions où la comparaison directe des intensités est déjà hautement informative, et où il y a peu de données pour que le réseau apprenne des filtres véritablement supérieurs aux descripteurs simples.
>
> Le véritable avantage des CNN apparaît dans des scénarios que les descripteurs artisanaux et les classificateurs simples ne peuvent pas aborder : des images plus grandes et plus réalistes, avec des milliers de catégories, une variation substantielle de pose, d'éclairage et de fond, et des ensembles d'entraînement massifs — exactement le régime dans lequel les modèles présentés dans la section *« Applications à Grande Échelle »*, plus loin, ont été entraînés. La leçon pédagogique qui traverse les Chapitres 7, 8 et 9 de ce livre est cohérente : **la sophistication d'une méthode doit être proportionnelle à la complexité du problème** — utiliser un CNN pour un problème qu'un $k\text{-NN}$ résout tout aussi bien est un gaspillage de ressources computationnelles, non une vertu.
>
> Cette même proportionnalité vaut pour les outils d'inspection utilisés tout au long du chapitre. Le `mm.showNet` a été construit à des fins **didactiques** et fonctionne bien sur des réseaux peu profonds comme `CNNDigitos`, mais ne passe pas à l'échelle pour des architectures profondes : chaque couche suivie devient une colonne dans la figure, et les couches convolutionnelles avec des centaines de canaux génèrent des mosaïques trop grandes pour une interprétation visuelle ; de plus, les *hooks* stockent toutes les activations en mémoire, et la *mise en page* suppose un flux séquentiel, ne représentant pas fidèlement les connexions résiduelles ou les ramifications (comme dans les *ResNets* ou les modules *Inception*). Ainsi, `showNet` doit être compris comme une lentille pédagogique pour les petits réseaux — analogue au rôle de `mm.showBoundBox` dans le débogage visuel des détections — et non comme un substitut aux outils orientés production, comme *TensorBoard* ou *torchviz*.

#### 9.5.1.2 Transfert d'apprentissage

Entraîner un CNN de zéro nécessite généralement une grande quantité de données étiquetées et des ressources computationnelles significatives, car le processus d'entraînement doit ajuster tous les paramètres du réseau. Dans de nombreuses applications, cependant, seul un ensemble réduit de données est disponible pour la tâche d'intérêt. Dans cette situation, le **transfert d'apprentissage** (*transfer learning*) réutilise les représentations apprises par un modèle préalablement entraîné sur une tâche source avec un grand volume de données, réduisant ainsi le coût d'entraînement et le besoin de nouvelles échantillons.

En vision par ordinateur, cette stratégie exploite l'organisation hiérarchique des CNN. Les couches initiales apprennent des caractéristiques visuelles de bas niveau, telles que les contours, les textures, les gradients d'intensité et les motifs de couleurs, qui restent utiles dans différents domaines. Les couches plus profondes combinent ces informations pour former des représentations progressivement plus abstraites et spécialisées, liées aux classes présentes dans la base d'entraînement.

Cette section examine dans quelles conditions le transfert d'apprentissage produit de bons résultats. La première expérience montre qu'un extracteur de petite taille et entraîné sur un domaine restreint peut conduire à un **transfert négatif** (*negative transfer*). La deuxième démontre pourquoi des modèles profonds pré-entraînés sur de grandes bases d'images atteignent des performances élevées sur de nouvelles tâches. Enfin, la troisième applique cette stratégie à un problème de diagnostic phytosanitaire, illustrant un scénario proche des applications réelles.

##### Expérience 1 — Limites d'un extracteur petit et spécialisé

La première expérience montre que le transfert d'apprentissage n'améliore pas toujours les performances d'un modèle. Pour ce faire, l'ensemble de chiffres manuscrits (`load_digits`) est divisé en deux domaines disjoints :

- **Domaine A (source) :** chiffres $0$ à $4$, utilisés pour entraîner une petite CNN ;
- **Domaine B (cible) :** chiffres $5$ à $9$, réindexés de $0$ à $4$, formant une nouvelle tâche avec seulement $20$ échantillons d'entraînement.

L'objectif consiste à évaluer l'effet de la réutilisation de l'extracteur de caractéristiques appris dans le Domaine A sans permettre son adaptation au Domaine B.

###### Bloco 1 : Division des domaines, rareté et affichage des échantillons

Ce bloc prépare l'ensemble de données pour l'expérience. Contrairement au projet précédent, qui utilisait tous les chiffres dans un problème de classification unique, la base est divisée en deux tâches indépendantes : une tâche source (Domaine A) et une tâche cible (Domaine B).

La [Figure 9.20](#fig-09-transfer-amostras) présente des exemples des deux domaines après la séparation des classes, la conversion en *tensors* **PyTorch** et le prétraitement.

1. **Séparation des classes :** Les masques booléens `mask_A` et `mask_B` séparent les exemples de chaque domaine. Ensuite, le code réindexe les étiquettes du Domaine B (`y[mask_B] - 5`) dans l'intervalle $[0,4]$, permettant ainsi aux deux modèles d'utiliser cinq classes de sortie.

2. **Rareté des données :** Le générateur `np.random.default_rng(0)` sélectionne uniquement $20$ échantillons pour l'entraînement du Domaine B, environ quatre par classe, simulant un scénario où l'entraînement à partir de zéro tend à souffrir d'*overfitting*.

3. **Conversion en *tensors* :** La fonction `para_tensor` convertit les images au format $(N,1,8,8)$ et les étiquettes en `torch.long`, compatibles avec les couches `nn.Conv2d` et la fonction de perte.

4. **Visualisation des échantillons :** Le code utilise `.squeeze().numpy()` pour convertir les *tensors* en matrices **NumPy**. La [Figure 9.20](#fig-09-transfer-amostras) présente des exemples des deux domaines et met en évidence la réindexation appliquée aux étiquettes du Domaine B.

In [20]:
# 1. Division du jeu de données en deux domaines disjoints
classes_A, classes_B = [0, 1, 2, 3, 4], [5, 6, 7, 8, 9]
mask_A, mask_B = np.isin(y, classes_A), np.isin(y, classes_B)

XA, yA = X[mask_A], y[mask_A]
XB, yB = X[mask_B], y[mask_B] - 5  # Réindexation des étiquettes dans l'intervalle [0, 4]

# Division en entraînement et test pour les deux domaines
XA_tr, XA_te, yA_tr, yA_te = train_test_split(
    XA, yA, test_size=0.25, random_state=42, stratify=yA
)
XB_tr, XB_te, yB_tr, yB_te = train_test_split(
    XB, yB, test_size=0.25, random_state=42, stratify=yB
)

# Simulation de pénurie extrême dans le domaine cible : seulement 20 échantillons d'entraînement
rng = np.random.default_rng(0)
idx_poucos = rng.choice(len(XB_tr), size=20, replace=False)
XB_tr_poucos, yB_tr_poucos = XB_tr[idx_poucos], yB_tr[idx_poucos]

# Fonction auxiliaire pour la conversion en tenseurs PyTorch
def para_tensor(Ximg, yarr):
    return torch.tensor(Ximg).unsqueeze(1), torch.tensor(yarr, dtype=torch.long)

XA_tr_t, yA_tr_t = para_tensor(XA_tr, yA_tr)
XA_te_t, yA_te_t = para_tensor(XA_te, yA_te)
XB_tr_t, yB_tr_t = para_tensor(XB_tr_poucos, yB_tr_poucos)
XB_te_t, yB_te_t = para_tensor(XB_te, yB_te)

# Affichage d'échantillons des deux domaines
n_amostras = 5
imgs_A = [img.squeeze().numpy() for img in XA_tr_t[:n_amostras]]
titles_A = [f"A: {label.item()}" for label in yA_tr_t[:n_amostras]]

imgs_B = [img.squeeze().numpy() for img in XB_tr_t[:n_amostras]]
titles_B = [f"B: {label.item()} (orig: {label.item()+5})" for label in yB_tr_t[:n_amostras]]

mm.show(
    imgs_A + imgs_B,
    titles=titles_A + titles_B,
    cols=n_amostras,
    figsize=(12, 4.5)
)

<Figure size 1800x675 with 10 Axes>

**Figure 9.20:** Échantillons des ensembles d


###### Bloc 2 : Architecture Modulaire et Routines Génériques

Pour faciliter le transfert d'apprentissage, l'architecture convolutive et la boucle d'entraînement ont été refactorisées par rapport à la classe `CNNDigitos` du projet précédent.

1. **Modularisation de l'architecture (Différence par rapport à `CNNDigitos`) :**
   * Dans le projet précédent, la classe `CNNDigitos` déclarait toutes les couches (`conv1`, `conv2`, `pool`, `fc1`, `fc2`) comme membres directs d'une seule classe monolithique.
   * Ici, l'architecture est séparée en deux composants : la classe `ExtratorConv` encapsule le bloc spatial convolutif ($2$ convolutions $3 \times 3$, $2$ *Max-Poolings* $2 \times 2$ et l'aplatissement en $64$ éléments), tandis que la classe `CNNCompleta` instancie cet extracteur dans `self.extrator` et ajoute la « tête » de classification (`fc1` et `fc2`).
   * Cette séparation est ce qui permet de copier l'état interne de l'extracteur (`state_dict()`) d'un modèle à un autre de manière isolée.

2. **Ajustement du nombre de classes de sortie :**
   Alors que `CNNDigitos` dans le projet précédent possédait $10$ *logits* dans la couche de sortie (`self.fc2 = nn.Linear(32, 10)`), la classe `CNNCompleta` reçoit `n_classes=5` dans le constructeur pour s'adapter à la division des domaines $A$ et $B$.

3. **Flexibilisation de la boucle d'entraînement (`treinar`) :**
   * Dans le projet précédent, la boucle d'entraînement itérait directement sur les attributs globaux du modèle (`modelo_cnn.parameters()`) et calculait des métriques spécifiques en ligne.
   * La fonction `treinar` abstrait ce processus et introduit le paramètre optionnel `parametros`. S'il est fourni, l'optimiseur *Adam* met à jour **uniquement** les paramètres de cette liste, ignorant les couches dont les gradients ont été désactivés. Cette flexibilité est cruciale pour exécuter l'entraînement avec un gel partiel du réseau.

4. **Isolation de l'évaluation (`calcular_acuracia`) :**
   Comme cela a été fait lors de la phase de test du projet précédent, la fonction place le modèle en mode `eval()` et utilise le contexte `torch.no_grad()` pour désactiver l'*autograd*, calculant la précision via `.argmax(dim=1)`.

In [21]:
# Définition du bloc convolutif réutilisable (même extraction que le projet précédent)
class ExtratorConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return x.view(x.size(0), -1)

# Architecture modulaire combinant l'extracteur et la tête de classification
class CNNCompleta(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.extrator = ExtratorConv()
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.extrator(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Routine générique d'entraînement avec optimisation sélective des paramètres
def treinar(modelo, X_t, y_t, epocas, lr, tam_lote=16, parametros=None):
    # Paramètres entraînables
    params = parametros if parametros is not None else modelo.parameters()  
    otim = optim.Adam(params, lr=lr)               # Optimiseur Adam
    crit = nn.CrossEntropyLoss()                   # Fonction de perte
    n_amostras = X_t.size(0)                       # Nombre d'échantillons
    for _ in range(epocas):                        # Répète par époque
        perm = torch.randperm(n_amostras)          # Mélange les échantillons
        for i in range(0, n_amostras, tam_lote):   # Parcourt les mini-lots
            idx = perm[i:i + tam_lote]             # Indices du lot
            otim.zero_grad()                       # Met à zéro les gradients
            perda = crit(modelo(X_t[idx]), y_t[idx])  # Calcule la perte
            perda.backward()                       # Rétropropagation
            otim.step()                            # Met à jour les poids

# Routine d'évaluation
def calcular_acuracia(modelo, X_t, y_t):
    modelo.eval()                                  # Mode évaluation
    with torch.no_grad():                          # Sans gradients
        pred = modelo(X_t).argmax(dim=1)           # Classes prédites
    return (pred == y_t).float().mean().item()     # Retourne la précision

###### Bloco 3: Pré-entraînement, Transfert et Analyse Comparative

Ce bloc exécute la comparaison entre l’ajustement du modèle à partir de zéro et l’application du transfert avec gel statique de l’extracteur. Pour une transparence totale de l’expérience, les tailles des ensembles d’entraînement et de test des deux domaines sont imprimées dans le terminal.

1. **Quantification des échantillons par domaine :**
   * **Domaine A (source, chiffres $0$ à $4$):** Compte $675$ échantillons d’entraînement ($75\%$) et $226$ de test ($25\%$), fournissant des données abondantes pour que le `modelo_origem` apprenne l’extracteur convolutif jusqu’à atteindre $100\%$ de précision.
   * **Domaine B (cible, chiffres $5$ à $9$):** Dispose de $224$ échantillons de test au total, mais son ensemble d’entraînement est intentionnellement réduit de $672$ à seulement **$20$ échantillons** (`XB_tr_poucos`), créant un scénario sévère de pénurie de données.

2. **Étape 1 : Pré-entraînement sur le Domaine A (Source) :**
   Le `modelo_origem` est entraîné à partir de zéro sur les $675$ échantillons des chiffres $0$ à $4$. Pendant $40$ époques, l’extracteur convolutif ajuste ses filtres pour identifier les traits caractéristiques de ces cinq premiers chiffres, atteignant $100\%$ de précision sur l’ensemble de test ($226$ échantillons).

3. **Étape 2 : Transfert de poids et gel :**  
   * On crée le `modelo_transferencia` pour résoudre la tâche du Domaine B (chiffres $5$ à $9$).
   * Les poids appris dans le Domaine A sont copiés via :
  
     - `load_state_dict(modelo_origem.extrator.state_dict())`
  
   * **Gel :** La boucle `for p in modelo_transferencia.extrator.parameters(): p.requires_grad = False` désactive le calcul des gradients dans les couches convolutives.
   * **Entraînement sélectif :** L’appel `treinar(...)` passe strictement les paramètres des couches denses (`params_cabeca`), ajustant la tête de classification avec seulement les $20$ échantillons d’entraînement.

4. **Étape 3 : Entraînement à partir de zéro sur le Domaine B (Contrôle expérimental) :**
   Le `modelo_do_zero` possède la même architecture, mais est entraîné à partir de zéro sur les mêmes $20$ échantillons du Domaine B, sans aucune réutilisation de poids, pendant les mêmes $40$ époques.

5. **Analyse des résultats ([Figure 9.21](#fig-09-transfer-learning)) :**
   * **Avec transfert gelé ($72,77\%$):** En réutilisant l’extracteur entraîné sur le Domaine A et en gelant ses paramètres, le réseau atteint $72,77\%$ de précision sur le test ($224$ échantillons) en ajustant uniquement les couches denses.
   * **Entraîné à partir de zéro ($76,79\%$):** L’entraînement à partir de zéro surpasse le transfert gelé sur l’ensemble de test du Domaine B.
   * **Cause de la différence :** En raison d’un modèle minuscule (seulement $16$ filtres convolutifs dans des matrices de $8 \times 8$), l’extracteur entraîné sur le Domaine A est devenu **hyperspécialisé** dans les formes géométriques des chiffres $0$ à $4$. En gelant rigidement ces quelques filtres, le modèle cible s’est trouvé limité à des détecteurs inadéquats pour $5$ à $9$. Le réseau entraîné à partir de zéro, même avec seulement $20$ échantillons, a réussi à adapter ses $16$ filtres directement aux traits du Domaine B.

In [22]:
# Fixar semente para reprodutibilidade
torch.manual_seed(42)

# Exibição do tamanho dos grupos de treino e teste
print("=== Detalhamento do Tamanho das Bases ===")
print(f"Domínio A (0-4) — Treino: {len(XA_tr_t)} amostras | Teste: {len(XA_te_t)} amostras")
print(f"Domínio B (5-9) — Treino completo: {len(XB_tr)} | Treino reduzido: {len(XB_tr_poucos)}",
      f"| Teste: {len(XB_te_t)} amostras\n")

# 1. Pré-treinamento na tarefa de origem (Domínio A: dígitos 0-4)
modelo_origem = CNNCompleta(n_classes=5)                       # Cria CNN

#######
treinar(modelo_origem, XA_tr_t, yA_tr_t, epocas=40, lr=1e-2)   # Treina modelo
         
acc_A = calcular_acuracia(modelo_origem, XA_te_t, yA_te_t)     # Mede acurácia
                          
print(f"Acurácia no domínio de origem A "                      # Exibe resultado
      f"(dígitos 0-4, {len(XA_te_t)} testes): " f"{acc_A:.4f}")

# 2. Transferência de Aprendizado (Extrator Congelado)
modelo_transferencia = CNNCompleta(n_classes=5)        # Cria CNN
modelo_transferencia.extrator.load_state_dict(         # Copia extrator
    modelo_origem.extrator.state_dict())

for p in modelo_transferencia.extrator.parameters():   # Percorre extrator
    p.requires_grad = False                            # Congela pesos

params_cabeca = list(modelo_transferencia.fc1.parameters())  # FC1
params_cabeca += list(modelo_transferencia.fc2.parameters()) # +FC2

#######
treinar(modelo_transferencia, XB_tr_t, yB_tr_t,              # Treina cabeça
         epocas=40, lr=1e-2, parametros=params_cabeca)

acc_transferencia = calcular_acuracia(modelo_transferencia, XB_te_t, yB_te_t) # Mede acurácia

# 3. Treinamento do Zero no Domínio B
modelo_do_zero = CNNCompleta(n_classes=5)                     # Cria CNN

#######
treinar(modelo_do_zero, XB_tr_t, yB_tr_t, epocas=40, lr=1e-2) # Treina modelo
         
acc_do_zero = calcular_acuracia(modelo_do_zero,  XB_te_t, yB_te_t) # Mede acurácia
                               

print(f"Domínio de destino B (dígitos 5-9), apenas {len(XB_tr_poucos)} ", 
      f"exemplos de treino ({len(XB_te_t)} testes):")
print(f"  Com transferência (extrator congelado): {acc_transferencia:.4f}")
print(f"  Treinando do zero (mesmos dados/épocas): {acc_do_zero:.4f}")

# Visualização comparativa
plt.figure(figsize=(4.5, 4))
plt.bar(["Do zero", "Transferência"], [acc_do_zero, acc_transferencia], 
        color=["#dc2626", "#16a34a"])
plt.ylim(0, max([acc_do_zero, acc_transferencia]) + 0.1)
plt.ylabel("Acurácia no domínio B (teste)")
plt.title(f"Efeito da Transferência ({len(XB_tr_poucos)} exemplos de treino)")

for i, v in enumerate([acc_do_zero, acc_transferencia]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

=== Detalhamento do Tamanho das Bases ===
Domínio A (0-4) — Treino: 675 amostras | Teste: 226 amostras
Domínio B (5-9) — Treino completo: 672 | Treino reduzido: 20 | Teste: 224 amostras



Acurácia no domínio de origem A (dígitos 0-4, 226 testes): 1.0000


Domínio de destino B (dígitos 5-9), apenas 20  exemplos de treino (224 testes):
  Com transferência (extrator congelado): 0.7277
  Treinando do zero (mesmos dados/épocas): 0.7679


<Figure size 1350x1200 with 1 Axes>

**Figure 9.21:** Comparação de acurácia no conjunto de teste do Domínio B (dígitos 5 a 9) sob restrição de dados (20 exemplos de treino): demonstração do impacto do congelamento rígido e da transferência negativa em redes de baixa capacidade.


###### Bloco 4 : Visualisation du flux d’activations avec `mm.showNet`

Pour confirmer que l’extraction de caractéristiques réutilisée préserve les transformations dimensionnelles étudiées dans le projet précédent, on utilise à nouveau la fonction `mm.showNet` de la bibliothèque `morph`. La [Figure 9.22](#fig-09-transfer-shownet) affiche le flux d’activations du `modelo_transferencia` lors du traitement d’un échantillon du Domaine B (chiffre $7$, réindexé pour la classe $2$).

1. **Préservation du flux convolutionnel :** Comme l’architecture `ExtratorConv` reproduit les mêmes couches de convolution et de *pooling* que la `CNNDigitos` du projet précédent, les dimensions des *tensors* intermédiaires restent en $(1, 8, 4, 4)$ dans le premier bloc.
2. **Inspection de la tête adaptée :** La différence par rapport au projet précédent apparaît dans la couche de sortie (`fc2`) : alors que le modèle du projet précédent projetait le vecteur intermédiaire en $10$ *logits* (classes de $0$ à $9$), le modèle de transfert projette le vecteur en $5$ *logits* (classes de $0$ à $4$), capturant les probabilités relatives du Domaine B.

In [23]:
# Sélectionne le premier échantillon de test du Domaine B
x_amostra_B = XB_te_t[0:1]  # Tenseur de dimension (1, 1, 8, 8)
classe_verdadeira = yB_te_t[0].item()
classe_original = classe_verdadeira + 5

# Inspection du flux d'activations dans le modèle de transfert
acts_transfer = mm.showNet(
    modelo_transferencia,
    x_amostra_B,
    titulo="Fluxo de ativações no modelo de transferência (Domínio B)",
    subtitulo=f"Amostra do dígito {classe_original} (rótulo reindexado: {classe_verdadeira})",
)

print("Couches capturées dans le modèle de transfert :\n", list(acts_transfer.keys()))

<Figure size 3213x578 with 7 Axes>

**Figure 9.22:** Flux d


Couches capturées dans le modèle de transfert :
 ['extrator.conv1', 'extrator.pool', 'extrator.conv2', 'extrator.pool #2', 'fc1', 'fc2']


###### Analyse de l'expérience 1

Le modèle entraîné de zéro atteint une précision supérieure à celle du modèle avec transfert d'apprentissage et extracteur figé. Ce résultat caractérise un cas de **transfert négatif** (*negative transfer*) et découle de trois facteurs :

1. **Faible capacité :** L'extracteur ne possède que $16$ filtres $3 \times 3$, insuffisants pour apprendre des représentations généralisables.

2. **Spécialisation au domaine :** L'entraînement avec les chiffres $0$ à $4$ produit des filtres peu discriminants pour les chiffres $5$ à $9$.

3. **Absence d'adaptation :** Le gel empêche l'extracteur d'ajuster ses filtres à la nouvelle tâche.

> ### 📝 Nota
>
> ###### 💡 Provocation pédagogique
>
> Cette expérience utilise un petit extracteur entraîné sur un domaine restreint. Le résultat serait-il différent si l'extracteur avait appris ses représentations sur une base de millions d'images et une grande diversité d'objets ?

##### Expérience 2 — Quand le Transfert Fonctionne Vraiment (*ResNet-18* Pré-entraînée)

La deuxième expérience reprend la même structure que la première — peu d'exemples d'entraînement, deux classes, comparaison entre stratégies —, mais remplace l'extracteur artisanal de $16$ filtres par la **_ResNet-18_**, une architecture de $18$ couches pré-entraînée sur *ImageNet* ($1,4$ million d'images, $1.000$ catégories), et le *dataset* synthétique de chiffres par des photographies réelles du **Oxford-IIIT Pet Dataset** (PARKHI, 2012).

La tâche : distinguer deux races canines — **Carlin** et **Boxer** — à partir de seulement $15$ photographies d'entraînement par classe.

> ### 💡 Dica
>
> ###### 🐶 Pourquoi ce scénario ?
> Le défi ici ne réside pas dans la similarité visuelle entre les races — le Carlin et le Boxer ont des tailles et des proportions bien distinctes —, mais dans la rareté des données : seulement $30$ photographies réelles au total, sans aucune image synthétique. C'est le type de problème à faible budget de données qui motive, en pratique, l'utilisation de réseaux pré-entraînés : il n'y a ni le temps ni les ressources pour photographier et étiqueter des milliers de chiens avant d'entraîner un classificateur à partir de zéro.

###### Bloc 1 : Chargement du Jeu de Données Réel et Échantillonnage Parcimonieux

1. **Source :** le *Oxford-IIIT Pet Dataset* (PARKHI, 2012) est chargé via `torchvision.datasets.OxfordIIITPet`, qui télécharge automatiquement les $7.349$ photographies et leurs étiquettes de race lors de la première exécution.
2. **Filtrage :** seules les deux races d’intérêt (`Pug`, `Boxer`) sont conservées.
3. **Parcimonie délibérée :** seules $15$ photographies d’entraînement par classe ($30$ au total) sont tirées au sort — le reste constitue l’ensemble de test, utilisé exclusivement pour l’évaluation.

La [Figure 9.23](#fig-09-pets-amostras) affiche des échantillons d’entraînement de chaque race.

In [24]:
import random

RACAS_ALVO = ["Pug", "Boxer"]
N_TREINO_POR_CLASSE = 15
N_TESTE_POR_CLASSE = 20

# 1. Téléchargement du jeu de données complet (37 races) — licence CC BY-SA 4.0
pets_completo = OxfordIIITPet(
    root="dados_pets", split="trainval", target_types="category", download=True
)
nomes_racas = pets_completo.classes
indices_alvo = [nomes_racas.index(r) for r in RACAS_ALVO]

# 2. Filtrage des deux races d'intérêt, séparées par classe
por_classe = {idx: [] for idx in indices_alvo}
for img, lbl in pets_completo:
    if lbl in indices_alvo:
        por_classe[lbl].append(img)

# 3. Échantillonnage : peu d'images d'entraînement, plus d'images de test
rng = random.Random(42)
imgs_treino, y_treino, imgs_teste, y_teste = [], [], [], []
for classe_idx, idx_original in enumerate(indices_alvo):
    imgs_raca = por_classe[idx_original][:]
    rng.shuffle(imgs_raca)
    imgs_treino += imgs_raca[:N_TREINO_POR_CLASSE]
    y_treino += [classe_idx] * N_TREINO_POR_CLASSE
    imgs_teste += imgs_raca[N_TREINO_POR_CLASSE : N_TREINO_POR_CLASSE + N_TESTE_POR_CLASSE]
    y_teste += [classe_idx] * N_TESTE_POR_CLASSE

print(f"Entraînement : {len(imgs_treino)} images | Test : {len(imgs_teste)} images")

amostras_pil = imgs_treino[:4] + imgs_treino[N_TREINO_POR_CLASSE:N_TREINO_POR_CLASSE + 4]
amostras_exibicao = [np.array(img.convert("RGB")) for img in amostras_pil]  # PIL -> ndarray
titulos_exibicao = [RACAS_ALVO[0]] * 4 + [RACAS_ALVO[1]] * 4
mm.show(amostras_exibicao, titles=titulos_exibicao, cols=4, figsize=(11, 6))

**Figure 9.23:** Échantillons réels d


###### Bloc 2 : Trois stratégies sur la même architecture

Pour isoler l'effet du transfert d'apprentissage, les trois stratégies réutilisent **exactement la même architecture** (*ResNet-18*), en ne variant que l'origine des poids et les paramètres qui restent entraînables :

1. **`do_zero` :** poids aléatoires (`weights=None`) — équivalent à entraîner l'architecture de la *ResNet-18* entièrement de zéro, comme dans le Bloc 2 de l'Expérience 1.
2. **`congelado` :** poids pré-entraînés sur *ImageNet*, avec `requires_grad = False` sur toutes les couches convolutionnelles — seule la nouvelle couche finale est entraînée.
3. **`fine_tuning` :** poids pré-entraînés sur *ImageNet* comme point de départ, mais **sans** gel — tout le réseau s'adapte au nouveau domaine, avec un taux d'apprentissage faible pour ne pas détruire les connaissances préalables.

Dans tous les cas, la couche finale `fc` est remplacée par `nn.Linear(fc.in_features, 2)`, correspondant aux deux races cibles.

In [25]:
transformacao_resnet = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def prepara_tensores(imgs, labels):
    X = torch.stack([transformacao_resnet(img.convert("RGB")) for img in imgs])
    y = torch.tensor(labels, dtype=torch.long)
    return X, y

X_tr, y_tr = prepara_tensores(imgs_treino, y_treino)
X_te, y_te = prepara_tensores(imgs_teste, y_teste)

def cria_modelo_pets(estrategia):
    pesos = None if estrategia == "do_zero" else models.ResNet18_Weights.DEFAULT
    modelo = models.resnet18(weights=pesos)
    if estrategia == "congelado":
        for p in modelo.parameters():
            p.requires_grad = False
    modelo.fc = nn.Linear(modelo.fc.in_features, len(RACAS_ALVO))
    return modelo

modelo_do_zero = cria_modelo_pets("do_zero")
modelo_congelado = cria_modelo_pets("congelado")
modelo_fine_tuning = cria_modelo_pets("fine_tuning")

###### Bloc 3 : Entraînement comparatif et analyse de la précision

En réutilisant les fonctions génériques `treinar` et `calcular_acuracia`, définies dans le Bloc 2 de l’Expérience 1, les trois modèles sont entraînés sur le même ensemble de $30$ photographies et évalués sur l’ensemble de test (images jamais vues pendant l’entraînement) :

* Le modèle `do_zero` a tendance à **surajuster** rapidement aux $30$ photographies d’entraînement, sans généraliser à l’ensemble de test — $30$ exemples sont drastiquement insuffisants pour ajuster les $11$ millions de paramètres de la *ResNet-18* à partir de zéro.
* Le modèle `congelado` devrait déjà atteindre une précision considérablement supérieure, car il réutilise, sans aucun ajustement, des caractéristiques visuelles génériques (bords, textures, contours) apprises sur *ImageNet* — seule la nouvelle couche linéaire doit être ajustée aux $30$ photographies.
* Le modèle `fine_tuning` tend à égaler ou à dépasser l’extracteur figé, car il part des mêmes connaissances préalables, mais permet en outre un ajustement fin de tout le réseau aux particularités visuelles des races.

La [Figure 9.24](#fig-09-pets-comparativo) résume les trois résultats.

In [26]:
torch.manual_seed(42)

configuracoes = [
    ("Do zero",              modelo_do_zero,      None, 2e-3),
    ("Extrator congelado",   modelo_congelado,    "fc", 1e-3),
    ("Fine-tuning completo", modelo_fine_tuning,  None, 1e-4),
]

resultados_pets = {}
for nome, modelo, alvo_params, taxa in configuracoes:
    parametros = modelo.fc.parameters() if alvo_params == "fc" else None
    treinar(modelo, X_tr, y_tr, epocas=15, lr=taxa, tam_lote=8, parametros=parametros)
    resultados_pets[nome] = calcular_acuracia(modelo, X_te, y_te)
    print(f"{nome}: {resultados_pets[nome]*100:.1f}%")

plt.figure(figsize=(5.5, 4))
cores = ["#dc2626", "#f59e0b", "#16a34a"]
plt.bar(resultados_pets.keys(), resultados_pets.values(), color=cores)
plt.ylim(0, 1.05)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Chute aleatório (50%)")
plt.ylabel("Acurácia no teste")
plt.title("Pug vs. Boxer — 15 fotos de treino/classe")
for i, v in enumerate(resultados_pets.values()):
    plt.text(i, v + 0.02, f"{v*100:.1f}%", ha="center")
plt.xticks(rotation=10)
plt.legend()
plt.tight_layout()
plt.show()

**Figure 9.24:** Comparaison de précision sur l


###### Bloc 4 : Inspection qualitative des prédictions

Comme dans l’Expérience 1 et dans la section suivante sur le diagnostic foliaire, il est instructif d’observer individuellement certaines prédictions du meilleur modèle (typiquement `fine_tuning` ou `congelado`) sur des photographies réelles de test, en comparant l’étiquette prédite avec la race réelle.

In [27]:
melhor_modelo = modelo_fine_tuning  # ou modelo_congelado, selon le résultat du Bloc 3
melhor_modelo.eval()

idx_amostras = list(range(4)) + list(range(N_TESTE_POR_CLASSE, N_TESTE_POR_CLASSE + 4))

imgs_pred, titulos_pred = [], []
with torch.no_grad():
    for idx in idx_amostras:
        entrada = X_te[idx].unsqueeze(0)
        pred_idx = melhor_modelo(entrada).argmax(dim=1).item()
        real_idx = y_te[idx].item()
        marcador = "✓" if pred_idx == real_idx else "✗"
        imgs_pred.append(np.array(imgs_teste[idx].convert("RGB")))  # PIL -> ndarray
        titulos_pred.append(f"{marcador} previsto: {RACAS_ALVO[pred_idx]}\n"
                             f"real: {RACAS_ALVO[real_idx]}")

mm.show(imgs_pred, titles=titulos_pred, cols=4, figsize=(12, 7))

**Figure 9.25:** Prédictions du modèle avec extracteur pré-entraîné (ResNet-18) sur des photographies réelles de test : étiquette prédite vs. race réelle, quatre échantillons de chaque classe.


In [28]:
 
# Nettoyage explicite des données téléchargées et libération de la mémoire
if FLAG_LIMPAR_DADOS:
    if os.path.exists('./dados_pets'):
        shutil.rmtree('./dados_pets')
        print('🧹 Répertoire de données temporaires ./dados_pets supprimé avec succès.')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

##### Expérience 3 — Diagnostic phytosanitaire par apprentissage par transfert

L'expérience précédente a montré qu'une *ResNet-18* pré-entraînée sur **ImageNet** peut s'adapter à une nouvelle tâche en utilisant peu d'échantillons. Désormais, la même stratégie est appliquée à un problème de diagnostic phytosanitaire. La *ResNet-18* doit classer des images de feuilles en trois catégories : `["folha_saudavel", "folha_doente", "sintoma_desconhecido"]`.

- **`folha_saudavel` :** feuille sans lésions visibles.
- **`folha_doente` :** feuille présentant des taches sombres simulant une maladie fongique.
- **`sintoma_desconhecido` :** feuille avec chlorose jaunâtre, représentant un motif différent de la maladie connue.

> ### 💡 Dica
>
> ###### 🌱 Pourquoi ce scénario ?
>
> Le diagnostic phytosanitaire constitue une application importante de la vision par ordinateur (VC) dans l'agriculture de précision. Un modèle pré-entraîné sur **ImageNet** peut réutiliser des caractéristiques telles que les bords, les textures et les motifs de couleur pour apprendre cette nouvelle tâche avec peu d'images.

###### Bloc 1 : Génération du *dataset* synthétique

Ce bloc génère un ensemble synthétique avec **30** images par classe pour l'entraînement et **8** pour la validation, totalisant respectivement **90** et **24** images.

1. **Génération de la feuille :** La fonction `desenha_folha_base` crée le contour de la feuille, en variant la taille, l'orientation et la tonalité de vert.

2. **Simulation de la maladie :** La fonction `aplica_manchas_doenca` ajoute des taches sombres irrégulières simulant des lésions fongiques.

3. **Simulation d'un autre symptôme :** La fonction `aplica_sintoma_desconhecido` ajoute des régions jaunâtres représentant un motif distinct de la maladie connue.

4. **Visualisation des échantillons :** La [Figure 9.26](#fig-09-folhas-amostras) présente des exemples des trois catégories de l'ensemble synthétique.

In [29]:
CLASSES_FOLHA = ["folha_saudavel", "folha_doente", "sintoma_desconhecido"]


def desenha_folha_base(tam_img, rng):
    '''Dessine le contour ovale d'une feuille verte avec nervure centrale,
    avec de petites variations de teinte, de taille et d'orientation entre les échantillons.'''
    img = np.full((tam_img, tam_img, 3), 245, dtype=np.uint8)  # fond clair
    cx, cy = tam_img // 2, tam_img // 2
    eixo_a = rng.randint(int(tam_img * 0.30), int(tam_img * 0.38))
    eixo_b = rng.randint(int(tam_img * 0.20), int(tam_img * 0.26))
    angulo = rng.uniform(-15, 15)
    verde = (rng.randint(40, 70), rng.randint(120, 160), rng.randint(40, 70))
    cv2.ellipse(img, (cx, cy), (eixo_a, eixo_b), angulo, 0, 360, verde, -1, cv2.LINE_AA)
    ang_rad = np.deg2rad(angulo)
    dx, dy = np.cos(ang_rad), np.sin(ang_rad)
    p1 = (int(cx - eixo_a * dx), int(cy - eixo_a * dy))
    p2 = (int(cx + eixo_a * dx), int(cy + eixo_a * dy))
    cv2.line(img, p1, p2, (25, 90, 25), 2, cv2.LINE_AA)  # nervure centrale
    return img, (cx, cy, eixo_a, eixo_b, angulo)


def aplica_manchas_doenca(img, centro_folha, rng, n_manchas=(4, 8)):
    '''Simule des lésions foliaires : taches sombres aux bords irréguliers
    (motif typique des maladies fongiques).'''
    cx, cy, eixo_a, eixo_b, _ = centro_folha
    for _ in range(rng.randint(*n_manchas)):
        raio = rng.randint(1, 3)
        px = cx + rng.randint(-int(eixo_a * 0.7), int(eixo_a * 0.7))
        py = cy + rng.randint(-int(eixo_b * 0.7), int(eixo_b * 0.7))
        cor_mancha = (rng.randint(50, 90), rng.randint(25, 45), rng.randint(10, 25))
        cv2.circle(img, (px, py), raio, cor_mancha, -1, cv2.LINE_AA)
        cv2.circle(img, (px, py), raio + 1, (120, 85, 30), 1, cv2.LINE_AA)  # halo
    return img


def aplica_sintoma_desconhecido(img, centro_folha, rng):
    '''Simule un motif distinct (marbrures jaunâtres/chlorose), différent
    des taches sombres de la maladie connue.'''
    cx, cy, eixo_a, eixo_b, _ = centro_folha
    for _ in range(rng.randint(3, 5)):
        eixo_m = (rng.randint(1, 3), rng.randint(2, 3))
        px = cx + rng.randint(-int(eixo_a * 0.6), int(eixo_a * 0.6))
        py = cy + rng.randint(-int(eixo_b * 0.6), int(eixo_b * 0.6))
        cor_clorose = (rng.randint(200, 235), rng.randint(195, 225), rng.randint(50, 90))
        ang_m = rng.uniform(0, 180)
        cv2.ellipse(img, (px, py), eixo_m, ang_m, 0, 360, cor_clorose, -1, cv2.LINE_AA)
    return img


def aplica_ruido_sal_pimenta(img, prop_ruido=0.02, rng=None):
    '''Applique du bruit sel (points blancs) et poivre (points noirs) aléatoires.
    prop_bruit : fraction de pixels modifiés (ex : 0.02 = 2 % des pixels).'''
    if prop_ruido <= 0:
        return img
    
    img_ruido = img.copy()
    num_pixels = int(prop_ruido * img.shape[0] * img.shape[1])
    n_sal = num_pixels // 2
    n_pimenta = num_pixels - n_sal

    # Applique Sel (Blanc - [255, 255, 255])
    for _ in range(n_sal):
        y = rng.randint(0, img.shape[0] - 1)
        x = rng.randint(0, img.shape[1] - 1)
        img_ruido[y, x] = [255, 255, 255]

    # Applique Poivre (Noir - [0, 0, 0])
    for _ in range(n_pimenta):
        y = rng.randint(0, img.shape[0] - 1)
        x = rng.randint(0, img.shape[1] - 1)
        img_ruido[y, x] = [0, 0, 0]

    return img_ruido


def gera_folha(classe_idx, tam_img=128, rng=None, prop_ruido=0.02):
    rng = rng or random.Random()
    img, geometria = desenha_folha_base(tam_img, rng)
    nome = CLASSES_FOLHA[classe_idx]
    
    if nome == "folha_doente":
        img = aplica_manchas_doenca(img, geometria, rng)
    elif nome == "sintoma_desconhecido":
        img = aplica_sintoma_desconhecido(img, geometria, rng)
        
    ruido_exp = rng.randint(-3, 3)  # légère variation d'exposition
    img = np.clip(img.astype(np.int16) + ruido_exp, 0, 255).astype(np.uint8)
    
    # Application du bruit Sel et Poivre
    img = aplica_ruido_sal_pimenta(img, prop_ruido=prop_ruido, rng=rng)
    
    return img


def gera_conjunto(n_por_classe, tam_img=128, seed=0, prop_ruido=0.02):
    rng = random.Random(seed)
    imgs, labels = [], []
    for classe_idx in range(len(CLASSES_FOLHA)):
        for _ in range(n_por_classe):
          imgs.append(gera_folha(classe_idx, tam_img=tam_img, rng=rng, prop_ruido=prop_ruido))
          labels.append(classe_idx)
    return imgs, labels


N_POR_CLASSE_TREINO, N_POR_CLASSE_VAL = 30, 8

imgs_treino, labels_treino = gera_conjunto(n_por_classe=N_POR_CLASSE_TREINO, seed=42, 
                                           prop_ruido=0.02)
imgs_val, labels_val = gera_conjunto(n_por_classe=N_POR_CLASSE_VAL, seed=123, prop_ruido=0.02)

print(f"Entraînement : {len(imgs_treino)} images ({N_POR_CLASSE_TREINO} par classe) | "
      f"Validation : {len(imgs_val)} images ({N_POR_CLASSE_VAL} par classe)")

# Affichage de 2 échantillons de chaque classe (6 images au total)
amostras_exibir, titulos_exibir = [], []
for classe_idx, nome in enumerate(CLASSES_FOLHA):
    for k in range(2):
        idx = classe_idx * N_POR_CLASSE_TREINO + k
        amostras_exibir.append(imgs_treino[idx])
        titulos_exibir.append(nome)

mm.show(amostras_exibir, titles=titulos_exibir, cols=3, figsize=(10, 7))

Entraînement : 90 images (30 par classe) | Validation : 24 images (8 par classe)


<Figure size 1500x1050 with 6 Axes>

**Figure 9.26:** Échantillons synthétiques du *dataset* de diagnostic foliaire : feuille saine, feuille malade (taches sombres) et symptôme inconnu (chlorose jaunâtre) avec bruit sel et poivre.


> ### 📝 Nota
>
> ###### 🧠 Piège Fréquent
>
> Le transfert d'apprentissage exige que chaque classe présente des motifs visuels distincts. Répéter la même image avec des étiquettes différentes empêche la couche de classification d'apprendre une frontière de décision, car l'extracteur génère pratiquement les mêmes caractéristiques pour tous les échantillons.
>
> Dans cette expérience, chaque image est générée de manière indépendante, avec des motifs visuels compatibles avec sa classe (feuille saine, lésions fongiques ou chlorose), fournissant ainsi des informations suffisantes pour l'entraînement de la couche de classification.

###### Bloco 2: Préparation des tenseurs et adaptation de l'architecture

Des modèles comme *ResNet-18* exigent des images colorées de $224 \times 224$ *pixels* normalisées selon les statistiques d'*ImageNet* ($\mu = [0,485; 0,456; 0,406]$ et $\sigma = [0,229; 0,224; 0,225]$).

1. **Transformation d'entrée (`transforms.Compose`):** on applique le redimensionnement et la normalisation standard à chaque image du *dataset* synthétique, produisant les tenseurs `X_treino`/`X_val` et les étiquettes `y_treino`/`y_val` — chaque exemple est une image réellement distincte, associée à l'étiquette correcte de sa classe.
2. **Gel de l'extracteur:** la boucle `for p in modelo_resnet.parameters(): p.requires_grad = False` désactive les gradients dans les couches convolutionnelles pré-entraînées.
3. **Nouvelle couche finale:** la couche `modelo_resnet.fc` est remplacée par une nouvelle instance `nn.Linear(modelo_resnet.fc.in_features, n_classes_destino)`, récemment initialisée et avec des gradients actifs par défaut. Le nombre de caractéristiques d'entrée est obtenu dynamiquement à partir de la couche originale elle-même (`in_features`, égal à $512$ dans ResNet-18), plutôt que d'être fixé manuellement dans le code — pratique recommandée, car elle rend le fragment réutilisable pour d'autres variantes de l'architecture sans modifications.

In [30]:
# 1. Pipeline de transformations attendues par ResNet
transformacao_resnet = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def prepara_tensores(imgs, labels):
    tensores = [transformacao_resnet(T.functional.to_tensor(img)) for img in imgs]
    X = torch.stack(tensores)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y


# 2. Conversion du dataset synthétique (Bloc 1) en tenseurs normalisés
X_treino, y_treino = prepara_tensores(imgs_treino, labels_treino)
X_val, y_val = prepara_tensores(imgs_val, labels_val)

loader_treino = DataLoader(TensorDataset(X_treino, y_treino), batch_size=16, shuffle=True)
loader_val = DataLoader(TensorDataset(X_val, y_val), batch_size=16, shuffle=False)

# 3. Chargement de ResNet-18 pré-entraînée et gel de l'extracteur
n_classes_destino = len(CLASSES_FOLHA)
modelo_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for parametro in modelo_resnet.parameters():
    parametro.requires_grad = False

# 4. Remplacement de la couche finale pour les 3 nouvelles classes cibles
modelo_resnet.fc = nn.Linear(modelo_resnet.fc.in_features, n_classes_destino)

print(f"Nouvelle couche finale : {modelo_resnet.fc}")

###### Bloco 3: Boucle d’Entraînement et Évaluation

Avec l’extracteur figé et la nouvelle couche de sortie correctement intégrée, on exécute l’ajustement fin (*fine-tuning*) de la nouvelle tête de classification.

1. **Optimisation ciblée :** l’optimiseur *Adam* reçoit strictement `modelo_resnet.fc.parameters()`, ne mettant à jour que la nouvelle couche de sortie — le reste du réseau reste figé, comme défini dans le Bloco 2.
2. **Exécution de la boucle :** à chaque époque, le modèle itère sur les lots d’entraînement, calcule la perte par entropie croisée et ajuste les poids de la couche finale ; ensuite, on évalue la précision sur l’ensemble de validation (images jamais vues durant l’entraînement).
3. **Courbes d’entraînement :** la [Figure 9.27](#fig-09-resnet-treino) suit l’évolution de la perte d’entraînement et de la précision de validation au fil des époques — comme les trois classes sont visuellement distinctes entre elles, on s’attend à une convergence réelle, bien au-dessus du seuil de $33\%$ correspondant à un choix aléatoire parmi $3$ classes.

In [31]:
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_resnet = modelo_resnet.to(dispositivo)

criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_resnet.fc.parameters(), lr=1e-3)

historico_perda, historico_acc = [], []
epocas = 10

for epoca in range(epocas):
    modelo_resnet.train()
    perda_acumulada, n_batches = 0.0, 0

    for X_batch, y_batch in loader_treino:
        X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)

        otimizador.zero_grad()
        saidas = modelo_resnet(X_batch)
        perda = criterio(saidas, y_batch)
        perda.backward()
        otimizador.step()

        perda_acumulada += perda.item()
        n_batches += 1
    historico_perda.append(perda_acumulada / n_batches)

    modelo_resnet.eval()
    acertos, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)
            predicoes = modelo_resnet(X_batch).argmax(dim=1)
            acertos += (predicoes == y_batch).sum().item()
            total += y_batch.size(0)
    acc = acertos / total
    historico_acc.append(acc)

    print(f"Époque {epoca+1}/{epocas} — perte : {historico_perda[-1]:.4f} — ",
          f" précision_val : {acc*100:.1f}%")

f=mm.showTrainCurves(
    historico_perda, historico_acc,
    titulo="Fine-Tuning da ResNet-18 — Diagnóstico Foliar",
    subtitulo="Apenas a nova camada linear (fc) é treinada; o extrator permanece congelado",
)

**Figure 9.27:** Courbes d


###### Bloco 4: Inspeção Qualitativa das Previsões

Além da curva de precisão agregada, é instrutivo observar individualmente algumas previsões do modelo no conjunto de validação, comparando o rótulo previsto com o rótulo real. A [Figure 9.28](#fig-09-resnet-predicoes) exibe duas amostras de cada classe.

In [32]:
modelo_resnet.eval()

# Deux échantillons de chaque classe dans l'ensemble de validation
idx_amostras = [0, N_POR_CLASSE_VAL, 2 * N_POR_CLASSE_VAL,
                1, N_POR_CLASSE_VAL + 1, 2 * N_POR_CLASSE_VAL + 1]

imgs_pred, titulos_pred = [], []
with torch.no_grad():
    for idx in idx_amostras:
        entrada = X_val[idx].unsqueeze(0).to(dispositivo)
        pred_idx = modelo_resnet(entrada).argmax(dim=1).item()
        real_idx = y_val[idx].item()
        marcador = "✓" if pred_idx == real_idx else "✗"
        imgs_pred.append(imgs_val[idx])
        titulos_pred.append(f"{marcador} previsto: {CLASSES_FOLHA[pred_idx]}\n"+
                            f"real: {CLASSES_FOLHA[real_idx]}")

mm.show(imgs_pred, titles=titulos_pred, cols=3, figsize=(10, 7))

**Figure 9.28:** Prédictions de la ResNet-18 ajustée sur des échantillons de validation : étiquette prédite vs. étiquette réelle, avec deux échantillons par classe.


###### Analyse de l’expérience 3

La *ResNet-18* atteint une haute précision même en utilisant un ensemble réduit d’images synthétiques. Ce résultat montre que les représentations apprises sur **ImageNet** restent utiles dans un domaine complètement différent, ne nécessitant que l’adaptation de la couche de classification.

L’expérience illustre également une situation courante dans les applications réelles, où la disponibilité de données étiquetées est limitée. Dans ces scénarios, le transfert d’apprentissage réduit le temps d’entraînement et permet d’obtenir des modèles performants même sans entraîner l’ensemble du réseau.

> ### 📝 Nota
>
> ###### 🧠 Synthèse comparative — Quand le transfert d’apprentissage fonctionne-t-il ?
>
> Les trois expériences montrent que le transfert d’apprentissage dépend de la capacité de généralisation de l’extracteur de caractéristiques.
>
> - **Expérience 1 :** un petit extracteur, entraîné sur un domaine restreint, apprend des représentations peu généralisables et peut produire un **transfert négatif**.
>
> - **Expérience 2 :** une *ResNet-18* pré-entraînée sur **ImageNet** transfère des représentations générales à une tâche de classification de races de chiens et de chats, atteignant une haute précision avec peu d’échantillons.
>
> - **Expérience 3 :** la même stratégie adapte le modèle à un problème de diagnostic phytosanitaire, montrant qu’un seul extracteur peut servir de base à différents domaines d’application.

##### Comparação das Abordagens de Transferência

A [Tableau 9.2](#tbl-comparativo-transferencia) resume os resultados obtidos nos três experimentos.

<a id="tbl-comparativo-transferencia"></a>

**Tabela 9.2:** Comparação entre os três cenários de transferência de aprendizado apresentados nesta seção.

| Aspecto | Experimento 1 | Experimento 2 | Experimento 3 |
| --- | --- | --- | --- |
| **Extrator** | CNN pequena | *ResNet-18* | *ResNet-18* |
| **Treinamento do extrator** | Dígitos ($0$–$4$) | **ImageNet** | **ImageNet** |
| **Capacidade de generalização** | Baixa | Alta | Alta |
| **Nova tarefa** | Dígitos ($5$–$9$) | Raças de cães e gatos | Diagnóstico fitossanitário |
| **Resultado** | Transferência negativa | Transferência positiva | Transferência positiva |


### 9.5.2 Détection d'objets

Les expériences précédentes ont montré comment le transfert d'apprentissage adapte des modèles pré-entraînés à des tâches de classification d'images. Le même principe sous-tend également les architectures de détection d'objets, où un extracteur de caractéristiques pré-entraîné fournit des représentations visuelles générales, tandis que des modules spécialisés localisent et classifient les objets dans l'image.

Les sections suivantes présentent la **Faster R-CNN** comme exemple de détecteur pré-entraîné utilisé directement pour l'inférence, puis une expérience complète d'ajustement fin avec l'architecture **YOLO**.

#### 9.5.2.1 Faster R-CNN : Détecteur pré-entraîné

La détection d'objets étend l'utilisation de modèles pré-entraînés à une tâche plus complexe que la classification. La **Faster R-CNN** utilise une CNN pré-entraînée, comme la *ResNet-50*, comme **extracteur de caractéristiques** (*backbone*) et ajoute des modules spécialisés pour localiser et classifier les objets.

Concernant cet extracteur, l'architecture intègre deux « têtes » principales :

- ***Region Proposal Network* (RPN) :** propose des régions de l'image ayant une forte probabilité de contenir des objets.
- **Tête de classification :** affine ces régions, attribue une classe à chaque objet et ajuste ses boîtes englobantes.

Le code suivant utilise une Faster R-CNN avec des poids pré-entraînés sur **COCO** pour détecter des objets dans une image, produisant leurs classes, coordonnées et scores de confiance.

> ### 📝 Nota
>
> ##### 🔍 Où se trouve le transfert d'apprentissage ici ?
>
> Contrairement aux expériences précédentes, cet exemple **n'effectue pas d'ajustement fin** (*fine-tuning*). Le modèle exécute uniquement l'inférence (`eval()`), réutilisant directement les poids du *backbone*, de la RPN et de la tête de classification entraînés sur **COCO**.
>
> L'adaptation à un nouveau domaine exigerait de remplacer la couche `box_predictor` par une nouvelle tête de classification, compatible avec les classes de l'application, et de l'entraîner sur un ensemble d'images annotées. Cette procédure suit le même principe présenté dans la section sur le transfert d'apprentissage et constitue le flux habituel pour des applications spécifiques, telles que la détection de nuisibles, de défauts de fabrication ou de véhicules.

1. **Catégories COCO :** Le code récupère les noms des classes à partir des méta-informations des poids (`FasterRCNN_ResNet50_FPN_Weights.DEFAULT.meta["categories"]`). Bien que cette liste contienne $91$ entrées pour des raisons historiques liées au format d'annotation de COCO, seules $80$ correspondent à des catégories d'objets.

2. **Inférence :** L'image chargée par `mm.read()` est convertie en *tensor* et traitée par le modèle en mode évaluation (`eval()`). Le code ne conserve que les détections dont la confiance est supérieure à $80\%$.

3. **Annotation de l'image :** Pour chaque objet détecté, le code dessine la boîte englobante (`cv2.rectangle`) et écrit la classe prédite ainsi que sa confiance (`cv2.putText`).

4. **Visualisation :** La [Figure 9.29](#fig-09-deteccao-pretreinada) présente l'image annotée avec les détections réalisées par le modèle.

In [33]:
# 1. Chargement de l'image et des noms des catégories du COCO
url_imagem = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
url_imagem = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = mm.read(url_imagem)

pesos_coco = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
categorias_coco = pesos_coco.meta["categories"]  # Mappage indice -> nom de la classe

# 2. Chargement du modèle Faster R-CNN pré-entraîné
modelo_detection = fasterrcnn_resnet50_fpn(weights=pesos_coco).eval()

# 3. Exécution de l'inférence sans calcul de gradients
with torch.no_grad():
    predicao = modelo_detection([to_tensor(img)])[0]

# 4. Filtrage des détections avec une confiance supérieure à 80%
limiar_confianca = 0.8
mascara_confianca = predicao["scores"] >= limiar_confianca

caixas_filtradas = predicao["boxes"][mascara_confianca].numpy()
scores_filtrados = predicao["scores"][mascara_confianca].numpy()
labels_filtrados = predicao["labels"][mascara_confianca].numpy()

img_com_caixas = img.copy()

# 5. Dessin des boîtes englobantes et des étiquettes de classe
for box, score, label_idx in zip(caixas_filtradas, scores_filtrados, labels_filtrados):
    x1, y1, x2, y2 = box.astype(int)
    nome_classe = categorias_coco[label_idx]
    texto_rotulo = f"{nome_classe}: {score:.2f}"

    # Dessine le rectangle rouge (RGB : 255, 0, 0) avec une épaisseur de 3 pixels
    cv2.rectangle(img_com_caixas, (x1, y1), (x2, y2), (255, 0, 0), 3)

    # Écrit la classe et la confiance au-dessus de la boîte englobante
    cv2.putText(
        img_com_caixas,
        texto_rotulo,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2,
        cv2.LINE_AA,
    )

# 6. Affichage graphique de l'image résultante
mm.show(img_com_caixas, title="Faster R-CNN (COCO) — Détection avec classe et confiance")

**Figure 9.29:** Résultat de l


#### 9.5.2.2 Détection d'objets et transfert d'apprentissage avec YOLO

Les sections précédentes ont appliqué le transfert d'apprentissage à des problèmes de **classification d'images**, où le modèle associe une seule étiquette à l'image entière. Dans cette section, le même principe est étendu à la **détection d'objets**, une tâche qui exige d'identifier simultanément **ce qui** est présent dans l'image et **où** se trouve chaque objet.

La sous-section précédente a présenté la **Faster R-CNN** comme exemple de détecteur pré-entraîné utilisé directement pour l'inférence, sans aucune adaptation au nouveau domaine. Dans cette expérience, le modèle passe par une étape d'**ajustement fin** (*fine-tuning*) : on part d'une architecture **YOLO** (*You Only Look Once*) pré-entraînée sur l'ensemble **COCO** et on adapte le réseau pour détecter et classer les objets d'un nouveau domaine.

Contrairement à la Faster R-CNN, qui effectue la détection en deux étapes, la famille **YOLO** adopte une architecture à étape unique (*single-stage detector*), estimant, en une seule propagation à travers le réseau, les boîtes englobantes (*bounding boxes*), la confiance de chaque détection et la classe correspondante. Cette stratégie réduit le coût computationnel et rend possibles des applications en temps réel.

À titre d'exemple, l'expérience utilise un ensemble synthétique de formes géométriques (triangles, carrés, étoiles, entre autres), avec des variations de couleur, de taille, de rotation et une dégradation par bruit de type sel et poivre.

##### Bloc 1 : Génération du *dataset* synthétique

Le bloc suivant génère un ensemble synthétique pour l’entraînement et l’évaluation du détecteur. Chaque image contient entre un et trois objets appartenant à l’une des neuf classes :

```python
CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
```

1. **Génération des formes :** Les fonctions `poligono_regular`, `poligono_estrela` et `poligono_cruz` construisent les coordonnées des objets. La fonction `desenha_objeto` dessine chaque forme avec une position, une taille, une orientation et une couleur aléatoires et calcule sa *bounding box*.

2. **Annotation au format YOLO :** La fonction `gera_imagem_ruidosa` génère entre un et trois objets par image et convertit chaque *bounding box* au format YOLO, représenté par la classe et les coordonnées normalisées du centre, de la largeur et de la hauteur.

3. **Dégradation de l’image :** La fonction `adiciona_ruido_sal_pimenta` ajoute un bruit impulsif, simulant des imperfections d’acquisition.

4. **Visualisation des échantillons :** La [Figure 9.30](#fig-09-yolo-amostras-iniciais) présente des exemples de l’ensemble synthétique avec les *bounding boxes* superposées par la fonction `mm.showBoundBox()`.

In [34]:
CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
N_LADOS = {'Triangle': 3, 'Square': 4, 'Pentagon': 5, 'Hexagon': 6, 'Heptagon': 7}

def poligono_regular(cx, cy, r, n_lados, rot_graus):
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + 2 * np.pi * np.arange(n_lados) / n_lados
    return np.stack([cx + r * np.cos(angs), cy + r * np.sin(angs)], axis=1)

def poligono_estrela(cx, cy, r_externo, rot_graus, n_pontas=5):
    r_interno = r_externo * 0.45
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + np.pi * np.arange(2 * n_pontas) / n_pontas
    raios = np.where(np.arange(2 * n_pontas) % 2 == 0, r_externo, r_interno)
    return np.stack([cx + raios * np.cos(angs), cy + raios * np.sin(angs)], axis=1)

def poligono_cruz(cx, cy, r, rot_graus, espessura_rel=0.35):
    w = r * espessura_rel
    base = np.array([
        (-w, -r), (w, -r), (w, -w), (r, -w), (r, w), (w, w),
        (w, r), (-w, r), (-w, w), (-r, w), (-r, -w), (-w, -w),
    ])
    theta = np.deg2rad(rot_graus)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return base @ R.T + np.array([cx, cy])

def desenha_objeto(img, classe_idx, cx, cy, tamanho, rotacao, cor):
    nome = CLASSES[classe_idx]
    if nome in N_LADOS:
        pts = poligono_regular(cx, cy, tamanho, N_LADOS[nome], rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Star':
        pts = poligono_estrela(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Cross':
        pts = poligono_cruz(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Circle':
        cv2.circle(img, (int(cx), int(cy)), int(tamanho), cor, -1)
        xs, ys = np.array([cx - tamanho, cx + tamanho]), np.array([cy - tamanho, cy + tamanho])
    else:  # Ellipse
        eixo = (int(tamanho), int(tamanho * 0.6))
        cv2.ellipse(img, (int(cx), int(cy)), eixo, rotacao, 0, 360, cor, -1)
        ang = np.deg2rad(rotacao)
        dx = np.hypot(eixo[0] * np.cos(ang), eixo[1] * np.sin(ang))
        dy = np.hypot(eixo[0] * np.sin(ang), eixo[1] * np.cos(ang))
        xs, ys = np.array([cx - dx, cx + dx]), np.array([cy - dy, cy + dy])
    return xs.min(), ys.min(), xs.max(), ys.max()

def adiciona_ruido_sal_pimenta(img, quantidade=0.05):
    img_ruidosa = img.copy()
    h, w, c = img_ruidosa.shape
    num_ruido = int(quantidade * h * w)
    
    # Sel (255, 255, 255)
    coords_sal = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_sal[0], coords_sal[1]] = [255, 255, 255]
    
    # Poivre (0, 0, 0)
    coords_pimenta = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_pimenta[0], coords_pimenta[1]] = [0, 0, 0]
    
    return img_ruidosa

def gera_imagem_ruidosa(tam_img=160, n_objetos=(1, 3), taxa_ruido=0.01, rng=None):
    rng = rng or random.Random()
    img_limpa = np.full((tam_img, tam_img, 3), 255, dtype=np.uint8)
    anotacoes = []
    
    for _ in range(rng.randint(*n_objetos)):
        classe_idx = rng.randrange(len(CLASSES))
        tamanho = rng.randint(tam_img // 10, tam_img // 5)
        cx = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        cy = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        rotacao = rng.uniform(0, 360)
        cor = tuple(rng.sample(range(30, 226), 3))
        
        x0, y0, x1, y1 = desenha_objeto(img_limpa, classe_idx, cx, cy, tamanho, rotacao, cor)
        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, tam_img), min(y1, tam_img)
        
        # Format YOLO : (classe, x_centre, y_centre, largeur, hauteur) normalisés
        xc, yc = (x0 + x1) / 2 / tam_img, (y0 + y1) / 2 / tam_img
        w, h = (x1 - x0) / tam_img, (y1 - y0) / tam_img
        anotacoes.append((classe_idx, xc, yc, w, h)) 
        
    img_ruidosa = adiciona_ruido_sal_pimenta(img_limpa, quantidade=taxa_ruido)
    return img_ruidosa, anotacoes

In [35]:
# Génération de 5 échantillons pour affichage initial en haut du projet
n_amostras_iniciais = 5
rng_demo = random.Random(42)

imgs_demo = []
titulos_demo = []

for idx in range(n_amostras_iniciais):
    img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_demo)
    
    # Enregistrement temporaire de l'annotation pour lecture native par mm.showBoundBox
    filename_temp = f"temp_label_{idx}.txt"
    with open(filename_temp, "w") as f:
        for c, xc, yc, w, h in anotacoes:
            f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")
            
    img_anotada = mm.showBoundBox(img_ruid, filename=filename_temp, fmt="yolo", show=False)
    imgs_demo.append(img_anotada)
    titulos_demo.append(f"Amostra {idx+1}")

# Affichage du panneau de 5 échantillons
mm.show(
    imgs_demo,
    titles=titulos_demo,
    cols=n_amostras_iniciais,
    figsize=(14, 3)
)

<Figure size 2100x450 with 5 Axes>

**Figure 9.30:** Échantillons initiaux du *dataset* synthétique bruité d


##### Bloc 2 : Organisation du *Dataset* et création du fichier `data.yaml`

Ce bloc organise l'ensemble de données au format attendu par la bibliothèque **Ultralytics YOLO**. Les images et les annotations sont réparties dans des répertoires séparés pour l'entraînement et la validation, tandis que le fichier `data.yaml` regroupe les informations nécessaires à l'entraînement du détecteur.

```text
shapes_dataset/
├── data.yaml
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

1. **Génération de l'ensemble de données :** Le code crée **90** images pour l'entraînement et **20** pour la validation. Pour chaque image, il enregistre un fichier `.txt` contenant une ligne par objet, au format YOLO (`classe`, `x_c`, `y_c`, `largeur`, `hauteur`), avec toutes les coordonnées normalisées.

2. **Organisation des fichiers :** Les images sont stockées dans `images/train` et `images/val`, tandis que les annotations correspondantes sont enregistrées dans `labels/train` et `labels/val`, en préservant le même nom de fichier.

3. **Création du fichier `data.yaml` :** Le code génère automatiquement le fichier de configuration contenant le chemin du *dataset*, les répertoires d'entraînement et de validation, ainsi que la correspondance entre les indices numériques et les noms des neuf classes.

In [36]:
base_dir = "shapes_dataset"
rng_global = random.Random(42)

for split, n_imgs in [("train", 90), ("val", 20)]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)
    
    for i in range(n_imgs):
        img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_global)
        cv2.imwrite(f"{base_dir}/images/{split}/{i:04d}.jpg", img_ruid)
        
        with open(f"{base_dir}/labels/{split}/{i:04d}.txt", "w") as f:
            for c, xc, yc, w, h in anotacoes:
                f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")

with open(f"{base_dir}/data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath(base_dir)}\n"
        "train: images/train\nval: images/val\nnames:\n"
    )
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Jeu de données bruité généré avec succès : 90 images d'entraînement et 20 de validation.")

Jeu de données bruité généré avec succès : 90 images d'entraînement et 20 de validation.


##### Bloco 3: Pré-traitement avec filtre médian

Ce bloc applique un pré-traitement pour réduire l'effet du bruit de type sel et poivre introduit lors de la génération du *dataset*. Le **filtre médian** (`cv2.medianBlur`) supprime ce type de dégradation tout en préservant mieux les contours des objets que les filtres de lissage conventionnels.

1. **Filtrage de l'image :** Le code applique un filtre médian avec une fenêtre $3 \times 3$ à l'image bruitée, réduisant les *pixels* impulsifs sans modifier les annotations de l'ensemble de données.

2. **Visualisation comparative :** La [Figure 9.31](#fig-09-yolo-pre-processamento) compare l'image originale et l'image filtrée, en maintenant les *bounding boxes* superposées grâce à la fonction `mm.showBoundBox()`.

In [37]:
# 1. Chargement du premier échantillon bruité du dataset
caminho_img = f"{base_dir}/images/train/0000.jpg"
caminho_label = f"{base_dir}/labels/train/0000.txt"

img_ruidosa = mm.read(caminho_img)

# 2. Prétraitement avec Filtre Médian (fenêtre 3x3)
img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)

# 3. Superposition des bounding boxes avec mm.showBoundBox
img_ruid_anotada = mm.showBoundBox(img_ruidosa, filename=caminho_label, fmt="yolo", show=False)
img_filt_anotada = mm.showBoundBox(img_filtrada, filename=caminho_label, fmt="yolo", show=False)

# 4. Affichage comparatif avec mm.show
mm.show(
    [img_ruid_anotada, img_filt_anotada],
    titles=[
        "1. Image Bruyante Originale (Sel et Poivre)",
        "2. Prétraitée (Filtre Médian 3x3)"
    ],
    cols=2,
    figsize=(9, 4)
)

<Figure size 1350x600 with 2 Axes>

**Figure 9.31:** Comparaison entre l


##### Bloco 4: Prétraitement du *dataset* et ajustement fin de YOLOv8

Ce bloc applique le filtre médian à l'ensemble d'images et réalise l'ajustement fin (*fine-tuning*) du détecteur **YOLOv8n** pré-entraîné sur l'ensemble **COCO**. Le filtrage réduit l'effet du bruit impulsif introduit lors de la génération des images, tandis que l'entraînement adapte les paramètres du réseau au nouveau domaine de formes géométriques.

1. **Filtrage par lots :** Le code parcourt les répertoires `train` et `val` et applique `cv2.medianBlur` avec une fenêtre $3 \times 3$ sur toutes les images, en conservant les annotations YOLO d'origine.

2. **Ajustement fin du détecteur :** Le réseau `YOLO("yolov8n.pt")`, initialement entraîné sur COCO, est adapté à l'ensemble géométrique via la fonction `.train()`. L'entraînement utilise des images avec une résolution de $320 \times 320$ *pixels* pendant $30$ époques.

3. **Évaluation du modèle :** La fonction `.val()` calcule les métriques de détection sur l'ensemble de validation, y compris la **précision** (*precision*), le **rappel** (*recall*) et la **mAP50** (*mean Average Precision* avec un seuil d'IoU égal à $0,5$).

In [38]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

base_dir = "shapes_dataset"
base_dir_filt = "shapes_dataset_filtrado"

# 1. Crée une COPIE filtrée du jeu de données dans un dossier séparé
#    (le jeu de données original dans base_dir reste bruité, intact)
for split in ["train", "val"]:
    pasta_imgs_orig = f"{base_dir}/images/{split}"
    pasta_labels_orig = f"{base_dir}/labels/{split}"
    pasta_imgs_filt = f"{base_dir_filt}/images/{split}"
    pasta_labels_filt = f"{base_dir_filt}/labels/{split}"

    os.makedirs(pasta_imgs_filt, exist_ok=True)
    os.makedirs(pasta_labels_filt, exist_ok=True)

    for nome_arq in os.listdir(pasta_imgs_orig):
        if nome_arq.endswith(".jpg"):
            img_ruidosa = cv2.imread(f"{pasta_imgs_orig}/{nome_arq}")
            img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)
            # écrit la version filtrée dans le dossier NOUVEAU, ne remplace pas l'original
            cv2.imwrite(f"{pasta_imgs_filt}/{nome_arq}", img_filtrada)

    # copie les étiquettes (elles ne changent pas avec le filtre)
    for nome_arq in os.listdir(pasta_labels_orig):
        shutil.copy(f"{pasta_labels_orig}/{nome_arq}", f"{pasta_labels_filt}/{nome_arq}")

# 2. data.yaml pointant vers le jeu de données FILTRÉ
with open(f"{base_dir_filt}/data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath(base_dir_filt)}\n"
        "train: images/train\nval: images/val\nnames:\n"
    )
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Prétraitement terminé : jeu de données filtré enregistré dans un dossier séparé.\n")

# Télécharger le modèle YOLOv8 pré-entraîné (yolov8n.pt) s'il n'est pas déjà présent
with contextlib.redirect_stdout(io.StringIO()), \
    contextlib.redirect_stderr(io.StringIO()):
    modelo_yolo = YOLO("yolov8n.pt")
print("Modèle YOLOv8 chargé.")

Prétraitement terminé : jeu de données filtré enregistré dans un dossier séparé.

Modèle YOLOv8 chargé.


In [39]:
# 3. Callback personnalisé pour imprimer uniquement l'époque en cours d'exécution
def on_train_epoch_start(trainer):
    epoch_atual = trainer.epoch + 1
    total_epochs = trainer.epochs
    # Écrit directement dans la sortie standard d'origine (en contournant le silencieux)
    sys.__stdout__.write(f"🔄 Processando Época {epoch_atual}/{total_epochs}...\n")
    sys.__stdout__.flush()

# Ajoute le callback au modèle
modelo_yolo.add_callback("on_train_epoch_start", on_train_epoch_start)

# 4. Gestionnaire de contexte pour silencer les déchets d'Ultralytics (C/C++ et Python)
@contextlib.contextmanager
def silenciar_logs():
    logger = logging.getLogger("ultralytics")
    disabled_state = logger.disabled
    logger.disabled = True
    
    with open(os.devnull, "w") as fnull:
        old_stdout_fd = os.dup(1)
        old_stderr_fd = os.dup(2)
        try:
            os.dup2(fnull.fileno(), 1)
            os.dup2(fnull.fileno(), 2)
            with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
                yield
        finally:
            os.dup2(old_stdout_fd, 1)
            os.dup2(old_stderr_fd, 2)
            os.close(old_stdout_fd)
            os.close(old_stderr_fd)
            logger.disabled = disabled_state

# Exécution de l'entraînement
print("--- Démarrage de l'entraînement YOLOv8 ---")
with silenciar_logs():
    resultados_treino = modelo_yolo.train(
        data=f"{base_dir_filt}/data.yaml",   # <-- entraîne sur le jeu de données filtré
        epochs=30,
        imgsz=320,
        batch=16,
        device=device,
        verbose=False,
        plots=False
    )
    metricas = modelo_yolo.val(verbose=False)

# 5. Métriques finales
precision = metricas.results_dict["metrics/precision(B)"]
recall = metricas.results_dict["metrics/recall(B)"]
map50 = metricas.results_dict["metrics/mAP50(B)"]

print("\n--- Performance optimisée du modèle YOLOv8 ---")
print(f"Précision : {precision*100:.2f}%")
print(f"Rappel (Recall) : {recall*100:.2f}%")
print(f"mAP à 50% (IoU 0,50) : {map50*100:.2f}%")

--- Démarrage de l'entraînement YOLOv8 ---



--- Performance optimisée du modèle YOLOv8 ---
Précision : 70.06%
Rappel (Recall) : 88.26%
mAP à 50% (IoU 0,50) : 87.32%


##### Bloc 5 : Comparaison d’inférence : Image bruitée et image restaurée

Après le réglage fin de YOLOv8 sur l’ensemble restauré, une comparaison visuelle est effectuée entre la détection appliquée directement sur une image dégradée par le bruit sal et poivre et la même image après le filtre médian.

L’objectif est d’observer comment une étape simple de prétraitement peut influencer la qualité des prédictions d’un détecteur déjà adapté au nouveau domaine.

1. **Inférence avec YOLO :** La fonction `modelo_yolo.predict()` exécute la détection sur les deux versions de l’image, en utilisant un seuil de confiance de $25\%$ (`conf=0.25`).

2. **Visualisation des prédictions :** La fonction `plot()` génère les images annotées avec les boîtes englobantes et les étiquettes prédites par le modèle. La [Figure 9.32](#fig-09-yolo-inferencia-comparativa) présente la comparaison entre les deux scénarios.

In [40]:
# 1. Chargement d'un échantillon de test original (sans le filtre sauvegardé en lot)
caminho_teste = f"{base_dir}/images/val/0002.jpg"
img_ruidosa_teste = mm.read(caminho_teste)

# 2. Application ponctuelle du Filtre Médian (3x3) pour comparaison
img_filtrada_teste = cv2.medianBlur(img_ruidosa_teste, ksize=3)

# 3. Inférence avec le modèle YOLOv8 entraîné
pred_ruidosa = modelo_yolo.predict(img_ruidosa_teste, conf=0.25, verbose=False)[0]
pred_filtrada = modelo_yolo.predict(img_filtrada_teste, conf=0.25, verbose=False)[0]

# 4. Extraction des matrices annotées par le générateur du YOLO (conversion BGR -> RGB)
img_pred_ruid = cv2.cvtColor(pred_ruidosa.plot(), cv2.COLOR_BGR2RGB)
img_pred_filt = cv2.cvtColor(pred_filtrada.plot(), cv2.COLOR_BGR2RGB)

# 5. Affichage comparatif standardisé via mm.show
mm.show(
    [img_pred_ruid, img_pred_filt],
    titles=[
        f"Inférence sur l'image bruitée ({len(pred_ruidosa.boxes)} objets)",
        f"Inférence sur l'image filtrée ({len(pred_filtrada.boxes)} objets)"
    ],
    cols=2,
    figsize=(10, 4)
)

**Figure 9.32:** Comparaison de l


In [41]:
# Nettoyage explicite des données téléchargées et libération de la mémoire
if FLAG_LIMPAR_DADOS:
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
        print('🧹 Diretório de dados temporários {} removido com sucesso.'.format(base_dir))

    if os.path.exists(base_dir_filt):
        shutil.rmtree(base_dir_filt)
        print('🧹 Diretório de dados temporários {} removido com sucesso.'.format(base_dir_filt))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

> ### 📝 Nota
>
> ###### 🧠 Un domaine bien plus éloigné que celui des chiffres
>
> Dans l'expérience de transfert d'apprentissage entre chiffres manuscrits (domaines A et B), la tâche source et la tâche cible partageaient des statistiques visuelles très proches : toutes deux étaient des traits en niveaux de gris sur fond uniforme. Ici, la distance entre les domaines est bien plus grande — le YOLO a été pré-entraîné sur des **photographies naturelles en couleur** de COCO (personnes, animaux, véhicules, objets du quotidien), et la tâche cible consiste en **formes géométriques synthétiques, de couleur unie et au contour bien défini**, sans texture, éclairage ni arrière-plan complexe.
>
> Néanmoins, le transfert d'apprentissage reste avantageux : les couches initiales d'un détecteur entraîné sur COCO apprennent des filtres génériques — détecteurs de bords, de coins et de régions de contraste — qui restent utiles pour délimiter le contour d'un triangle ou d'une étoile, même si le contenu visuel final est assez distinct. C'est pourquoi permettre l'ajustement fin (*fine-tuning*) de toutes les couches, combiné à un prétraitement cohérent entre l'entraînement et l'inférence pour atténuer le bruit sel et poivre, permet d'atteindre des taux de précision élevés dans la détection d'objets du nouveau domaine.

##### Utilisation d’un Ensemble de Données Réel

Le *pipeline* ci-dessus a été entièrement construit autour du **format
d’annotation YOLO** (classe, $x_{centre}$, $y_{centre}$, largeur, hauteur,
normalisés par la largeur et la hauteur de l’image), précisément afin qu’il
puisse être réutilisé sans modification si le lecteur dispose d’un ensemble
d’images réelles annotées de la même manière — par exemple, un ensemble
d’images d’objets géométriques photographiés ou rendus, chacun accompagné
d’un fichier `.txt` correspondant au même format utilisé ici. Pour cela, il
suffirait de :

1. Organiser les images réelles dans `shapes_dataset/images/train` et
   `shapes_dataset/images/val`, ainsi que les fichiers `.txt` d’annotation
   correspondants dans les dossiers `labels/train` et `labels/val` (un fichier
   d’annotation par image, même nom de base, extension `.txt`) ;
2. Ajuster le fichier `data.yaml` si le nombre ou les noms des classes
   diffèrent ;
3. Exécuter les mêmes cellules d’entraînement, d’ajustement fin et de
   visualisation déjà présentées, sans aucune autre modification de code.

Cette séparation entre **génération/organisation des données** et
**entraînement du modèle** est, en pratique, la raison pour laquelle les
formats d’annotation standardisés (comme celui du YOLO) sont si largement
adoptés : ils permettent de remplacer l’ensemble de données d’entrée —
synthétique par réel, un domaine par un autre — tout en maintenant inchangé
tout le reste du *pipeline* de transfert d’apprentissage.

### 9.5.3 Segmentation d'objets

Le même principe de transfert d'apprentissage sous-tend également les architectures de segmentation d'images, dans lesquelles un extracteur de caractéristiques pré-entraîné fournit des représentations visuelles générales, tandis qu'une tête spécialisée effectue la classification dense, pixel par pixel.

Les sections suivantes présentent **DeepLabV3** comme exemple de segmentateur pré-entraîné utilisé directement pour l'inférence, puis l'architecture **U-Net**, entraînée de zéro et comparée à une ligne de base morphologique classique.

#### 9.5.3.1 DeepLabV3 : Segmentateur Pré-entraîné

En segmentation sémantique, l’objectif ne se limite pas à la localisation des objets par des boîtes englobantes (*bounding boxes*). Le réseau attribue une classe à chaque *pixel* de l’image, produisant une carte d’étiquettes ayant la même résolution que l’entrée. Des architectures telles que **DeepLabV3**, avec un *backbone* **ResNet-50**, utilisent un extracteur de caractéristiques pré-entraîné et une tête spécialisée pour réaliser cette classification dense.

1. **Chargement et inférence :** Le modèle `deeplabv3_resnet50(weights="DEFAULT")` charge des poids pré-entraînés sur l’ensemble **Pascal VOC**, qui définit $21$ classes de segmentation. Le code convertit l’image en *tensor*, ajoute la dimension de *batch* (`unsqueeze(0)`) et exécute l’inférence.

2. **Carte des classes :** La sortie du modèle possède la dimension $(1, 21, H, W)$, contenant une valeur pour chaque classe à chaque *pixel*. L’opération `.argmax(dim=1)` sélectionne la classe ayant la réponse la plus élevée à chaque position, générant une matrice bidimensionnelle d’étiquettes de dimensions $(H, W)$.

3. **Visualisation :** Le code convertit la carte d’étiquettes au format attendu par `mm.show()`, qui affiche le résultat de la segmentation sur la [Figure 9.33](#fig-09-segmentacao-pretreinada)..

In [42]:
# 1. Chargement de l'image (retourne numpy.ndarray)
url_imagem = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
url_imagem = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = mm.read(url_imagem)

# 2. Chargement du modèle DeepLabV3 pré-entraîné en mode évaluation
modelo_segmentacao = deeplabv3_resnet50(weights="DEFAULT").eval()

# 3. Exécution de l'inférence sans calcul de gradients
with torch.no_grad():
    tensor_entrada = to_tensor(img).unsqueeze(0)  # Format (1, C, H, W)
    saida = modelo_segmentacao(tensor_entrada)["out"]
    
    # Sélection de la classe avec la plus grande probabilité par pixel (argmax sur l'axe des canaux)
    mapa_classes = saida.argmax(dim=1).squeeze(0).byte().cpu().numpy()

# 4. Affichage de la carte de segmentation sémantique
mm.show(
    [img,mapa_classes],
    title=["Image originale","Segmentation sémantique (DeepLabV3)"]
)

**Figure 9.33:** Carte de segmentation sémantique générée par le modèle DeepLabV3 pré-entraîné : classification pixel par pixel représentée dans une matrice 2D affichée.


#### 9.5.3.2 Segmentação Semântica com Arquitetura U-Net

A subseção anterior apresentou um modelo pré-treinado (*DeepLabV3*) produzindo diretamente um rótulo de classe por *pixel*. Esta seção completa a sequência de projetos práticos — classificação e detecção — abordando a segmentação semântica implementada e treinada do zero com a **U-Net**, a arquitetura de referência introduzida por Ronneberger (2015)..

Na classificação, o mapa de características final era achatado (*flatten*) em um vetor, descartando a informação espacial em favor de um único rótulo por imagem. A U-Net, por sua vez, produz uma saída com a **mesma resolução espacial da entrada**: um mapa bidimensional no qual cada *pixel* recebe sua própria classificação. Essa exigência — preservar detalhe espacial de alta resolução ao mesmo tempo em que se constrói contexto semântico em camadas profundas — motiva a arquitetura de codificador-decodificador com conexões de atalho (*skip connections*).

O cenário utilizado simula a segmentação de nódulos em exames médicos sintéticos: imagens em escala de cinza contêm uma região circular ("nódulo") sobreposta a um fundo, ambos contaminados por ruído gaussiano e com médias de intensidade muito próximas — um desafio intencional de baixo contraste, ideal para demonstrar o ganho do aprendizado espacial em relação à limiarização pontual.

##### Bloc 1 : Générateur de l'ensemble synthétique de nodules et affichage initial

Le générateur suivant produit des paires (image, masque) : l'image contient une région circulaire d'intensité légèrement supérieure à celle du fond, toutes deux affectées par le même écart-type de bruit gaussien. Le masque binaire délimite exactement la région du nodule et sert de vérité de référence (*ground truth*).

1. **Construction du nodule :** La fonction `gera_imagem_com_nodulo` superpose le nodule au fond sur une matrice $64 \times 64$ et applique un bruit gaussien.
2. **Visualisation avec `mm.show` :** La [Figure 9.34](#fig-09-unet-dataset) illustre les deux premiers échantillons et leurs masques respectifs.

In [43]:
TAM_IMG = 64


def gera_imagem_com_nodulo(
    tam=TAM_IMG,
    raio_min=7,
    raio_max=15,
    media_fundo=95,
    media_nodulo=118,
    sigma_ruido=26,
    rng=None,
):
    rng = rng or np.random.default_rng()
    fundo = rng.normal(media_fundo, sigma_ruido, (tam, tam))
    nodulo = rng.normal(media_nodulo, sigma_ruido, (tam, tam))
    mascara = np.zeros((tam, tam), dtype=np.uint8)

    raio = int(rng.integers(raio_min, raio_max))
    cx = int(rng.integers(raio + 4, tam - raio - 4))
    cy = int(rng.integers(raio + 4, tam - raio - 4))

    cv2.circle(mascara, (cx, cy), raio, 255, -1)
    imagem = np.clip(np.where(mascara > 0, nodulo, fundo), 0, 255).astype(
        np.uint8
    )
    return imagem, mascara


# Génération des ensembles d'entraînement et de validation
rng_dados = np.random.default_rng(42)
N_TREINO, N_VAL = 160, 40

imgs_treino, masks_treino = zip(
    *[gera_imagem_com_nodulo(rng=rng_dados) for _ in range(N_TREINO)]
)
imgs_val, masks_val = zip(
    *[gera_imagem_com_nodulo(rng=rng_dados) for _ in range(N_VAL)]
)

print(
    f"Ensemble d'entraînement : {N_TREINO} images | Ensemble de validation : {N_VAL} images\n"
)

# Affichage d'échantillons initiaux 
mm.show(
    [imgs_treino[0], masks_treino[0], imgs_treino[1], masks_treino[1]],
    titles=["Image 1", "Masque 1", "Image 2", "Masque 2"],
    cols=4,
    figsize=(11, 3),
)

Ensemble d'entraînement : 160 images | Ensemble de validation : 40 images



<Figure size 1650x450 with 4 Axes>

**Figure 9.34:** Échantillons du jeu de données synthétique de nodules : image en niveaux de gris sous bruit et masque binaire de référence correspondant.


##### Bloco 2: Ligne de base classique (filtrage, Otsu et morphologie)

Avant d'employer l'U-Net, on évalue la performance d'un *pipeline* morphologique classique construit avec la bibliothèque `morph` : un **lisseur gaussien** (`mm.blur`), un **seuillage d'Otsu** (`mm.threshold`) et une **ouverture morphologique** (`mm.open`) pour l'élimination des bruits isolés.

1. **Métrique d'Intersection sur Union (IoU) :** La fonction `iou_mascaras` calcule le degré de chevauchement *pixel par pixel* entre la prédiction et le masque réel.
2. **Exécution et comparatif :** La [Figure 9.35](#fig-09-unet-classico) affiche le résultat de la segmentation classique sur une image de test, démontrant les limitations du seuil global sous faible contraste.

In [44]:
def iou_mascaras(predita, referencia):
    p, r = predita > 0, referencia > 0
    intersecao = np.logical_and(p, r).sum()
    uniao = np.logical_or(p, r).sum()
    return intersecao / uniao if uniao else 1.0


def segmenta_classico(imagem, elemento_estrutural):
    suavizada = mm.blur(imagem, 7)
    binaria = mm.threshold(suavizada)
    return mm.open(binaria, elemento_estrutural)


elemento_estrutural = mm.sedisk(5)
ious_classico = [
    iou_mascaras(segmenta_classico(img, elemento_estrutural), mask)
    for img, mask in zip(imgs_val, masks_val)
]
iou_classico_medio = float(np.mean(ious_classico))
print(
    f"IoU moyen (ligne de base classique) sur la validation : {iou_classico_medio:.4f}\n"
)

predicao_classica_exemplo = segmenta_classico(imgs_val[0], elemento_estrutural)

mm.show(
    [imgs_val[0], masks_val[0], predicao_classica_exemplo],
    titles=["Image", "Masque de référence", "Prédiction classique"],
    cols=3,
    figsize=(9, 3.2),
)

IoU moyen (ligne de base classique) sur la validation : 0.7516



<Figure size 1350x480 with 3 Axes>

**Figure 9.35:** Ligne de base classique de segmentation : lissage, seuillage d


##### Bloco 3: Construção da Arquitetura U-Net e Funções de Perda

A U-Net é uma arquitetura em formato de "U" (daí o nome), pensada especificamente para segmentação de imagens. Ela é formada por dois caminhos que trabalham em conjunto:

- 🔽 **Codificador (encoder):** desce pela imagem, reduzindo a resolução espacial a cada etapa enquanto extrai características cada vez mais abstratas (bordas → texturas → formas → contexto).
- 🔼 **Decodificador (decoder):** sobe de volta, reconstruindo a resolução original através de convoluções transpostas (*upsampling*), até gerar uma máscara do mesmo tamanho da imagem de entrada.

O elemento que torna a U-Net especial são as **conexões de atalho** (*skip connections*): elas levam os mapas de características do codificador diretamente para a etapa correspondente do decodificador, na mesma resolução. Isso evita que detalhes finos — como contornos e bordas — se percam durante a compressão espacial.

**Fluxo geral da arquitetura:**

```text
Entrada
  │
  ▼
Codificador (Conv → Conv → Pool) × 3
  │
  ├──── conexões de atalho reinjetam mapas de alta resolução  ────┐
  ▼                                                               │
Base (bottleneck)                                                 │
  │                                                               │
  ▼                                                               │
Decodificador (Upsample → Concat → Conv → Conv) × 3  ◄────────────┘
  │
  ▼
Saída 1×1 (logits)
```

**Componentes principais:**

1. **Bloco Convolucional Base (`BlocoConv`)**
   A unidade fundamental repetida em toda a rede. Aplica duas convoluções $3 \times 3$ em sequência, cada uma seguida de ativação ReLU, com *padding* que preserva as dimensões espaciais da entrada. É esse bloco que aparece tanto no codificador quanto no decodificador.

2. **Convolução Transposta (`nn.ConvTranspose2d`)**
   É a operação responsável pelo *upsampling* no decodificador: ao invés de reduzir a resolução espacial (como o `MaxPool2d` faz no codificador), ela a aumenta, aprendendo os pesos necessários para "desfazer" a compressão e recuperar gradualmente o tamanho original da imagem.

3. **Perda Combinada (BCE + Dice)**
   A função `perda_segmentacao` soma duas métricas complementares:
   - **Entropia Cruzada Binária (BCE):** avalia o acerto *pixel a pixel*.
   - **Coeficiente de Dice:** avalia a *sobreposição global* entre a máscara prevista e a real.

   Juntas, elas equilibram precisão local com fidelidade da forma segmentada como um todo.

In [45]:
class BlocoConv(nn.Module):

    def __init__(self, canais_entrada, canais_saida):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(canais_entrada, canais_saida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(canais_saida, canais_saida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.rede(x)


class UNetCompacta(nn.Module):

    def __init__(self, canais_entrada=1, base=8):
        super().__init__()
        self.enc1 = BlocoConv(canais_entrada, base)
        self.enc2 = BlocoConv(base, base * 2)
        self.enc3 = BlocoConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)

        self.fundo = BlocoConv(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(
            base * 8, base * 4, kernel_size=2, stride=2
        )
        self.dec3 = BlocoConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(
            base * 4, base * 2, kernel_size=2, stride=2
        )
        self.dec2 = BlocoConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(
            base * 2, base, kernel_size=2, stride=2
        )
        self.dec1 = BlocoConv(base * 2, base)

        self.saida = nn.Conv2d(base, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        f = self.fundo(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(f), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.saida(d1)


def para_tensores(imagens, mascaras):
    X = (
        torch.tensor(np.stack(imagens), dtype=torch.float32).unsqueeze(1)
        / 255.0
    )
    Y = (
        torch.tensor(np.stack(mascaras), dtype=torch.float32).unsqueeze(1)
        / 255.0
    )
    return X, Y


def perda_dice(logits, alvo, eps=1e-6):
    probs = torch.sigmoid(logits)
    intersecao = (probs * alvo).sum(dim=(1, 2, 3))
    uniao = probs.sum(dim=(1, 2, 3)) + alvo.sum(dim=(1, 2, 3))
    dice = (2 * intersecao + eps) / (uniao + eps)
    return 1 - dice.mean()


def perda_segmentacao(logits, alvo):
    return nn.functional.binary_cross_entropy_with_logits(
        logits, alvo
    ) + perda_dice(logits, alvo)

##### Bloc 4 : Entraînement de la U-Net et Évaluation des Performances

L'entraînement est exécuté sur $35$ époques en utilisant l'optimiseur *Adam*. À chaque époque, on surveille la Perte d'Entraînement et l'indice IoU moyen sur l'ensemble de validation.

1. **Boucle d'Entraînement :** Les paramètres sont mis à jour avec des *mini-lots* de $16$ échantillons.
2. **Évolution Graphique :** La [Figure 9.36](#fig-09-unet-treinamento) affiche le graphique avec la progression de la perte et de l'IoU.

In [46]:
modelo_unet = UNetCompacta()
print(
    f"Paramètres entraînables de l'U-Net : {sum(p.numel() for p in modelo_unet.parameters())}"
)

X_treino_unet, Y_treino_unet = para_tensores(imgs_treino, masks_treino)
X_val_unet, Y_val_unet = para_tensores(imgs_val, masks_val)

otimizador_unet = optim.Adam(modelo_unet.parameters(), lr=1e-3)
n = X_treino_unet.size(0)
tam_lote = 16
epocas_unet = 35
historico_perda_unet, historico_iou_unet = [], []

for epoca in range(epocas_unet):
    if epoca % 5 == 0:  # Imprime toutes les 5 époques
        print(epoca + 1, "/", epocas_unet)
    modelo_unet.train()
    perm = torch.randperm(n)
    perda_epoca = 0.0

    for i in range(0, n, tam_lote):
        idx = perm[i : i + tam_lote]
        otimizador_unet.zero_grad()
        logits = modelo_unet(X_treino_unet[idx])
        perda = perda_segmentacao(logits, Y_treino_unet[idx])
        perda.backward()
        otimizador_unet.step()
        perda_epoca += perda.item() * len(idx)

    modelo_unet.eval()
    with torch.no_grad():
        predicao_val = (torch.sigmoid(modelo_unet(X_val_unet)) > 0.5).float()
        intersecao = (predicao_val * Y_val_unet).sum(dim=(1, 2, 3))
        uniao = ((predicao_val + Y_val_unet) > 0).float().sum(dim=(1, 2, 3))
        iou_epoca = (intersecao / uniao.clamp(min=1e-6)).mean().item()

    historico_perda_unet.append(perda_epoca / n)
    historico_iou_unet.append(iou_epoca)

iou_unet_final = historico_iou_unet[-1]
print(f"IoU moyen final de l'U-Net en validation : {iou_unet_final:.4f}")

# Graphique de l'évolution de l'entraînement
r = mm.showTrainCurves(historico_perda_unet, historico_iou_unet,
        titulo="Treinamento da U-Net — Segmentação de Nódulos Sintéticos",
        subtitulo="Perda no conjunto de treino e IoU médio no conjunto de validação \
            ao longo de 35 épocas")

Paramètres entraînables de l'U-Net : 120681
1 / 35


6 / 35


11 / 35


16 / 35


21 / 35


26 / 35


31 / 35


IoU moyen final de l'U-Net en validation : 0.8876


<Figure size 1190x714 with 2 Axes>

**Figure 9.36:** Évolution de l


##### Bloc 5 : Comparatif Quantitatif et Qualitatif (Classique vs. U-Net)

La comparaison entre l’approche classique et la U-Net met en évidence la supériorité de l’apprentissage de représentations dans les scénarios à faible contraste.

1. **Tableau de métriques :** Le graphique à barres dans la [Figure 9.37](#fig-09-unet-comparativo) contraste le IoU moyen des deux méthodes lors de la validation.
2. **Visualisation qualitative :** La comparaison visuelle sur trois échantillons démontre comment les connexions de raccourci permettent de récupérer le contour du nodule, même sous un bruit prononcé.

In [47]:
# 1.  comparatif 
print(f"IoU moyen (classique Lissage + Otsu) : {iou_classico_medio:.4f}")
print(f"IoU moyen (U-Net) : {iou_unet_final:.4f}")

# 2. Affichage qualitatif côte à côte sur 3 échantillons via mm.show
imgs_comparativas = []
titulos_comparativos = []

for i in range(3):
    with torch.no_grad():
        pred_unet = (
            (torch.sigmoid(modelo_unet(X_val_unet[i : i + 1])) > 0.5)
            .float()
            .squeeze()
            .numpy()
            * 255
        )

    pred_classico = segmenta_classico(imgs_val[i], elemento_estrutural)

    imgs_comparativas.extend(
        [imgs_val[i], masks_val[i], pred_classico, pred_unet.astype(np.uint8)]
    )

    t_prefix = f"Amostra {i+1}"
    titulos_comparativos.extend(
        [
            f"{t_prefix}: Imagem",
            f"{t_prefix}: Referência",
            f"{t_prefix}: Clássico",
            f"{t_prefix}: U-Net",
        ]
    )

mm.show(
    imgs_comparativas,
    titles=titulos_comparativos,
    cols=4,
    figsize=(11, 7.5),
)

IoU moyen (classique Lissage + Otsu) : 0.7516
IoU moyen (U-Net) : 0.8876


<Figure size 1650x1125 with 12 Axes>

**Figure 9.37:** Comparatif quantitatif (IoU moyen) et qualitatif entre l


> ### 📝 Nota
>
> ###### 🧠 Pourquoi l'U-Net surpasse-t-elle le seuillage fixe ?
>
> Le seuillage d'Otsu applique une valeur de coupure globale sur l'intensité locale. Lorsque la différence de moyenne entre le nodule et le fond est faible par rapport au bruit gaussien, cette règle commet des erreurs systématiques au niveau des bords.
>
> L'U-Net contourne cette limitation en combinant le contexte sémantique large extrait par l'encodeur avec les détails spatiaux fins préservés par les connexions de saut. Cela permet d'identifier la présence du nodule et de délimiter ses contours avec précision, même sous un bruit intense.

### 9.5.4 Ingénierie des données pour la vision par ordinateur (*Roboflow*)

Une U-Net peut être entraînée à partir d'images annotées sur des plateformes d'ingénierie des données pour la vision par ordinateur, telles que *Roboflow*. Ces plateformes permettent d'organiser des ensembles de données, de réaliser des annotations, d'appliquer des étapes de prétraitement et d'augmentation des données, d'entraîner des modèles et d'exporter les données dans différents formats. Dans cette section, toutefois, l'exemple reprend la **détection d'objets géométriques**, en utilisant des ensembles de données déjà présentés dans cet ouvrage.

> ### ❗ Dépendance à la connexion et à la clé API
>
> Les cellules de cette section exigent une connexion à Internet et une clé API gratuite de *Roboflow* (`app.roboflow.com`). Dans l'*espace de travail* du projet, accédez à **⚙ → API Roboflow** et copiez la **clé API privée**.
>
> Créez, dans ce dossier, le fichier `chave_roboflow.txt` contenant **uniquement la clé**, sans guillemets. Un modèle de ce fichier est disponible dans `chave_roboflow.txt.exemplo`.
>
> Ajoutez `chave_roboflow.txt` au fichier `.gitignore`, car il contient une information d'authentification qui ne doit être ni versionnée ni partagée.

#### 9.5.4.1 Détection d'objets avec *Roboflow*

*Roboflow* permet d'effectuer une inférence sur des images locales, permettant
d'évaluer les performances du modèle hébergé dans l'identification des objets
d'intérêt.

Outre l'inférence, le *dataset* peut être exporté au format
`png-mask-semantic`, dans lequel chaque image est accompagnée d'un masque de
segmentation sémantique. Dans ce masque, chaque pixel représente la classe à
laquelle appartient l'objet correspondant. Les paires image-masque sont utilisées
comme données d'entraînement pour la `UNetCompacta`.

#### 9.5.4.2 Connexion, *téléchargement* et vérification du *dataset*

Le code établit une connexion avec le *Workspace* `mctest` et le projet
`geometric-test00`, version 6, puis effectue le *téléchargement* du *dataset* vers
`dados/datasetRoboFlow`. Ensuite, il vérifie le *shape* des images dans chaque
*split*. La fonction de vérification est réutilisée ultérieurement dans la section.

In [48]:
from roboflow import Roboflow
from pathlib import Path
from PIL import Image
from collections import Counter

def contar_shapes(raiz, splits=("train", "valid", "test")):
    """Compte la forme (hauteur, largeur, canaux) des images par split."""
    for split in splits:
        pasta = raiz / split / "images"
        if not pasta.exists():
            print(f"{split}: dossier introuvable")
            continue

        shapes = Counter()
        for arquivo in pasta.iterdir():
            if arquivo.is_file():
                with Image.open(arquivo) as img:
                    shapes[(img.height, img.width, len(img.getbands()))] += 1

        txt = ", ".join(f"{s}: {n}" for s, n in shapes.items())
        print(f"{split}: {txt}")


chave = Path("chave_roboflow.txt")

if not chave.exists():
    print("Clé Roboflow introuvable : clé_roboflow.txt")
else:
    with open(chave) as f:
        api_key = f.read().strip()

    # Projet :
    # https://app.roboflow.com/mctest/geometric-test00/models
    # geometric-test00/6

    rf = Roboflow(api_key=api_key)
    projeto = rf.workspace("mctest").project("geometric-test00")
    versao = projeto.version(6)

    print(
        f"ID : {versao.version} | Nom : {versao.name} | "
        f"Images : {versao.images}"
    )

    raiz = Path("dados/datasetRoboFlow")
    versao.download("yolov8", location=str(raiz))

    print("Dataset :", raiz)
    contar_shapes(raiz)

#### 9.5.4.3 Téléchargement du *dataset* de manière reproductible (alternative)

En alternative au *dataset* obtenu via *Roboflow*, on peut utiliser un
*dataset* disponible dans un dépôt GitHub, également organisé selon les mêmes
*splits* et contenant les mêmes classes d'objets. Les ensembles de
données ne sont toutefois pas identiques : les images du GitHub ont une résolution
de `608×608` pixels, tandis que les images exportées par *Roboflow* sont en
`640×640` pixels.

Le code ci-dessous effectue le téléchargement du *dataset* depuis GitHub, s'il n'est
pas encore disponible localement, et réutilise `contar_shapes` pour vérifier
les dimensions des images dans chaque *split*.

In [49]:
import os

if not os.path.exists("dados/dataset"):
    cmd = (
        "git clone --no-checkout --depth 1 --filter=blob:none "
        "https://github.com/fzampirolli/pdi-vc.git tmp_repo && "
        "cd tmp_repo && git sparse-checkout set all/cap09/dados/dataset "
        "&& git checkout && cd .. && mkdir -p dados && "
        "cp -r tmp_repo/all/cap09/dados/dataset dados/dataset && "
        "rm -rf tmp_repo"
    )
    !{cmd}

if not chave.exists():
    print("Chave do Roboflow não encontrada: chave_roboflow.txt")
else:
  versao_recente = projeto.versions()[-1]
  print(f"Versão mais recente: {versao_recente.version.split('/')[-1]}")

  contar_shapes(Path("dados/dataset"))

#### 9.5.4.4 Comparaison d'une image de chaque *dataset*

Les deux *datasets* possèdent des images avec des résolutions différentes : `608×608` sur GitHub et `640×640` sur *Roboflow*. Pour une comparaison visuelle directe, les images sont redimensionnées à la même dimension avant d'être affichées côte à côte ([Figure 9.38](#fig-09-datasets-comparacao)).

In [50]:
import cv2

if not chave.exists():
    print("Clé Roboflow introuvable : clé_roboflow.txt")
else:
    caminho1 = next((raiz / "train/images").iterdir())
    caminho2 = next((Path("dados/dataset") / "train/images").iterdir())

    img1 = mm.read(str(caminho1))
    img2 = mm.read(str(caminho2))

    # Redimensionne les deux à la même taille (la plus petite des deux)
    largura = min(img1.shape[1], img2.shape[1])
    altura = min(img1.shape[0], img2.shape[0])
    img1_r = cv2.resize(img1, (largura, altura))
    img2_r = cv2.resize(img2, (largura, altura))

    mm.show(
        [img1_r, img2_r],
        title=[f"Roboflow {img1.shape}", f"GitHub {img2.shape}"],
    )

**Figure 9.38:** Une image de chaque *dataset*, redimensionnées pour comparaison


#### 9.5.4.5 Inférence avec le modèle entraîné

Le modèle entraîné dans la version 6 est utilisé pour effectuer l’inférence sur
une image de test locale. La prédiction prend en compte les seuils de confiance et
de chevauchement employés par la suppression des non-maxima (*Non-Maximum
Suppression*, NMS), et le résultat est présenté sur l’image annotée de la
[Figure 9.39](#fig-09-roboflow-detection).

In [51]:
# version.model est déprécié ; utiliser version.models()

if not chave.exists():
    print("Clé Roboflow introuvable : chave_roboflow.txt")
else:
    modelo = versao.models()[0]

    img_path = "dados/dataset/test/images/00001.jpg"
    pred = modelo.predict(img_path, confidence=40, overlap=30)
    resp = pred.json()  # inclut les boîtes détectées dans resp["predictions"]

    altura, largura, _ = mm.read(img_path).shape
    print("Dimensions de l'image de test :", (altura, largura))

    pred.save("resultado.jpg")  # image annotée

    mm.show(
        mm.read("resultado.jpg"),
        title="Résultat de l'inférence avec le modèle Roboflow",
        figsize=(6, 6)
    )

**Figure 9.39:** Résultat de l


#### 9.5.4.6 Avaliação das predições com IoU e classe

O *Roboflow* retorna cada caixa com as coordenadas do centro (`x`, `y`) em
*pixels* e a classe prevista, enquanto os rótulos locais
(`dados/dataset/test/labels/00001.txt`) seguem o formato YOLO, com centro e
dimensões normalizados no intervalo `[0, 1]`. Antes da comparação por meio de
`mm.IoU`, as caixas devem ser convertidas para o mesmo formato, com as
coordenadas do canto superior esquerdo e as dimensões expressas em
*pixels*.

Uma predição só é considerada correta quando a classe prevista coincide com
a classe da caixa real e sua *Intersection over Union* (IoU) é maior ou
igual ao limiar definido, adotando-se, neste exemplo, `0,5` (50%) como valor
padrão.

> ### ❗ Importante
>
> ##### O `class_id` do *Roboflow* não corresponde ao índice dos rótulos locais
>
> Na exportação, o *Roboflow* reordena as classes em **ordem alfabética** no
> `data.yaml`, independentemente da ordem utilizada no projeto original,
> mantida no `data.yaml` do GitHub. Assim, `class_id = 0` corresponde a
> `Circulo` no retorno da API, enquanto o mesmo índice corresponde a
> `Triangulo` nos rótulos locais.
>
> Por isso, a comparação deve ser feita pelo **nome da classe**
> (`p["class"]`), convertendo-o posteriormente para o índice correspondente
> na lista local. Os `class_id` não devem ser comparados diretamente.

In [52]:
# resp, largura e altura foram definidos na célula anterior

# Ordem das classes usada nas labels locais (dados/dataset/*/labels/*.txt)
CLASSES_LOCAIS = ["Triangulo", "Quadrado", "Pentagono", "Hexagono",
                   "Heptagono", "Circulo", "Elipse"]

def predicao_correta(pred: tuple, real: tuple, limiar: float = 0.5) -> bool:
    """True se mesma classe e mm.IoU(caixa_pred, caixa_real) >= limiar."""
    classe_pred, caixa_pred = pred
    classe_real, caixa_real = real
    return classe_pred == classe_real and mm.IoU(caixa_pred, caixa_real) >= limiar

def caixa_roboflow(p: dict) -> tuple:
    """Converte predição do Roboflow para (classe, (x, y, w, h))."""
    caixa = (p["x"] - p["width"] / 2, p["y"] - p["height"] / 2,
             p["width"], p["height"])
    # usa o nome da classe (não o class_id!) para casar com a ordem local
    classe = CLASSES_LOCAIS.index(p["class"])
    return (classe, caixa)

def carrega_labels_yolo(caminho_txt: str, largura: int, altura: int) -> list:
    """Lê rótulos YOLO (normalizados) e converte para (classe, (x, y, w, h))."""
    caixas = []
    with open(caminho_txt) as f:
        for linha in f:
            classe, xc, yc, w, h = map(float, linha.split())
            w_px, h_px = w * largura, h * altura
            x_px = xc * largura - w_px / 2
            y_px = yc * altura - h_px / 2
            caixas.append((int(classe), (x_px, y_px, w_px, h_px)))
    return caixas

if not chave.exists():
    print("Chave do Roboflow não encontrada: chave_roboflow.txt")
else:
    caixas_pred = [caixa_roboflow(p) for p in resp["predictions"]]
    caixas_real = carrega_labels_yolo(
        "dados/dataset/test/labels/00001.txt", largura, altura
    )

    acertos = sum(
        any(predicao_correta(cp, cr, limiar=0.5) for cr in caixas_real)
        for cp in caixas_pred
    )
    print(f"{acertos}/{len(caixas_pred)} predições corretas (classe + IoU >= 50%)")

Pour visualiser le résultat, chaque prédiction est dessinée sur l'image :
en vert lorsqu'il s'agit d'une réussite (classe + IoU ≥ seuil) et en rouge lorsqu'il s'agit
d'une erreur, comme le montre la [Figure 9.40](#fig-09-roboflow-acertos)..

In [53]:
import cv2

VERDE, VERMELHO = (0, 255, 0), (255, 0, 0)

if not chave.exists():
    print("Clé Roboflow introuvable : chave_roboflow.txt")
else:
    img_acertos = mm.read(img_path).copy()

    if not chave.exists():
        print("Clé Roboflow introuvable : chave_roboflow.txt")
    else:
        for cp in caixas_pred:
            classe_pred, (x, y, w, h) = cp
            correta = any(predicao_correta(cp, cr, limiar=0.5) for cr in caixas_real)
            cor = VERDE if correta else VERMELHO

            p1, p2 = (int(x), int(y)), (int(x + w), int(y + h))
            cv2.rectangle(img_acertos, p1, p2, cor, 2)
            cv2.putText(img_acertos, str(classe_pred), (p1[0], p1[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, cor, 2)

    mm.show(
        img_acertos,
        title=f"{acertos}/{len(caixas_pred)} prédictions correctes "
            f"(classe + IoU >= 50%)",
        figsize=(6, 6)
    )

**Figure 9.40:** Prédictions correctes (vert) et incorrectes (rouge)


### 9.5.5 Applications géométriques et intégrées

Le chapitre se termine en intégrant deux piliers de la VC : la **géométrie projective** (étudiée dans les chapitres 6 et 8) et l'**apprentissage profond**. La combinaison de ces approches soutient des applications pratiques dans le monde réel, comme illustré ci-dessous.

#### 9.5.5.1 Réalité Augmentée avec Marqueurs et Homographie

Imaginez une caméra pointée vers une table sur laquelle quelqu'un a collé un petit marqueur ArUco. Selon l'angle de la caméra, ce marqueur apparaît **pivoté, incliné, en perspective** — jamais parfaitement carré. C'est précisément cette distorsion que l'homographie sait « lire » et corriger (ou, dans notre cas, répliquer vers une nouvelle image).

Le flux complet d'une application de RA basée sur des marqueurs suit trois étapes :

1. **Mise en situation :** le marqueur est inséré dans une scène réelle, subissant une transformation de perspective (simulant l'angle de la caméra).
2. **Détection :** l'algorithme localise le marqueur dans la scène et récupère les coordonnées exactes de ses 4 coins avec `cv2.aruco.ArucoDetector`.
3. **Substitution :** grâce à l'homographie entre le marqueur « idéal » et le marqueur « détecté », on projette une **nouvelle image virtuelle** exactement sur la zone du marqueur — comme si celui-ci s'était transformé en une fenêtre vers un autre contenu, comme le montre la [Figure 9.41](#fig-09-realidade-aumentada).

> ### 📝 Nota
>
> **Détail technique important :** l'ArUco nécessite une marge blanche autour du motif (la « zone de silence ») pour que le détecteur puisse différencier le marqueur du fond. C'est pourquoi le code ci-dessous ajoute une bordure avec `cv2.copyMakeBorder` et utilise une interpolation `cv2.INTER_NEAREST` lors de la déformation de l'image, évitant ainsi que la rotation ne floute les petits carrés noirs et blancs et n'empêche la détection.

In [54]:
# 1. Génération du marqueur ArUco synthétique, déjà avec une marge blanche (zone calme)
# → cette marge est essentielle pour que le détecteur puisse « voir » le marqueur
dic = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_bruto = cv2.aruco.generateImageMarker(dic, 7, 200)
aruco_gray = cv2.copyMakeBorder(
    aruco_bruto, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255
)  # 200 -> 280px, avec 40px de cadre blanc de chaque côté
aruco = cv2.cvtColor(aruco_gray, cv2.COLOR_GRAY2BGR)
lado = aruco.shape[0]  # 280

# 2. Scène réelle de fond, en utilisant une image d'exemple de skimage.data
fundo = cv2.cvtColor(skdata.coffee(), cv2.COLOR_RGB2BGR)
cena = cv2.resize(fundo, (640, 480))

# Coins du marqueur « de face » (src) et sa position tournée/inclinée dans la scène (dst)
src = np.float32([[0, 0], [lado, 0], [lado, lado], [0, lado]])
dst = np.float32([[190, 160], [420, 70], [470, 330], [150, 370]])  # rotation + perspective

# 3. Projette le marqueur (avec des bords nets) sur la scène réelle
H_cena, _ = cv2.findHomography(src, dst)
mask_cena = cv2.warpPerspective(
    np.full((lado, lado), 255, np.uint8), H_cena, (640, 480),
    flags=cv2.INTER_NEAREST,
)
cena[mask_cena > 0] = cv2.warpPerspective(
    aruco, H_cena, (640, 480), flags=cv2.INTER_NEAREST
)[mask_cena > 0]

# 4. Détection du marqueur dans la scène (comme le ferait une caméra)
det = cv2.aruco.ArucoDetector(dic, cv2.aruco.DetectorParameters())
corners, ids, _ = det.detectMarkers(cena)

assert ids is not None and len(corners) > 0, \
    "Marcador não detectado — confira iluminação/contraste da cena."

# 5. Image virtuelle (une autre image de skimage.data) qui va « remplacer » le marqueur
# Remarque : nous utilisons le carré INTÉRIEUR du marqueur (sans la marge) comme zone de projection,
# donc l'homographie de l'image virtuelle utilise 'src' original (200x200), pas 'side' avec bordure
src_interno = np.float32([[0, 0], [200, 0], [200, 200], [0, 200]])
virtual = cv2.resize(
    cv2.cvtColor(skdata.camera(), cv2.COLOR_RGB2BGR), (200, 200)
)

# Les coins détectés correspondent au marqueur AVEC la marge (280x280),
# donc nous recalculons H_ra en utilisant 'src' avec marge, pour garder la projection cohérente
H_ra, _ = cv2.findHomography(src, corners[0][0])

# L'homographie détectée est appliquée à l'image virtuelle (redimensionnée à 'side')
virtual_grande = cv2.resize(virtual, (lado, lado))
ra = cena.copy()
mask_ra = cv2.warpPerspective(
    np.full((lado, lado), 255, np.uint8), H_ra, (640, 480), flags=cv2.INTER_NEAREST
)
ra[mask_ra > 0] = cv2.warpPerspective(
    virtual_grande, H_ra, (640, 480), flags=cv2.INTER_NEAREST
)[mask_ra > 0]

# Affichage : scène avec marqueur vs. scène avec Réalité Augmentée appliquée
mm.show(
    [cv2.cvtColor(cena, cv2.COLOR_BGR2RGB), cv2.cvtColor(ra, cv2.COLOR_BGR2RGB)],
    titles=["Marqueur dans la Scène Réelle (tourné)", "Superposition Virtuelle via Homographie"],
    cols=2,
    figsize=(9, 4),
)

<Figure size 1350x600 with 2 Axes>

**Figure 9.41:** Réalité augmentée basée sur un marqueur ArUco : marqueur inséré dans une scène réelle et tourné, détecté et remplacé par une image virtuelle via homographie.


**Résumé du *pipeline* :**

| Étape | Fonction | Clé |
|---|---|---|
| 1. Génération | Marqueur ArUco avec marge blanche | `generateImageMarker` + `copyMakeBorder` |
| 2. Mise en scène | Insère un marqueur déformé dans la scène | `findHomography` + `warpPerspective` |
| 3. Détection | Localise le marqueur et retourne les coins | `ArucoDetector.detectMarkers` |
| 4. Remplacement | Projette une image virtuelle sur le marqueur | `findHomography` + `warpPerspective` |

> 💡 **Leçon :** le détecteur qui « ne trouve rien » est courant en vision par ordinateur. Demandez-vous toujours : **« ai-je fourni suffisamment de contraste et d'espace ? »** — cela vaut pour ArUco, les codes QR et la reconnaissance faciale.

#### 9.5.5.2 Photogrammétrie et référence d'échelle

La **photogrammétrie** permet d'estimer les dimensions physiques d'objets à partir d'images numériques. Pour cela, on utilise un objet de référence aux dimensions connues, placé dans la même scène que l'objet d'intérêt. Cette procédure établit une relation entre les distances mesurées en *pixels* et leurs dimensions correspondantes dans le monde réel.

Considérons, par exemple, une carte de crédit, dont les dimensions suivent la norme internationale ISO/IEC 7810. Comme sa largeur est exactement de **8,56 cm**, il suffit de déterminer combien de *pixels* cette largeur occupe dans l'image pour calculer le facteur de conversion entre *pixels* et centimètres. Si la carte correspond à 140 *pixels*, alors chaque *pixel* représentera approximativement **0,061 cm**. Ce même facteur d'échelle peut être appliqué pour estimer les dimensions de tout autre objet situé dans le même plan de la scène, comme illustré dans la [Figure 9.42](#fig-09-fotogrametria).

La procédure peut être divisée en deux étapes principales :

1. **Segmentation et *bounding box* :** localiser, dans l'image, à la fois l'objet de référence et l'objet d'intérêt, en utilisant des techniques telles que la segmentation par couleur, le seuillage, la détection de contours ou des méthodes de détection d'objets.
2. **Conversion en dimensions physiques :** calculer le rapport $\mathrm{cm/pixel}$ à partir de la largeur connue de l'objet de référence et l'utiliser pour convertir les mesures de l'objet d'intérêt de *pixels* en centimètres.

> ### 📝 Nota
>
> **Condition pour des mesures fiables**
>
> La conversion entre *pixels* et centimètres suppose que l'objet de référence et l'objet d'intérêt soient approximativement dans le même plan et à la même distance de la caméra. Dans ces conditions, l'échelle reste pratiquement constante sur toute l'image. Les différences de profondeur, l'inclinaison de la caméra ou les distorsions de l'objectif peuvent introduire des erreurs dans les mesures estimées.

In [55]:
COR_REFERENCIA = (200, 200, 200)  # Carte de référence (grise)
COR_OBJETO = (60, 60, 220)  # Objet cible (rouge)

# Dessin de la scène synthétique
cena_medicao = np.full((300, 500, 3), 255, dtype=np.uint8)
cv2.rectangle(
    cena_medicao, (30, 200), (30 + 140, 200 + 88), COR_REFERENCIA, -1
)
cv2.rectangle(cena_medicao, (250, 100), (250 + 220, 100 + 150), COR_OBJETO, -1)


def caixa_delimitadora_por_cor(imagem_bgr, cor_bgr, tolerancia=40):
    diferenca = np.abs(imagem_bgr.astype(int) - np.array(cor_bgr)).sum(axis=2)
    mascara = (diferenca < tolerancia).astype(np.uint8) * 255
    contornos, _ = cv2.findContours(
        mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    maior_contorno = max(contornos, key=cv2.contourArea)
    return cv2.boundingRect(maior_contorno)


x_ref, y_ref, w_ref_px, h_ref_px = caixa_delimitadora_por_cor(
    cena_medicao, COR_REFERENCIA
)
x_obj, y_obj, w_obj_px, h_obj_px = caixa_delimitadora_por_cor(
    cena_medicao, COR_OBJETO
)

# Calcul d'échelle physique
LARGURA_REFERENCIA_CM = 8.56
razao_cm_por_px = LARGURA_REFERENCIA_CM / w_ref_px
largura_obj_cm = w_obj_px * razao_cm_por_px
altura_obj_cm = h_obj_px * razao_cm_por_px

# Dessin des boîtes et mesures estimées
resultado = cena_medicao.copy()
cv2.rectangle(
    resultado, (x_ref, y_ref), (x_ref + w_ref_px, y_ref + h_ref_px), (0, 180, 0), 2
)
cv2.rectangle(
    resultado, (x_obj, y_obj), (x_obj + w_obj_px, y_obj + h_obj_px), (0, 180, 0), 2
)

cv2.putText(
    resultado,
    f"{LARGURA_REFERENCIA_CM:.2f} cm",
    (x_ref, y_ref - 8),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.55,
    (0, 120, 0),
    2,
)

cv2.putText(
    resultado,
    f"{largura_obj_cm:.1f} x {altura_obj_cm:.1f} cm",
    (x_obj, y_obj - 8),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (0, 120, 0),
    2,
)

mm.show(
    [
        cv2.cvtColor(cena_medicao, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB),
    ],
    titles=[
        "Scène originale",
        "Mesure par référence d'échelle (centimètres)",
    ],
    cols=2,
    figsize=(9, 4),
)

<Figure size 1350x600 with 2 Axes>

**Figure 9.42:** Mesure de dimensions physiques réelles à l


**Résumé du *pipeline* :**

| Étape | Ce qu'elle fait | Fonction clé |
|---|---|---|
| 1. Scène synthétique | Dessine la carte de référence et l'objet cible avec des couleurs distinctes | `cv2.rectangle` |
| 2. Segmentation | Isole chaque objet par couleur et extrait son contour | `cv2.findContours` |
| 3. Boîte englobante | Obtient la boîte englobante (position et taille en pixels) de chaque objet | `cv2.boundingRect` |
| 4. Mise à l'échelle | Convertit les pixels en centimètres en utilisant la largeur connue de la carte | Règle de trois : $\text{cm/pixel} = \dfrac{8{,}56}{w_{ref\_px}}$ |
| 5. Annotation | Dessine les boîtes et affiche les mesures estimées sur l'image | `cv2.rectangle` + `cv2.putText` |

> 💡 **Application dans le monde réel :** c'est exactement la technique utilisée par les *apps* de *e-commerce* qui estiment la taille d'un produit à partir d'une photo prise à côté d'une carte, par les systèmes agricoles qui mesurent des fruits sur des tapis roulants, et même par les expertises forensiques qui calculent les dimensions de traces sur des scènes de crime — tout repose sur la même idée : **une règle connue dans la photo elle-même.**

## 9.6 Résumé

Ce chapitre, qui clôt la deuxième partie de l’ouvrage, a présenté :

* **Convolution et *pooling* appris :** la même opération mathématique de
  convolution qu’au chapitre 3, mais avec des *kernels* traités comme des
  paramètres ajustés par l’entraînement, au lieu d’être définis manuellement ;

* **Le partage des poids et la hiérarchie des caractéristiques** comme
  propriétés qui rendent les CNN efficaces et capables d’apprendre des
  représentations de plus en plus abstraites dans des couches successives ;

* **L’entraînement d’une CNN de zéro**, avec des performances comparables — et pas
  nécessairement supérieures — aux classificateurs classiques du chapitre 7
  sur une base petite et simple, soulignant que le choix de la méthode doit
  être proportionnel à la complexité réelle du problème ;

* **Le transfert d’apprentissage**, démontré expérimentalement comme une
  stratégie efficace pour des tâches avec peu de données étiquetées, réutilisant
  un extracteur de caractéristiques déjà entraîné sur une tâche connexe, et
  ses limites, mises en évidence par le transfert négatif entre des domaines
  très distincts ;

* **Des applications à grande échelle** avec des modèles pré-entraînés de classification,
  de détection (*Faster R-CNN*) et de segmentation (*DeepLabV3*), suivant le même
  principe de transfert d’apprentissage à l’échelle industrielle ;

* **Le transfert d’apprentissage appliqué à la détection d’objets**, en ajustant
  un YOLO pré-entraîné sur COCO pour localiser et classer des objets
  géométriques synthétiques, illustrant le même principe de gel partiel
  dans un domaine source et cible encore plus éloignés l’un de l’autre ;

* **La segmentation sémantique avec U-Net**, implémentée et entraînée de zéro sur
  un ensemble synthétique à faible contraste, surpassant une ligne de base
  classique de seuillage grâce aux connexions de raccourci entre l’encodeur et
  le décodeur ;

* **L’ingénierie des données pour la vision par ordinateur avec *Roboflow***, en utilisant
  un *dataset* et un modèle pré-entraîné pour effectuer une inférence sur la détection
  d’objets géométriques, ainsi qu’en comparant des ensembles de données avec des
  résolutions différentes, mais avec les mêmes classes d’objets ;

* L’intégration de la **géométrie computationnelle** (homographie et calibrage) et de
  l’**apprentissage profond** dans deux applications réelles qui clôturent l’ouvrage :
  la **réalité augmentée** et la **photogrammétrie**.

## 9.7 🤖 Utilisation de Gemini Notebook comme tuteur complémentaire

Dans cette édition, l'utilisation de **Gemini Notebook** est encouragée comme outil
d'apprentissage complémentaire. Fondé sur l'intelligence artificielle, le système
utilise exclusivement les documents fournis par l'auteur comme source de
connaissance, produisant des réponses alignées sur le contenu et l'approche
adoptés tout au long de ce chapitre.

> ### ❗ 🎓 Étudiez avec le tuteur intelligent
>
> [🚀 ACCÉDER À GEMINI NOTEBOOK : CHAPITRE 09](https://notebooklm.google.com/notebook/88495fa2-9139-45da-8d1b-9a9b2e609d36)
>
> #### 🌐 Langue et langage de programmation
>
> Le projet de ce chapitre dans Gemini Notebook a été construit uniquement avec le texte en **portugais** et les exemples de code en **Python**. Si vous étudiez à partir de l'édition en anglais ou en français, ou si vous suivez le parcours en C++, les réponses du tuteur peuvent ne pas correspondre exactement à la version que vous lisez.
>
> #### ⚠️ Avis concernant le contenu généré par l'IA
>
> Bien qu'il s'agisse d'un outil précieux de soutien à l'étude, Gemini Notebook peut
> éventuellement produire des réponses incomplètes, imprécises ou incorrectes.
> Il est recommandé de valider les informations en consultant le matériel du chapitre,
> les livres, les articles scientifiques et d'autres sources académiques fiables. Dans
> la mesure du possible, exécutez et expérimentez les exemples pratiques présentés
> tout au long du texte afin de consolider la compréhension des concepts.

## 9.8 Liste d'exercices

Les exercices suivants consolident les concepts présentés dans ce chapitre à
travers des adaptations, des expériences et des extensions des algorithmes
développés tout au long du texte, en utilisant les bibliothèques `PyTorch`,
`ultralytics` et la bibliothèque didactique `morph`.

1. **(10%)** Étudiez l'impact de la profondeur dans une architecture
   convolutionnelle. En partant du réseau à deux couches du Projet Pratique 1,
   ajoutez une troisième couche convolutionnelle avec 32 filtres avant les
   couches entièrement connectées. Entraînez la nouvelle architecture en
   conservant le même nombre d'époques et la même répartition des données.
   Comparez la précision sur l'ensemble de test et le nombre total de
   paramètres entraînables par rapport au réseau original, en discutant si
   l'augmentation de la profondeur a apporté un bénéfice mesurable pour des
   images de dimension $8 \times 8$.

2. **(15%)** Évaluez le seuil de données nécessaires dans le domaine cible pour
   que l'entraînement d'un CNN de zéro devienne compétitif avec le
   **transfert d'apprentissage**. En faisant varier le nombre d'échantillons
   d'entraînement disponibles dans le domaine B pour $\{5, 10, 20, 40, 80\}$,
   mesurez la précision de test pour les deux stratégies. Présentez les
   résultats dans un graphique en lignes et déterminez à partir de quel volume
   de données l'entraînement de zéro atteint des performances équivalentes à
   celles de l'extracteur pré-entraîné.

3. **(15%)** Étudiez la stratégie de **réglage fin partiel**
   (*fine-tuning*) par rapport au gel total des poids. Dans le scénario de
   transfert d'apprentissage entre domaines de chiffres, décongelez la
   deuxième couche convolutionnelle (`conv2`) de l'extracteur afin qu'elle soit
   mise à jour conjointement avec la tête de classification pendant
   l'entraînement dans le domaine B. Comparez la précision obtenue avec le gel
   total et avec l'entraînement de zéro, en discutant du compromis entre la
   capacité d'adaptation et le risque de sur-ajustement (*overfitting*).

4. **(20%)** Évaluez l'influence de la profondeur du gel (*freeze*) sur les
   performances des détecteurs **YOLO** soumis au transfert d'apprentissage. En
   utilisant l'ensemble de données synthétique de formes géométriques,
   exécutez le réglage fin en faisant varier le paramètre de gel du *backbone*
   pour $\{0, 5, 10, 15\}$. Enregistrez la métrique $\text{mAP}_{50}$ sur
   l'ensemble de validation pour chaque configuration, présentez les données
   dans un tableau et discutez si le gel partiel est avantageux lorsque les
   domaines source (COCO) et cible (formes géométriques) sont significativement
   distincts.

5. **(20%)** Étudiez l'importance des **connexions de raccourci** (*skip
   connections*) dans l'architecture **U-Net** pour la segmentation sémantique.
   Implémentez une variante `UNetSemAtalhos` qui fonctionne comme un
   *autoencodeur* convolutionnel traditionnel, en supprimant les concaténations
   entre les étages de l'encodeur et du décodeur. Entraînez les deux modèles
   sur la même base de nodules synthétiques, comparez l'IoU moyen sur
   l'ensemble de validation et présentez visuellement la différence dans la
   précision des contours segmentés par chaque méthode.

6. **(20%) — Défi : segmentation sémantique avec *Roboflow*.** Utilisez un
   projet *Roboflow* de type *instance segmentation*, contenant les sept
   classes de formes géométriques utilisées dans ce chapitre. Exportez
   l'*ensemble de données* au format `coco-segmentation` et développez une
   procédure pour convertir les polygones stockés dans les fichiers
   `_annotations.coco.json` en masques sémantiques multiclasses, dans lesquels
   chaque pixel reçoit l'indice de la classe correspondante et la valeur `0`
   représente le fond. Utilisez les images et les masques résultants pour
   entraîner la `UNetCompacta`. Évaluez l'IoU moyen sur l'ensemble de test et
   comparez visuellement les masques prédits avec les annotations originales.
   Discutez des principales difficultés rencontrées lors de la conversion des
   annotations COCO en masques et des effets des objets superposés ou
   appartenant à différentes classes.

7. **(Bonus – 10%)** Développez un système interactif qui combine la
   **détection d'objets (YOLO)** avec la **mesure par référence d'échelle
   (photogrammétrie)**. Entraînez le détecteur pour identifier deux classes
   dans une scène : une « Carte de Référence » (dimension connue de
   $8{,}56\text{ cm} \times 5{,}39\text{ cm}$) et un « Objet Cible ». Lors de
   l'inférence sur une nouvelle image, utilisez la dimension en pixels de la
   *boîte englobante* de la carte détectée pour convertir les dimensions de la
   boîte de l'objet cible en centimètres. Affichez l'image traitée avec les
   étiquettes de classe, la probabilité de confiance et les dimensions
   physiques estimées superposées.

8. **(Bonus – 10%)** Développez un système d'**estimation de profondeur par
   vision stéréo** à partir de deux images de la même scène obtenues depuis des
   positions différentes, simulant une paire de caméras stéréo. Considérez que
   la distance entre les deux positions de capture (*baseline*) est connue.

   Utilisez l'une des méthodes de **détection d'objets** présentées dans le
   chapitre, comme YOLO, pour localiser les objets d'intérêt dans les deux
   images. Pour chaque détection, établissez la correspondance entre le même
   objet dans les deux positions et déterminez sa **disparité**. À partir de la
   disparité, de la *baseline* et des paramètres de la caméra, utilisez la
   géométrie stéréo pour estimer la distance de chaque objet par rapport aux
   caméras.

   Comme extension de la bibliothèque didactique `morph`, modifiez la méthode
   `showBoundBox` afin qu'en plus de la **classe** et de la **confiance de la
   détection**, elle présente sur chaque *boîte englobante* la **distance
   estimée de l'objet**. Le résultat doit permettre de visualiser directement,
   sur les images, la classe, la précision (confiance) et la profondeur de
   chaque objet détecté.

   Présentez les deux images avec les détections, les correspondances entre
   les objets, l'image de disparité et une représentation de la profondeur
   estimée. Discutez de la manière dont la distance entre les caméras, la
   précision de la détection et de la correspondance, la résolution des images
   et la position de l'objet dans la scène influencent la qualité de
   l'estimation.

   Pour la validation, utilisez au moins un objet dont la distance à la caméra
   est connue. Comparez la profondeur estimée avec la valeur réelle et
   rapportez l'**erreur absolue** et l'**erreur relative**. Discutez également
   des limites de la méthode lorsqu'un objet n'est pas correctement détecté
   dans les deux images ou lorsque la correspondance entre les régions
   observées est ambiguë.

## 9.9 Clôture de la Partie II

Ce chapitre conclut la Partie II du livre et clôt la séquence de contenus initiée au chapitre 6, consacrée à la représentation, la détection, la description et la mise en correspondance de caractéristiques dans les images. Tout au long de ces chapitres, des méthodes classiques de vision par ordinateur (VC) basées sur des **caractéristiques conçues manuellement**, telles que Sobel, LBP, HOG, ORB et Haar Cascade, ainsi que des méthodes fondées sur des **caractéristiques apprises automatiquement**, représentées par les CNN, ont été présentées.

Les exemples et expériences développés montrent qu'aucune de ces approches n'est universellement supérieure. Le choix de la technique la plus adaptée dépend des caractéristiques du problème, de la disponibilité des données d'entraînement, des exigences de précision et des contraintes computationnelles de l'application. Dans des problèmes bien structurés et avec peu de données, les descripteurs classiques offrent souvent des solutions simples et efficaces. En revanche, des tâches plus complexes tendent à bénéficier de la capacité d'apprentissage offerte par les CNN.

Plusieurs directions d'étude peuvent approfondir les concepts présentés dans cette partie du livre, parmi lesquelles :

- **Les architectures modernes de CNN**, telles que ResNet, EfficientNet et les *Vision Transformers*, qui élargissent la capacité de représentation et les performances dans les tâches de classification et de reconnaissance visuelle ;
- **La détection et la segmentation d'objets**, avec un accent sur les familles YOLO et les modèles de segmentation basés sur des *prompts*, comme *Segment Anything* ;
- **La reconstruction tridimensionnelle et le SLAM** (*Simultaneous Localization and Mapping*), qui utilisent plusieurs images pour estimer la géométrie de la scène et la trajectoire de caméras en mouvement ;
- **Les modèles génératifs d'images**, tels que les réseaux adversariaux génératifs (GAN) et les modèles de diffusion, capables de synthétiser des images réalistes à partir d'exemples ou de descriptions textuelles.

Les fondements développés tout au long de la Partie II constituent la base de ces domaines et d'autres domaines avancés de la VC, dans lesquels la représentation adéquate des informations visuelles demeure un élément central pour l'analyse et la compréhension des images.

## Références du chapitre

Les concepts et algorithmes présentés dans ce chapitre s’appuient sur des références classiques et contemporaines de la littérature sur l’apprentissage profond appliqué à la vision par ordinateur :

* Mcculloch (1943), pour la proposition de la première abstraction mathématique et logique du neurone artificiel, posant les fondements conceptuels du traitement neuronal computationnel.
* Rosenblatt (1958), pour la formulation originale du **Perceptron**, modèle précurseur du neurone artificiel utilisé dans les architectures modernes d’apprentissage profond.
* Goodfellow (2016) et Lecun (2015), pour les fondements des réseaux de neurones, de la convolution, des fonctions d’activation et de l’entraînement des modèles profonds.
* Bishop (2006), pour les concepts rigoureux de reconnaissance des formes, de probabilité, d’estimation du maximum de vraisemblance et de méthodes statistiques appliquées à l’apprentissage automatique.
* Ronneberger (2015), pour l’architecture **U-Net**, utilisée dans la segmentation sémantique avec des connexions de raccourci entre l’encodeur et le décodeur.
* Redmon (2016), pour l’architecture **YOLO** (*You Only Look Once*), utilisée dans les expériences de détection d’objets avec transfert d’apprentissage.
* Ren (2015), pour l’architecture **Faster R-CNN**, employée comme modèle pré-entraîné de détection d’objets.
* He (2016), pour l’architecture **ResNet**, base de divers extracteurs de caractéristiques pré-entraînés utilisés dans ce chapitre.
* Chen (2018), pour l’architecture **DeepLabV3**, utilisée comme modèle pré-entraîné de segmentation sémantique.
* Kirillov (2023), pour le modèle **Segment Anything (SAM)**, mentionné comme piste d’étude pour la segmentation promptable.
* {google} (2025), concernant l’outil **Gemini Notebook**, utilisé dans l’élaboration de l’infographie de synthèse du chapitre et mis à disposition comme support complémentaire à l’étude.

## Références du Chapitre


BISHOP, C. M. **Pattern Recognition and Machine Learning**. New York, NY, USA, Springer, 2006.

CHEN, Liang-Chieh *et al*. **Encoder-Decoder with Atrous Separable Convolution for Semantic Image Segmentation**. Cham, Springer International Publishing, 2018.

GOODFELLOW, Ian; BENGIO, Yoshua; COURVILLE, Aaron. **Deep Learning**. Cambridge, MA, USA, MIT Press, 2016.

HE, Kaiming *et al*. **Deep Residual Learning for Image Recognition**. 2016.

KINGMA, D. P.; BA, J. **Adam: A Method for Stochastic Optimization**. San Diego, CA, USA, 2015.

KIRILLOV, Alexander *et al*. **Segment Anything**. 2023.

LECUN, Y.; BENGIO, Y.; HINTON, G. **Deep learning**. Nature Publishing Group, 2015.

MCCULLOCH, W. S.; PITTS, W. **A Logical Calculus of the Ideas Immanent in Nervous Activity**. 1943.

PARKHI, Omkar M. *et al*. **Cats and Dogs**. IEEE, 2012.

REDMON, Joseph *et al*. **You Only Look Once: Unified, Real-Time Object Detection**. 2016.

REN, Shaoqing *et al*. **Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks**. Curran Associates, Inc., 2015.

RONNEBERGER, O.; FISCHER, P.; BROX, T. **U-Net: Convolutional Networks for Biomedical Image Segmentation**. Cham, Switzerland, Springer, 2015.

ROSENBLATT, F. **The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain**. 1958.

{GOOGLE}. **{NotebookLM}**. 2025.

*Référence introuvable pour : 50*